# Hydraulisch-thermische Netzstudie: Großwärmespeicher MAN/UPM im Fernwärmenetz Stadtbach

**Stand:** Etappe 4 von 4 (Kapitel 1–24 vollständig). Von oben nach unten ausführbar; Rechenzeit ca. 15–20 Minuten (Kalibrierung aus Zwischenspeicher).

## 1. Management Summary
> *Die Zusammenfassung wird in Kapitel 24 aus den berechneten Ergebnissen erzeugt, dort angezeigt und in `results/studie_MAN_UPM/management_summary.md` gespeichert (ebenso die Antworten auf die 20 Fragen: `antworten_20_fragen.md`). Ohne vollständigen Lauf gibt es keine belastbare Aussage zur Speicherwirkung.*

**Wo steht was:** Daten/Qualität/Heizkurve Kap. 4–8, Netzmodell und Kalibrierung Kap. 9–14, Speicher/Szenarien/Optimierung Kap. 15–19, pandapipes Kap. 20, Unsicherheit Kap. 21, Antworten Kap. 22–23.

## 2. Zielsetzung und Systemgrenzen
**Frage:** Kann ein direkt eingebundener, druckbeaufschlagter Heißwasserspeicher bei MAN/UPM kritische Druck- und Temperatursituationen im bestehenden und ausgebauten Netz vermeiden oder reduzieren – und welche Speicher-, Pumpen-, Anschluss- und Betriebsparameter sind dafür nötig?

**Systemgrenze:** Fernwärmenetz Stadtbach (Netz Mitte/Ost/Süd/West) mit Erzeugern HKW, GT-Ost, BMHKW, AVA, Heizwerk Süd/West; Speicher ohne trennenden Wärmeübertrager.

**Grundsatz Speicherphysik:** Ein direkt eingebundener Speicher erzeugt *keine* Förderhöhe. Er kann Druckverluste nur über lokale Einspeisung und reduzierte Transportströme senken. Aktive Druckanhebung erfordert eine Pumpe. Diese Unterscheidung wird in Modell und Ergebnisdarstellung getrennt ausgewiesen.

**Arbeitsannahmen (mit dem Auftraggeber abgestimmt):** (i) Speicher als *Druckspeicher* wegen bis zu 120 °C im Winter; (ii) MAN-Anschlusspunkt wird als Annahme aus den Plänen festgelegt und als Sensitivität variiert; (iii) lineare Vorlauf-Heizkurve 120 °C bei −10 °C bis 90 °C bei +20 °C als Vorgabe.

## 3. Reproduzierbarkeit und Softwareumgebung

In [ ]:
import os, sys, re, json, math, platform, hashlib, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# ---------------- zentrale Konfiguration ----------------
SEED = 20250101
np.random.seed(SEED)

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "calion").is_dir() and (p / "configs").is_dir():
            return p
    raise FileNotFoundError("Repository-Wurzel (Ordner 'calion' + 'configs') nicht gefunden.")

ROOT      = find_repo_root()
DATA_DIR  = ROOT / "data" / "Stadtbach" / "upload"      # NDA: nicht versionieren!
RESULTS   = ROOT / "results" / "studie_MAN_UPM"
FIG_DIR   = RESULTS / "abbildungen"
for d in (RESULTS, FIG_DIR): d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    year=2025,
    p_unit="bar (Überdruck, Annahme – siehe Annahmenkatalog A-01)",
    heizkurve_vorgabe=dict(T_low_out=-10.0, T_VL_at_low=120.0, T_high_out=20.0, T_VL_at_high=90.0),
    cp_kJ_kgK=4.19, rho_kg_m3=971.8,        # Näherung bei ~80 °C; später temperaturabhängig
    dt_h=1.0,
)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
COL = dict(vl="#c0392b", rl="#2874a6", ok="#1e8449", warn="#d68910", grau="#7f8c8d", mess="#34495e")

print("Python", platform.python_version(), "|", platform.platform())
for m in ("numpy","pandas","scipy","matplotlib","pyarrow","openpyxl","yaml","pyomo","pandapipes"):
    try:
        mod = __import__(m); print(f"{m:11s}", getattr(mod, "__version__", "?"))
    except Exception as e:
        print(f"{m:11s} NICHT installiert ({type(e).__name__})")
print("Repo-Wurzel:", ROOT.name, "| Datenordner vorhanden:", DATA_DIR.exists())

## 4. Repository- und Dateiinventur
Rekursive Bestandsaufnahme der für die Studie relevanten Verzeichnisse. Die Bedeutung wird – wo möglich – am **Inhalt** geprüft (YAML: Knoten/Leitungen; Parquet: Spalten/Zeilen; PDF: Seitenzahl), nicht am Dateinamen.
Schreibweise „Stadtbach": tolerante, case-insensitive Suche (der Auftrag nennt `stadtabcach`).

In [ ]:
import yaml
# --- 4.1 Tolerante Ordnersuche
def fuzzy_dirs(base, pattern=r"stadt.?a?b?a?c?h"):
    rx = re.compile(pattern, re.I)
    return sorted({str(p.relative_to(ROOT)) for p in base.rglob("*")
                   if p.is_dir() and rx.search(p.name) and ".git" not in p.parts})
print("Gefundene Stadtbach-Ordner:")
for d in fuzzy_dirs(ROOT): print("  ", d)

# --- 4.2 Inventur
SCAN = [ROOT/"configs", ROOT/"docs", ROOT/"paper2_package", ROOT/"scripts", ROOT/"tools", ROOT/"calion",
        ROOT/"revision", DATA_DIR]
EXT = {".yaml",".yml",".csv",".parquet",".json",".geojson",".shp",".dxf",".dwg",".pdf",".xlsx",".xls",
       ".ipynb",".md",".docx",".pptx",".py"}
rows=[]
for base in SCAN:
    if not base.exists(): continue
    for p in base.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXT and ".git" not in p.parts and "_build" not in p.parts:
            rel = p.relative_to(ROOT).as_posix()
            if not re.search(r"stadt|WV6|640|acron|Import_Data|storage_geometry|pandapipes|Lastg", rel, re.I):
                continue
            st = p.stat()
            rows.append(dict(pfad=rel, typ=p.suffix.lower().lstrip("."), groesse_kB=round(st.st_size/1024,1),
                             geaendert=datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d")))
inv = pd.DataFrame(rows).sort_values("pfad").reset_index(drop=True)
print(len(inv), "relevante Dateien")

In [ ]:
# --- 4.3 Inhaltsprüfung + Bewertung
def inspect(rel):
    p = ROOT/rel; ext = p.suffix.lower()
    try:
        if ext in (".yaml",".yml"):
            d = yaml.safe_load(p.read_text(encoding="utf-8", errors="ignore"))
            n = d.get("network", {}) if isinstance(d, dict) else {}
            if n.get("nodes") and n.get("pipes"):
                return f"YAML-Netz: {len(n['nodes'])} Knoten, {len(n['pipes'])} Leitungen"
            return "YAML (kein Netz)"
        if ext == ".parquet":
            df = pd.read_parquet(p); return f"Parquet {df.shape[0]}x{df.shape[1]}"
        if ext == ".csv":
            df = pd.read_csv(p, nrows=5, sep=None, engine="python"); return f"CSV, Spalten: {list(df.columns)[:4]}…"
        if ext == ".pdf":
            txt = p.read_bytes(); npg = len(re.findall(rb"/Type\s*/Page[^s]", txt)); return f"PDF, {npg} Seite(n), Raster (kein Text erwartet)"
        if ext == ".xlsx":
            xl = pd.ExcelFile(p); return f"Excel, Blätter: {xl.sheet_names[:3]}"
    except Exception as e:
        return f"Lesefehler: {type(e).__name__}"
    return ""

def bewerte(rel):
    r = rel.lower()
    if "upload/a_daten/acron_stundenwerte_wert" in r: return ("Druck/Temperatur/Durchfluss/Leistung 2025 (stündlich)", "verwendet", "Kalibrierung/Validierung, Heizkurve", "Zeitumstellung: 1 Doppelstunde verworfen; viele Lücken")
    if "upload/a_daten/acron_stundenwerte_minmax" in r: return ("Min/Max zu ACRON-Reihen", "verwendet (später)", "Druckspitzen/-minima", "nicht in Etappe 1 ausgewertet")
    if "acron_signalindex" in r: return ("Zuordnung Signal -> Originaldatei/Station", "verwendet", "Messstellenzuordnung", "-")
    if "import_data_stadtbach_15min" in r: return ("Außentemperatur, V_n-Zähler, WRG (15 min)", "verwendet", "Außentemperatur, Kundenlasten", "Einheiten der V_n-Kanäle ungeklärt")
    if "combined" in r: return ("Aggregierte Kundenlasten (7 gemessen + 17 geschätzt)", "nur Referenz", "Vergleich zur alten Config", "17 Reihen zonal geschätzt -> keine Validierungsziele")
    if "b_plaene" in r: return ("Netzpläne WV640/650/660 (Raster)", "verwendet (Etappe 2)", "Topologie, Armaturen, Höhen, PN", "verkleinerte Raster; Text nicht extrahierbar")
    if "lastg" in r: return ("Lastgänge 2020–2022", "nicht verwendet (Etappe 1)", "evtl. Ausbau-/Lastform", "älter als 2025")
    if r.endswith("stadtbach_topo.yaml") and "configs/paper_2/" in r and "archiev" not in r: return ("Aktuelles YAML-Netz (Baum)", "verwendet als Ausgangspunkt", "Sektions-Skelett, Erzeuger", "Baum statt Masche; Druckniveau nicht messkonform")
    if "archiev" in r and "stadtbach" in r: return ("Ältere Config-Stände", "nicht verwendet", "-", "veraltet; Kommentare widersprüchlich")
    if "storage_geometry" in r: return ("Speichergeometrie (atmosphärische Grube)", "nicht verwendet", "Widerspruch zum Druckspeicher", "atmosph. 95 °C-Grenze; nicht passend")
    if "20250922_stadtbach.ipynb" in r: return ("Alter Pyomo-MILP (keine pandapipes)", "nicht verwendet", "-", "keine pandapipes-Studie vorhanden")
    if "pandapipes" in r: return ("pandapipes-Crosscheck (Memmingen)", "Vorlage", "Vorgehen für Kontrolle", "nur Memmingen")
    if r.endswith(".py") and "stadtbach" in r: return ("Hilfsskript", "nicht verwendet", "-", "-")
    return ("Sonstiges/Dokumentation", "nicht verwendet", "-", "-")

inv["inhalt_geprueft"] = inv.pfad.map(inspect)
b = inv.pfad.map(bewerte)
inv["bedeutung"], inv["status"], inv["begruendung"], inv["qualitaet"] = [b.map(lambda t: t[i]) for i in range(4)]
inv.to_csv(RESULTS/"inventur_dateien.csv", index=False)
with pd.option_context("display.max_colwidth", 70, "display.width", 250):
    display(inv[inv.status.str.startswith(("verwendet","nur","Vorlage"))][["pfad","typ","groesse_kB","inhalt_geprueft","status"]])
print("Status-Übersicht:", inv.status.value_counts().to_dict())

**Befund 4.4 – fehlende Quellen:** Suche nach Stadtbach-DXF/DWG/Shape/GeoJSON: siehe nächste Zelle. Die Pläne liegen nur als verkleinerte Raster-PDFs vor.

In [ ]:
geo = inv[inv.typ.isin(["dxf","dwg","shp","geojson"])]
print("Geometriedateien (Stadtbach):", "keine" if geo.empty else geo.pfad.tolist())
print("DXF im Repo (nur Memmingen erwartet):", [p.relative_to(ROOT).as_posix() for p in ROOT.rglob("*.dxf") if ".git" not in p.parts])
print("Stadtbach-pandapipes-Studie im Repo:", "nein – nur Memmingen-Crosscheck" if not any("pandapipes" in x and "stadtbach" in x.lower() for x in inv.pfad) else "ja")

## 5. Quellen-, Konflikt- und Annahmenkatalog

### 5.1 Annahmenkatalog
Herkunftsklassen: **M** direkt gemessen · **P** Plan/Betriebsunterlage · **Y** YAML/bestehendes Modell · **H** Hersteller · **K** kalibriert · **A** fachlicher Standardwert · **S** Sensitivitätsparameter. Der Katalog wird in späteren Etappen fortgeschrieben.

In [ ]:
katalog = pd.DataFrame([
 # id, parameter, wert, einheit, klasse, quelle, gueltig, unsicherheit, wirkung, sens_bereich
 ("A-01","Druckdefinition ACRON", "Überdruck", "bar", "A","Plausibilität: HKW-RL-Median 8,5 bar; Absolutdruck unwahrscheinlich niedrig für Ruhedruck 3,2–6,5 bar Ü", "2025","hoch: Faktor 1,013 bar","Siedesicherheit, Speicherdruck","abs vs. Ü (Δ=1,013 bar)"),
 ("A-02","Durchflusseinheit ACRON", "m³/h (≈ t/h)", "m³/h", "A","Leistungsbilanz MAN (Abschnitt 6.4)", "2025","mittel","Massenströme","t/h vs. m³/h (Δρ≈3–4 %)"),
 ("A-03","Heizkurve Hauptrechnung", "T_VL=110−T_a, begrenzt 90…120 °C", "°C", "S","Auftraggeber-Vorgabe (vorläufig)", "Winter/Übergang","mittel","Vorlauftemperatur","YAML-Kurve (122→70 °C) als Alternative"),
 ("A-04","Speichertyp", "Druckspeicher, direkt eingebunden", "-", "P","Auftraggeber: bis 120 °C im Winter nötig", "-","-","Betriebsdruck ≥ p_sat(120 °C)+Marge","-"),
 ("A-05","MAN-Anschlusspunkt", "wird in Etappe 2 aus WV650 festgelegt", "-", "A","Auftraggeber: 'Du kannst den MAN-Punkt annehmen'", "-","hoch","Speicherhydraulik","Varianten am Netzknoten"),
 ("A-06","Netz-Spreizung Dimensionierung", "99/60 °C", "°C", "Y","Stadtbach_topo.yaml", "Übergangszeit","mittel","Erzeugerkapazität","Messung 2025 (Abschnitt 8)"),
 ("A-07","Transportgrenzen SL5/SL6/SL7 (separiert)", "750/500/300 t/h", "t/h", "P","WV650 laut Auftrag (noch nicht am Plan geprüft)", "Stand 2022","mittel","Separierter Betrieb","±20 %"),
 ("A-08","Mindest-Ruhedruck bei 120 °C", "MITTE–CITY 6,5 / MITTE 3,2 / SÜD 3,2 / OST 2,5", "bar Ü", "P","WV650 laut Auftrag (noch nicht am Plan geprüft)", "Stand 2022","gering","Druckhaltung","-"),
 ("A-09","Kundendifferenzdruck minimal", "0,7", "bar", "Y","Stadtbach_topo.yaml delta_p_min_consumer_bar", "-","hoch (Messung 0–4 bar)","Druckverletzungen","0,3 … 1,5 bar"),
 ("A-10","Rohrrauheit", "0,5 (YAML) – wirkt dort nicht", "mm", "Y","Stadtbach_topo.yaml; im Framework nur gespeichert, nicht verwendet", "-","hoch","Druckverlust","0,1 … 1,0 mm (Colebrook im Notebook)"),
], columns=["id","parameter","wert","einheit","klasse","quelle_fundstelle","gueltigkeit","unsicherheit","auswirkung","sensitivitaetsbereich"])
katalog.to_csv(RESULTS/"annahmenkatalog.csv", index=False)
with pd.option_context("display.max_colwidth", 60, "display.width", 250): display(katalog[["id","parameter","wert","einheit","klasse","unsicherheit"]])

### 5.2 Konfliktkatalog (Widersprüche zwischen Quellen)
Keine Quelle wird stillschweigend überschrieben; für jeden Widerspruch wird eine Arbeitsannahme festgelegt und die Alternative als Sensitivität gerechnet.

In [ ]:
konflikte = pd.DataFrame([
 ("K-01","Heizkurve","Vorgabe: 120 °C@−10 °C … 90 °C@+20 °C","YAML: 122 °C@−15 °C … 70 °C@+30 °C (+Saisonwerte)","Abschnitt 8: Vergleich mit Messung","Vorgabe für Hauptrechnung, YAML als Sensitivität"),
 ("K-02","Netztopologie","Auftrag/Plan: teilweise vermascht, Sektionen","YAML: 33 Knoten/32 Leitungen, reiner Baum; Header nennt 32/31; Paper-Tabelle nennt 'meshed'","Etappe 2","Sektionsmodell mit Maschen aus WV650/660"),
 ("K-03","Speichertyp","Auftraggeber: Druckspeicher bis 120 °C","storage_geometry.yaml: atmosphärische Grube (95 °C, bis 120 000 m³); Topo-YAML: 10 bar/5 000 m³","Kap. 15","Druckspeicher; atmosphärische Variante entfällt"),
 ("K-04","Speicherenergie","5 000 m³ @ 10 K ≈ 56–58 MWh","YAML-Leiter bis 84,5 MWh (≈ 15 K)","Kap. 15","aus Volumen und ΔT berechnen; 84,5 MWh nicht übernehmen"),
 ("K-05","Druckniveau","Messung 2025: HKW-VL Median 10,7 bar, RL 8,5 bar","YAML: VL-Sollwert 16 bar, RL am Boden 0,5 bar","Kap. 13/14","Messwerte als Anker; YAML-Sollwert verwerfen"),
 ("K-06","Erzeugerleistungen (Kommentare)","YAML-Assets: HKW 127, HWS 32, HWW 104 MW","YAML-Header/Serienmap: HKW 75, HWS 40, HWW 64 MW","Etappe 2/3","Asset-Werte (Üzeit, 99/60 °C); Kommentare veraltet"),
 ("K-07","Hydraulik im Framework","Darcy-Weisbach mit Rauheit (Auftrag)","Framework: konstantes λ=0,02; Δp nur als untere Hüllkurve","Etappe 2","eigener Netzlöser im Notebook"),
], columns=["id","thema","quelle_A","quelle_B","pruefung","arbeitsannahme"])
konflikte.to_csv(RESULTS/"konfliktkatalog.csv", index=False)
with pd.option_context("display.max_colwidth", 55, "display.width", 250): display(konflikte)

## 6. Import und Prüfung der Messdaten
Rohdaten werden **nie überschrieben**: `acron_raw` bleibt unverändert, `acron` ist die bereinigte Arbeitskopie.
Geprüft werden Zeitstempel/Zeitzone/Sommerzeit, Duplikate, Lücken, Nullwerte, eingefrorene Werte, Ausreißer, Vorzeichen und Einheiten.

In [ ]:
# --- 6.1 Import
A = DATA_DIR/"A_Daten"
sig      = pd.read_csv(A/"acron_signalindex.csv")
acron_raw= pd.read_parquet(A/"acron_stundenwerte_wert.parquet")
imp      = pd.read_parquet(A/"Import_Data_stadtbach_15min.parquet").set_index("Datum")

print("ACRON:", acron_raw.shape, acron_raw.index.min(), "→", acron_raw.index.max())
print("Index eindeutig:", acron_raw.index.is_unique, "| Frequenz stündlich:",
      (acron_raw.index.to_series().diff().dropna() == pd.Timedelta("1h")).mean().round(4))
missing_h = pd.date_range(f"{CFG['year']}-01-01", f"{CFG['year']}-12-31 23:00", freq="h").difference(acron_raw.index)
print("Fehlende Stunden im Jahr 2025:", list(missing_h), "(30.03. 02:00 existiert lokal nicht – Sommerzeitbeginn; die doppelte Stunde 26.10. 02:00 wurde beim Upload auf den ersten Wert reduziert; Zeitstempel naiv lokal)")
print("Import 15 min:", imp.shape, imp.index.min(), "→", imp.index.max(), "| eindeutig:", imp.index.is_unique)

def kanal(name):
    n = name
    for k,v in [("Druck_VL","p_VL"),("Druck_RL","p_RL"),("DiffDruck","dp"),("Temp_VL","T_VL"),("Temp_RL","T_RL"),
                ("Durchfluss","m_dot"),("Waermeleistung","Q"),("Drehzahl","n_pumpe")]:
        if k in n: return v
    return "sonstige"
meta = pd.DataFrame({"kanal": [kanal(c) for c in acron_raw.columns]}, index=acron_raw.columns)
meta = meta.join(sig.set_index("key")[["ordner","title"]])
print(meta.kanal.value_counts().to_dict())

In [ ]:
# --- 6.2 Qualitätsanalyse + Bereinigung (regelbasiert, dokumentiert)
def longest_const_run(s):
    s = s.dropna()
    if s.empty: return 0
    return int((s.diff() != 0).cumsum().value_counts().max())

acron = acron_raw.copy()
log = []
p_cols = [c for c in acron.columns if meta.kanal[c] in ("p_VL","p_RL","dp")]
for c in p_cols:
    s = acron[c]
    nb = pd.concat([s.shift(1), s.shift(-1)], axis=1).median(axis=1)         # Nachbarn ±1 h
    if meta.kanal[c] == "dp":
        bad = (s <= 0.0) & (nb > 0.5)
    else:
        bad = (s < 0.5) & (nb > 2.0)                                          # Nullwerte bei intakten Nachbarn
    if bad.any():
        acron.loc[bad, c] = np.nan; log.append((c, "Druck/Δp ≈ 0 bei intakten Nachbarn → NaN", int(bad.sum())))
t_cols = [c for c in acron.columns if meta.kanal[c] in ("T_VL","T_RL")]
for c in t_cols:
    s = acron[c]; bad = (s < 5) | (s > 135)
    if bad.any(): acron.loc[bad, c] = np.nan; log.append((c, "Temperatur außerhalb 5…135 °C → NaN", int(bad.sum())))
m_cols = [c for c in acron.columns if meta.kanal[c] == "m_dot"]
for c in m_cols:
    if c.startswith("HKW_Durchfluss_SL"): continue     # Vorzeichen = Flussrichtung (bewusst erlaubt)
    s = acron[c]; bad = s < -1e-9
    if bad.any(): acron.loc[bad, c] = np.nan; log.append((c, "negativer Durchfluss → NaN", int(bad.sum())))

cleanlog = pd.DataFrame(log, columns=["signal","regel","anzahl"])
print(f"Bereinigungsregeln angewandt auf {len(cleanlog)} Signale, {cleanlog.anzahl.sum()} Werte → NaN")

rows=[]
for c in acron.columns:
    s0, s = acron_raw[c], acron[c]
    rows.append(dict(signal=c, kanal=meta.kanal[c], station=meta.ordner[c],
        abdeckung_pct=round(100*s.notna().mean(),1), luecken_pct=round(100*s.isna().mean(),1),
        min=s.min(), q05=s.quantile(.05), median=s.median(), q95=s.quantile(.95), max=s.max(),
        nullwerte_roh=int((s0==0).sum()), laengster_konst_lauf_h=longest_const_run(s0),
        ausreisser_3sigma=int(((s-s.mean()).abs() > 3*s.std()).sum()) if s.std()>0 else 0,
        bereinigt=int(s0.notna().sum()-s.notna().sum())))
qual = pd.DataFrame(rows).set_index("signal")
qual["eingefroren_verdacht"] = (qual.laengster_konst_lauf_h >= 72) & ~qual.kanal.isin(["n_pumpe"])
# Validierungstauglichkeit: Abdeckung ≥ 80 % und kein Stillstand ≥ 168 h (Pumpendrehzahlen ausgenommen, dort ist Stillstand normal)
qual["validierungstauglich"] = (qual.abdeckung_pct >= 80) & ((qual.laengster_konst_lauf_h < 168) | qual.kanal.isin(["n_pumpe"]))
qual.round(2).to_csv(RESULTS/"datenqualitaet_acron.csv")
cleanlog.to_csv(RESULTS/"bereinigungslog.csv", index=False)

### 6.3 Ergebnis der Qualitätsprüfung

In [ ]:
print("Abdeckung je Kanal (Median über Signale, %):")
print(qual.groupby("kanal").abdeckung_pct.agg(["count","median","min"]).round(1))
print("\nSignale mit < 80 % Abdeckung:")
print(qual[qual.abdeckung_pct < 80][["kanal","station","abdeckung_pct"]].sort_values("abdeckung_pct").to_string())
print("\nEingefrorene Reihen (≥ 72 h konstant, Verdacht auf Sensor-/Übertragungsausfall):")
print(qual[qual.eingefroren_verdacht][["kanal","station","laengster_konst_lauf_h"]].sort_values("laengster_konst_lauf_h", ascending=False).to_string())
print("\nValidierungstauglich (Abdeckung ≥ 80 %, kein Stillstand ≥ 168 h):", int(qual.validierungstauglich.sum()), "von", len(qual), "Signalen; je Kanal:", qual.groupby("kanal").validierungstauglich.sum().to_dict())
print("\nBereinigung (Top 15):"); print(cleanlog.sort_values("anzahl", ascending=False).head(15).to_string(index=False))

### 6.4 Einheiten- und Konsistenzprüfung
* **Durchfluss:** Aus `Q = V̇·cp·ρ·ΔT` wird an Stationen mit Leistung, Durchfluss, VL- und RL-Temperatur die implizite Einheit geprüft (Erwartung: m³/h).
* **Druck:** VL > RL, Δp ≈ VL − RL (soweit alle drei Signale vorhanden).

In [ ]:
# Einheitenprüfung Durchfluss: Q[MW] ≈ V[m³/h]·ρ·cp·ΔT/3.6e6 (ρ≈960, cp≈4,2 kJ/kgK)
rows=[]
for st in sorted(set(meta.ordner)):
    q  = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="Q"]
    v  = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="m_dot"]
    tv = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="T_VL"]
    tr = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="T_RL"]
    if q and v and tv and tr:
        d = pd.concat([acron[q[0]], acron[v[0]], acron[tv[0]]-acron[tr[0]]], axis=1, keys=["Q","V","dT"]).dropna()
        d = d[(d.Q>0.05)&(d.V>5)&(d.dT>5)]
        if len(d) > 100:
            q_calc = d.V*960*4.2*d.dT/3.6e6
            rows.append((st, len(d), round((d.Q/q_calc).median(),3), round((d.Q/q_calc).quantile(.1),2), round((d.Q/q_calc).quantile(.9),2)))
einh = pd.DataFrame(rows, columns=["station","n","Q_gemessen/Q_aus_V·ΔT (Median)","q10","q90"])
print(einh.to_string(index=False))
print("\n→ Kundenstationen: Verhältnis ≈ 1,04 (Näherungskonstanten ρ=960, cp=4,2) bestätigt m³/h. Abweichend: HKW (≈15×), HWW sekundär (≈2,2×), GT-Ost (≈0,7×) – dort ist Definition von Durchfluss bzw. Leistung (Teilstrom/Bezugsstelle) ungeklärt → Rückfrage; diese Signale werden nicht als Validierungsziel verwendet.")

# Druckkonsistenz
pc = []
for st in sorted(set(meta.ordner)):
    pv = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="p_VL"]
    pr = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="p_RL"]
    dd = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="dp"]
    if pv and pr:
        d = (acron[pv[0]]-acron[pr[0]]).dropna()
        row = dict(station=st, VL_kleiner_RL_pct=round(100*(d<0).mean(),1), dp_VL_RL_median=round(d.median(),2))
        if dd:
            e = (acron[dd[0]]-(acron[pv[0]]-acron[pr[0]])).dropna(); row["dp_gemessen_minus_(VL−RL)_median"]=round(e.median(),2)
        pc.append(row)
print(); print(pd.DataFrame(pc).to_string(index=False))

## 7. Explorative Datenanalyse
Außentemperatur, Netztemperaturen, Drücke am HKW und Differenzdrücke an den Kundenstationen 2025.

In [ ]:
Ta = imp["outdoor_temp_C"].resample("1h").mean()
Ta = Ta.reindex(acron.index)
fig, ax = plt.subplots(4,1, figsize=(12,10), sharex=True)
ax[0].plot(Ta, color=COL["grau"], lw=.6); ax[0].set_ylabel("Außentemp. [°C]")
ax[1].plot(acron["HKW_Temp_VL"], color=COL["vl"], lw=.6, label="HKW Vorlauf"); ax[1].plot(acron["HKW_Temp_RL"], color=COL["rl"], lw=.6, label="HKW Rücklauf")
ax[1].set_ylabel("Temperatur [°C]"); ax[1].legend(loc="upper right", ncol=2)
ax[2].plot(acron["HKW_Druck_VL"], color=COL["vl"], lw=.6, label="HKW Druck VL"); ax[2].plot(acron["HKW_Druck_RL"], color=COL["rl"], lw=.6, label="HKW Druck RL")
ax[2].set_ylabel("Druck [bar Ü]"); ax[2].legend(loc="upper right", ncol=2)
ax[3].plot(acron["HKW_Durchfluss_gesamt"], color=COL["mess"], lw=.6); ax[3].set_ylabel("HKW Durchfluss ges. [m³/h]")
fig.suptitle("Netzzustand am HKW, 2025", y=.995); fig.tight_layout(); fig.savefig(FIG_DIR/"07_hkw_zeitreihen.png"); plt.show()

In [ ]:
# Differenzdruck-Dauerlinien der Kundenstationen: kritische Stellen
dp_cols = [c for c in acron.columns if meta.kanal[c]=="dp" and meta.ordner[c] not in ("PSS","PSW","HKW","BMHKW","GT-Ost","AVA","Heizwerk_Sued","HWW_prim_Einsp")]
dpq = pd.DataFrame({c:{"q01":acron[c].quantile(.01),"median":acron[c].median(),"anteil_<0,7bar_%":100*(acron[c]<0.7).mean()} for c in dp_cols}).T.round(2)
dpq = dpq.sort_values("q01")
display(dpq.head(12))
fig, ax = plt.subplots(figsize=(9,5))
for c in dpq.index[:6]:
    s = acron[c].dropna().sort_values(ascending=False).reset_index(drop=True)
    ax.plot(np.linspace(0,100,len(s)), s, label=c.replace("_DiffDruck",""))
ax.axhline(0.7, color=COL["warn"], ls="--", label="0,7 bar (YAML-Annahme)")
ax.set_xlabel("Zeitanteil [%]"); ax.set_ylabel("Differenzdruck [bar]"); ax.set_title("Differenzdruck-Dauerlinien, sechs kritischste Stationen"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR/"07_dp_dauerlinien.png"); plt.show()

## 8. Prüfung der Heizkurve
Vergleich von (a) der **Vorgabe** (Auftrag), (b) der **YAML-Kurve**, (c) der **gemessenen** Vorlauftemperatur am Erzeugeraustritt und (d) der **statistisch aus Messdaten** ermittelten Kurve.

Vorgabe: $T_{VL,soll}=\min\big(120\,^\circ\mathrm{C},\ \max(90\,^\circ\mathrm{C},\ 110\,^\circ\mathrm{C}-T_{a})\big)$
YAML: linear von 122 °C bei −15 °C bis 70 °C bei +30 °C.

In [ ]:
def hk_vorgabe(Ta):
    return np.clip(110.0 - np.asarray(Ta, float), 90.0, 120.0)
def hk_yaml(Ta):
    return np.clip(122.0 - (122.0-70.0)/(30.0-(-15.0))*(np.asarray(Ta, float)+15.0), 70.0, 122.0)

# Tests: Endpunkte
assert hk_vorgabe(-10)==120 and hk_vorgabe(20)==90 and hk_vorgabe(-20)==120 and hk_vorgabe(30)==90
assert abs(hk_yaml(-15)-122)<1e-9 and abs(hk_yaml(30)-70)<1e-9
print("Heizkurven-Endpunkttests bestanden.")

hk = pd.DataFrame({"Ta":Ta, "VL_HKW":acron["HKW_Temp_VL"], "VL_MAN":acron.get("MAN_Temp_VL_Station"),
                   "VL_PPS":acron.get("PPS_Temp_VL"), "VL_PSW":acron.get("PSW_Temp_VL"),
                   "V_ges":acron["HKW_Durchfluss_gesamt"]}).dropna(subset=["Ta","VL_HKW"])
hk["soll_vorgabe"] = hk_vorgabe(hk.Ta); hk["soll_yaml"] = hk_yaml(hk.Ta)
hk["mon"] = hk.index.month; hk["stunde"] = hk.index.hour
hk["saison"] = hk.mon.map(lambda m: "Winter" if m in (12,1,2) else "Frühjahr" if m in (3,4,5) else "Sommer" if m in (6,7,8) else "Herbst")
hk["last"] = pd.qcut(hk.V_ges, 3, labels=["niedrig","mittel","hoch"])

# Statistische Kurve: Median/Quantile je 2-K-Klasse + lineare Anpassung (−10…+20 °C)
hk["bin"] = (np.floor(hk.Ta/2)*2+1)
st = hk.groupby("bin").VL_HKW.agg(n="count", q10=lambda s: s.quantile(.1), median="median", q90=lambda s: s.quantile(.9)).query("n>=20")
sel = st.loc[(st.index>=-10)&(st.index<=20)]
k, d0 = np.polyfit(sel.index, sel["median"], 1)
print(f"Statistische Kurve (Median, −10…20 °C): T_VL = {d0:.1f} {k:+.3f}·T_a  → bei −10 °C: {d0+k*-10:.1f} °C, bei +20 °C: {d0+k*20:.1f} °C")
for nm, f in (("Vorgabe",hk_vorgabe),("YAML",hk_yaml)):
    r = hk.VL_HKW - f(hk.Ta); print(f"{nm:8s}: Bias {r.mean():+.1f} K, RMSE {np.sqrt((r**2).mean()):.1f} K, MAE {r.abs().mean():.1f} K")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,5))
xs = np.linspace(-13,32,100)
ax[0].scatter(hk.Ta, hk.VL_HKW, s=3, alpha=.15, color=COL["grau"], label="Messung HKW, stündlich")
ax[0].fill_between(st.index, st.q10, st.q90, color=COL["vl"], alpha=.15, label="10–90 %-Band")
ax[0].plot(st.index, st["median"], "o-", color=COL["vl"], ms=3, label="Median je 2-K-Klasse")
ax[0].plot(xs, hk_vorgabe(xs), color=COL["ok"], lw=2, label="Vorgabe (Auftrag)")
ax[0].plot(xs, hk_yaml(xs), color=COL["warn"], lw=2, ls="--", label="YAML-Kurve")
ax[0].set(xlabel="Außentemperatur [°C]", ylabel="Vorlauftemperatur [°C]", ylim=(60,130), title="Heizkurve: Vorgabe, YAML und Messung")
ax[0].legend(fontsize=8)
res = (hk.VL_HKW - hk.soll_vorgabe)
hk.assign(res=res).boxplot(column="res", by="saison", ax=ax[1], grid=False)
ax[1].axhline(0, color="k", lw=.8); ax[1].set(title="Abweichung Messung − Vorgabe nach Jahreszeit", xlabel="", ylabel="ΔT_VL [K]"); plt.suptitle("")
fig.tight_layout(); fig.savefig(FIG_DIR/"08_heizkurve.png"); plt.show()

In [ ]:
# Abweichungen nach Tageszeit, Lastniveau, Netzbereich
hk["res"] = hk.VL_HKW - hk.soll_vorgabe
print("Nach Tageszeit (Median Abweichung, K):")
print(hk.groupby(pd.cut(hk.stunde,[-1,5,11,17,23],labels=["Nacht 0–5","Vormittag 6–11","Nachmittag 12–17","Abend 18–23"]),observed=True).res.median().round(1).to_string())
print("\nNach Lastniveau (HKW-Durchfluss-Terzile):"); print(hk.groupby("last",observed=True).res.median().round(1).to_string())
bereich = pd.DataFrame({b: (hk[c] - hk.soll_vorgabe) for b,c in (("HKW (Mitte)","VL_HKW"),("MAN","VL_MAN"),("Pumpstation Süd","VL_PPS"),("Pumpstation West","VL_PSW")) if hk[c].notna().any()})
print("\nNach Netzbereich (Median Abweichung zur Vorgabe, K; Winter):")
print(bereich[hk.saison=="Winter"].median().round(1).to_string())
kalt = hk[hk.Ta < -5]
print(f"\nKältester Bereich (T_a < −5 °C, n={len(kalt)} h): gemessener Vorlauf Median {kalt.VL_HKW.median():.1f} °C, max {kalt.VL_HKW.max():.1f} °C; Vorgabe 115–120 °C")
print("Spitzen ≥ 120 °C:", int((hk.VL_HKW>=120).sum()), "Stunden; ≥ 122 °C:", int((hk.VL_HKW>=122).sum()), "Stunden")

### 8.4 Zwischenbewertung Heizkurve (aus den Rechenergebnissen oben)

In [ ]:
rm = {n: float(np.sqrt(((hk.VL_HKW - f(hk.Ta))**2).mean())) for n, f in (("Vorgabe", hk_vorgabe), ("YAML", hk_yaml))}
bias = {n: float((hk.VL_HKW - f(hk.Ta)).mean()) for n, f in (("Vorgabe", hk_vorgabe), ("YAML", hk_yaml))}
best = min(rm, key=rm.get)
print(f"Hauptrechnung: Heizkurve '{best}' (RMSE {rm[best]:.1f} K, Bias {bias[best]:+.1f} K) gegenüber '{[k for k in rm if k!=best][0]}' (RMSE {max(rm.values()):.1f} K, Bias {bias[[k for k in rm if k!=best][0]]:+.1f} K).")
print(f"Statistische Kurve T_VL = {d0:.1f} {k:+.3f}·T_a weicht bei −10 °C um {d0+k*-10-120:+.1f} K und bei +20 °C um {d0+k*20-90:+.1f} K von der Vorgabe ab.")
print("Die YAML-Kurve wird als Sensitivität mitgeführt (Konflikt K-01). Es gab 2025 keinen Speicherbetrieb; eine Abhängigkeit vom Speicher entfällt.")
print("Hinweis: Nur 182 Stunden mit T_a < −5 °C und minimal −12,9 °C im Jahr 2025 – der Auslegungsfall (−14 … −16 °C) ist NICHT durch Messung abgedeckt und wird durch die Kurve extrapoliert.")

---
**Ende Etappe 1.** Folgt: Kapitel 9–10 (Netzpläne, kanonisches Netzmodell) – danach Halt zur Prüfung der abgelesenen Topologie.

## 9. Analyse der Netzpläne und der YAML-Topologie
**Quellen:** Arbeitsblätter *WV 640 Bl. 3* (Erzeugerausspeisungen, Stand 14.06.2023), *WV 650* (Netz Mitte/Süd/Nordost/Ost, Stand 25.03.2022 bzw. Datei 2022-10-28) und *WV 660* (Netz West/Nordwest, Stand 25.03.2022).
Die Pläne liegen nur als **verkleinerte Rasterbilder** vor. Alle Werte in diesem Kapitel wurden **manuell bzw. per OCR aus den Bildern abgelesen** und sind hier als Datentabellen hinterlegt (Herkunftsklasse **P**). Ablesefehler sind möglich – die Tabellen sind zur Prüfung durch den Auftraggeber gedacht.

### 9.1 Erzeuger, Pumpstationen und Druckhaltung (WV 640 Bl. 3)

In [ ]:
# Höhen [m NN] = Bezugshöhe der Anlage; Ruhedruck [bar Ü] und Abschaltdrücke [bar Ü] gemäß WV640 Bl.3
erzeuger = pd.DataFrame([
 # anlage, z_NN, dp_intern, P_AT14, m_AT14, T_VL/T_RL AT14, P_UeZ, m_UeZ, T_VL/T_RL UeZ, Tmax_VL, Pumpen, Anordnung, Ruhedruck, Ruhedruck_bez_NN, ab_RL_lt, ab_VL_gt, Bezug_NN_Abschalt
 ("HKW",      473.77, 1.1, 127.0, 2350, "107/60 (Parallelbetr.)", None, None, None, 128, "2x1000 + 2x900 t/h @ 73,5 m FIS", "im VL", 8.4, 474.68, None, None, None),
 ("AVA",      466.69, 0.8,  40.0,  530, "125/60",  45.0,  990, "99/60", 128, "2x500 t/h @ 90 m FIS",  "im RL", 7.5, 467.84, 5.0, 19.5, 467.84),
 ("GT-Ost",   467.98, 1.3,  31.0,  410, "125/60",  35.0,  720, "102/60",128, "2x725 t/h @ 90 m FIS",  "im RL", 6.8, 474.98, 4.4, 20.5, 474.98),
 ("BM-HKW",   467.08, 0.9,  13.0,  190, "120/60",  14.5,  400, "91/60", 120, "2x206 t/h @ 115 m FIS", "im RL", None, None, 4.8, 22.0, 473.75),
 ("HWS",      490.51, 0.8,  40.0,  550, "128/65",  32.0,  550, "110/60",128, "3x280 m3/h @ 58,5 m FIS","im RL", 6.8, 488.16, 4.0, 12.0, 488.05),
 ("HWW",      477.99, None, 64.0,  960, "120/65 (2 Kessel + 2 WÜ Prim-Einsp.)", 24.0, 516, "100/60 (2 WÜ Prim-Einsp.)", 120, "3x480 m3/h @ 75 m FIS", "im RL", 6.8, 475.18, 4.6, None, 475.18),
], columns=["anlage","z_NN_m","dp_intern_bar","P_max_AT-14_MW","m_max_AT-14_t/h","T_VL/T_RL_AT-14","P_max_UeZeit_MW","m_max_UeZeit_t/h","T_VL/T_RL_UeZeit",
            "Tmax_VL_C","Umwaelzung_installiert","Pumpenanordnung","Ruhedruck_bar_Ue","Ruhedruck_Bezug_mNN","abschalt_RL_unter_bar_Ue","abschalt_VL_ueber_bar_Ue","abschalt_Bezug_mNN"])
pumpst = pd.DataFrame([
 ("PSW", 471.55, "P1+2: 700 m3/h @ 38 m FIS (kein Parallelbetr.); P3: 325 m3/h @ 30 m FIS", "im VL, Richtung HWW", 0.2, 5.0, 18.0, 471.16),
 ("PSS", 481.45, "P1+2: 2x429 m3/h @ 40 m FIS (Richtung HWS); P3: 215 m3/h @ 27 m FIS (Richtung HKW)", "im VL", 0.2, 5.0, 18.0, 482.00),
], columns=["pumpstation","z_NN_m","Umwaelzung_installiert","Anordnung","dp_intern_bar","abschalt_Saugseite_unter_bar_Ue","abschalt_Druckseite_ueber_bar_Ue","Bezug_mNN"])
extras = pd.DataFrame([
 ("HKW","Max. Differenzdruck Austritt Werk","4,0 bar"), ("HKW","Ruhedruckbalken min/normal/max","6,1 / 8,4 / 9,1 bar Ü (Bezug 474,68 m NN)"),
 ("HKW","max. P (Umlenkbetrieb, Parallelb. mit Ost)","143 MW"), ("Ost-Erzeuger","max. Netz-Massenstrom parallel AVA/GT/BM","1550 t/h (SL5 750, SL6 500, SL7 300)"),
 ("Ost-Erzeuger","max. Differenzdruck Austritt Werk","7,5 bar"), ("HWW","Prim-Netz max. Vorlauftemperatur","110 °C"),
 ("HWW","WÜ Prim-Ausspeisung AT-14","31 MW/485 t/h + 9 MW/140 t/h (120/65 °C)"), ("HWW","WÜ Prim-Ausspeisung ÜZeit","12 MW/258 t/h (100/60 °C)"),
 ("HWW","2 Kessel (sekundär, direkt)","40 MW/625 t/h (120/65 °C)"), ("HWW","Differenzdruckbereich Eintritt WÜ","0,7 … 2,5 bar (max. 3,0); Überströmventil öffnet bei Δp > 3,5 bar"),
], columns=["anlage","groesse","wert"])
erzeuger.to_csv(RESULTS/"plan_erzeuger.csv", index=False); pumpst.to_csv(RESULTS/"plan_pumpstationen.csv", index=False)
with pd.option_context("display.width",250,"display.max_columns",30,"display.max_colwidth",40):
    display(erzeuger[["anlage","z_NN_m","P_max_AT-14_MW","P_max_UeZeit_MW","Tmax_VL_C","Pumpenanordnung","Ruhedruck_bar_Ue","abschalt_RL_unter_bar_Ue","abschalt_VL_ueber_bar_Ue"]])
    display(pumpst); display(extras)

**Konsistenzprüfungen gegen die YAML-Config und die Messung.**
* Die YAML-Assets (127/35/14,5/45/32/104 MW) entsprechen den Übergangszeit-Werten des Plans (HKW 127; GT-Ost 35; BM-HKW 14,5; AVA 45; HWS 32; HWW 24+… ).
* **HWW 104 MW (YAML)** = 64 MW (2 Kessel + 2 WÜ Prim-Einspeisung) + 40 MW (2 Kessel sekundär). Nur die 64 MW bzw. 24 MW (ÜZeit) speisen in das *Primärnetz* ein.
* Die **Ruhedruckdaten** (HKW 8,4 bar Ü) bestätigen die Überdruck-Annahme A-01 durch den Vergleich mit der Messung im nächsten Abschnitt.

In [ ]:
# --- Ruhedruck-Druckhöhe: H = z_Bezug + p/(ρ g)  (Netzpegel, Bezug für Höhenvergleiche)
RHO_COLD, G = 983.0, 9.81           # Wasser ~60 °C
def p_to_head_m(p_bar): return p_bar*1e5/(RHO_COLD*G)
rd = erzeuger.dropna(subset=["Ruhedruck_bar_Ue"]).copy()
rd["Druckhoehe_mNN"] = rd.Ruhedruck_Bezug_mNN + p_to_head_m(rd.Ruhedruck_bar_Ue)
print("Ruhedruck-Druckhöhe (Netzpegel) je Anlage [m NN]:"); print(rd[["anlage","Ruhedruck_bar_Ue","Ruhedruck_Bezug_mNN","Druckhoehe_mNN"]].round(1).to_string(index=False))
print("\nHinweis: Inselbetrieb-/Notbetrieb-Ruhedrücke einzelner Anlagen sind Werte für den *getrennten* Betrieb; im Verbund gilt der Ruhedruckbalken am HKW.")

# --- Messung 2025 gegen Plan: Ruhedruck HKW und Abschaltgrenzen der Erzeuger
def stat(col, f): return float(f(acron[col].dropna())) if col in acron else np.nan
chk = []
chk.append(("HKW Rücklauf (Ruhedruckbalken min/max)", "6,1 … 9,1", stat("HKW_Druck_RL",np.min), stat("HKW_Druck_RL",np.max), stat("HKW_Druck_RL",np.median), "Plan-Fenster liegt um die Messung -> Überdruck bestätigt"))
for sig, lim, an in (("AVA_Druck_RL",5.0,"AVA"),("GT-Ost_Druck_RL",4.4,"GT-Ost"),("BMHKW_Druck_RL",4.8,"BM-HKW"),("Heizwerk_Sued_Druck_RL",4.0,"HWS"),("HWW_Druck_RL",4.6,"HWW (sekundär, Messort ungeklärt)")):
    chk.append((f"{an}: Rücklauf-Untergrenze {lim} bar", f"≥ {lim}", stat(sig,np.min), stat(sig,np.max), stat(sig,np.median), f"Reserve nach unten: {stat(sig,np.min)-lim:+.1f} bar"))
for sig, lim, an in (("Heizwerk_Sued_Druck_RL",12.0,"HWS"),("HWW_Druck_RL",8.3,"HWW (sekundär, Messort ungeklärt)")):
    chk.append((f"{an}: Rücklauf-Obergrenze {lim} bar", f"≤ {lim}", stat(sig,np.min), stat(sig,np.max), stat(sig,np.median), f"Reserve nach oben: {lim-stat(sig,np.max):+.1f} bar"))
for sig, lim, an in (("AVA_Druck_VL",19.5,"AVA"),("GT-Ost_Druck_VL",20.5,"GT-Ost"),("BMHKW_Druck_VL",22.0,"BM-HKW"),("PSW_Druck_VL_nach_Pumpe",18.0,"PSW Druckseite"),("PPS_Druck_VL_nach_Pumpe",18.0,"PSS Druckseite")):
    chk.append((f"{an}: Vorlauf-/Druckseiten-Obergrenze {lim} bar", f"≤ {lim}", stat(sig,np.min), stat(sig,np.max), stat(sig,np.median), f"Reserve nach oben: {lim-stat(sig,np.max):+.1f} bar"))
chk = pd.DataFrame(chk, columns=["Größe","Plan-Grenze","Messung_min","Messung_max","Messung_median","Bewertung"])
display(chk.round(2)); chk.round(2).to_csv(RESULTS/"pruefung_druck_plan_vs_messung.csv", index=False)

### 9.2 Sektionen und Netzbereiche (WV 650 / WV 660)
Die vier HKW-Stammleitungen **SL1–SL4** versorgen im *separierten* Betrieb je eine Sektion. Die Ost-Erzeuger (AVA, GT-Ost, BM-HKW) speisen über **SL5, SL6, SL7** in die Sektionen SL1 bzw. SL3 ein (gesicherte Transportkapazität 750/500/300 t/h). Das Westnetz (PN16) wird über drei Leitungen SEK1–SEK3 aus dem Heizwerk West versorgt.

In [ ]:
# Netzvolumen [m3], Kundenanlagen [Stck], Anschlussleistung [MW] laut Plan (Stand 17.03.2022)
sek650 = pd.DataFrame([
 ("SL1","SL1-West",239,9,2.7),("SL1","SL1-Mitte",315,10,1.9),("SL1","SL1-Nord",0,0,0.0),("SL1","SL1-Ost",425,14,1.6),
 ("SL1","SL5-West mit MAN",123,1,32.0),("SL1","SL5-Mitte",480,4,2.4),("SL1","SL5-Ost",613,23,8.2),
 ("SL2","DOM-West",231,77,10.4),("SL2","DOM-Mitte",222,168,28.1),("SL2","DOM-Ost",205,89,13.4),
 ("SL2","CITY-West",179,84,29.4),("SL2","CITY-Mitte",214,166,33.5),("SL2","CITY-Ost",59,35,6.7),("SL2","CITY-Nord",39,71,14.5),("SL2","CITY-Süd",286,67,14.7),
 ("SL3","STAWA-West",160,67,13.5),("SL3","STAWA-Nord",164,25,8.0),("SL3","STAWA-Ost",326,50,16.7),("SL3","STAWA-Süd",141,56,12.9),("SL3","STAWA-Don Bosco",178,17,7.7),
 ("SL3","SL6-Ost",751,25,5.8),("SL3","SL6-Süd",529,73,3.6),("SL3","SL7-West",99,0,0.0),("SL3","SL7-Mitte",281,7,10.1),("SL3","SL7-Ost",372,0,2.3),
 ("SL4","SL4-Nord",346,11,2.1),("SL4","SL4-Süd",259,26,10.0),("SL4","ULRICH-Ost",63,78,9.1),("SL4","ULRICH-West",127,90,17.4),
 ("SL4","SÜD-Hochfeld-West",217,35,4.2),("SL4","SÜD-Hochfeld-Ost",388,27,9.3),("SL4","SÜD-Technopark",115,21,8.0),("SL4","SÜD-UNI",98,2,8.3),
 ("SL4","SÜD-Univiertel-Süd",141,38,19.1),("SL4","SÜD-Univiertel-Mitte",19,7,4.3),("SL4","SÜD-Univiertel-Nord",124,22,10.9),("SL4","SÜD-RWS",237,6,3.7),
 ("SL4","SÜD-Inno-Park",61,6,4.1),("SL4","SÜD-Antonsviertel",49,4,1.4)],
 columns=["sektion","teilgebiet","netzvol_m3","anlagen","p_anschluss_MW"])
sek650["netz"] = "Verbund (PN25)"
sek660 = pd.DataFrame([
 ("SEK1","Neusäß",30,14,1.6),("SEK1","UNI-MED",21,1,2.5),("SEK1","SEK1-West",356,12,23.0),("SEK1","SEK1-Mitte",147,9,2.8),("SEK1","SEK1-Süd",236,100,6.8),("SEK1","SEK6-Gesamt",131,31,8.1),
 ("SEK2","SEK2-Ost",127,128,4.0),("SEK2","SEK2-Nord",132,44,7.2),("SEK2","SEK5-Nord",54,15,0.5),("SEK2","SEK5-Süd",41,29,3.3),
 ("SEK3","SEK3-Nord",228,58,5.8),("SEK3","SEK3-Süd",151,181,8.0),("SEK3","SEK4-Gesamt",127,37,2.1)],
 columns=["sektion","teilgebiet","netzvol_m3","anlagen","p_anschluss_MW"])
sek660["netz"] = "West/Nordwest (PN16)"
sek = pd.concat([sek650, sek660], ignore_index=True)

# Summenprüfung gegen die im Plan ausgewiesenen Sektions-/Gesamtsummen
plan_summen = {"SL1":(2195,61,48.9),"SL2":(1434,757,150.7),"SL3":(3000,313,80.6),"SL4":(2244,373,111.9),"SEK1":(921,167,44.8),"SEK2":(354,216,15.0),"SEK3":(506,276,15.9)}
pruef = sek.groupby("sektion")[["netzvol_m3","anlagen","p_anschluss_MW"]].sum().round(1)
pruef = pruef.join(pd.DataFrame(plan_summen, index=["vol_plan","anl_plan","MW_plan"]).T)
pruef["Δ_MW"] = (pruef.p_anschluss_MW - pruef.MW_plan).round(1)
display(pruef)
tot650 = sek650.p_anschluss_MW.sum(); tot660 = sek660.p_anschluss_MW.sum()
print(f"Summe Verbund: {tot650:.1f} MW (Plan 392,0) | Netzvolumen {sek650.netzvol_m3.sum()} m3 (Plan 8.874) | Summe West: {tot660:.1f} MW (Plan 75,8)")
assert abs(tot650-392.0) < 0.6 and abs(tot660-75.8) < 0.3, "Transkriptionsfehler in Sektionstabelle"
sek.to_csv(RESULTS/"plan_sektionen.csv", index=False)
print("→ Leistungen und Netzvolumina konsistent mit den Plan-Summen (Rundung ±0,1 MW).")
print("Hinweis: Die Anlagenzahl der Sektion SL3 ist im Plan intern uneinheitlich (Teilgebiete 215+98+13 = 326, Sektionssumme 313); die Leistung ist davon nicht betroffen.")

**Auffällig:** *SL5-West mit MAN* ist im Plan **eine einzige Kundenanlage mit 32,0 MW Anschlussleistung** – das ist 8,2 % der Gesamtanschlussleistung (392 MW) des Verbundnetzes und sitzt am Ende der Trasse SL5, unmittelbar vor der Einbindung in die SL1-Sektion (Armaturen 342.1–342.3 „Stadtbachstraße, MAN östlich der Pforte“).

### 9.3 Armaturen und Trennstellen
Transkription der Tabelle „Einbauorte der Armaturen zur Netztrennung“ (WV650: 103 Zeilen, WV660: 21 Zeilen; per OCR mit Plausibilisierung, **kosmetische OCR-Reste in den Straßennamen möglich**, Nummern, DN und Normalstatus wurden geprüft).

In [ ]:
ARMATUREN_ROWS = [["01", "1529b", "329EZ", "Franziskanergasse9 (HKW SL1)", "MAS", 350, "offen", "WV650"], ["02", "1529b", "329EZ", "Franziskanergasse9 (HKW SL2)", "MAH", 350, "offen", "WV650"], ["03", "1529b", "329EZ", "Franziskanergasse9 (HKWSL3)", "MAH", 350, "offen", "WV650"], ["04", "1529b", "329EZ", "Franziskanergasse9 (HKWSL4)", "MAH", 400, "offen", "WV650"], ["001.2", "1428b", "001S", "Thomm Straße (Ecke Herwart Straße)", "MAH", 400, "offen", "WV650"], ["007.1", "1527d", "007S", "Senkelbach Straße", "MAH", 400, "offen", "WV650"], ["035.1", "1528c", "035S", "Jesuitengasse13", "MAH", 200, "offen", "WV650"], ["042.1", "1528d", "042S", "Hoher Weg30", "MAH", 350, "offen", "WV650"], ["050.1", "1529d", "050S", "Bei Sankt Max", "MAH", 300, "offen", "WV650"], ["050.2", "1529d", "050S", "Bei Sankt Max", "MAH", 250, "offen", "WV650"], ["080.2", "1626d", "080S", "Schletterer Straße (Richtung SL2)", "MAH", 350, "offen", "WV650"], ["080.3", "1626d", "080S", "Schletterer Straße (Richtung Fuggerschule)", "MAH", 150, "offen", "WV650"], ["105.2", "1627c", "105S", "Frölich Straße (Richtung Diako)", "MAH", 125, "offen", "WV650"], ["105.3", "1627c", "105S", "Frölich Straße (Richtung Hauptbahnhof)", "MAH", 300, "offen", "WV650"], ["112.1", "1627d", "112G", "Gutenberg Straße 1 (Maria-Theresia-Gymn.im Keller)", "MAH", 150, "offen", "WV650"], ["114.2", "1628a", "114S", "Kaserm Straße (Richtung Theater Straße)", "MAH", 300, "offen", "WV650"], ["114.3", "1628a", "114S", "Kasern Straße (Richtung Schletter Straße)", "MAH", 350, "offen", "WV650"], ["141.1", "1628b", "141G", "Hoher Weg 1 (swa-Betriebsgebaude im Keller)", "MAH", 80, "zu", "WV650"], ["165.1", "1628d", "165G", "Anna Straße3", "MAH", 80, "offen", "WV650"], ["166.1", "1729d", "166S", "Forster Straße", "MAH", 250, "offen", "WV650"], ["170.1", "1628d", "170G", "Rathausplatz8", "MAH", 65, "offen", "WV650"], ["185.1", "1629a", "185G", "Leonhardsberg (in der Tiefgarage)", "MAH", 200, "offen", "WV650"], ["187.1", "1629a", "187G", "Leonhardsberg 15 (Stadtbad Heizzentrale)", "MAH", 250, "zu", "WV650"], ["195.1", "1628a", "195G", "Grottenau6", "MAH", 80, "offen", "WV650"], ["203.1", "1629c", "203G", "Barfufer Straße (Brücke-Nord)", "MAH", 200, "offen", "WV650"], ["204.1", "1629c", "204G", "Rathausplatz2a (Verwaltungsgebäude Il)", "MAH", 200, "offen", "WV650"], ["207.1", "1629c", "207S", "Barfußer Straße (Brücke-Suid)", "AH", 200, "zu", "WV650"], ["210.1", "1629d", "210S", "Au Berer Gang", "MAH", 150, "zu", "WV650"], ["214.1", "1630a", "214G", "Rauwolff Straße5", "MAH", 100, "offen", "WV650"], ["230.1", "1728c", "230G", "Halder Straße 3 (Stadtsparkasse)", "MAH", 125, "offen", "WV650"], ["233.1", "1728a", "233S", "Fugger Straße2 (Mc Donalds)", "MAH", 150, "offen", "WV650"], ["252.1", "1728c", "252S", "Konrad-Adenauer-Allee 13-15", "MAH", 150, "offen", "WV650"], ["253.1", "1728d", "253S", "Hall Straße (Ecke K.-Adenauer-Allee 21 im Gehweg)", "MAH", 200, "offen", "WV650"], ["275.1", "1728d", "275G", "Maximilian Straße 69 (Standesamt)", "MAH", 200, "offen", "WV650"], ["276.1", "1729d", "276S", "Am Schwall (Schacht beim Glascontainer)", "MAH", 200, "offen", "WV650"], ["278.1", "1729d", "278S", "Am Schwal (Brücke)", "MAK", 350, "offen", "WV650"], ["291.1", "1829d", "291G", "Beim Rabenbad6", "MAH", 150, "offen", "WV650"], ["294.1", "1929b", "294S", "Rote-Torwall-Straße-Trennung SL4", "AS", 350, "offen", "WV650"], ["297.1", "2029b", "297P", "An der Hochschule (Pumpstation Süd)", "MAH", 350, "offen", "WV650"], ["300.1", "1628a", "300G", "Hafnerberg2", "MAH", 150, "offen", "WV650"], ["325.1", "1729a", "325G", "Hunoldsgraben15 (inder Tiefgarage)", "MAH", 125, "offen", "WV650"], ["338.1", "1531c", "338G", "Lechhauser Straße 20", "MAH", 250, "offen", "WV650"], ["338.2", "1531c", "338G", "Lechhauser Straße20", "MAH", 350, "offen", "WV650"], ["338.3", "1531c", "338G", "Lechhauser Straße20", "MAH", 250, "offen", "WV650"], ["342.1", "1329c", "342S", "Stadtbach Straße (MAN östlich der Pforte)", "MAH", 400, "offen", "WV650"], ["342.2", "1329c", "342S", "Stadtbach Straße (MAN östlich der Pforte)", "MAH", 300, "offen", "WV650"], ["342.3", "1329c", "342S", "Stadtbach Straße (MAN östlich der Pforte)", "MAH", 400, "offen", "WV650"], ["346.1", "1625d", "346P", "Holzbach Straße14 (Pumpstation West)", "MAH", 350, "offen", "WV650"], ["368.1", "1731c", "368E", "Nagahama-Alle (Am Provinopark 1a)", "AH", 250, "offen", "WV650"], ["441.5", "1628c", "441S", "Fugger Straße (ehem.Fu Bgangertunnel)", "MAH", 150, "offen", "WV650"], ["476.1", "2428d", "476G", "Universitats Straße 10 (UNI)", "MAH", 300, "offen", "WV650"], ["502.1", "2029d", "502S", "Scherlin Straße (Haltestelle)", "MAH", 350, "offen", "WV650"], ["502.2", "2029d", "502S", "Schertlin Straße (Haltestelle)", "MAH", 250, "offen", "WV650"], ["515.1", "0934b", "515EZ", "Beim Grenzgraben 10 (GT-Ost)", "MAS", 350, "offen", "WV650"], ["517.1", "2329a", "517S", "Alter Postweg 86a (RWS)", "MAH", 350, "offen", "WV650"], ["525.1", "2428d", "525S", "Alter Postweg120", "MAH", 300, "offen", "WV650"], ["539.2", "2529a", "539S", "Salomon-ldler-Straße 22", "MAH", 150, "offen", "WV650"], ["541.1", "2529a", "541S", "Salomon-ldler-Straße 18", "MAH", 150, "offen", "WV650"], ["555.1", "2530a", "555S", "Josef-Priller-Straße 10", "MAH", 150, "offen", "WV650"], ["555.2", "2530a", "555S", "Josef-Priller-Straße10", "MAH", 100, "offen", "WV650"], ["561.1", "2329c", "561EZ", "Alter Postweg93 (Heizwerk Süd Südeinspeisung1)", "MAH", 350, "offen", "WV650"], ["561.2", "2329c", "561EZ", "Alter Postweg93 (Heizwerk Süd Südeinspeisung2)", "MAH", 200, "offen", "WV650"], ["561.3", "2329c", "561EZ", "Alter Postweg93 (Heizwerk Süd Betriebswarme)", "MAH", 100, "offen", "WV650"], ["564.1", "1629a", "564G", "Metzplatz (im Keller des Gebäudes nordliche Seite)", "MAH", 65, "offen", "WV650"], ["591.2", "2129d", "591S", "Johann-Georg-Halske-Straße", "MAH", 200, "offen", "WV650"], ["603.4", "1623b", "603EZ", "Flandern Straße 10 (Heizwerk West Primareinspeisung)", "MAS", 350, "offen", "WV650"], ["735.1", "1828d", "735S", "Theodeor-Heuss-Platz MAH2", "?", 200, "offen", "WV650"], ["750.1", "0935a", "750S", "Pottmeser Straße (Richtung GT-Ost)", "MAH", 350, "offen", "WV650"], ["750.3", "0935a", "750S", "Pottmeser Straße (Richtung Pottmeser Straße)", "MAH", 350, "offen", "WV650"], ["769.1", "1831b", "769E", "Schafflerbach Straße", "AH", 65, "zu", "WV650"], ["773.1", "1034a", "773S", "Affinger Straße (Richtung Biomasse-HKW)", "MAH", 350, "offen", "WV650"], ["773.2", "1034a", "773S", "Affinger Straße (Richtung Affinger Straße)", "MAH", 300, "offen", "WV650"], ["773.3", "1034a", "773S", "Affinger Straße (Richtung Neuburger Straße)", "MAH", 350, "offen", "WV650"], ["777.1", "1132a", "777S", "Kurt-Schumacher-Straße (Richtung Pottmeser Straße)", "MAH", 350, "offen", "WV650"], ["777.2", "1132a", "777S", "Kurt-Schumacher-Straße (Richtung Neuburger Straße)", "MAH", 350, "offen", "WV650"], ["777.3", "1132a", "777S", "Kurt-Schumacher-Straße (Richtung Medienzentrum)", "MAH", 250, "offen", "WV650"], ["783.1", "1832b", "783S", "Reichenberger Straße", "MAH", 250, "offen", "WV650"], ["783.2", "1832b", "783S", "Reichenberger Straße-Richtung KUKA", "MAH", 250, "offen", "WV650"], ["783.3", "1832b", "783S", "Reichenberger Straße-Richtung Schwabencenter", "MAH", 200, "offen", "WV650"], ["784.1", "0835a", "784EZ", "Beim Grenzgraben 20 (Biomasse-HKW)", "MAH", 250, "offen", "WV650"], ["785.1", "1728d", "785G", "Maximilian Straße46 (Schaezler Palais)", "MAH", 125, "offen", "WV650"], ["809.1", "0934a", "809S", "Am Mittleren Moos (Richtung GT-Ost)", "MAH", 400, "offen", "WV650"], ["809.2", "0934a", "809S", "Am Mittleren Moos/Ulstett Straße (SL5 Richtung Westen)", "MAH", 400, "offen", "WV650"], ["810.1", "0834c", "810EZ", "Am Mitleren Moos 60 (AVA-Ubemahmestation)", "MAH", 350, "offen", "WV650"], ["818.1", "1733b", "818S", "Proviantbach Straße Richtung Westen", "MAH", 125, "offen", "WV650"], ["818.2", "1733b", "818S", "Proviantbach Straße Richtung Osten", "MAH", 250, "offen", "WV650"], ["818.3", "1733b", "818S", "Proviantbach Straße Richtung Suiden", "MAH", 250, "offen", "WV650"], ["832.1", "1131b", "832S", "Neuburger Straße (Richtung Steinerne Furt)", "MAH", 350, "offen", "WV650"], ["832.2", "1131b", "832S", "Neuburger Straße (Richtung Neuburger Straße Süd)", "MAH", 350, "offen", "WV650"], ["832.3", "1131b", "832S", "Neuburger Straße (Richtung Hans-Böckler-Straße)", "MAH", 350, "offen", "WV650"], ["833.1", "1131a", "833S", "Hans-Böckler-Straße (Richtung GT-Ost)", "MAH", 400, "offen", "WV650"], ["833.3", "1131a", "833S", "Hans-Böckler-Straße (Richtung MAN)", "MAH", 400, "offen", "WV650"], ["834.1", "2628b", "834S", "Am Technologiezentrum", "MAH", 200, "offen", "WV650"], ["860.1", "2229c", "860E", "Alter Postweg (nordl.Parkplatz RWS)", "AH", 200, "offen", "WV650"], ["861.1", "1528b", "861G", "Frauentor Straße 26", "MAH", 100, "offen", "WV650"], ["864.1", "2429c", "864G", "Alter Postweg115", "MAH", 200, "offen", "WV650"], ["887.1", "1428a", "887S", "Thomm Straße/Trennung Am Pfannenstiel (Nordspange)", "MAH", 300, "offen", "WV650"], ["888.1", "1334d", "888S", "Südtiroler Straße/Kalterer Straße", "MAH", 350, "offen", "WV650"], ["891.1", "1731b", "891S", "Provino Straße", "MAH", 250, "offen", "WV650"], ["894.1", "2028c", "894S", "Windprecht Straßesidlich Scherlin Straße", "MAH", 250, "offen", "WV650"], ["895.1", "1827b", "895S", "Ladehofe", "MAH", 300, "offen", "WV650"], ["995.1", "1634a", "995G", "Zugspitz Straße/KUKA", "MAH", 250, "offen", "WV650"], ["603.1", "1623b", "603EZ", "Flandem Straße10 (Heizwerk West SEK1)", "MAS", 300, "offen", "WV660"], ["603.2", "1623b", "603EZ", "Flandem Straße10 (Heizwerk West SEK2)", "MAS", 300, "offen", "WV660"], ["603.3", "1623b", "603EZ", "Flandem Straße10 (Heizwerk West SEK3)", "MAS", 300, "offen", "WV660"], ["605.1", "1321b", "605S", "Dr.-Dürrwanger-Straße-Richtung SEK1", "MAH", 250, "offen", "WV660"], ["605.2", "1321b", "605S", "Dr.-Dürrwanger-Straße-Richtung SEK2", "MAH", 200, "offen", "WV660"], ["613.1", "1323d", "613S", "Ulmer Straße", "MAH", 250, "offen", "WV660"], ["622.1", "1523b", "622S", "Am Exerzierplatz", "AH", 200, "zu", "WV660"], ["622.2", "1523b", "622S", "Am Exerzierplatz", "AS", 200, "zu", "WV660"], ["631.1", "1622b", "631S", "Columbus Straße7-Richtung Heizwerk West", "AS", 250, "offen", "WV660"], ["631.2", "1622b", "631S", "Columbus Straße7-Richtung Sheridanpark", "MAH", 250, "offen", "WV660"], ["631.3", "1622b", "631S", "Columbus Straße7-Richtung Zentralklinikum", "AS", 250, "offen", "WV660"], ["631.4", "1622b", "631S", "Columbus Straße7-Richtung Reesepark", "AS", 250, "offen", "WV660"], ["650.1", "1522b", "650S", "Carl-Schurz-Straße 16", "MAH", 200, "offen", "WV660"], ["734.1", "1922c", "734E", "General-Cramer-Weg", "AH", 150, "offen", "WV660"], ["797.1", "1822d", "797S", "Grasiger Weg-Richtung Heizwerk West", "MAH", 200, "offen", "WV660"], ["797.2", "1822d", "797S", "Grasiger Weg-Richtung Sheridanpark", "MAH", 150, "offen", "WV660"], ["797.3", "1822d", "797S", "Grasiger Weg-Richtung Schacht850", "MAH", 200, "offen", "WV660"], ["816.1", "1223c", "816S", "Bertha-von-Suttner-Straße", "MAH", 200, "offen", "WV660"], ["850.1", "1822c", "850S", "Siegfried-Aufhäuser-Straße", "MAH", 200, "offen", "WV660"], ["850.3", "1822c", "850S", "Siegfried-Aufhäuser-Straße", "MAH", 150, "offen", "WV660"], ["870.1", "1219a", "870S", "Geschwister-Schönert-Straße-Richtung Neusäß", "MAH", 200, "offen", "WV660"], ["870.2", "1219a", "870S", "Geschwister-Schönert-Straße-Richtung UNI-MED", "MAH", 200, "offen", "WV660"]]

armaturen = pd.DataFrame(ARMATUREN_ROWS, columns=["armatur_nr","schaltplan","anlage","strasse","typ","dn_mm","normalstatus","planblatt"])
SEPARIERUNG_650 = ["141.1","300.1","112.1","080.2","080.3","203.1","166.1","735.1","253.1","785.1","325.1","187.1","832.3","777.3","809.1"]
armaturen["separierung_schliesst"] = armaturen.armatur_nr.isin(SEPARIERUNG_650)
armaturen["man_relevant"] = armaturen.armatur_nr.isin(["342.1","342.2","342.3","833.1","833.3","809.2","001.2"])
armaturen.to_csv(RESULTS/"plan_armaturen.csv", index=False)
print(len(armaturen), "Armaturen | Normalstatus:", armaturen.normalstatus.value_counts().to_dict())
print("Normal ZU (im Plan als geschlossen geführt – bitte prüfen):", armaturen.loc[armaturen.normalstatus=='zu',['armatur_nr','strasse','dn_mm']].values.tolist())
print("\nMAN-relevante Armaturen:"); display(armaturen.loc[armaturen.man_relevant,["armatur_nr","strasse","typ","dn_mm","normalstatus"]])
print("Armaturen der Separierung (WV650, Befehl 'Separierung' schließt diese Notabsperrgruppen):")
display(armaturen.loc[armaturen.separierung_schliesst,["armatur_nr","strasse","dn_mm","normalstatus"]])
# Vollständigkeit der Separierungsliste
fehlend = sorted(set(SEPARIERUNG_650) - set(armaturen.armatur_nr)); print("Separierungs-Armaturen nicht in der Tabelle:", fehlend or "keine")

### 9.4 Transportgrenzen und Plausibilität der Trassen-Nennweiten
Die im Plan genannten gesicherten Transportkapazitäten im separierten Betrieb (SL5 750, SL6 500, SL7 300 t/h) werden mit den in der Armaturenliste abgelesenen Nennweiten der jeweiligen Trasse auf Strömungsgeschwindigkeit geprüft.

In [ ]:
RHO_HOT = 960.0
def v_ms(t_h, dn_mm): return (t_h*1000/3600)/(RHO_HOT*np.pi*(dn_mm/1000)**2/4)
def q_mw(t_h, dT=39.0): return t_h*1000/3600*4.19*dT/1000
tg = pd.DataFrame([("SL5",750,400,"Armaturen 001.2, 342.1/.3, 809.x: DN400"),("SL6",500,350,"Armaturen 338.2, 832.x, 750.x: DN350"),("SL7",300,250,"Armaturen 818.2/.3, 995.1: DN250 (888.1: DN350)")],
                  columns=["trasse","gesicherte_Kapazitaet_t/h","DN_Plan_mm","Fundstelle"])
tg["v_bei_Grenze_m/s"] = [v_ms(t,d) for t,d in zip(tg["gesicherte_Kapazitaet_t/h"], tg.DN_Plan_mm)]
tg["Q_bei_dT39K_MW"] = [q_mw(t) for t in tg["gesicherte_Kapazitaet_t/h"]]
display(tg.round(2))
print(f"Summe SL5+SL6+SL7: {tg['gesicherte_Kapazitaet_t/h'].sum()} t/h ≙ {tg.Q_bei_dT39K_MW.sum():.0f} MW bei ΔT=39 K; YAML-Trasse hkw_to_ost: DN600 (Sammelleitung) mit v=3 m/s ≙ 135 MW")
assert 1.0 < tg["v_bei_Grenze_m/s"].min() and tg["v_bei_Grenze_m/s"].max() < 2.5
print("→ Die gesicherten Kapazitäten entsprechen Geschwindigkeiten von etwa 1,5–1,8 m/s in den abgelesenen Nennweiten: die Zuordnung Trasse ↔ DN ist plausibel.")

### 9.5 Vergleich: YAML-Topologie gegen Pläne

In [ ]:
ycfg = yaml.safe_load((ROOT/"configs/paper_2/Stadtbach_topo.yaml").read_text(encoding="utf-8"))
yn, yp = ycfg["network"]["nodes"], ycfg["network"]["pipes"]
E, N = len(yp), len(yn); comps = 1
print(f"YAML: {N} Knoten, {E} Leitungen, unabhängige Schleifen = E − N + Komponenten = {E-N+comps} (0 = Baum)")
vgl = pd.DataFrame([
 ("Netzstruktur","reiner Baum, 1 bidirektionale Trasse","Verbundnetz: 4 Stammleitungen SL1–4 ab HKW, 3 Ost-Trassen SL5–7, Vermaschung über Notabsperr-Armaturen","hoch"),
 ("Ost-Anbindung","1 Sammelleitung hkw_to_ost (DN600, 5,93 km) bis zu einem 'j_ost'","3 getrennte Trassen SL5/SL6/SL7 (DN400/350/250) mit Kopplungen bei 832/777/809 und Einbindung bei 001.2, 338, 818","hoch"),
 ("MAN","Verbraucher j_man, DN400, 850 m ab HKW, direkt am HKW","32-MW-Einzelkunde am SL5-Ende, Armaturen 342.1–3 (DN400/300/400), ~0,3–0,4 km vor Einbindung 001.2 in SL1","hoch"),
 ("Westnetz","direkte Rohrverbindung HKW→PSW→HWW→Kunden (ein Druckniveau)","PN16-Netz, hydraulisch über Wärmeübertrager von PN25 getrennt (Pfersee/PSW, Kapellenschule); Versorgung über SEK1–SEK3 aus HWW","hoch"),
 ("Druckniveau","VL-Sollwert 16 bar am HKW","Ruhedruckbalken 8,4 bar Ü; Pumpen im VL (HKW/PSW/PSS) bzw. im RL (Ost-Erzeuger, HWS, HWW)","hoch"),
 ("Höhen","keine","Bezugshöhen der Erzeuger/Pumpstationen 466,7 … 490,5 m NN (Δ ≈ 24 m ≈ 2,4 bar)","hoch"),
 ("PN-Zonen","keine","PN16 (West/Nordwest), PN25 (Verbund)","hoch"),
 ("Verbrauchsstationen","24 einzelne Stichleitungen (DN125–400) je Station","Sektionen mit 1.504 Kundenanlagen; Stationen = Messstellen an Sektionsleitungen","mittel"),
 ("Erzeugerleistungen","127/35/14,5/45/32/104 MW","identisch (ÜZeit-Werte, WV640); HWW gesamt 104 = 64 Prim + 40 sek.","hoch"),
], columns=["thema","YAML","Plan","Sicherheit_Befund"])
with pd.option_context("display.max_colwidth",95,"display.width",250): display(vgl)
vgl.to_csv(RESULTS/"vergleich_yaml_plan.csv", index=False)

## 10. Aufbau des kanonischen Netzmodells (Sektionsmodell v0.1)
**Auflösung:** Ein Straßen- und Rohrabschnittsmodell ist aus den vorhandenen Rasterplänen nicht ableitbar (keine Rohrlängen/-durchmesser je Abschnitt). Es wird ein **Sektionsmodell** mit Vorlauf- und Rücklaufnetz aufgebaut, das die reale Vermaschung (SL1–SL7, Koppelstellen, Ost-Trassen) abbildet. Rohrlängen sind nur für wenige Trassen belastbar; die übrigen Werte sind als *Ersatzleitungen* gekennzeichnet und werden in Kapitel 13 gegen Messdaten kalibriert.

**Modellierungsentscheidungen (Arbeitsannahmen, bitte prüfen):**
1. **MAN-Anschlusspunkt:** Knoten `MAN` auf der Trasse SL5 zwischen Koppelstelle `K001` (001.2) und `SL5M` (Hans-Böckler-Straße). Der 32-MW-Kunde ist Verbraucher an diesem Knoten; der Speicher wird als zusätzliche Anschlussstelle **am selben Knoten** modelliert (Kapitel 15).
2. **Westnetz (PN16)** wird hydraulisch als *getrennt* behandelt. Die Kopplung zum Verbund erfolgt an `HWW` über Wärmeübertrager (nur thermische Kopplung, Massenströme sind getrennt).
3. **Vermaschung:** Verbindungen zwischen den Sektionen sind Kanten mit Normalstatus *offen*; die Separierung schließt sie.
4. **Höhen:** nur für Erzeuger und Pumpstationen aus WV640 bekannt; für alle übrigen Knoten fehlt die Höhe (`z_NN_m = NaN`) → in Kapitel 13 als Kalibrierkonstante (Höhen-Offset) je Sektion.

In [ ]:
# ------------------------------- Knoten -------------------------------
def _n(i,name,typ,sek,netz,z=np.nan,p_MW=np.nan,x=0,y=0,quelle="",sicher="mittel"):
    return dict(node_id=i,name=name,typ=typ,sektion=sek,netz=netz,PN=(16 if netz=="West" else 25),z_NN_m=z,p_anschluss_MW=p_MW,x_km=x,y_km=y,quelle=quelle,sicherheit=sicher)
knoten = pd.DataFrame([
 _n("HKW","Heizkraftwerk Sammelschiene","erzeuger","-","Mitte",473.77,0,0.0,0.0,"WV640","hoch"),
 _n("K001","Koppelstelle SL1/SL5 (Thommstr., Arm. 001.2)","koppelstelle","SL1","Mitte",np.nan,0,-0.6,0.9,"WV650","mittel"),
 _n("MAN","MAN (Stadtbachstr., Arm. 342.1-3) – Speicher-Anschluss","verbraucher","SL1","Mitte",np.nan,32.0,-0.2,1.4,"WV650 Tabelle SL5-West mit MAN","hoch"),
 _n("SL5M","SL5-Mitte (Hans-Böckler-Str., Arm. 833)","verbraucher","SL1","Ost",np.nan,2.4,0.9,2.5,"WV650","mittel"),
 _n("KNEU","Koppelstelle SL5/SL6 (Neuburger Str., Arm. 832/777)","koppelstelle","SL1/SL3","Ost",np.nan,0,1.7,3.1,"WV650","mittel"),
 _n("SL5O","SL5-Ost (Am Mittleren Moos, Arm. 809.2)","verbraucher","SL1","Ost",np.nan,8.2,3.2,3.7,"WV650","mittel"),
 _n("AVA","AVA-Übernahme","erzeuger","-","Ost",466.69,0,4.2,4.1,"WV640","hoch"),
 _n("GT","GT-Ost","erzeuger","-","Ost",467.98,0,4.6,3.8,"WV640","hoch"),
 _n("BM","Biomasse-HKW","erzeuger","-","Ost",467.08,0,4.9,4.4,"WV640","hoch"),
 _n("SL6S","SL6-Gebiet (Ost+Süd)","verbraucher","SL3","Ost",np.nan,9.4,2.6,1.9,"WV650","mittel"),
 _n("K338","Koppelstelle SL6/SL3 (Lechhauser Str., Arm. 338)","koppelstelle","SL3","Ost",np.nan,0,1.3,0.4,"WV650","mittel"),
 _n("SL7G","SL7-Gebiet (West+Mitte+Ost)","verbraucher","SL3","Ost",np.nan,12.4,4.2,1.0,"WV650","mittel"),
 _n("K818","Koppelstelle SL7/SL3 (Proviantbachstr., Arm. 818)","koppelstelle","SL3","Ost",np.nan,0,2.3,-0.4,"WV650","mittel"),
 _n("SL1G","SL1-Gebiet (West/Mitte/Nord/Ost) ohne SL5","verbraucher","SL1","Mitte",np.nan,6.2,-1.5,0.4,"WV650","mittel"),
 _n("SEC2","Sektion SL2 (DOM + CITY)","verbraucher","SL2","Mitte",np.nan,150.7,-0.7,-0.3,"WV650","mittel"),
 _n("SEC3","Sektion SL3-STAWA","verbraucher","SL3","Mitte",np.nan,58.8,0.9,-0.2,"WV650","mittel"),
 _n("SEC4","Sektion SL4 (SL4 + ULRICH)","verbraucher","SL4","Süd",np.nan,38.6,0.3,-1.2,"WV650","mittel"),
 _n("PSS","Pumpstation Süd","pumpstation","SL4","Süd",481.45,0,0.5,-3.2,"WV640","hoch"),
 _n("SUED","Sektion SÜD","verbraucher","SL4","Süd",np.nan,73.3,0.8,-3.8,"WV650","mittel"),
 _n("HWS","Heizwerk Süd","erzeuger","SL4","Süd",490.51,0,0.6,-4.5,"WV640","hoch"),
 _n("PSW","Pumpstation West","pumpstation","SL1","Mitte",471.55,0,-2.6,-0.6,"WV640","hoch"),
 _n("HWW","Heizwerk West (Primärseite / WÜ)","erzeuger","-","Mitte",477.99,0,-4.0,-0.3,"WV640","hoch"),
 _n("WEST","Westnetz PN16 (SEK1–SEK3, sekundär)","verbraucher","SEK1-3","West",np.nan,75.8,-4.8,0.8,"WV660","mittel"),
])
knoten.to_csv(RESULTS/"modell_knoten.csv", index=False)
print(len(knoten), "Knoten | Typen:", knoten.typ.value_counts().to_dict())
print(f"Σ Anschlussleistung Knoten = {knoten.p_anschluss_MW.sum():.1f} MW (Plan Verbund 392,0 + West 75,8 = 467,8)")

In [ ]:
# ------------------------------- Leitungen / Kanten -------------------------------
# laenge_quelle: Y = YAML-Wert, P = YAML-Summe nach Plan-Trassenanteil aufgeteilt, S = Schätzung (Sicherheit niedrig)
def _e(i,a,b,art,dn,L,lq,sich,stat,arm,qmax=np.nan,bidir=True):
    return dict(edge_id=i,von=a,nach=b,art=art,dn_mm=dn,laenge_m=L,laenge_quelle=lq,sicherheit=sich,normalstatus=stat,armaturen=arm,max_t_h_separiert=qmax,bidirektional=bidir)
# SL5-Gesamtlänge 5.930 m (YAML, HKW->Ost) nach Pixelanteilen der Plan-Trasse verteilt (HKW-K001 14,4 %, K001-MAN 6,7 %, MAN-SL5M 38,5 %, SL5M-KNEU 4,4 %, KNEU-AVA 36,1 %)
kanten = pd.DataFrame([
 _e("SL1a","HKW","K001","Stammleitung SL1",400,854,"P","niedrig","offen","01, 001.2"),
 _e("SL5a","K001","MAN","Trasse SL5",400,394,"P","niedrig","offen","342.1, 342.3"),
 _e("SL5b","MAN","SL5M","Trasse SL5",400,2285,"P","niedrig","offen","833.1, 833.3",750),
 _e("SL5c","SL5M","KNEU","Trasse SL5",400,258,"P","niedrig","offen","832.x",750),
 _e("SL5d","KNEU","SL5O","Trasse SL5",400,1800,"P","niedrig","offen","809.2",750),
 _e("SL5e","SL5O","AVA","Erzeugeranschluss",350,870,"Y","mittel","offen","810.1, 809.x",750),
 _e("SL5f","SL5O","GT","Erzeugeranschluss",350,60,"Y","mittel","offen","809.1, 515.1",750),
 _e("SL5g","SL5O","BM","Erzeugeranschluss",250,60,"Y","mittel","offen","784.1, 773.1",750),
 _e("SL6a","KNEU","SL6S","Trasse SL6",350,1500,"S","niedrig","offen","777.x, 832.1",500),
 _e("SL6b","SL6S","K338","Trasse SL6",350,1500,"S","niedrig","offen","338.2",500),
 _e("SL6c","KNEU","GT","Trasse SL6 (Ost)",350,3000,"S","niedrig","offen","773.x, 750.x",500),
 _e("SL7a","K818","SL7G","Trasse SL7",250,2500,"S","niedrig","offen","818.2, 818.3",300),
 _e("SL7b","SL7G","GT","Trasse SL7 (Ost)",250,3500,"S","niedrig","offen","888.1, 995.1",300),
 _e("SL2","HKW","SEC2","Stammleitung SL2",350,600,"S","niedrig","offen","02"),
 _e("SL3","HKW","SEC3","Stammleitung SL3",350,800,"S","niedrig","offen","03, 050.x"),
 _e("SL4a","HKW","SEC4","Stammleitung SL4",400,1200,"S","niedrig","offen","04, 294.1"),
 _e("SL4b","SEC4","PSS","Stammleitung SL4",400,3270,"P","niedrig","offen","297.1"),
 _e("SL4c","PSS","SUED","Stammleitung SL4",400,610,"Y","mittel","offen","502.x"),
 _e("SL4d","SUED","HWS","Erzeugeranschluss",350,900,"S","niedrig","offen","561.1/.2"),
 _e("SL1b","K001","SL1G","Stammleitung SL1",350,1400,"S","niedrig","offen","001.2, 007.1"),
 _e("SL1c","SL1G","PSW","Stammleitung SL1",350,1400,"S","niedrig","offen","080.x, 346.1"),
 _e("W1","PSW","HWW","Prim-Leitung West",350,960,"Y","mittel","offen","346.1, 603.4"),
 # Verbundkanten (Vermaschung): im Normalbetrieb offen, bei Separierung geschlossen
 _e("M_SL1_SL2","SL1G","SEC2","Verbund SL1↔SL2",350,700,"S","niedrig","offen","080.2, 080.3, 141.1, 300.1, 112.1"),
 _e("M_SL2_SL3","SEC2","SEC3","Verbund SL2↔SL3",200,600,"S","niedrig","offen","203.1, 214.1, 050.1/.2"),
 _e("M_SL2_SL4","SEC2","SEC4","Verbund SL2↔SL4",200,800,"S","niedrig","offen","735.1, 785.1, 325.1, 253.1, 187.1"),
 _e("M_SL3_SL4","SEC3","SEC4","Verbund SL3↔SL4",250,700,"S","niedrig","offen","166.1"),
 _e("K_SL6_SL3","K338","SEC3","Landung SL6 in SL3",350,300,"S","niedrig","offen","338.1-3"),
 _e("K_SL7_SL3","K818","SEC3","Landung SL7 in SL3",250,600,"S","niedrig","offen","818.1-3, 783.x"),
], columns=None)
kanten["laenge_km"] = kanten.laenge_m/1000
kanten.to_csv(RESULTS/"modell_kanten.csv", index=False)
print(len(kanten), "Kanten; Längenquelle:", kanten.laenge_quelle.value_counts().to_dict(), "| Σ Länge:", round(kanten.laenge_m.sum()/1000,1), "km")

# --------- Topologie-Kennzahlen (scipy statt networkx, keine zusätzliche Abhängigkeit)
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
idx = {n:i for i,n in enumerate(knoten.node_id)}
A = coo_matrix((np.ones(len(kanten)), ([idx[a] for a in kanten.von],[idx[b] for b in kanten.nach])), shape=(len(idx),)*2)
ncomp, lab = connected_components(A, directed=False)
print(f"Zusammenhangskomponenten: {ncomp} (Knoten je Komponente: {np.bincount(lab).tolist()})")
print(f"Unabhängige Schleifen (E − N + K): {len(kanten)-len(knoten)+ncomp}  → das Modell ist vermascht (YAML: 0)")
print("Die 2. Komponente ist der Knoten WEST (PN16): bewusst nur thermisch über Wärmeübertrager an HWW gekoppelt, keine hydraulische Kante.")
assert all(k in idx for k in list(kanten.von)+list(kanten.nach))

In [ ]:
# --------- Schaubild des Sektionsmodells
fig, ax = plt.subplots(figsize=(12,8.5))
pos = knoten.set_index("node_id")[["x_km","y_km"]]
farbe = {"erzeuger":COL["warn"],"pumpstation":COL["ok"],"verbraucher":COL["mess"],"koppelstelle":COL["grau"]}
for _,e in kanten.iterrows():
    a,b = pos.loc[e.von], pos.loc[e.nach]
    verbund = e.edge_id.startswith("M_")
    ax.plot([a.x_km,b.x_km],[a.y_km,b.y_km], color=COL["vl"] if not verbund else COL["rl"], lw=1+e.dn_mm/150, ls="--" if verbund else "-", alpha=.85, zorder=1)
for _,n in knoten.iterrows():
    s = 120+ (np.sqrt(n.p_anschluss_MW)*25 if n.p_anschluss_MW==n.p_anschluss_MW else 0)
    ax.scatter(n.x_km,n.y_km,s=s,color=farbe[n.typ],edgecolor="k",zorder=3, marker="*" if n.node_id=="MAN" else "o")
    ax.annotate(n.node_id, (n.x_km,n.y_km), xytext=(6,6), textcoords="offset points", fontsize=9, fontweight="bold" if n.node_id=="MAN" else None)
ax.set(xlabel="schematisch Ost →  [km, nicht maßstäblich]", ylabel="schematisch Nord →  [km]", title="Sektionsmodell v0.1 (Vorlauf/Rücklauf je Kante); gestrichelt: Verbundkanten (Normalstatus offen); ★ MAN/Speicher")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([],[],marker="o",ls="",color=c,label=l,markeredgecolor="k") for l,c in farbe.items()]+[Line2D([],[],color=COL["rl"],ls="--",label="Verbundkante")], loc="lower right")
ax.set_aspect("equal"); fig.tight_layout(); fig.savefig(FIG_DIR/"10_sektionsmodell.png"); plt.show()

### 10.2 Zuordnung der Messstellen zum Modell
Zuordnung der 34 ACRON-Stationen (Ordner) zu Modellknoten. **Sicherheit:** *hoch* = Anlage im Plan eindeutig identifiziert; *mittel* = Zuordnung über Straßenname/Sektion, plausibel; *niedrig* = Sektion nur vermutet. Unsichere Zuordnungen fließen als Sensitivität in Kapitel 13/21 ein.

In [ ]:
zuord = {
 "HKW":("HKW","hoch"),"AVA":("AVA","hoch"),"GT-Ost":("GT","hoch"),"BMHKW":("BM","hoch"),"MAN":("MAN","hoch"),
 "PSS":("PSS","hoch"),"PSW":("PSW","hoch"),"Heizwerk_Sued":("HWS","hoch"),"HWW_prim_Einsp":("HWW","hoch"),"HWW_Kessel":("HWW","mittel"),"HWW_sek_Aussp":("WEST","mittel"),
 "Hans-Boeckler-Str":("SL5M","hoch"),"Am_Mittleren_Moos":("SL5O","hoch"),"Kurt-Schumacher-Str":("SL5O","mittel"),
 "Lechhauser_Str":("SEC3","mittel"),"Don_Bosco":("SEC3","hoch"),"KUKA":("SL7G","mittel"),
 "Hoher_Weg":("SEC2","hoch"),"Hunoldsgraben":("SEC2","mittel"),"Kreissparkasse":("SEC2","mittel"),"Fuggerstr":("SEC2","mittel"),"Schlettererstr":("SL1G","niedrig"),
 "Theodor-Heuss-Platz":("SEC4","hoch"),"UNI":("SUED","hoch"),"SIGMA_Technopark":("SUED","hoch"),"Fraunhofer":("SUED","niedrig"),"Beethovenpark":("SUED","niedrig"),
 "Klinikum":("WEST","hoch"),"Josefinum":("WEST","mittel"),"Lise-Meitner-Str":("WEST","niedrig"),"August-Wessels-Str":("WEST","niedrig"),
 "Hooverstr":("WEST","mittel"),"Grasiger_Weg":("WEST","mittel"),"Siegfried_Aufhaeuser-Str":("WEST","mittel"),
}
stationen = sorted(set(meta.ordner))
fehl = [s for s in stationen if s not in zuord and s.replace("-","_") not in zuord and s.replace("_","-") not in zuord]
zuord["Siegfried-Aufhaeuser-Str"] = zuord["Siegfried_Aufhaeuser-Str"]
messz = []
for s in stationen:
    k, sich = zuord.get(s, (None,"offen"))
    cols = [c for c in acron.columns if meta.ordner[c]==s]
    messz.append(dict(station=s, modellknoten=k, sicherheit=sich, kanaele=", ".join(sorted({meta.kanal[c] for c in cols})),
                      n_signale=len(cols), abdeckung_median_pct=round(float(qual.loc[cols,"abdeckung_pct"].median()),1)))
messz = pd.DataFrame(messz)
messz.to_csv(RESULTS/"messstellenzuordnung.csv", index=False)
print("Nicht zugeordnete Stationen:", messz[messz.modellknoten.isna()].station.tolist() or "keine")
print("Sicherheit:", messz.sicherheit.value_counts().to_dict())
with pd.option_context("display.width",200): display(messz)

### 10.3 Kritische Plausibilitätsprüfung des Modells gegen die Messdaten
Ziel: prüfen, ob die aus den Plänen abgelesene Struktur mit dem gemessenen Betrieb vereinbar ist, **bevor** darauf gerechnet wird. Vier Tests: (a) Flussrichtungen an den HKW-Stammleitungen, (b) hydraulische Druckhöhen der Anlagen, (c) hydraulische Entkopplung des Westnetzes, (d) Druckverlust-Ansatz Δp = c₀ + k·Q|Q| je Stammleitung und Vergleich mit den Modell-Ersatzleitungen.

In [ ]:
# (a) Flussrichtungen (+ = aus dem HKW ins Netz)
fr = []
for k in ("SL1","SL2","SL3","SL4"):
    s_ = acron[f"HKW_Durchfluss_{k}"].dropna()
    fr.append((k, round(100*(s_<0).mean()), round(s_.median()), round(s_.min()), round(s_.max())))
fr = pd.DataFrame(fr, columns=["Stammleitung","Anteil_Rückfluss_ins_HKW_%","Median_m3/h","min","max"]); display(fr)
print("→ SL1 und SL3 führen überwiegend Wärme *in* die HKW-Sammelschiene (Ost-Einspeisung über SL5 bzw. SL6/SL7), SL2 und SL4 nur aus dem HKW heraus: passt zur abgelesenen Struktur (SL5→SL1, SL6/SL7→SL3).")

# (b) hydraulische Druckhöhen der Anlagen  h = z + p/(ρ g)
zA = {"HKW":473.77,"AVA":466.69,"GT-Ost":467.98,"BMHKW":467.08,"Heizwerk_Sued":490.51}
hh = pd.DataFrame([(st, z0, acron[f"{st}_Druck_VL"].median(), acron[f"{st}_Druck_RL"].median(),
                    z0+acron[f"{st}_Druck_VL"].median()*1e5/(960*9.81), z0+acron[f"{st}_Druck_RL"].median()*1e5/(960*9.81)) for st,z0 in zA.items()],
                  columns=["Anlage","z_Plan_mNN","p_VL_bar","p_RL_bar","Druckhöhe_VL_mNN","Druckhöhe_RL_mNN"]).round(1)
display(hh)
print("→ HKW-Rücklauf-Druckhöhe ≈ Ruhedruck-Druckhöhe (561,8 m): Höhenbezug und Überdruck-Annahme konsistent. AVA/BM-HKW liegen im Betrieb ~10–15 m über dem HKW (Fluss Ost→West). GT-Ost steht überwiegend still (Median = Netzdruck).")
print("→ Die geodätischen Unterschiede (bis ~24 m ≙ 2,4 bar) sind größer als die Zieldruckdifferenzen (0,7–4 bar): Höhen sind für jeden Knoten zu kalibrieren.")

# (c) Entkopplung Westnetz: Partialkorrelation der Rücklaufdrücke nach Herausrechnen von Außentemperatur und Last
Xc = pd.concat([Ta, acron["HKW_Durchfluss_gesamt"]], axis=1).dropna()
def resid(s_):
    d = pd.concat([s_, Xc], axis=1).dropna(); X = np.c_[np.ones(len(d)), d.iloc[:,1:].values]
    return d.iloc[:,0] - X @ np.linalg.lstsq(X, d.iloc[:,0], rcond=None)[0]
Rr = pd.DataFrame({k: resid(acron[v]) for k,v in {"HKW_RL":"HKW_Druck_RL","HWW_prim_RL":"HWW_Druck_RL_Primeinsp","HWW_sek_RL":"HWW_Druck_RL","Klinikum_RL":"Klinikum_Druck_RL","Grasiger_RL":"Grasiger_Weg_Druck_RL","Sued_RL":"Heizwerk_Sued_Druck_RL"}.items()})
print("Streuung (σ) des Rücklaufdrucks [bar]: HWW-sekundär", round(acron.HWW_Druck_RL.std(),2), "| HKW", round(acron.HKW_Druck_RL.std(),2), "| HWW-primär", round(acron.HWW_Druck_RL_Primeinsp.std(),2))
display(Rr.corr().round(2).loc[["Klinikum_RL","Grasiger_RL","HWW_sek_RL"],["HKW_RL","HWW_prim_RL","Sued_RL","HWW_sek_RL","Klinikum_RL","Grasiger_RL"]])
print("→ Der Westnetz-Rücklauf ist praktisch konstant (σ≈0,05 bar, eigene Druckhaltung ~6,6 bar) und nach Herausrechnen der gemeinsamen Last unkorreliert mit dem Verbund-Rücklauf (|r|≤0,06). Die West-Stationen korrelieren untereinander (0,5). Das stützt eine hydraulische Entkopplung (Wärmeübertrager).")

In [ ]:
# (d) Druckverlust-Ansatz je Stammleitung: Δp = c0 + k·Q|Q|   (c0 ≙ geodätischer Höhen-/Sensor-Offset, Q in m3/h)
def fit_dp(dp, Q):
    d = pd.concat([dp, Q], axis=1, keys=["dp","Q"]).dropna(); X = np.c_[np.ones(len(d)), d.Q*d.Q.abs()/1e4]
    b = np.linalg.lstsq(X, d.dp, rcond=None)[0]; r2 = 1-((d.dp-X@b)**2).sum()/((d.dp-d.dp.mean())**2).sum(); return b[0], b[1], r2, len(d)
def coef_bar_per_m(dn_mm, lam=0.02, rho=960.0):
    D = dn_mm/1000; A = np.pi*D**2/4; return lam/D*rho/2/(3600*A)**2/1e5      # bar je m Rohr je (m3/h)^2 (einfache Leitung)
tests = [("SL1 → MAN (VL)", acron.MAN_Druck_VL_Station-acron.HKW_Druck_VL, -acron.HKW_Durchfluss_SL1, 400, 854+394),
         ("SL2 → Hoher Weg", acron.Hoher_Weg_Druck_VL-acron.HKW_Druck_VL, -acron.HKW_Durchfluss_SL2, 350, 600),
         ("SL3 → Lechhauser Str.", acron.Lechhauser_Str_Druck_VL-acron.HKW_Druck_VL, -acron.HKW_Durchfluss_SL3, 350, 800),
         ("SL4 → Theodor-Heuss-Pl.", acron["Theodor-Heuss-Platz_Druck_VL"]-acron.HKW_Druck_VL, -acron.HKW_Durchfluss_SL4, 400, 1200)]
rows = []
for nm, dp, Q, dn, Lmod in tests:
    c0, k, r2, n = fit_dp(dp, Q); Leq = (k*1e-4)/coef_bar_per_m(dn)
    rows.append((nm, n, round(c0,2), round(k,3), round(r2,2), dn, round(Leq), Lmod))
fitdf = pd.DataFrame(rows, columns=["Pfad","n","c0_bar","k_bar/(1e4 m3/h)²","R²","DN_Stammleitung","L_äquivalent_m","L_Modell_v0.1_m"]); display(fitdf)
fitdf.to_csv(RESULTS/"pruefung_druckverlust_fit.csv", index=False)
print("→ Die aus den Messdaten abgeleiteten äquivalenten Längen (≈1,0–2,6 km) liegen um den Faktor 1,2–3 über den geschätzten Modell-Längen v0.1 (0,6–1,25 km): die Ersatzleitungen sind zu kurz und werden kalibriert. SL2 und SL4 werden gut erklärt (R² 0,5–0,6), SL3 schwach (0,23), SL1→MAN kaum (0,13): der SL1-Fluss enthält Abzweige (SL1-West, PSW), und die Druckdifferenz MAN–HKW liegt nahe der Sensorauflösung (0,1 bar). Die Hydraulik um MAN ist damit aus den Messdaten nur schwach bestimmt.")
print("→ c0 von −0,8 … −1,6 bar ≙ 8–16 m Höhen-/Sensoroffset gegenüber dem HKW: bestätigt die Notwendigkeit der Höhenkalibrierung.")

In [ ]:
# MAN-Last und Erzeugersignale
man = acron[["MAN_Waermeleistung_Station","MAN_Durchfluss_Station"]].dropna()
print(f"MAN gemessen: Q_max {man.MAN_Waermeleistung_Station.max():.1f} MW, 99 %-Quantil Winter {acron.MAN_Waermeleistung_Station[acron.index.month.isin([12,1,2])].quantile(.99):.1f} MW, Q_max/Anschlussleistung = {man.MAN_Waermeleistung_Station.max()/32.0:.0%}; V̇_max {man.MAN_Durchfluss_Station.max():.0f} m3/h")
v342 = man.MAN_Durchfluss_Station.max()/3600/(np.pi*0.3**2/4)
print(f"→ In der DN300-Armatur 342.2 (Abzweig zu MAN) entspricht V̇_max nur {v342:.2f} m/s: der MAN-Kundenanschluss ist hydraulisch schwach belastet; die 32 MW sind Anschlussleistung, nicht Gleichzeitigkeitslast.")
hw = acron.filter(regex="^HWW_(Waermeleistung|Durchfluss)").describe().loc[["count","50%","max"]].round(1).T; display(hw)
print("→ HWW_Waermeleistung_Primeinsp erreicht 95,5 MW, der Plan nennt für die Prim-Einspeisung max. 64 MW (95 ≈ 64 + 31 MW Prim-Ausspeisung): Bedeutung/Vorzeichen des Signals ungeklärt. Für AVA existiert kein Durchflusssignal, für GT-Ost nur 35 % Abdeckung.")

**Ergebnis der Plausibilitätsprüfung**
* *Bestätigt:* Rückflussrichtungen SL1/SL3, Druckhöhen der Ost-Erzeuger, Ruhedruck-Bezug, Entkopplung des Westnetzes, Größenordnung der Rohrwiderstände (Modell-Längen jedoch zu kurz).
* *Zweifelhaft / offen:* Armaturen 141.1 und 187.1 (im Plan „ZU“ und zugleich in der Separierungsliste), MAN als T-Abzweig DN300 statt Streckenknoten, Höhen/Sensorhöhen, Bedeutung einzelner Erzeugersignale, fehlende Schaltzustands-Historie, schwache Identifizierbarkeit der Strecke HKW–MAN.

### 10.4 Verbleibende Aggregationen und Einschränkungen
| Aggregation | hydraulische Aussage, die dadurch eingeschränkt ist |
|---|---|
| Sektionen (SL2, SL3, SL4, SÜD, West) als Einzelknoten mit Ersatzleitung | Druckverteilung *innerhalb* der Sektion, Verhalten einzelner Straßen/Stationen; Kundendifferenzdruck nur aus Messwerten, nicht aus Rohrnetz |
| Trassenlängen SL5–SL7 aus YAML-Summen bzw. geschätzt | absolute Druckverluste in den Ost-Trassen (Kalibrierung in Kap. 13 erforderlich) |
| Höhen nur an Erzeugern/Pumpstationen | geodätischer Anteil der Druckdifferenzen zwischen Knoten; wird als Offset kalibriert |
| Westnetz als ein Lastknoten hinter WÜ | Druckverhältnisse im Westnetz; SEK-Separierung (WV660) in dieser Etappe nicht abgebildet |
| Verbundkanten je Sektionspaar gebündelt | Einzelne Armaturenschaltungen (z. B. 080.2 vs. 080.3) werden gemeinsam geschaltet |

## 10.5 Datengestützte Ableitung fehlender Größen und Modell v0.2
Da Betreiberauskünfte nicht verfügbar sind, werden die offenen Punkte aus Kap. 10.3 **aus den Messdaten abgeleitet**. Jede Ableitung wird mit einem unabhängigen Test abgesichert; wo das nicht möglich ist, wird die Größe als Kalibrierparameter oder Sensitivität geführt.

| Offener Punkt | Ableitung | Absicherung |
|---|---|---|
| Bedeutung der Erzeugersignale | Energiebilanzen V̇·ΔT, Korrelation, Regression | Bestimmtheitsmaß, Vergleich mit Plan |
| Höhen der Knoten | Druckdifferenz zum HKW bei Schwachlast (Reibung ≈ 0) | Vorlauf- und Rücklauf-Offset müssen übereinstimmen; Vergleich mit Plan-Höhen der Erzeuger |
| Rohrlängen/Ersatzleitungen | Anpassung Δp = c₀ + k·Q\|Q\| | Größenordnung gegen Geometrie |
| Maßstab der Pläne | Blattgröße und Rasterauflösung | Vergleich mit YAML-Trassenlänge |
| Armaturen 141.1/187.1 | Wirkung auf Konnektivität | jede Verbundkante hat parallele Armaturen |
| Kritische Ereignisse/Lastniveau | Gesamtlast aus Erzeugung, Lastgänge 2020–2022 | Regression gegen Außentemperatur |

In [ ]:
# ---------- 10.5.1 Bedeutung der Erzeugersignale (Evidenz)
K_MW = 960*4.2/3.6e6                       # MW je (m3/h · K)
sl_heat = sum(acron[f"HKW_Durchfluss_SL{i}"] for i in (1,2,3,4))*K_MW*(acron.HKW_Temp_VL-acron.HKW_Temp_RL)
sl_pos  = sum(acron[f"HKW_Durchfluss_SL{i}"].clip(lower=0) for i in (1,2,3,4))
ev = []
ev.append(("HKW_Waermeleistung", "Netto-Wärme der HKW-Sammelschiene ins Netz = Σ vorzeichenbeh. SL1–SL4·ΔT", f"r = {acron.HKW_Waermeleistung.corr(sl_heat):.2f} mit Σ(SL·ΔT)"))
ev.append(("HKW_Durchfluss_gesamt", "Summe der positiven SL-Flüsse (Abgabe aus dem HKW)", f"r = {acron.HKW_Durchfluss_gesamt.corr(sl_pos):.2f} mit Σ max(SL,0)"))
d_ = pd.concat([acron.HWW_Waermeleistung_Sek_Aussp, acron.HWW_Waermeleistung_Primeinsp, acron.HWW_Waermeleistung_Kessel], axis=1, keys=["sek","prim","kessel"]).dropna()
Xr = np.c_[np.ones(len(d_)), d_.prim, d_.kessel]; br = np.linalg.lstsq(Xr, d_.sek, rcond=None)[0]; r2 = 1-((d_.sek-Xr@br)**2).sum()/((d_.sek-d_.sek.mean())**2).sum()
ev.append(("HWW_Waermeleistung_Primeinsp", "Wärme, die das Westnetz über die Wärmeübertrager aus dem Primärnetz bezieht (Vorzeichen +)", f"Q_Sek = {br[1]:.2f}·Q_Prim + {br[2]:.2f}·Q_Kessel, R² = {r2:.2f}"))
prim_spike = acron.HWW_Waermeleistung_Primeinsp > 40
ev.append(("… Ausreißer", f"{int(prim_spike.sum())} Stunden > 40 MW (15.07.2025 15–16 Uhr) sind Messfehler/Sondersituation; Winter-Maximum {acron.HWW_Waermeleistung_Primeinsp[acron.index.month.isin([12,1,2])].max():.0f} MW", "maskiert (NaN)"))
ev.append(("HWW_Waermeleistung_Sek_Aussp", "Gesamtwärme an das Westnetz (Wärmeübertrager + Kessel)", f"Sek-Fluss·ΔT / Messung = {((acron.HWW_Durchfluss_Sek1+acron.HWW_Durchfluss_Sek2+acron.HWW_Durchfluss_Sek3)*K_MW*(acron.HWW_Temp_VL-acron.HWW_Temp_RL)/acron.HWW_Waermeleistung_Sek_Aussp).median():.2f}"))
for st_ in ("AVA","BMHKW","GT-Ost"):
    pass
ev = pd.DataFrame(ev, columns=["Signal","abgeleitete Bedeutung","Evidenz"]); 
with pd.option_context("display.max_colwidth",95,"display.width",250): display(ev)

# abgeleitete Zeitreihen (Rohdaten bleiben unverändert)
abg = pd.DataFrame(index=acron.index)
abg["HKW_netto_MW"]   = acron.HKW_Waermeleistung
abg["HWW_prim_MW"]    = acron.HWW_Waermeleistung_Primeinsp.mask(prim_spike)
abg["HWW_sek_MW"]     = acron.HWW_Waermeleistung_Sek_Aussp
abg["HWW_kessel_MW"]  = (abg.HWW_sek_MW - abg.HWW_prim_MW)             # Identität Q_Sek = Q_Prim + Q_Kessel
abg["Ost_MW"]         = acron.AVA_Waermeleistung + acron.BMHKW_Waermeleistung + acron["GT-Ost_Waermeleistung"]
abg["HWS_MW"]         = acron.Heizwerk_Sued_Durchfluss*K_MW*(acron.Heizwerk_Sued_Temp_VL-acron.Heizwerk_Sued_Temp_RL)
abg["AVA_V_m3h"]      = acron.AVA_Waermeleistung/(K_MW*(acron.AVA_Temp_VL-acron.AVA_Temp_RL).where(lambda x: x>5))
# Methodentest V̇ = Q/(K·ΔT) an BM-HKW (dort gemessen)
bm = (acron.BMHKW_Waermeleistung/(K_MW*(acron.BMHKW_Temp_VL-acron.BMHKW_Temp_RL))/acron.BMHKW_Durchfluss).replace([np.inf,-np.inf],np.nan)
print(f"Methodentest AVA-Durchfluss aus Q/(K·ΔT): am BM-HKW (gemessener Durchfluss vorhanden) Verhältnis Median = {bm[(acron.BMHKW_Durchfluss>20)].median():.2f} → Verfahren zulässig")
# Gesamterzeugung Verbund + West
abg["Erzeugung_ges_MW"] = abg.HKW_netto_MW + abg.Ost_MW + abg.HWS_MW.fillna(0) + abg.HWW_kessel_MW
abg.to_csv(RESULTS/"abgeleitete_zeitreihen.csv")
print(f"Gesamterzeugung Verbund + West 2025: Median {abg.Erzeugung_ges_MW.median():.0f} MW, Winter-P95 {abg.Erzeugung_ges_MW[abg.index.month.isin([12,1,2])].quantile(.95):.0f} MW, Maximum {abg.Erzeugung_ges_MW.max():.0f} MW")

In [ ]:
# ---------- 10.5.2 Lastniveau, Lastgänge 2020–2022 und Auslegungslast
Gs = abg.Erzeugung_ges_MW
cmask = (Ta < 15) & Gs.notna()
bfit = np.polyfit(Ta[cmask], Gs[cmask], 1)
lg = pd.read_excel(DATA_DIR/"A_Daten"/"250519_tex_Lastgänge 2020_2022.xlsx", header=None, skiprows=4).iloc[:, :5]
lg.columns = ["Datum","Zeit","y2022","y2021","y2020"]
lg = lg[pd.to_numeric(lg.y2022, errors="coerce").notna()]
lg_max = {y: float(pd.to_numeric(lg[c], errors="coerce").where(lambda s: s<400).max()) for y,c in (("2020","y2020"),("2021","y2021"),("2022","y2022"))}
res_ = (Gs - (bfit[1]+bfit[0]*Ta))[cmask]
q99, qmax = float(res_.quantile(.99)), float(res_.max())
P14 = bfit[1]+bfit[0]*-14
last = pd.DataFrame({"Größe":["Mittelwert-Regression 2025 bei −10 °C","Mittelwert-Regression 2025 bei −14 °C","… + 99 %-Residuum (Stundenspitzen)","… + maximales Residuum 2025","Messmaximum 2025 (stündlich, bei mildem Wetter)","Lastgang-Maximum 2020 / 2021 / 2022 (ältere Erzeugungsdaten)"],
                     "MW":[bfit[1]+bfit[0]*-10, P14, P14+q99, P14+qmax, Gs.max(), f"{lg_max['2020']:.0f} / {lg_max['2021']:.0f} / {lg_max['2022']:.0f}"]})
last["MW"] = last.MW.map(lambda v: f"{v:.0f}" if not isinstance(v,str) else v); display(last)
print(f"Steigung {bfit[0]:.2f} MW/K, Achsenabschnitt {bfit[1]:.0f} MW; Korrelation Erzeugung–Ta = {Gs.corr(Ta):.2f}; Residuum P99 = {q99:.0f} MW, max = {qmax:.0f} MW")
daily = pd.DataFrame({"Ta":Ta,"P":Gs}).resample("D").mean().dropna()
print("Höchster Tagesbedarf 2025:", daily.P.idxmax().date(), f"{daily.P.max():.0f} MW (Tagesmittel) bei Ta = {daily.Ta[daily.P.idxmax()]:.1f} °C; kältester Tag: {daily.Ta.idxmin().date()} mit {daily.Ta.min():.1f} °C")
print(f"→ Auslegungsfall −14 °C: Mittelwert ≈ {P14:.0f} MW, Stundenspitze ≈ {P14+q99:.0f} MW (Bandbreite {P14:.0f}–{P14+qmax:.0f} MW). Die Lastgänge 2020–2022 (Max. 214 MW, 2021) liegen im unteren Teil dieser Bandbreite – die Erzeugung 2025 enthält Netzerweiterungen und Messunsicherheit (HWS/Kessel abgeleitet). Zentralwert für S1: Mittelwert {P14:.0f} MW (≈ Lastgang 2021, 214 MW); Sensitivität 210…{P14+q99:.0f} MW (Stundenspitzen-Residuum).")
LAST_MODELL = dict(a=float(bfit[1]), b=float(bfit[0]), P_mittel_14C=float(P14), P_spitze_14C=float(P14+q99), P_min=210.0, P_max=float(P14+q99))

In [ ]:
# ---------- 10.5.3 Höhen aus Schwachlast: Δp(Station − HKW) ≈ −ρ g Δz (Reibung ≈ 0)
import re as _re
def _norm(x): return _re.sub(r"[^a-z]", "", x.lower())
zn = {_norm(k):v for k,v in zuord.items()}; zn.update({"lisemeitnerstr":zuord["Lise-Meitner-Str"], "listemeitnerstr":zuord["Lise-Meitner-Str"], "amittlerenmoos":zuord["Am_Mittleren_Moos"]})
def stations_of(kind):
    out = {}
    for c in acron.columns:
        m = _re.match(rf"^(.+?)_Druck_{kind}(_Station)?$", c)
        if m and not any(t in c for t in ("Pumpe","Primeinsp")) and not c.startswith(("PPS","PSW","HWW")): out[m.group(1)] = c
    return out
vl_c, rl_c = stations_of("VL"), stations_of("RL")
low = (acron.HKW_Durchfluss_gesamt < acron.HKW_Durchfluss_gesamt.quantile(.10)) & acron.index.month.isin([6,7,8])
rows = []
for stn in sorted(set(vl_c)|set(rl_c)):
    dv = (acron[vl_c[stn]]-acron.HKW_Druck_VL)[low].median() if stn in vl_c else np.nan
    dr = (acron[rl_c[stn]]-acron.HKW_Druck_RL)[low].median() if stn in rl_c else np.nan
    knoten_id = zn.get(_norm(stn), (None,None))[0] if stn!="HKW" else "HKW"
    rows.append((stn, knoten_id, dv, dr))
g = pd.DataFrame(rows, columns=["station","knoten","dp_VL_bar","dp_RL_bar"])
g["netz"] = g.knoten.map(knoten.set_index("node_id").netz)
g["dz_m"] = -g[["dp_VL_bar","dp_RL_bar"]].mean(axis=1)*1e5/(971*9.81)             # Δz relativ zum HKW
g["dz_unsicherheit_m"] = np.where(g.dp_VL_bar.notna() & g.dp_RL_bar.notna(), 3.0, 5.0)
g["ost_stroemungsbeeinflusst"] = g.netz.eq("Ost")
both_ = g.dropna(subset=["dp_VL_bar","dp_RL_bar"])
both_v = both_[~both_.netz.eq("West") & ~both_.netz.eq("Ost")]
print(f"Test 1 – Vorlauf- und Rücklauf-Offset gleich (geodätisch)? Alle {len(both_)} Stationen: r = {both_.dp_VL_bar.corr(both_.dp_RL_bar):.2f}; Verbund Mitte/Süd ({len(both_v)}): Median |VL−RL| = {(both_v.dp_VL_bar-both_v.dp_RL_bar).abs().median():.2f} bar (≙ {((both_v.dp_VL_bar-both_v.dp_RL_bar).abs().median())*1e5/(971*9.81):.1f} m)")
# Test 2: Plan-Höhen der Erzeuger (Δz relativ zum HKW)
plan_dz = {"AVA":466.69-473.77,"BMHKW":467.08-473.77,"GT-Ost":467.98-473.77,"Heizwerk_Sued":490.51-473.77}
t2 = pd.DataFrame([(k, v, g.set_index("station").dz_m.get(k, np.nan)) for k,v in plan_dz.items()], columns=["Anlage","dz_Plan_m","dz_abgeleitet_m"]); t2["Abweichung_m"] = t2.dz_abgeleitet_m - t2.dz_Plan_m
display(t2.round(1))
print("→ Abweichungen zu den Plan-Höhen: AVA −0,8 m, BM-HKW +2,3 m, Heizwerk Süd −3,5 m (alle innerhalb der geschätzten Unsicherheit von ±3–5 m); GT-Ost (+16 m) steht überwiegend still bzw. hat eine RL-Pumpe und ist unzuverlässig (wird nicht verwendet). Für West-Stationen ist der Offset kein geodätischer Wert, sondern die Differenz zur eigenen Druckhaltung (Kap. 10.3).")
g.round(2).to_csv(RESULTS/"abgeleitete_hoehen.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12,4.6))
ax[0].scatter(both_.dp_VL_bar, both_.dp_RL_bar, c=np.where(both_.netz=="West",COL["warn"],np.where(both_.netz=="Ost",COL["rl"],COL["ok"])), s=40, edgecolor="k")
lim = [both_[["dp_VL_bar","dp_RL_bar"]].min().min()-.2, both_[["dp_VL_bar","dp_RL_bar"]].max().max()+.2]; ax[0].plot(lim, lim, "k--", lw=1)
ax[0].set(xlabel="Δp Vorlauf zum HKW [bar]", ylabel="Δp Rücklauf zum HKW [bar]", title="Schwachlast: Vorlauf- und Rücklauf-Offset (grün Verbund, blau Ost, orange West)")
gv = g[g.netz.isin(["Mitte","Süd","Ost"])].dropna(subset=["dz_m"]).sort_values("dz_m")
ax[1].barh(gv.station, gv.dz_m, color=[COL["rl"] if o else COL["ok"] for o in gv.ost_stroemungsbeeinflusst]); ax[1].set(xlabel="abgeleitete Höhe relativ zum HKW [m]", title="Abgeleitete hydraulische Höhen (Verbund)")
fig.tight_layout(); fig.savefig(FIG_DIR/"10_hoehen_abgeleitet.png"); plt.show()

In [ ]:
# ---------- 10.5.4 Maßstab der Pläne und Rohrlängen
pdf_mm = (900.2, 1500.1); px = (3544, 5906)
mm_px = pdf_mm[0]/px[0]; m_px = mm_px*10000/1000                               # m je Pixel bei 1:10 000
sl5_px = 2180.0                                       # abgegriffener Polygonzug HKW→001.2→MAN→833→832→809 (Rasterpixel)
sl5_len = sl5_px*m_px
print(f"Blattgröße 900×1500 mm bei {px[0]}×{px[1]} px → {mm_px:.3f} mm/px; bei Maßstab 1:10 000 → {m_px:.2f} m/px")
print(f"SL5-Polygonzug HKW→AVA-Abzweig: {sl5_px:.0f} px ≙ {sl5_len/1000:.2f} km  (YAML: 5,93 km; Abweichung {100*(sl5_len/5930-1):+.0f} %)")
print("→ Maßstab 1:10 000 ist Standard für solche Arbeitsblätter und wird durch die unabhängige YAML-Trassenlänge auf ≈7 % bestätigt. Die Rasterzelle ist damit ≈0,7 km (nicht 1 km).")
# Rohrlängen v0.2: Stammleitungen SL2/SL3/SL4 aus der Druckverlust-Anpassung (äquivalente Länge), SL5 aus Geometrie
leq = fitdf.set_index("Pfad")["L_äquivalent_m"]
print("Äquivalente Längen aus Messung [m]:", leq.to_dict())

In [ ]:
# ---------- 10.5.5 Konnektivität der Verbundkanten (Wirkung von 141.1/187.1)
bund = {"M_SL1_SL2":["080.2","080.3","141.1","300.1","112.1"], "M_SL2_SL3":["203.1","214.1","050.1","050.2"], "M_SL2_SL4":["735.1","785.1","325.1","253.1","187.1"], "M_SL3_SL4":["166.1"]}
am = armaturen.set_index("armatur_nr")
rows = []
for kid, lst in bund.items():
    st_ = [am.normalstatus.get(x, "?") for x in lst]
    rows.append((kid, ", ".join(f"{x} ({s})" for x,s in zip(lst, st_)), sum(s=="offen" for s in st_), len(lst), sum(am.dn_mm.get(x,0)**2 for x,s in zip(lst,st_) if s=="offen")**.5))
kv = pd.DataFrame(rows, columns=["Verbundkante","Armaturen (Normalstatus)","offen","gesamt","äquiv. DN offen [mm]"]); display(kv.round(0))
print("→ Jede Verbundkante besitzt mindestens eine im Normalbetrieb offene Armatur: die Konnektivität ist unabhängig davon, ob 141.1/187.1 im Normalbetrieb offen oder zu sind. Die Leitfähigkeit der Kanten wird kalibriert (Sensitivität: 141.1/187.1 offen vs. zu).")

In [ ]:
# ---------- 10.5.6 Modell v0.2: Knoten und Kanten
kn2 = knoten.copy(); ka2 = kanten.copy()
# (1) MAN als T-Abzweig: Durchgangsknoten JMAN auf SL5, Abzweig DN300 (Armatur 342.2) zum Kunden MAN
kn2 = pd.concat([kn2, pd.DataFrame([dict(node_id="JMAN", name="T-Abzweig SL5 (Arm. 342.1/342.3, DN400)", typ="koppelstelle", sektion="SL1", netz="Mitte", PN=25, z_NN_m=np.nan, p_anschluss_MW=0.0, x_km=-0.3, y_km=1.35, quelle="WV650 Armaturen 342.x", sicherheit="mittel")])], ignore_index=True)
ka2 = ka2[~ka2.edge_id.isin(["SL5a","SL5b"])]
neu = pd.DataFrame([
 dict(edge_id="SL5a",von="K001",nach="JMAN",art="Trasse SL5",dn_mm=400,laenge_m=394,laenge_quelle="P",sicherheit="niedrig",normalstatus="offen",armaturen="342.1",max_t_h_separiert=np.nan,bidirektional=True),
 dict(edge_id="SL5b",von="JMAN",nach="SL5M",art="Trasse SL5",dn_mm=400,laenge_m=2285,laenge_quelle="P",sicherheit="niedrig",normalstatus="offen",armaturen="342.3, 833.x",max_t_h_separiert=750,bidirektional=True),
 dict(edge_id="MAN_A",von="JMAN",nach="MAN",art="Kundenanschluss MAN",dn_mm=300,laenge_m=150,laenge_quelle="S",sicherheit="niedrig",normalstatus="offen",armaturen="342.2",max_t_h_separiert=np.nan,bidirektional=True)])
ka2 = pd.concat([ka2, neu], ignore_index=True)
# (2) Ersatzlängen aus Messung (Kalibrier-Startwerte)
def setL(eid, L, q): ka2.loc[ka2.edge_id==eid, ["laenge_m","laenge_quelle"]] = [float(L), q]
setL("SL2", leq["SL2 → Hoher Weg"], "K"); setL("SL3", leq["SL3 → Lechhauser Str."], "K")
setL("SL4a", leq["SL4 → Theodor-Heuss-Pl."], "K"); setL("SL4b", 4470-leq["SL4 → Theodor-Heuss-Pl."], "P")
# (3) Unsicherheitsbereiche der Länge (Sensitivität)
band = {"Y":.15,"P":.25,"K":.30,"S":.50}
ka2["laenge_min_m"] = ka2.laenge_m*(1-ka2.laenge_quelle.map(band)); ka2["laenge_max_m"] = ka2.laenge_m*(1+ka2.laenge_quelle.map(band))
# (4) abgeleitete Höhen an die Knoten
hz = g.dropna(subset=["knoten","dz_m"]).groupby("knoten").dz_m.mean()
kn2["z_abgeleitet_mNN"] = kn2.node_id.map(lambda n: 473.77 + hz.get(n, np.nan))
kn2["z_Quelle"] = np.where(kn2.z_NN_m.notna(), "Plan (WV640)", np.where(kn2.z_abgeleitet_mNN.notna(), "abgeleitet (Schwachlast)", "unbekannt → Kalibrierparameter"))
kn2.loc[kn2.netz=="West","z_abgeleitet_mNN"] = np.nan
kn2["z_modell_mNN"] = kn2.z_NN_m.fillna(kn2.z_abgeleitet_mNN)
kn2.to_csv(RESULTS/"modell_v02_knoten.csv", index=False); ka2.to_csv(RESULTS/"modell_v02_kanten.csv", index=False)
idx2 = {n:i for i,n in enumerate(kn2.node_id)}
A2 = coo_matrix((np.ones(len(ka2)), ([idx2[a] for a in ka2.von],[idx2[b] for b in ka2.nach])), shape=(len(idx2),)*2)
nc2, lab2 = connected_components(A2, directed=False)
print(f"Modell v0.2: {len(kn2)} Knoten, {len(ka2)} Kanten, Komponenten {nc2} (WEST separat), unabhängige Schleifen {len(ka2)-len(kn2)+nc2}")
print("Höhenquelle der Knoten:", kn2.z_Quelle.value_counts().to_dict())
print("Längenquelle:", ka2.laenge_quelle.value_counts().to_dict(), "(Y YAML, P Plan-Anteil, K aus Messung kalibriert, S Schätzung)")
assert nc2 == 2 and (len(ka2)-len(kn2)+nc2) == 7

### 10.5.7 Randbedingungen aus Messung und Lösbarkeit
Das Netzmodell wird als **Zustandsschätzproblem** gelöst: Für jede Stunde sind Erzeugereinspeisungen und einige Kundenlasten gemessen, die übrigen Sektionslasten, Verbundflüsse und die Druckverteilung werden so bestimmt, dass Massenbilanz, Druckverlustgesetz und Messwerte bestmöglich erfüllt werden.

In [ ]:
rb = pd.DataFrame([
 ("HKW","Einspeisung ins Netz: SL1–SL4 (Fluss, vorzeichenbehaftet), T_VL/T_RL, p_VL/p_RL","gemessen","hoch"),
 ("AVA","Q gemessen; V̇ aus Q/(K·ΔT) (Verfahren am BM-HKW getestet)","gemessen/abgeleitet","hoch"),
 ("GT","Q gemessen (100 %), V̇ nur 35 % → aus Q/ΔT","gemessen/abgeleitet","mittel"),
 ("BM","Q und V̇ gemessen","gemessen","hoch"),
 ("HWS","V̇, T → Q; p_VL/p_RL","gemessen","hoch"),
 ("HWW/WEST","Q_Prim (Spike maskiert), Q_Sek, Sek1–3-Flüsse, p","gemessen","hoch"),
 ("MAN","Q, V̇ (69 % Abdeckung), p_VL, Δp, T","gemessen","hoch"),
 ("Sektionen SL1G, SEC2, SEC3, SEC4, SUED, SL5M, SL5O, SL6S, SL7G","Last unbekannt → Schätzgröße (Prior: Anteil an Anschlussleistung; Summe = Gesamterzeugung − Verluste)","geschätzt","mittel"),
 ("Verbundkanten (4 Kanten + SL5↔SL6)","Fluss unbekannt → Leitfähigkeit kalibriert","kalibriert","niedrig"),
 ("Höhen","Plan (Erzeuger), abgeleitet (Stationen), sonst Kalibrierparameter","abgeleitet","mittel"),
], columns=["Knoten/Größe","Datenbasis","Klasse","Verlässlichkeit"])
with pd.option_context("display.max_colwidth",100,"display.width",250): display(rb)
n_p = sum(1 for c in acron.columns if meta.kanal[c] in ("p_VL","p_RL","dp") and qual.loc[c,"validierungstauglich"])
n_q = sum(1 for c in acron.columns if meta.kanal[c] in ("m_dot","Q") and qual.loc[c,"validierungstauglich"])
n_unb_stat = len(ka2) + int(kn2.z_modell_mNN.isna().sum())
print(f"Je Stunde: {n_p} validierungstaugliche Druck- und {n_q} Fluss-/Leistungssignale; unbekannt: 9 Sektionslasten + 5 Verbundflüsse je Stunde, statisch {n_unb_stat} Parameter (Rohr-k, Höhen).")
print("→ Die Aufgabe ist über ~8 700 Stunden stark überbestimmt (statische Parameter werden über die gesamte Zeitreihe bestimmt); je Stunde ist sie nur mit Regularisierung (Prior der Lasten) lösbar. Schwach bestimmt bleiben: Hydraulik HKW–MAN (R²=0,13), SL6/SL7-Längen, Verbundkantenleitfähigkeiten.")

### 10.5.8 Kalibrier-/Validierungsaufteilung und kritische Ereignisse
* **Kalibrierung:** 01.01.–30.09.2025 (enthält die Spitzenlast Februar 2025 mit ≈ 183 MW Tagesmittel). **Validierung:** 01.10.–31.12.2025 (enthält die kältesten Tage 22.11. und 31.12. mit ≈ 174 MW). Damit sind Kalibrier- und Validierungszeitpunkte zeitlich getrennt, beide enthalten Hochlastphasen.
* **Kritische Ereignisse:** In den Messdaten 2025 werden weder Kundendifferenzdrücke unter 0,7 bar noch Erzeuger-Abschaltdrücke erreicht (kleinste Reserve: HWS-Rücklauf, +0,8 bar). Die im Auftrag beschriebenen kritischen Lagen bei sehr niedrigen Außentemperaturen (≈ −14 °C, Erzeugung ≈ 215–270 MW) liegen **außerhalb** des Messbereichs 2025 (kältester Tag −7,3 °C) und werden mit dem kalibrierten Modell durch Lastextrapolation simuliert (Szenario S1, Lastbandbreite siehe 10.5.2). Historische Ereignisse aus früheren Jahren (Lastspitze 2021: 213,6 MW) sind ohne Druck- und Temperaturmessungen nicht nachrechenbar.

## 10.6 Geometrie aus den Plänen (Bildanalyse) und Modell v0.3
Die Trassenlängen aus Kap. 10.5 waren teils geschätzt. Die Pläne sind Rasterbilder im Maßstab 1:10 000 (2,54 m je Pixel, Kap. 10.5.4); die Trassen sind farbig gezeichnet (SL1 türkis, SL3 blau, SL4 lachsrot, SL5 magenta, SL6 cyan, SL7 grün). Vorgehen:
1. **Ankerpunkte:** Positionen der Armaturenbeschriftungen und Anlagen wurden per Texterkennung (RapidOCR) auf der WV650-Karte bestimmt und sind unten als Tabelle hinterlegt (Genauigkeit ≈ ±50 m, da die Beschriftung neben dem Symbol steht).
2. **Farbmasken** je Trasse und **kürzester Weg** entlang der gezeichneten Linie zwischen zwei Ankern (Wegsuche auf dem Pixelgitter). Die Länge ist die Wegstrecke × Maßstab.
3. **Kontrolle:** Anteil des Weges, der tatsächlich auf der Linie liegt (soll > 90 % sein), Luftlinie < Weglänge, Sichtprüfung im Overlay.

In [ ]:
import pymupdf, cv2
from skimage.graph import route_through_array
# --- Ankerpunkte im WV650-Raster [Pixel x, y] (Texterkennung, geprüft)
ANKER = {"HKW":(1550,2810), "K001":(1305,2528), "MAN_342_1":(1501,2474), "MAN_342_3":(1525,2441), "G833_3":(1892,1800), "G832_3":(1980,1837),
         "G809_2":(2591,1286), "AVA_810_1":(2596,1204), "GT":(2814,1274), "BM":(2878,1092), "G338_2":(1950,2854), "G750_1":(2823,1326),
         "G773":(2654,1556), "G818_2":(2534,3282), "G888_1":(2877,2404), "PSW_346_1":(652,3170), "PSS_297_1":(1603,3930)}
FARBEN = {"SL1":(8,104,96), "SL3":(0,48,235), "SL4":(240,128,128), "SL5":(200,10,96), "SL6":(0,176,240), "SL7":(120,176,92)}
pdf650 = next((DATA_DIR/"B_Plaene").glob("WV650*.pdf"))
doc_ = pymupdf.open(pdf650); pix_ = pymupdf.Pixmap(doc_, doc_[0].get_images()[0][0])
if pix_.n > 3: pix_ = pymupdf.Pixmap(pymupdf.csRGB, pix_)
IMG = np.frombuffer(pix_.samples, np.uint8).reshape(pix_.h, pix_.w, pix_.n)[..., :3]
print("Plan WV650:", IMG.shape, "| Maßstab:", round(m_px,2), "m/Pixel")
KERN = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
def farbmaske(rgb, tol=60):
    m = np.zeros(IMG.shape[:2], bool); c = np.array(rgb, np.int32)
    for r0 in range(0, IMG.shape[0], 500):                      # zeilenweise (Speicher)
        blk = IMG[r0:r0+500].astype(np.int32); m[r0:r0+500] = ((blk-c)**2).sum(-1) < tol**2
    return m
MASK = {k: farbmaske(v) for k,v in FARBEN.items()}; MASKD = {k: cv2.dilate(v.astype(np.uint8), KERN) > 0 for k,v in MASK.items()}
print("Linienpixel je Trasse:", {k: int(v.sum()) for k,v in MASK.items()})
def _snap(m, p, r=90):
    x, y = p; ys, xs = np.nonzero(m[max(0,y-r):y+r, max(0,x-r):x+r])
    if len(xs) == 0: return None
    d_ = (xs+max(0,x-r)-x)**2 + (ys+max(0,y-r)-y)**2; i = d_.argmin(); return (int(xs[i]+max(0,x-r)), int(ys[i]+max(0,y-r)))
def trasse(farbe, a_name, b_name, pad=250):
    m = MASKD[farbe]; a = _snap(m, ANKER[a_name]); b = _snap(m, ANKER[b_name])
    if a is None or b is None: return None
    x0 = max(0, min(a[0],b[0])-pad); x1 = min(m.shape[1], max(a[0],b[0])+pad); y0 = max(0, min(a[1],b[1])-pad); y1 = min(m.shape[0], max(a[1],b[1])+pad)
    cost = np.where(m[y0:y1, x0:x1], 1.0, 60.0)
    path, _ = route_through_array(cost, (a[1]-y0, a[0]-x0), (b[1]-y0, b[0]-x0), fully_connected=True, geometric=True)
    p = np.array(path); L = np.sqrt((np.diff(p, axis=0)**2).sum(1)).sum()
    return dict(farbe=farbe, von=a_name, nach=b_name, laenge_m=L*m_px, luftlinie_m=np.hypot(b[0]-a[0], b[1]-a[1])*m_px, auf_linie=float(m[y0:y1, x0:x1][p[:,0], p[:,1]].mean()), pfad=(p[:, ::-1]+[x0, y0]))
SEGMENTE = [("SL1a","SL1","HKW","K001"), ("SL1bc","SL1","K001","PSW_346_1"), ("SL5a","SL5","K001","MAN_342_1"), ("SL5b","SL5","MAN_342_3","G833_3"), ("SL5c","SL5","G833_3","G832_3"),
            ("SL5d","SL5","G832_3","G809_2"), ("SL5e","SL5","G809_2","AVA_810_1"), ("SL5f","SL5","G809_2","GT"), ("SL5g","SL6","GT","BM"), ("SL6ab","SL6","G338_2","G832_3"),
            ("SL6cd","SL6","G832_3","G750_1"), ("SL7","SL7","G773","G818_2"), ("SL4","SL4","HKW","PSS_297_1"), ("SL3","SL3","HKW","G338_2")]
TR = {sid: trasse(f, a_, b_) for sid, f, a_, b_ in SEGMENTE}
tr_tab = pd.DataFrame([(sid, r["farbe"], r["von"], r["nach"], round(r["laenge_m"]), round(r["luftlinie_m"]), round(100*r["auf_linie"])) for sid, r in TR.items()],
                      columns=["Segment","Trasse","von","nach","Länge_m","Luftlinie_m","auf_Linie_%"])
tr_tab["Umwegfaktor"] = (tr_tab["Länge_m"]/tr_tab["Luftlinie_m"]).round(2); display(tr_tab)
assert (tr_tab["auf_Linie_%"] >= 85).all() and (tr_tab["Länge_m"] >= tr_tab["Luftlinie_m"]).all(), "Trassenverfolgung unplausibel"
sl5_ges = sum(TR[k]["laenge_m"] for k in ("SL5a","SL5b","SL5c","SL5d","SL5e")) + TR["SL1a"]["laenge_m"]
print(f"SL5 vom HKW bis AVA: {sl5_ges/1000:.2f} km (YAML: 5,93 km; Modell v0.1: 5,93 km) | HKW→PSW (SL1): {(TR['SL1a']['laenge_m']+TR['SL1bc']['laenge_m'])/1000:.2f} km (YAML 2,63) | HKW→PSS (SL4): {TR['SL4']['laenge_m']/1000:.2f} km (YAML 4,47) | SL6 338→GT: {(TR['SL6ab']['laenge_m']+TR['SL6cd']['laenge_m'])/1000:.2f} km | SL7 773→818: {TR['SL7']['laenge_m']/1000:.2f} km")
tr_tab.to_csv(RESULTS/"trassenlaengen_plan.csv", index=False)

In [ ]:
# ---------- Modell v0.3: Längen aus Plangeometrie, Topologiekorrekturen
kn3 = kn2.copy(); ka3 = ka2.copy()
# (1) Abzweig SL7 an SL6 (Armatur 773): eigener Knoten N773 auf der SL6-Linie zwischen KNEU und GT
kn3 = pd.concat([kn3, pd.DataFrame([dict(node_id="N773", name="Abzweig SL7 an SL6 (Affinger Str., Arm. 773.x)", typ="koppelstelle", sektion="SL3", netz="Ost", PN=25, z_NN_m=np.nan, p_anschluss_MW=0.0,
                                         x_km=3.6, y_km=3.4, quelle="WV650 Karte", sicherheit="mittel", z_abgeleitet_mNN=np.nan, z_Quelle="unbekannt → Kalibrierparameter", z_modell_mNN=np.nan)])], ignore_index=True)
def set_edge(eid, **kw):
    i = ka3.index[ka3.edge_id == eid][0]
    for k_, v_ in kw.items(): ka3.loc[i, k_] = v_
def add_edge(**kw): 
    global ka3; ka3 = pd.concat([ka3, pd.DataFrame([kw])], ignore_index=True)
Lg = {k: v["laenge_m"] for k, v in TR.items()}
set_edge("SL1a", laenge_m=Lg["SL1a"], laenge_quelle="G")
set_edge("SL1b", laenge_m=Lg["SL1bc"]/2, laenge_quelle="G"); set_edge("SL1c", laenge_m=Lg["SL1bc"]/2, laenge_quelle="G")
for e_ in ("SL5a","SL5b","SL5c","SL5d","SL5e","SL5f"): set_edge(e_, laenge_m=Lg[e_], laenge_quelle="G")
set_edge("SL5g", von="GT", nach="BM", laenge_m=Lg["SL5g"], laenge_quelle="G", dn_mm=250, armaturen="784.1")          # BM-HKW hängt am GT-Ende der SL6 (nicht am SL5-Ost-Knoten)
set_edge("SL5f", dn_mm=350, armaturen="809.1, 515.1")
set_edge("SL6a", laenge_m=Lg["SL6ab"]/2, laenge_quelle="G"); set_edge("SL6b", laenge_m=Lg["SL6ab"]/2, laenge_quelle="G")
a_, b_ = np.hypot(*np.subtract(ANKER["G773"], ANKER["G832_3"])), np.hypot(*np.subtract(ANKER["G750_1"], ANKER["G773"]))
set_edge("SL6c", von="KNEU", nach="N773", laenge_m=Lg["SL6cd"]*a_/(a_+b_), laenge_quelle="G")
add_edge(edge_id="SL6d", von="N773", nach="GT", art="Trasse SL6 (Ost)", dn_mm=350, laenge_m=Lg["SL6cd"]*b_/(a_+b_), laenge_quelle="G", sicherheit="mittel", normalstatus="offen", armaturen="750.x", max_t_h_separiert=500, bidirektional=True)
d7a = np.hypot(*np.subtract(ANKER["G888_1"], ANKER["G818_2"])); d7b = np.hypot(*np.subtract(ANKER["G773"], ANKER["G888_1"]))
set_edge("SL7a", laenge_m=Lg["SL7"]*d7a/(d7a+d7b), laenge_quelle="G"); set_edge("SL7b", von="SL7G", nach="N773", laenge_m=Lg["SL7"]*d7b/(d7a+d7b), laenge_quelle="G")
set_edge("SL4b", laenge_m=Lg["SL4"] - float(ka3.loc[ka3.edge_id=="SL4a","laenge_m"].iloc[0]), laenge_quelle="G")            # SL4a bleibt kalibrierte Ersatzlänge
set_edge("SL3", laenge_m=Lg["SL3"], laenge_quelle="G")
for e_ in ka3.edge_id[ka3.laenge_quelle=="G"]: set_edge(e_, sicherheit="mittel")
band = {"Y":.15,"P":.25,"K":.30,"S":.50,"G":.12}
ka3["laenge_km"] = ka3.laenge_m/1000; ka3["laenge_min_m"] = ka3.laenge_m*(1-ka3.laenge_quelle.map(band)); ka3["laenge_max_m"] = ka3.laenge_m*(1+ka3.laenge_quelle.map(band))
idx3 = {n:i for i,n in enumerate(kn3.node_id)}
A3 = coo_matrix((np.ones(len(ka3)), ([idx3[a] for a in ka3.von],[idx3[b] for b in ka3.nach])), shape=(len(idx3),)*2)
nc3, _ = connected_components(A3, directed=False)
print(f"Modell v0.3: {len(kn3)} Knoten, {len(ka3)} Kanten, Komponenten {nc3}, unabhängige Schleifen {len(ka3)-len(kn3)+nc3}")
print("Längenquelle:", ka3.laenge_quelle.value_counts().to_dict(), "(G Plangeometrie, Y YAML, P Plan-Anteil, K Messung kalibriert, S Schätzung)")
assert nc3 == 2 and len(ka3)-len(kn3)+nc3 == 7
vgl = pd.DataFrame({"Kante":ka2.edge_id.values[:0]})
alt = ka2.set_index("edge_id").laenge_m; neu = ka3.set_index("edge_id").laenge_m
cmp3 = pd.DataFrame({"v0.2_m":alt, "v0.3_m":neu}).dropna().round(0); cmp3["Änderung_%"] = (100*(cmp3["v0.3_m"]/cmp3["v0.2_m"]-1)).round(0)
display(cmp3[cmp3["Änderung_%"].abs() > 5].sort_values("Änderung_%"))
kn3.to_csv(RESULTS/"modell_v03_knoten.csv", index=False); ka3.to_csv(RESULTS/"modell_v03_kanten.csv", index=False)
kn2, ka2 = kn3, ka3                      # ab hier gilt Modell v0.3 (Variablennamen bleiben zur Kompatibilität)

In [ ]:
# ---------- Overlay der verfolgten Trassen (Sichtprüfung)
ov = IMG.copy(); farb_bgr = {"SL1":(0,140,0), "SL3":(255,200,0), "SL4":(200,0,200), "SL5":(255,0,0), "SL6":(0,0,255), "SL7":(0,170,0)}
for sid, r in TR.items():
    pts = r["pfad"].astype(np.int32).reshape(-1,1,2); cv2.polylines(ov, [pts], False, (0,0,0), 18); cv2.polylines(ov, [pts], False, farb_bgr[r["farbe"]][::-1], 10)
for nm, (x_, y_) in ANKER.items(): cv2.circle(ov, (x_, y_), 16, (0,0,0), -1)
crop = ov[900:4100, 400:3100]; crop = cv2.resize(crop, (crop.shape[1]//3, crop.shape[0]//3), interpolation=cv2.INTER_AREA)
fig, ax = plt.subplots(figsize=(9,10)); ax.imshow(crop); ax.axis("off"); ax.set_title("Verfolgte Trassen im Plan WV650 (Overlay: SL1 grün, SL3 gelb, SL4 violett, SL5 rot, SL6 blau, SL7 grün)")
fig.tight_layout(); fig.savefig(FIG_DIR/"10_trassen_overlay.png", dpi=110); plt.show()

## 10.7 Abschließende kritische Prüfung der Annahmen
Vor dem Aufbau der Rechenmodelle werden alle tragenden Annahmen einmal gegen die Daten geprüft. Zentrale Frage: **wohin fließt die Ost-Erzeugung?** Der Osten erzeugt im Winter 60 % der Einspeisung, über die HKW-Sammelschiene fließt aber nur wenig. Entweder verbrauchen die Ost-/STAWA-Sektionen den Großteil selbst, oder der Überschuss erreicht die Innenstadt über die Verbundkanten. Die Kalibrierung (Kap. 13) entscheidet zugunsten hoher Ost-Auslastung; der Übertragungsfluss bleibt aus den Daten unbestimmt.

In [ ]:
w_ = acron.index.month.isin([12,1,2])
dT_h = acron.HKW_Temp_VL - acron.HKW_Temp_RL
Q_sl = {i: acron[f"HKW_Durchfluss_SL{i}"]*K_MW*dT_h for i in (1,2,3,4)}
einsp = abg.HKW_netto_MW + abg.Ost_MW + abg.HWS_MW.fillna(0)               # Einspeisung ins Verbundnetz
last_v = einsp - abg.HWW_prim_MW                                            # Kundenlast Verbund (ohne WÜ West, inkl. Verluste)
pa = knoten.set_index("node_id").p_anschluss_MW
ost_sek = ["SL1G","MAN","SL5M","SL5O","SEC3","SL6S","SL7G"]                 # vom Osten direkt erreichte Sektionen (SL1 + SL3)
mitte_sek = ["SEC2","SEC4","SUED"]                                          # SL2 + SL4
s13 = pa[ost_sek].sum()/(pa[ost_sek].sum()+pa[mitte_sek].sum()); s24 = 1-s13
transfer = abg.Ost_MW + Q_sl[1] + Q_sl[3] - s13*last_v                     # Wärme, die vom Osten über Verbundkanten nach SL2/SL4 muss
rows = [("Ost-Erzeuger (AVA+GT+BM)", abg.Ost_MW[w_].median()), ("HKW netto", abg.HKW_netto_MW[w_].median()), ("Heizwerk Süd", abg.HWS_MW[w_].median()),
        ("Einspeisung Verbund", einsp[w_].median()), ("WÜ Westnetz (aus Primärnetz)", abg.HWW_prim_MW[w_].median()), ("Kundenlast Verbund (inkl. Verluste)", last_v[w_].median()),
        ("HKW → SL2 + SL4", (Q_sl[2]+Q_sl[4])[w_].median()), ("HKW → SL1 + SL3", (Q_sl[1]+Q_sl[3])[w_].median())]
display(pd.DataFrame(rows, columns=["Größe (Winter, Median)","MW"]).round(1))
sens = [(f, (abg.Ost_MW + Q_sl[1] + Q_sl[3] - f*last_v)[w_].median()) for f in (0.25, s13, 0.40)]
print(f"Lastanteil der Ost-Sektionen an der Anschlussleistung: {s13:.2f}")
print("Über Verbundkanten von Ost nach SL2/SL4 zu übertragende Wärme (Winter-Median) bei Lastanteil 0,25 / %.2f / 0,40: %.0f / %.0f / %.0f MW" % (s13, sens[0][1], sens[1][1], sens[2][1]))
print("→ Bei gleicher Auslastung aller Sektionen müssten 30–50 MW über Verbundkanten nach SL2/SL4 fließen. Die Auslastung ist aber nicht einheitlich (Kap. 13: Innenstadt ≈ 0,2, Ost/STAWA deutlich höher); der Übertragungsfluss ist aus den Daten nicht eindeutig bestimmt (Bandbreite ≈ 0–40 MW) und wird als Sensitivität geführt.")

# Pumpstationen: Druckgewinn
pss_gain = (acron.PPS_Druck_VL_nach_Pumpe - acron.PPS_Druck_VL_vor_Pumpe)
psw_gain = (acron.PSW_DiffDruck_VL_nach_Pumpe - acron.PSW_DiffDruck_VL_vor_Pumpe)
print(f"PSS-Druckgewinn (nach − vor Pumpe): Median {pss_gain.median():.2f} bar, P95 {pss_gain.quantile(.95):.2f} bar, Anteil > 0,3 bar: {(pss_gain>0.3).mean():.0%}")
print(f"PSW-Druckgewinn (Δp-Differenz):     Median {psw_gain.median():.2f} bar, P95 {psw_gain.quantile(.95):.2f} bar, Anteil > 0,3 bar: {(psw_gain>0.3).mean():.0%}")
d_ = pd.concat([acron.PSW_Durchfluss_RL_nach_Mitte, acron.HWW_Durchfluss_Primeinsp], axis=1, keys=["psw","hww"]).dropna(); d_ = d_[d_.psw < 700]
print(f"PSW-Durchfluss / HWW-Primärfluss: Median-Verhältnis {(d_.psw/d_.hww).median():.2f} → konsistent (Primärkreis HKW–PSW–HWW)")

In [ ]:
audit = pd.DataFrame([
 ("Druckdefinition Überdruck (bar)", "HKW-Rücklauf im Median 8,5 bar gegen Ruhedruckbalken 8,4 bar Ü", "bestätigt", "Siedesicherheit: p_abs = p_ü + 1,013"),
 ("Durchfluss in m³/h", "Q_Station/(V̇·ρ·cp·ΔT) ≈ 1,04 an 8 Kundenstationen", "bestätigt", "Massenströme"),
 ("Struktur SL1–SL4 / SL5–SL7 / Verbund", "Rückflussanteile SL1 80 %, SL3 71 %; Ost speist 60 % der Winter-Einspeisung; Übertragungsfluss über Verbundkanten aus Daten unbestimmt", "Struktur bestätigt, Flussgrößen offen", "Vermaschung vorhanden; Flussverteilung kalibriert/Sensitivität"),
 ("Westnetz hydraulisch entkoppelt", "σ(RL) = 0,05 bar; Partialkorrelation mit Verbund ≈ 0", "bestätigt", "West thermisch über WÜ gekoppelt"),
 ("Signalbedeutung HKW/HWW", "r = 1,00 bzw. R² = 1,00 (Bilanzidentitäten)", "bestätigt", "Randbedingungen"),
 ("Höhen aus Schwachlast", "VL/RL-Offset r = 0,93; Plan-Höhen ±3,5 m (außer GT)", "plausibel (±3–5 m)", "Druckniveau der Stationen"),
 ("Maßstab 1:10 000", "SL5-Länge −7 % gegen YAML", "plausibel", "Rohrlängen"),
 ("Trassenlängen SL2/SL3/SL4", "aus Druckverlust-Fit (R² 0,23–0,58)", "kalibrierbar, schwach für SL3", "Δp im Verbund"),
 ("Hydraulik HKW–MAN", "R² = 0,13", "schwach bestimmt", "Unsicherheit am Speicherstandort"),
 ("Kundenlasten je Sektion", "nur 7 Stationen gemessen (Σ ≈ 21 MW im Winter von ≈ 113 MW)", "geschätzt (Anteile kalibriert)", "Flussverteilung im Verbund"),
 ("Pumpstationen PSS/PSW", "Druckgewinn im Median 0 bar, P95 ≈ 1,3 bar", "Bypass, nur bei Spitzen aktiv", "Speicherpumpe wirkt nur zusätzlich zu bestehenden Pumpen"),
 ("Auslegungslast −14 °C", "216 MW (Mittel) … 269 MW (Spitze); 2021: 214 MW", "unsicher (±25 %)", "Szenarien S1/S5"),
 ("Schaltzustände 2025", "nicht dokumentiert", "unbekannt → Ausreißerbehandlung", "unerklärte Residuen"),
 ("Plan-Widersprüche (141.1/187.1)", "Konnektivität unberührt", "unkritisch", "Leitfähigkeit"),
], columns=["Annahme","Prüfung","Urteil","Wirkung"])
with pd.option_context("display.max_colwidth",90,"display.width",250): display(audit)
audit.to_csv(RESULTS/"annahmenaudit.csv", index=False)

## 11. Dokumentation der mathematischen Gleichungen
Alle Größen in SI-Einheiten (kg/s, m, Pa; Anzeige in bar und °C). **Druckdefinitionen:** $p_{\ddot u}$ Überdruck, $p_{abs}=p_{\ddot u}+p_{atm}$ mit $p_{atm}=1{,}013$ bar, Ruhedruck = statischer Druck ohne Umwälzung, $\Delta p_{Kunde}=p_{VL}-p_{RL}$, Förderhöhe $H_P$ als Druckhöhe der Pumpe.

### 11.1 Massenerhaltung (Vorlauf und Rücklauf getrennt, Vermaschung)
Für jeden Knoten $n$ im Vorlauf- bzw. Rücklaufnetz gilt mit Kantenmassenströmen $\dot m_e$ (positiv von `von` nach `nach`) und der Einspeisung $\dot m_{n}^{ein}$ (Quelle positiv, Last negativ):
$$\sum_{e:\,\mathrm{von}(e)=n}\dot m_e-\sum_{e:\,\mathrm{nach}(e)=n}\dot m_e=\dot m_n^{ein},\qquad \mathbf A\,\dot{\mathbf m}=\dot{\mathbf m}^{ein}.$$
Flussrichtungsumkehr ist durch das Vorzeichen von $\dot m_e$ erlaubt. Im Rücklaufnetz gilt dieselbe Bilanz mit $\dot m^{ein}_{RL}=-\dot m^{ein}_{VL}$.
*Umsetzung:* Inzidenzmatrix `Ainc`, Funktion `solve_network` (Kap. 12.2).

### 11.2 Wärmeleistung
$$\dot Q=\dot m\,c_p\,(T_{VL}-T_{RL}),$$ mit temperaturabhängigem $c_p$ (Tabelle, Fehler gegenüber konstantem $c_p=4{,}19$ kJ/(kg·K) in Kap. 12.1 bewertet). *Umsetzung:* `K_MW` (Näherung, im Datenkapitel), `cp_w`.

### 11.3 Rohrdruckverlust (Darcy–Weisbach) und Druckhöhe
Druckhöhe $H=z+\dfrac{p_{\ddot u}}{\rho g}$. Für Kante $e$ mit Länge $L$, Innendurchmesser $D$, Querschnitt $A$:
$$H_{von}-H_{nach}=r_e\,\dot m_e|\dot m_e|-H_{P,e},\qquad r_e=\frac{\lambda_e\,L_e}{D_e\,2g\,\rho^2A_e^2}\,k_g ,$$
$$Re=\frac{\dot m D}{\rho A\mu},\qquad \lambda=\lambda(Re,\varepsilon/D)\ \text{(Swamee–Jain; laminar }64/Re\text{; Übergang linear)}.$$
$k_g$ ist der **Kalibriermultiplikator** je Leitungsgruppe (fasst lokale Verluste $\zeta_i$, Krümmer, Armaturen und die Unsicherheit von $L$ und $D$ zusammen; Herkunft: kalibriert). Der geodätische Anteil steckt in $z$ (Druckhöhe), nicht in $r_e$. Höhen: Plan (Erzeuger) bzw. abgeleitet (Stationen). Der Druck folgt aus $p_{\ddot u}=\rho g\,(H-z)$. **Randbedingungen:** Die Druckhöhe ist an zwei Knoten vorgegeben – am HKW (Ruhedruckbalken, gemessen) und am Ost-Sammler (AVA-Übernahme, gemessen); die Ost-Erzeuger regeln ihren Differenzdruck mit Rücklaufpumpen (WV640), der Massenstrom der beiden Randknoten stellt sich hydraulisch ein und wird mit der Messung verglichen.
*Umsetzung:* `edge_resistance`, `swamee_jain`, `solve_network`.

### 11.4 Pumpen
Vorhandene Pumpstationen PSS und PSW werden nicht über Kennlinien, sondern über die **gemessene Druckerhöhung** $H_{P,e}(t)$ abgebildet (keine Kennlinien vorhanden). Elektrische Leistung: $P_{el}=\dfrac{\dot m\,g\,H_P}{\eta}$, $\eta=0{,}75$ (Annahme, Sensitivität 0,6–0,8). Für die Speicherpumpe wird $H_P$ in Kap. 15 als Variable behandelt.

### 11.5 Temperaturmischung und Wärmeverlust (Kap. 12/17)
$$\sum_i\dot m_i h_i=\dot m_{aus}h_{aus},\qquad T_{aus}=T_{amb}+(T_{ein}-T_{amb})\exp\!\Big(-\frac{U\!L}{\dot m\,c_p}\Big),$$ Umgebung $T_{amb}=10$ °C, $U$ aus YAML/Plan (Klasse Y). Die thermische Simulation folgt in Etappe 4 zusammen mit den Szenarien; die Hydraulik wird hier stationär pro Stunde gelöst (quasistationär, keine Druckstöße).

### 11.6 Verbraucher
$\dot m=\dfrac{\dot Q}{c_p(T_{VL}-T_{RL})}$, Mindestdifferenzdruck $\Delta p_{Kunde}\ge\Delta p_{min}$ (Sensitivität 0,3–1,5 bar; YAML 0,7 bar).

### 11.7 Speicher (Kap. 15, hier Definition)
Energiebilanz je Zeitschritt: $E_{t+1}=E_t+\eta_{ch}\dot Q_{ch}\Delta t-\dot Q_{dis}\Delta t/\eta_{dis}-\dot Q_{V}\Delta t$, $E_{min}\le E_t\le E_{max}$, $E_{max}=\rho\,c_p\,V\,\Delta T\cdot\eta_{strat}$. Hydraulisch: Speicherzweig am Knoten `MAN` mit Massenstrom $\dot m_S$ (Laden negativ, Entladen positiv als Einspeisung) und Druckbilanz $H_{MAN}-H_S=\zeta_S\dot m_S|\dot m_S|-H_{P,S}$ (Ventil bzw. Pumpe).

### 11.8 Betriebsgrenzen
Siedesicherheit $p_{abs}\ge p_{sat}(T)+\Delta p_{S}$ mit $p_{sat}$ nach Wagner–Pruss (IAPWS 1992), $\Delta p_S=1$ bar (Sensitivität 0,5–2 bar); Mindestdrücke und Abschaltgrenzen der Erzeuger aus WV640; PN16/PN25; Transportgrenzen SL5/SL6/SL7 (750/500/300 t/h); Erzeuger-, Pumpen- und Speichergrenzen.

### 11.9 Zustandsschätzung und Kalibrierung
Je Stunde $t$: Einspeisungen aus Messung, unbekannte Sektionslasten $d_{s,t}=w_s\,R_t$ mit Restlast $R_t=\sum\dot m^{ein}-\dot m_{MAN}-\dot m_{W\ddot U}$ und statischen Anteilen $w_s$ ($\sum w_s=1$). Kalibrierparameter $\theta=(\ln k_g,\ \text{logit}(w_s))$; Minimierung
$$\min_\theta\sum_{t,i}\Big(\frac{p^{mod}_{i,t}-p^{mes}_{i,t}}{\sigma_p}\Big)^2+\sum_{t,e}\Big(\frac{\dot V^{mod}_{e,t}-\dot V^{mes}_{e,t}}{\sigma_V}\Big)^2+\text{Ridge}(\theta).$$
Parameter bleiben in physikalisch plausiblen Bereichen ($k_g\in[0{,}2;5]$).

### 11.10 Verwendete Module des Frameworks (Repository `calion`)
Das Framework wird für **Konstanten und Referenzfunktionen** genutzt und gegen die eigene Implementierung getestet (Kap. 12.1). Die Hydraulik (Druck, Vermaschung, Höhen) wird **nicht** aus dem Framework übernommen, weil dort der Druckabfall nur als untere Hüllkurve mit konstantem $\lambda=0{,}02$ modelliert ist und Höhen/Vermaschung fehlen (Kap. 4/5).

## 12. Implementierung des Modells

### 12.1 Stoffwerte, Siededruck und Referenztests

In [ ]:
import time
G = 9.81   # m/s²
def rho_w(T):      # Kell (1975), kg/m3, T in degC
    T = np.asarray(T, float)
    return (999.83952 + 16.945176*T - 7.9870401e-3*T**2 - 46.170461e-6*T**3 + 105.56302e-9*T**4 - 280.54253e-12*T**5)/(1+16.879850e-3*T)
def mu_w(T):       # Pa s (Vogel-Gleichung)
    T = np.asarray(T, float); return 2.414e-5*10**(247.8/(T+273.15-140.0))
def psat_bar(T):   # Wagner-Pruss (IAPWS 1992), bar
    T = np.asarray(T, float)+273.15; Tc, pc = 647.096, 220.64
    a = (-7.85951783, 1.84408259, -11.7866497, 22.6807411, -15.9618719, 1.80122502); tau = 1-T/Tc
    return pc*np.exp(Tc/T*(a[0]*tau + a[1]*tau**1.5 + a[2]*tau**3 + a[3]*tau**3.5 + a[4]*tau**4 + a[5]*tau**7.5))


def cp_w(T):       # kJ/(kg K), gesättigtes Wasser (IAPWS-IF97-Tabellenwerte), lineare Interpolation
    Tt = [0,20,40,60,80,100,120,140,160]; cc = [4.217,4.182,4.179,4.185,4.197,4.216,4.245,4.285,4.339]
    return np.interp(np.asarray(T, float), Tt, cc)

# ---- Tests gegen Referenzwerte (IAPWS/Handbuchwerte)
ref_rho = {20:998.2, 60:983.2, 100:958.4, 120:943.1}
for T_, r_ in ref_rho.items(): assert abs(rho_w(T_)-r_) < 1.0, (T_, rho_w(T_), r_)
assert abs(mu_w(20)*1e3-1.002) < 0.03 and abs(mu_w(100)*1e3-0.282) < 0.01
for T_, p_ in {100:1.0142, 120:1.9867, 150:4.7616}.items(): assert abs(psat_bar(T_)-p_) < 0.002, (T_, psat_bar(T_))
print("Stoffwerte bestanden: ρ (20/60/100/120 °C) =", rho_w([20,60,100,120]).round(1), "kg/m³ | μ(20/100 °C) =", (mu_w([20,100])*1e3).round(3), "mPa·s | p_sat(100/120/150 °C) =", psat_bar([100,120,150]).round(3), "bar")
print(f"Siedesicherheit: p_abs,min bei 120 °C = p_sat + 1 bar = {psat_bar(120)+1.0:.2f} bar abs = {psat_bar(120)+1.0-1.013:.2f} bar Ü; bei 128 °C: {psat_bar(128)+1.0-1.013:.2f} bar Ü")
print(f"Fehler konstantes cp=4,19 gegenüber Tabelle (60…120 °C): {100*(4.19/cp_w(np.array([60,90,120]))-1).round(3)} %")

# ---- Vergleich mit pandapipes (Stoffwerte Wasser) und Framework-Modul
try:
    import pandapipes as pp
    net_ = pp.create_empty_network(fluid="water"); fl_ = net_.fluid
    Tk = np.array([293.15, 333.15, 373.15, 393.15])
    print("pandapipes-Wasser ρ [kg/m³]:", np.round(fl_.get_density(Tk),1), "| eigene:", rho_w(Tk-273.15).round(1))
except Exception as e:
    print("pandapipes-Stoffwertvergleich nicht möglich:", type(e).__name__, e, "(wird in Kap. 20 erneut geprüft)")
sys.path.insert(0, str(ROOT))
from calion.models import network_physics as fw
print("Framework-Modul:", Path(fw.__file__).relative_to(ROOT).as_posix())
assert abs(fw.heat_kw_to_mdot_kg_s(1000.0, 4.19, 40.0) - 1000.0/(4.19*40.0)) < 1e-9
assert abs(fw.mdot_to_velocity_m_s(100.0, 300.0, 960.0) - 100/(960*np.pi*0.15**2)) < 1e-9
assert abs(fw.pipe_heat_loss_mw(0.3, 1000.0, 100.0, 10.0) - 0.3*1000*90/1e6) < 1e-12
print("Framework-Funktionen (heat_kw_to_mdot_kg_s, mdot_to_velocity_m_s, pipe_heat_loss_mw) stimmen mit der eigenen Rechnung überein.")

# ---- Speicherenergie (Gl. 11.7) und Test
def storage_step(E, Q_ch, Q_dis, dt_h, eta_ch=0.98, eta_dis=0.98, Q_loss=0.0):
    return E + eta_ch*Q_ch*dt_h - Q_dis/eta_dis*dt_h - Q_loss*dt_h
def storage_capacity_MWh(V_m3, dT_K, T_mean_C=90.0, eta_strat=1.0):
    return rho_w(T_mean_C)*cp_w(T_mean_C)*V_m3*dT_K*eta_strat/3.6e6
E0 = 20.0; E1 = storage_step(E0, 10.0, 0.0, 1.0, Q_loss=0.1)
assert abs(E1 - (20 + 0.98*10 - 0.1)) < 1e-12
assert 55 < storage_capacity_MWh(5000, 10) < 59
print(f"Speicherbilanz-Test bestanden; Kapazität 5 000 m³ bei 10 K = {storage_capacity_MWh(5000,10):.1f} MWh (bestätigt 56–58 MWh, nicht 84,5 MWh).")

### 12.2 Hydraulischer Netzlöser (stationär, vermascht, Batch über Stunden)

In [ ]:
def swamee_jain(Re, eps_rel):
    Re = np.maximum(Re, 1.0)
    turb = 0.25/np.log10(eps_rel/3.7 + 5.74/Re**0.9)**2
    lam_l = 64.0/Re
    lam_t = 64/2300 + (turb-64/2300)*(Re-2300)/1700
    return np.where(Re < 2300, lam_l, np.where(Re < 4000, lam_t, turb))

def edge_resistance(m, L, D, mult, T, rough):
    """r in ΔH = r·m|m| [m Wassersäule je (kg/s)^2]; Reibungsbeiwert aus Re(m)."""
    rho, mu = rho_w(T), mu_w(T); A = np.pi*D**2/4
    Re = rho*np.abs(m)/(rho*A)*D/mu * 1.0
    Re = np.abs(m)/A*D/mu
    lam = swamee_jain(Re, rough/D)
    return lam*L/(D*2*G*rho**2*A**2)*mult

def solve_network(Ainc, m_inj, L, D, mult, T, ref, H_ref, gain=None, rough=1e-4, omega=0.6, mmin=0.05, iters=300, tol=1e-5):
    """Stationäre Hydraulik mit einem oder mehreren Druckrandknoten.
    Ainc (N,E): +1 = Kante verlässt Knoten, -1 = erreicht Knoten. m_inj (B,N): Einspeisung [kg/s] (+Quelle, −Last) an den FREIEN Knoten;
    ref: Index bzw. Liste der Druckrandknoten (Druckhöhe H_ref (B,) bzw. (B,nref) vorgegeben; deren Einspeisung ergibt sich aus der Hydraulik).
    Rückgabe H (B,N) Druckhöhe [m], m (B,E) Massenstrom [kg/s] (+ von→nach), it, Kantenresiduum [m], Bilanzfehler an freien Knoten [kg/s]"""
    N, E = Ainc.shape; B = m_inj.shape[0]
    ref = np.atleast_1d(ref); H_ref = np.asarray(H_ref, float)
    if H_ref.ndim == 1: H_ref = H_ref[:, None]
    gain = np.zeros((B, E)) if gain is None else gain
    free = np.array([i for i in range(N) if i not in set(ref.tolist())]); Af = Ainc[free]; Ar = Ainc[ref]
    m = np.linalg.lstsq(Af, m_inj[:, free].T, rcond=None)[0].T           # Startwert (Minimum-Norm)
    for it in range(iters):
        r = edge_resistance(m, L, D, mult, T, rough)
        C = 1.0/(r*np.maximum(np.abs(m), mmin))                            # linearisierte Leitfähigkeit
        M = np.einsum('ne,be,ke->bnk', Af, C, Af) + 1e-9*np.eye(len(free))
        rhs = m_inj[:, free] - np.einsum('ne,be,be->bn', Af, C, gain) - np.einsum('ne,be,re,br->bn', Af, C, Ar, H_ref)
        Hf = np.linalg.solve(M, rhs[..., None])[..., 0]
        H = np.zeros((B, N)); H[:, free] = Hf; H[:, ref] = H_ref
        m_new = C*(H @ Ainc + gain)
        d = np.max(np.abs(m_new-m))
        m = omega*m_new + (1-omega)*m
        if d < tol: break
    r = edge_resistance(m, L, D, mult, T, rough)
    H_e = H @ Ainc
    res = np.max(np.abs(H_e + gain - r*np.abs(m)*m), axis=1)                # Kantengleichung [m]
    bal = np.max(np.abs((m @ Ainc.T - m_inj)[:, free]), axis=1)             # Knotenbilanz an freien Knoten [kg/s]
    return H, m, it, res, bal

In [ ]:
# ---- Tests des Lösers gegen analytische Lösungen
# (1) zwei parallele Rohre DN200/DN300: gleiche Druckhöhendifferenz
Ainc_t = np.array([[1,1],[-1,-1]], float); L_t = np.array([1000.,1000.]); D_t = np.array([0.2,0.3])
H_, m_, it_, res_, bal_ = solve_network(Ainc_t, np.array([[100.,-100.]]), L_t, D_t, np.ones(2), 80.0, ref=1, H_ref=np.array([50.0]))
r_ = edge_resistance(m_, L_t, D_t, np.ones(2), 80.0, 1e-4); dH_ = r_*np.abs(m_)*m_
assert abs(m_.sum()-100) < 1e-6 and abs(dH_[0,0]-dH_[0,1]) < 1e-3 and m_[0,1] > m_[0,0]
# (2) Ring mit drei Knoten: Knotenbilanz und Kantengleichung
Ainc_r = np.array([[1,0,-1],[-1,1,0],[0,-1,1]], float)
H_, m_, it_, res_, bal_ = solve_network(Ainc_r, np.array([[50.,-20.,-30.]]), np.array([500.,700.,600.]), np.array([.25,.2,.25]), np.ones(3), 80., 0, np.array([100.]))
assert bal_.max() < 1e-3 and res_.max() < 1e-2
# (3) Schleifengesetz: Summe der Druckhöhenverluste um die Masche = 0
r_ = edge_resistance(m_, np.array([500.,700.,600.]), np.array([.25,.2,.25]), np.ones(3), 80., 1e-4)
assert abs((r_*np.abs(m_)*m_).sum()) < 1e-2
# (4) Höhenunabhängigkeit: Druckhöhe H ändert sich nicht, wenn nur z (Druckumrechnung) geändert wird -> trivial durch Formulierung; Druckvorzeichen
print(f"Löser-Tests bestanden (Parallelrohre, Ring, Schleifengesetz); Iterationen {it_}, Bilanzfehler {bal_.max():.1e} kg/s.")

### 12.3 Netzdaten und Randbedingungen aus Messung

In [ ]:
NODES = [n for n in kn2.node_id if n != "WEST"]; nidx = {n:i for i,n in enumerate(NODES)}
E_IDS = list(ka2.edge_id); eidx = {e:i for i,e in enumerate(E_IDS)}
Ainc = np.zeros((len(NODES), len(E_IDS)))
for j,e in ka2.reset_index(drop=True).iterrows(): Ainc[nidx[e.von], j] = 1.0; Ainc[nidx[e.nach], j] = -1.0
L_E = ka2.laenge_m.values.astype(float); D_E = (ka2.dn_mm.values/1000.0).astype(float)
GRUPPEN = {"SL1":["SL1a","SL1b","SL1c"],"SL5":["SL5a","SL5b","SL5c","SL5d"],"OSTSTUB":["SL5e","SL5f","SL5g"],"SL6":["SL6a","SL6b","SL6c","SL6d"],"SL7":["SL7a","SL7b"],
           "SL2":["SL2"],"SL3":["SL3"],"SL4":["SL4a","SL4b","SL4c","SL4d"],"M12":["M_SL1_SL2"],"M23":["M_SL2_SL3"],"M24":["M_SL2_SL4"],"M34":["M_SL3_SL4"],
           "LAND":["K_SL6_SL3","K_SL7_SL3"],"MANA":["MAN_A"],"W1":["W1"]}
assert sorted(sum(GRUPPEN.values(), [])) == sorted(E_IDS), "Kanten ohne Gruppe"
GN = list(GRUPPEN); grp_of_edge = np.zeros(len(E_IDS), int)
for gi,g_ in enumerate(GN):
    for e in GRUPPEN[g_]: grp_of_edge[eidx[e]] = gi
SEKN = ["SL5M","SL5O","SL6S","SL7G","SL1G","SEC2","SEC3","SEC4","SUED"]
prior_w = np.array([pa[s] for s in SEKN]); prior_w = prior_w/prior_w.sum()
REFS = [nidx["HKW"], nidx["AVA"]]                      # Druckrandknoten: HKW (Verbund) und Ost-Sammler (AVA-Übernahme); Ost-Erzeuger regeln Δp, Massenstrom stellt sich ein
FIXED = ["MANA","W1"]                                    # ohne Messinformation: Multiplikator fest 1,0
GN_FREE = [g_ for g_ in GN if g_ not in FIXED]; NGF = len(GN_FREE)
OST_ST = ["Am_MIttleren_Moos","Hans-Boeckler-Str","Kurt-Schumacher-Str","KUKA","AVA_REF_VL","AVA_REF_RL"]   # Ost-Stationen: Schwachlast-Höhe strömungsbeeinflusst -> Offset kalibriert
EXCL_CAL = ["Beethovenpark"]                              # Ende einer 2,9 km langen DN150-Stichleitung: im Sektionsmodell nicht abbildbar (nur Validierung, informativ)
def unpack(theta):
    k = np.ones(len(GN)); k[[GN.index(g_) for g_ in GN_FREE]] = np.exp(theta[:NGF])
    u = theta[NGF:NGF+len(SEKN)]; w = prior_w*np.exp(u); w = w/w.sum()
    dz = dict(zip(OST_ST, theta[NGF+len(SEKN):]))
    return k, w, dz

# --- Stationen für den Druckvergleich (Verbund, ohne West/Pumpen/GT)
zHKW = 473.77
plan_z = {"AVA":466.69, "BMHKW":467.08, "Heizwerk_Sued":490.51, "HKW":zHKW}
zmap = g.set_index("station").dz_m
MESS = []
for stn in sorted(set(vl_c)|set(rl_c)):
    knt = zn.get(_norm(stn), (None,None))[0] if stn != "HKW" else "HKW"
    if knt is None or knt not in nidx or stn in ("GT-Ost","HKW","AVA","PPS","PSW","HWW") or kn2.set_index("node_id").netz.get(knt) == "West": continue
    zst = plan_z.get(stn, zHKW + zmap.get(stn, np.nan))
    if not np.isfinite(zst): continue
    MESS.append(dict(station=stn, node=nidx[knt], z=zst, vl=vl_c.get(stn), rl=rl_c.get(stn), sicherheit=zuord.get(stn, ("",""))[1] if stn in zuord else "hoch"))
MESS = pd.DataFrame(MESS)
print(len(MESS), "Vergleichsstationen:", MESS.station.tolist())

# --- Randbedingungen je Stunde
def rand_bed():
    T_vl = acron.HKW_Temp_VL.clip(60,125); T_rl = acron.HKW_Temp_RL.clip(35,85)
    rho_v = rho_w(T_vl)/3600.0
    m_hkw = sum(acron[f"HKW_Durchfluss_SL{i}"] for i in (1,2,3,4))*rho_v
    m_ava = abg.AVA_V_m3h*rho_v
    gt_dT = (acron["GT-Ost_Temp_VL"]-acron["GT-Ost_Temp_RL"]).where(lambda x: x>5)
    m_gt  = (acron["GT-Ost_Waermeleistung"]/(K_MW*gt_dT)).fillna(0.0).clip(lower=0)*rho_v
    m_bm  = acron.BMHKW_Durchfluss*rho_v
    m_hws = acron.Heizwerk_Sued_Durchfluss*rho_v
    m_wue = acron.HWW_Durchfluss_Primeinsp.mask(prim_spike)*rho_v
    m_man = acron.MAN_Durchfluss_Station*rho_v
    bc = pd.DataFrame(dict(T_vl=T_vl, T_rl=T_rl, m_hkw=m_hkw, m_ava=m_ava, m_gt=m_gt, m_bm=m_bm, m_hws=m_hws, m_wue=m_wue, m_man=m_man,
                           pVL_hkw=acron.HKW_Druck_VL, pRL_hkw=acron.HKW_Druck_RL, pVL_ava=acron.AVA_Druck_VL, pRL_ava=acron.AVA_Druck_RL, Ta=Ta))
    bc["R"] = bc.m_hkw + bc.m_ava + bc.m_gt + bc.m_bm + bc.m_hws - bc.m_wue - bc.m_man
    # Druckgewinn der Pumpstationen (Kanten SL4c bzw. W1), nur zur Kontrolle: Median ≈ 0
    bc["gain_pss_bar"] = (acron.PPS_Druck_VL_nach_Pumpe - acron.PPS_Druck_VL_vor_Pumpe).fillna(0.0).clip(lower=0)
    bc["gain_psw_bar"] = (acron.PSW_DiffDruck_VL_nach_Pumpe - acron.PSW_DiffDruck_VL_vor_Pumpe).fillna(0.0).clip(lower=0)
    need = ["T_vl","T_rl","m_hkw","m_ava","m_gt","m_bm","m_hws","m_wue","m_man","pVL_hkw","pRL_hkw","pVL_ava","pRL_ava"]
    bc["valid"] = bc[need].notna().all(axis=1) & (bc.R > 2.0)
    return bc
BC = rand_bed()
print(f"Stunden mit vollständigen Randbedingungen: {int(BC.valid.sum())} von {len(BC)}; Restlast R: Median {BC.R[BC.valid].median():.0f} kg/s, max {BC.R[BC.valid].max():.0f} kg/s")

### 12.4 Simulationsfunktion (Vorlauf- und Rücklaufnetz)

In [ ]:
def simulate(theta, hours, rough=1e-4, with_flows=True):
    k_g, w, dz = unpack(theta)
    bc = BC.loc[hours]; B = len(bc)
    mult = k_g[grp_of_edge]
    inj = np.zeros((B, len(NODES)))
    inj[:, nidx["GT"]] = bc.m_gt; inj[:, nidx["BM"]] = bc.m_bm; inj[:, nidx["HWS"]] = bc.m_hws        # HKW und AVA: Druckrandknoten (Einspeisung ergibt sich)
    inj[:, nidx["HWW"]] = -bc.m_wue; inj[:, nidx["MAN"]] = -bc.m_man
    for s_, w_s in zip(SEKN, w): inj[:, nidx[s_]] -= w_s*bc.R.values
    rho_v = rho_w(bc.T_vl.values)[:, None]; rho_r = rho_w(bc.T_rl.values)[:, None]
    zref = zHKW
    zava_v = plan_z["AVA"] + dz.get("AVA_REF_VL", 0.0); zava_r = plan_z["AVA"] + dz.get("AVA_REF_RL", 0.0)     # effektive Offsets am Ost-Sammler (Pumpen/Sensorhöhe)
    Hv_ref = np.c_[zref + bc.pVL_hkw.values*1e5/(rho_v[:,0]*G), zava_v + bc.pVL_ava.values*1e5/(rho_v[:,0]*G)]
    Hr_ref = np.c_[zref + bc.pRL_hkw.values*1e5/(rho_r[:,0]*G), zava_r + bc.pRL_ava.values*1e5/(rho_r[:,0]*G)]
    gain = np.zeros((B, len(E_IDS)))
    gain[:, eidx["SL4c"]] = bc.gain_pss_bar.values*1e5/(rho_v[:,0]*G); gain[:, eidx["W1"]] = bc.gain_psw_bar.values*1e5/(rho_v[:,0]*G)
    Hv, mv, itv, resv, balv = solve_network(Ainc, inj, L_E, D_E, mult, bc.T_vl.values[:, None], REFS, Hv_ref, gain, rough)
    Hr, mr, itr, resr, balr = solve_network(Ainc, -inj, L_E, D_E, mult, bc.T_rl.values[:, None], REFS, Hr_ref, None, rough)
    ok = (balv < 0.05) & (resv < 0.05) & (balr < 0.05) & (resr < 0.05)
    return dict(Hv=Hv, Hr=Hr, mv=mv, mr=mr, rho_v=rho_v[:,0], rho_r=rho_r[:,0], ok=ok, w=w, k=k_g, dz=dz, it=(itv, itr), inj_all=mv @ Ainc.T)

def station_pressures(sim):
    pv = np.full((len(sim["Hv"]), len(MESS)), np.nan); pr = pv.copy()
    for j,r in MESS.iterrows():
        zz = r.z + sim["dz"].get(r.station, 0.0)
        pv[:, j] = sim["rho_v"]*G*(sim["Hv"][:, r.node]-zz)/1e5; pr[:, j] = sim["rho_r"]*G*(sim["Hr"][:, r.node]-zz)/1e5
    return pv, pr

theta0 = np.zeros(NGF + len(SEKN) + len(OST_ST))
hrs_test = BC.index[BC.valid][:50:5]
t0_ = time.time(); s0 = simulate(theta0, hrs_test); dt_ = time.time()-t0_
pv0, pr0 = station_pressures(s0)
print(f"Testlauf mit Startparametern: {len(hrs_test)} Stunden in {dt_:.2f} s, konvergiert: {s0['ok'].mean():.0%}, Iterationen VL/RL: {s0['it']}")
# Konsistenztests: Massenerhaltung an der HKW-Sammelschiene, Vorlauf > Rücklauf, Druckvorzeichen
assert np.all(np.isfinite(pv0))
mb = s0["inj_all"].sum(axis=1)
dh = (s0["inj_all"][:, nidx["HKW"]] - BC.loc[hrs_test, "m_hkw"].values)
print(f"Gesamt-Massenbilanz Σ Einspeisung (inkl. Druckrandknoten): max. {np.abs(mb).max():.2e} kg/s | HKW-Einspeisung Modell − Messung (Startparameter): Median {np.median(dh):+.0f} kg/s | Vorlaufdruck > Rücklaufdruck an allen Stationen: {bool((pv0 > pr0).all())}")
assert np.abs(mb).max() < 1e-2

## 13. Kalibrierung
**Zeitliche Trennung:** Kalibrierung mit Stunden vom 01.01. bis 30.09.2025 (Stichprobe, geschichtet nach Last, inkl. Spitzenlast Februar), Validierung mit 01.10. bis 31.12.2025 (Kap. 14). Kalibriert werden 15 Leitungsgruppen-Multiplikatoren $k_g$ und 9 Lastanteile $w_s$. Höhen bleiben auf den abgeleiteten Werten (Sensitivität in Kap. 21). Ergebnisse werden im Ergebnisordner zwischengespeichert (`kalibrierung.json`, deterministisch).

In [ ]:
from scipy.optimize import least_squares
import hashlib
SIG_P, SIG_V = 0.3, 25.0                       # bar, m3/h (Messunsicherheit inkl. Höhenfehler)
def sample_hours(t0, t1, n_reg, n_top, seed=SEED):
    ok = BC.index[BC.valid & (BC.index >= t0) & (BC.index <= t1)]
    rng = np.random.default_rng(seed)
    reg = rng.choice(ok, size=min(n_reg, len(ok)), replace=False)
    top = BC.loc[ok].R.nlargest(n_top).index
    return pd.DatetimeIndex(sorted(set(reg) | set(top)))
H_CAL = sample_hours("2025-01-01", "2025-09-30 23:00", 260, 40)
print(f"Kalibrierstichprobe: {len(H_CAL)} Stunden (Restlast {BC.R[H_CAL].min():.0f}…{BC.R[H_CAL].max():.0f} kg/s)")

SL_MAP = {"SL1":"SL1a","SL2":"SL2","SL3":"SL3","SL4":"SL4a"}        # Messung HKW_Durchfluss_SLx ↔ Kanten ab HKW
def residuals(theta, hours, detail=False, for_cal=False):
    sim = simulate(theta, hours); pv, pr = station_pressures(sim); ok = sim["ok"]
    out = []; recs = []
    for j,r in MESS.iterrows():
        if for_cal and r.station in EXCL_CAL: continue
        for kind, mod, col in (("VL", pv[:, j], r.vl), ("RL", pr[:, j], r.rl)):
            if col is None: continue
            meas = acron.loc[hours, col].values
            e = (mod - meas)
            e = np.where(np.isfinite(e) & ok, e, np.nan)
            out.append(e/SIG_P)
            if detail: recs.append(pd.DataFrame(dict(zeit=hours, station=r.station, art="p_"+kind, modell=mod, messung=meas, ok=ok)))
        if detail and r.vl is not None and r.rl is not None:
            recs.append(pd.DataFrame(dict(zeit=hours, station=r.station, art="dp_Kunde", modell=pv[:, j]-pr[:, j], messung=acron.loc[hours, r.vl].values-acron.loc[hours, r.rl].values, ok=ok)))
    for sl, ed in SL_MAP.items():
        mod = sim["mv"][:, eidx[ed]]/sim["rho_v"]*3600.0; meas = acron.loc[hours, f"HKW_Durchfluss_{sl}"].values
        e = np.where(np.isfinite(meas) & ok, mod-meas, np.nan); out.append(e/SIG_V)
        if detail: recs.append(pd.DataFrame(dict(zeit=hours, station=f"HKW_{sl}", art="V_m3h", modell=mod, messung=meas, ok=ok)))
    rv = sim["rho_v"]
    for nm, node, col in (("HKW_netto", "HKW", "m_hkw"), ("AVA_V", "AVA", "m_ava")):
        mod = sim["inj_all"][:, nidx[node]]/rv*3600.0; meas = BC.loc[hours, col].values/rv*3600.0
        e = np.where(np.isfinite(meas) & ok, mod-meas, np.nan); out.append(e/SIG_V)
        if detail: recs.append(pd.DataFrame(dict(zeit=hours, station=nm, art="V_m3h", modell=mod, messung=meas, ok=ok)))
    if detail: return pd.concat(recs, ignore_index=True), sim
    return np.concatenate(out)

def objective(theta):
    r = np.nan_to_num(residuals(theta, H_CAL, for_cal=True), nan=0.0)          # feste Vektorlänge (nicht konvergierte Stunden → 0)
    ridge = np.r_[theta[:NGF]/0.7, theta[NGF:NGF+len(SEKN)]/0.5, theta[NGF+len(SEKN):]/5.0]*0.05*np.sqrt(len(r))
    return np.r_[r, ridge]

MESH = ["M12","M23","M24","M34","LAND"]                                   # Verbundkanten bündeln mehrere parallele Leitungen: Multiplikator bis 0,02 zulässig
OSTS = ["SL5M","SL5O","SL6S","SL7G","SL1G"]
lo = np.r_[[np.log(0.02) if g_ in MESH else np.log(0.2) for g_ in GN_FREE], [-0.9 if s_ in OSTS else -1.6 for s_ in SEKN], np.array([-25.0 if 'AVA_REF' in s_ else -10.0 for s_ in OST_ST])]
hi = np.r_[np.full(NGF, np.log(5.0)), [1.0 if s_ in OSTS else 0.7 for s_ in SEKN], np.array([25.0 if 'AVA_REF' in s_ else 10.0 for s_ in OST_ST])]
sig_in = hashlib.sha1(("v5" + str(len(H_CAL)) + str(H_CAL[0]) + str(H_CAL[-1]) + str(L_E.sum()) + str(SIG_P) + str(SEED)).encode()).hexdigest()[:12]
cache = RESULTS/"kalibrierung.json"
theta_cal = None
if cache.exists():
    cj = json.loads(cache.read_text())
    if cj.get("signatur") == sig_in: theta_cal = np.array(cj["theta"]); print("Kalibrierung aus Zwischenspeicher geladen (Signatur", sig_in, ")")
if theta_cal is None:
    t0_ = time.time()
    t1_ = time.time(); r0 = objective(theta0); print(f"Startwert: Summe der Fehlerquadrate {np.sum(r0**2):.0f}; Rechenzeit je Auswertung {time.time()-t1_:.1f} s")
    sol = least_squares(objective, theta0, bounds=(lo, hi), method="trf", x_scale=1.0, diff_step=1e-3, max_nfev=60, verbose=0)
    theta_cal = sol.x; print(f"Optimierung beendet: Status {sol.status} ({sol.message}), Auswertungen {sol.nfev}, Zeit {time.time()-t0_:.0f} s, Kosten {sol.cost:.0f}")
    cache.write_text(json.dumps(dict(signatur=sig_in, theta=list(map(float, theta_cal)), gruppen=GN_FREE, sektionen=SEKN, ost_stationen=OST_ST, cost=float(sol.cost))))

In [ ]:
k_cal, w_cal, dz_cal = unpack(theta_cal)
free_idx = [GN.index(g_) for g_ in GN_FREE]
at_b = np.zeros(len(GN), bool); at_b[free_idx] = np.isclose(theta_cal[:NGF], lo[:NGF], atol=1e-3) | np.isclose(theta_cal[:NGF], hi[:NGF], atol=1e-3)
par = pd.DataFrame({"Leitungsgruppe":GN, "k_g":k_cal.round(2), "Status":["fest (keine Messinformation)" if g_ in FIXED else ("an Grenze" if at_b[i] else "kalibriert") for i,g_ in enumerate(GN)]})
par["L_Modell_km"] = [L_E[grp_of_edge==i].sum()/1000 for i in range(len(GN))]; par["L_effektiv_km"] = par.L_Modell_km*par.k_g
display(par.round(2))
shares = pd.DataFrame({"Sektion":SEKN, "Prior (Anschlussleistung)":prior_w.round(3), "kalibriert":w_cal.round(3)}); shares["Last_Winter_MW"] = (shares.kalibriert*last_v[w_].median()).round(1)
shares["Anschlussleistung_MW"] = [pa[s_] for s_ in SEKN]; shares["Auslastung_Winter"] = (shares.Last_Winter_MW/shares.Anschlussleistung_MW).round(2); display(shares)
print("Höhen-Offsets der Ost-Stationen und effektive Offsets am Ost-Sammler (VL/RL) [m]:", {k_: round(float(v_),1) for k_,v_ in dz_cal.items()})
rms0 = np.sqrt(np.nanmean(residuals(theta0, H_CAL, for_cal=True)**2)); rms1 = np.sqrt(np.nanmean(residuals(theta_cal, H_CAL, for_cal=True)**2))
print(f"RMS der normierten Residuen (Kalibrierstichprobe): Start {rms0:.2f} → kalibriert {rms1:.2f}  (1,0 = Fehler entspricht der angesetzten Messunsicherheit)")
par.to_csv(RESULTS/"kalibrierung_gruppen.csv", index=False); shares.to_csv(RESULTS/"kalibrierung_lastanteile.csv", index=False)

In [ ]:
# Identifizierbarkeit: Jacobi-Matrix an der Lösung (Finite Differenzen), Standardfehler aus (J^T J)^-1
def jac_fd(theta, hours, h=1e-3):
    r0_ = residuals(theta, hours, for_cal=True); J = []
    for i in range(len(theta)):
        t_ = theta.copy(); t_[i] += h; J.append((residuals(t_, hours, for_cal=True)-r0_)/h)
    J = np.array(J).T; m_ = np.isfinite(J).all(axis=1) & np.isfinite(r0_); return J[m_], r0_[m_]
J_, r_ = jac_fd(theta_cal, H_CAL[::2])
sv = np.linalg.svd(J_/np.sqrt(len(J_)), compute_uv=False)
cov = np.linalg.pinv(J_.T@J_)*np.mean(r_**2); se = np.sqrt(np.diag(cov))
ident = pd.DataFrame({"Parameter":GN_FREE+["u_"+s_ for s_ in SEKN]+["dz_"+s_ for s_ in OST_ST], "Wert (ln k / u / dz)":theta_cal.round(2), "Standardfehler":se.round(2), "Sensitivität |J|":np.linalg.norm(J_, axis=0).round(0)})
ident["Bewertung"] = np.where(ident.Standardfehler > 0.7, "schwach bestimmt", np.where(ident.Standardfehler > 0.3, "mäßig", "gut bestimmt"))
display(ident)
print(f"Konditionszahl der skalierten Jacobi-Matrix: {sv[0]/sv[-1]:.0f}; kleinster Singulärwert {sv[-1]:.3f}")
ident.to_csv(RESULTS/"kalibrierung_identifizierbarkeit.csv", index=False)

## 14. Validierung an Messdaten (getrennter Zeitraum 01.10.–31.12.2025)
Bewertet werden Vorlaufdruck, Rücklaufdruck und die Stammleitungsflüsse am HKW. Kennzahlen je Messstelle: mittlerer Fehler (Bias), MAE, RMSE, relativer Fehler, 95 %-Quantil des Absolutfehlers, Korrelation und Anzahl gültiger Vergleichswerte. Temperaturen, Differenzdruck und Wärmeleistung der Kundenstationen werden nach Einbau der thermischen Simulation (Etappe 4) validiert.

In [ ]:
H_VAL = sample_hours("2025-10-01", "2025-12-31 23:00", 300, 40)
ext_days = [d for d in ("2025-11-22","2025-12-31") if len(BC.loc[d]) > 0]
H_EXT = pd.DatetimeIndex([t for t in BC.index[BC.valid] if str(t.date()) in ext_days])
H_VAL = pd.DatetimeIndex(sorted(set(H_VAL) | set(H_EXT)))
recs, simv = residuals(theta_cal, H_VAL, detail=True); recs = recs[recs.ok & recs.messung.notna() & recs.modell.notna()].copy()
recs["fehler"] = recs.modell - recs.messung
def kenn(g_):
    e = g_.fehler; return pd.Series(dict(n=len(g_), Bias=e.mean(), MAE=e.abs().mean(), RMSE=np.sqrt((e**2).mean()), rel_pct=100*e.abs().mean()/max(abs(g_.messung.mean()),1e-9),
                                      P95_abs=e.abs().quantile(.95), r=g_.modell.corr(g_.messung)))
val = recs.groupby(["art","station"]).apply(kenn).round(2)
val.to_csv(RESULTS/"validierung_kennzahlen.csv")
print(f"Validierungsstichprobe: {len(H_VAL)} Stunden, davon {len(H_EXT)} an den Extremtagen {ext_days}; konvergiert: {simv['ok'].mean():.0%}")
for art in ("p_VL","p_RL","dp_Kunde","V_m3h"):
    v_ = val.loc[art]; einh = "bar" if art.startswith("p") else "m³/h"
    print(f"\n{art} [{einh}]  – Gesamt: RMSE {np.sqrt((recs[recs.art==art].fehler**2).mean()):.2f}, MAE {recs[recs.art==art].fehler.abs().mean():.2f}, Bias {recs[recs.art==art].fehler.mean():+.2f}, r = {recs[recs.art==art].modell.corr(recs[recs.art==art].messung):.2f}")
    display(v_.sort_values("RMSE", ascending=False))

In [ ]:
# Kalibrier- gegenüber Validierungsfehler (Überanpassung?) und Abbildungen
rc, _ = residuals(theta_cal, H_CAL, detail=True); rc = rc[rc.ok & rc.messung.notna()].copy(); rc["fehler"] = rc.modell-rc.messung
cmp_ = pd.DataFrame({"Kalibrierung":rc.groupby("art").fehler.apply(lambda e: np.sqrt((e**2).mean())), "Validierung":recs.groupby("art").fehler.apply(lambda e: np.sqrt((e**2).mean()))}).round(2)
cmp_["Einheit"] = ["m³/h" if i.startswith("V") else "bar" for i in cmp_.index]; display(cmp_)
print("Anteil der Modellstunden mit Vorlaufdruck > Rücklaufdruck (alle Stationen):", f"{(recs[recs.art=='dp_Kunde'].modell>0).mean():.1%}", "| gemessen:", f"{(recs[recs.art=='dp_Kunde'].messung>0).mean():.1%}")
fig, ax = plt.subplots(1, 3, figsize=(15,4.6))
for a_, art, tit in zip(ax, ("p_VL","p_RL","V_m3h"), ("Vorlaufdruck [bar]","Rücklaufdruck [bar]","Stammleitungsfluss HKW [m³/h]")):
    d_ = recs[recs.art==art]; a_.scatter(d_.messung, d_.modell, s=6, alpha=.35, color=COL["mess"]); lim = [min(d_.messung.min(),d_.modell.min()), max(d_.messung.max(),d_.modell.max())]
    a_.plot(lim, lim, "k--", lw=1); a_.set(xlabel="Messung", ylabel="Modell", title=tit)
fig.suptitle("Validierung 01.10.–31.12.2025: Modell gegen Messung"); fig.tight_layout(); fig.savefig(FIG_DIR/"14_validierung_streu.png"); plt.show()
# Residuen über Last und Außentemperatur
rr = recs[recs.art=="p_VL"].merge(BC[["R","Ta"]], left_on="zeit", right_index=True)
fig, ax = plt.subplots(1, 2, figsize=(12,4.2))
ax[0].scatter(rr.R, rr.fehler, s=6, alpha=.3, color=COL["vl"]); ax[0].axhline(0, color="k", lw=.8); ax[0].set(xlabel="Restlast R [kg/s]", ylabel="Fehler Vorlaufdruck [bar]", title="Residuen über der Last")
ax[1].scatter(rr.Ta, rr.fehler, s=6, alpha=.3, color=COL["rl"]); ax[1].axhline(0, color="k", lw=.8); ax[1].set(xlabel="Außentemperatur [°C]", ylabel="Fehler Vorlaufdruck [bar]", title="Residuen über der Außentemperatur")
fig.tight_layout(); fig.savefig(FIG_DIR/"14_validierung_residuen.png"); plt.show()
# Extremtage: Zeitreihe MAN und schlechteste Stationen
bad = val.loc["p_VL"].RMSE.sort_values(ascending=False).head(5); print("Am schlechtesten erklärte Vorlaufdruck-Stationen (RMSE, bar):", bad.round(2).to_dict())

In [ ]:
# Aussagegüte je Region (aus den Validierungsfehlern)
grp = {"Ost (Druckregelung durch Erzeuger)":["Am_MIttleren_Moos","BMHKW","Kurt-Schumacher-Str","Hans-Boeckler-Str","KUKA"], "Stichleitungsende (nicht abbildbar)":["Beethovenpark"]}
def region(st_):
    for k_, v_ in grp.items():
        if st_ in v_: return k_
    return "Mitte/Süd inkl. MAN"
recs["region"] = recs.station.map(region)
tab = recs[recs.art.isin(["p_VL","p_RL","dp_Kunde"])].groupby(["region","art"]).fehler.apply(lambda e: np.sqrt((e**2).mean())).unstack().round(2)[["p_VL","p_RL","dp_Kunde"]]
def guete(v): return "gut (≤ 0,35 bar)" if v <= 0.35 else ("mäßig (≤ 0,7 bar)" if v <= 0.7 else "schwach (> 0,7 bar)")
tab["Aussagegüte Vorlaufdruck"] = tab.p_VL.map(guete); display(tab)
man = val.loc["p_VL"].loc["MAN"]
print(f"MAN (Speicherstandort): p_VL RMSE {man.RMSE:.2f} bar, P95 {man.P95_abs:.2f} bar, r = {man.r:.2f}; Auslegungs-Unsicherheit für Druckaussagen dort: ±{man.P95_abs:.1f} bar (95 %).")

### 14.1 Bewertung der Validierung
* **Belastbar:** Vorlauf- und Rücklaufdruck in Mitte/Süd einschließlich **MAN** (Vorlaufdruck RMSE ≈ 0,07 bar, r ≈ 0,99) werden gut reproduziert; Kalibrier- und Validierungsfehler liegen nahe beieinander (keine wesentliche Überanpassung, Vorlaufdruck 0,60 → 0,58 bar über alle Stationen).
* **Eingeschränkt belastbar:** Der Ost-Bereich (SL5–SL7, Ost-Erzeuger) wird nur mäßig bis schwach erklärt (Vorlaufdruck RMSE ≈ 0,8 bar, Differenzdruck RMSE ≈ 1,7 bar). Ursache: Die Ost-Erzeuger regeln den Differenzdruck mit Rücklaufpumpen; das Sektionsmodell bildet diese Regelung nur über zwei effektive Höhenoffsets am Ost-Sammler ab (mit gemessener Geometrie nur noch ≈ −0,3 m Vorlauf, +7,7 m Rücklauf). Aussagen zu Drücken und Flussverteilung im Osten haben deshalb eine Unsicherheit von etwa ±1 bar; sie werden in den Szenarien als Sensitivität geführt.
* **Nicht abbildbar:** Stichleitungsenden wie Beethovenpark (2,9 km DN150) – der Fehler ist dort lokal und wird in den Szenarien nicht bewertet.
* **Flüsse:** Die Stammleitungsflüsse am HKW werden mit RMSE ≈ 55–110 m³/h getroffen, die Einspeisung am HKW (r = 0,89) mit Bias −74 m³/h. Die Verteilung der Last auf die Sektionen ist **nicht eindeutig identifizierbar** (Standardfehler der Lastanteile groß, Kap. 13); die Auslastung der Innenstadt ergibt sich zu ≈ 0,17–0,18, die der Ost-/STAWA-Sektionen deutlich höher.
* **Grenzen der Aussage:** Schaltzustände 2025 unbekannt; Höhen der Stationen ±3–5 m (≙ ±0,3–0,5 bar); Kundenlasten der Sektionen geschätzt.
* **Konsequenz für die Speicherstudie:** Wirkungen des Speichers werden als **Differenz gegenüber dem Referenzfall** ausgewiesen (systematische Modellfehler heben sich weitgehend auf); absolute Druckniveaus im Osten sind nicht belastbar. Aussagen für MAN, Innenstadt und Süd sind mit ±0,2–0,3 bar belastbar.

## 15. Definition der Speicheranbindung

### 15.1 Vorgehen und Abgrenzung
Der Speicher ist ein **direkt eingebundener Druckspeicher** ohne Wärmeübertrager. Oben (Kopf) liegt das heiße Wasser (bis 120 °C), unten (Fuß) das kalte Rücklaufwasser; der Kopf ist mit dem lokalen **Vorlauf**, der Fuß mit dem lokalen **Rücklauf** am Knoten MAN (T-Abzweig der SL5, Arm. 342.x) verbunden. Eine Einbindungsplanung liegt in den Unterlagen **nicht** vor; deshalb werden zwei gekennzeichnete Varianten untersucht:

| Variante | Einbindung | Laden | Entladen |
|---|---|---|---|
| **A** | Regelventile, **keine** aktive Druckerhöhung | über Netz-Δp (Vorlauf → Kopf, Fuß → Rücklauf), Regelventil drosselt | **hydraulisch nicht möglich** (siehe Gl. 15.1) |
| **B** | regelbare Lade-/Entladepumpe (reversibel oder getrennte Stränge) | Pumpe nur bei zu kleinem Netz-Δp | Pumpe hebt von der Rücklauf- auf die Vorlaufdruckhöhe |

**Gl. 15.1 (Entladen).** Beim Entladen muss Wasser aus dem Speicher in den Vorlauf strömen (Speicherdruck ≥ p_VL,MAN) und gleichzeitig Rücklaufwasser in den Speicher fließen (Speicherdruck ≤ p_RL,MAN). Da p_VL > p_RL, ist das ohne Pumpe ein Widerspruch. Die erforderliche Förderhöhe ist
$$H_{P} = (H_{VL}-H_{RL})_{MAN} + h_{L,VL}(\dot m) + h_{L,RL}(\dot m),\qquad P_{el} = \frac{\dot m\, g\, H_P}{\eta_P}$$
**Gl. 15.2 (Laden ohne Pumpe).** Laden ist möglich, solange $(H_{VL}-H_{RL})_{MAN} - h_{L,VL} - h_{L,RL} \ge \Delta p_{Ventil,min}/(\rho g)$.

Die Wirkung eines Speichers wird **nicht** pauschal als „druckstabilisierend“ bezeichnet. Sie wird über folgende Mechanismen getrennt ausgewiesen (Kap. 18.2): lokale Wärmeeinspeisung, reduzierter Massenstrom auf vorgelagerten Leitungen, geringere Druckverluste, aktive Förderhöhe der Speicherpumpe, geänderte Rücklauftemperatur, geänderter Erzeuger-/Pumpeneinsatz, zeitliche Lastverschiebung.

**Modellierung im Netz.** Der Speicher wirkt im Netzmodell als Massenstrom-Einspeisung $\dot m_{S}$ (>0 Entladen) im Vorlaufknoten MAN und gleich große Entnahme im Rücklaufknoten MAN (Laden: umgekehrt). Der Netzlöser (Kap. 12.2) bleibt unverändert; die Speicherpumpe ergibt sich aus den Druckhöhen beider Knoten (Gl. 15.1).

### 15.2 Simulationsengine für Szenarien (Erweiterung von Kap. 12.4)
`solve_case` löst Vorlauf und Rücklauf für beliebige Randbedingungen (Erzeugerdrücke, Lasten je Sektion, Speicherstrom, geschlossene Kanten, Multiplikatoren, Pumpenskalierung, Wegfall des AVA-Druckrandes). Test: Die Engine reproduziert exakt die kalibrierte Simulation.

In [ ]:
# ---------------------------------------------------------------- Knotenhöhen (auch für Verzweigungsknoten ohne Messung)
Z_N = kn2.set_index("node_id").z_modell_mNN.reindex(NODES).astype(float)
Z_FILL = Z_N.isna()
for _ in range(20):                                                      # fehlende Höhen: Mittel der Nachbarknoten (iterativ)
    for n in NODES:
        if np.isnan(Z_N[n]):
            nb = [ka2.nach.iloc[j] if ka2.von.iloc[j] == n else ka2.von.iloc[j] for j in range(len(ka2)) if n in (ka2.von.iloc[j], ka2.nach.iloc[j])]
            zz = [Z_N[x] for x in nb if x in Z_N.index and not np.isnan(Z_N[x])]
            if zz: Z_N[n] = float(np.mean(zz))
Z_NODE = Z_N.values
KUNDEN = ["MAN","SL5M","SL5O","SL6S","SL7G","SL1G","SEC2","SEC3","SEC4","SUED","HWW"]      # Knoten mit Kundenanschlüssen (Δp-Kriterium)
KUNDEN_IDX = np.array([nidx[k] for k in KUNDEN])
print("Höhen der Verzweigungsknoten (interpoliert, ±5 m):", Z_N[Z_FILL].round(1).to_dict())

# ---------------------------------------------------------------- Speicherstandort (Standard: MAN); für die Standortvariation (Anhang A) umschaltbar
STOR_NODE = "MAN"
import contextlib
@contextlib.contextmanager
def with_stor_node(node):
    global STOR_NODE
    old = STOR_NODE; STOR_NODE = node
    try: yield
    finally: STOR_NODE = old

@contextlib.contextmanager
def with_theta(th):
    global theta_cal
    old = theta_cal; theta_cal = th
    try: yield
    finally: theta_cal = old

# ---------------------------------------------------------------- Fälle (Randbedingungen je Stunde)
CASE_COLS = ["T_vl","T_rl","Ta","pVL_hkw","pRL_hkw","pVL_ava","pRL_ava","m_gt","m_bm","m_hws","m_wue","m_man","R","gain_pss_bar","gain_psw_bar"]
def case_from_bc(hours):
    return BC.loc[hours, CASE_COLS].copy().reset_index(drop=True)

def solve_case(cs, theta=None, cfg=None, m_stor=None, Tsrc=None, rough=1e-4):
    """Löst Vorlauf- und Rücklaufnetz für alle Zeilen von cs (Batch).
    cs: DataFrame (CASE_COLS), optional Spalten 'w_<Sektion>' (Lastanteile), 'f_<Sektion>' (Lastfaktor je Sektion, Standard 1).
    m_stor: Speichermassenstrom am MAN-Knoten [kg/s], >0 Entladen (Einspeisung in den Vorlauf), <0 Laden.
    cfg: dict(closed=[Kanten], ava_ref=True, mult={Kante: Faktor}, pump_scale=1.0, k_scale={Gruppe: Faktor}, rough=…)."""
    cfg = dict(cfg or {}); theta = theta_cal if theta is None else theta
    k_g, w, dz = unpack(theta); B = len(cs)
    mult = k_g[grp_of_edge].copy()
    for gname, fac in cfg.get("k_scale", {}).items(): mult[[eidx[e] for e in GRUPPEN[gname]]] *= fac
    for e, fac in cfg.get("mult", {}).items(): mult[eidx[e]] *= fac
    keep = np.array([e not in cfg.get("closed", []) for e in E_IDS]); Ak = Ainc[:, keep]
    inj = np.zeros((B, len(NODES)))
    inj[:, nidx["GT"]] = cs.m_gt; inj[:, nidx["BM"]] = cs.m_bm; inj[:, nidx["HWS"]] = cs.m_hws
    inj[:, nidx["HWW"]] = -cs.m_wue; inj[:, nidx["MAN"]] = -cs.m_man
    for j, s_ in enumerate(SEKN):
        ws = cs[f"w_{s_}"].values if f"w_{s_}" in cs else w[j]
        fs = cs[f"f_{s_}"].values if f"f_{s_}" in cs else 1.0
        inj[:, nidx[s_]] -= ws*fs*cs.R.values
    if m_stor is not None: inj[:, nidx[STOR_NODE]] += np.asarray(m_stor, float)
    rho_v = rho_w(cs.T_vl.values); rho_r = rho_w(cs.T_rl.values)
    zava_v = plan_z["AVA"] + dz["AVA_REF_VL"]; zava_r = plan_z["AVA"] + dz["AVA_REF_RL"]
    Hv_ref = np.c_[zHKW + cs.pVL_hkw.values*1e5/(rho_v*G), zava_v + cs.pVL_ava.values*1e5/(rho_v*G)]
    Hr_ref = np.c_[zHKW + cs.pRL_hkw.values*1e5/(rho_r*G), zava_r + cs.pRL_ava.values*1e5/(rho_r*G)]
    ref = REFS
    if not cfg.get("ava_ref", True): ref = [nidx["HKW"]]; Hv_ref = Hv_ref[:, :1]; Hr_ref = Hr_ref[:, :1]; inj[:, nidx["AVA"]] = 0.0
    gain = np.zeros((B, len(E_IDS)))
    ps = cfg.get("pump_scale", 1.0)
    gain[:, eidx["SL4c"]] = ps*cs.gain_pss_bar.values*1e5/(rho_v*G); gain[:, eidx["W1"]] = ps*cs.gain_psw_bar.values*1e5/(rho_v*G)
    gk = gain[:, keep]
    Hv, mvk, itv, resv, balv = solve_network(Ak, inj, L_E[keep], D_E[keep], mult[keep], cs.T_vl.values[:, None], ref, Hv_ref, gk, cfg.get("rough", rough))
    Hr, mrk, itr, resr, balr = solve_network(Ak, -inj, L_E[keep], D_E[keep], mult[keep], cs.T_rl.values[:, None], ref, Hr_ref, None, cfg.get("rough", rough))
    mv = np.zeros((B, len(E_IDS))); mr = mv.copy(); mv[:, keep] = mvk; mr[:, keep] = mrk
    ok = (balv < 0.05) & (resv < 0.05) & (balr < 0.05) & (resr < 0.05)
    return dict(Hv=Hv, Hr=Hr, mv=mv, mr=mr, rho_v=rho_v, rho_r=rho_r, ok=ok, inj=inj, inj_all=mv @ Ainc.T, gain=gain, keep=keep, cs=cs, w=w, mult=mult, cfg=cfg)

def node_pressures(res):
    """Überdruck [bar] an allen Knoten (Vorlauf, Rücklauf) aus Druckhöhe und Knotenhöhe."""
    pv = res["rho_v"][:, None]*G*(res["Hv"] - Z_NODE[None, :])/1e5; pr = res["rho_r"][:, None]*G*(res["Hr"] - Z_NODE[None, :])/1e5
    return pv, pr

# ---------------------------------------------------------------- Test: Engine = simulate() (Kalibriersimulation)
hh_ = BC.index[BC.valid][100:400:30]; sim_ref = simulate(theta_cal, hh_); cs_ = case_from_bc(hh_); res_ = solve_case(cs_)
assert np.allclose(sim_ref["Hv"], res_["Hv"], atol=1e-6) and np.allclose(sim_ref["mr"], res_["mr"], atol=1e-6)
print(f"Test bestanden: Szenario-Engine reproduziert die kalibrierte Simulation aus Kap. 12.4 ({len(hh_)} Stunden, max. Δ Druckhöhe {np.abs(sim_ref['Hv']-res_['Hv']).max():.1e} m).")

### 15.3 Temperaturmodell (quasistationär)
Aufwind-Mischung an den Knoten und exponentieller Wärmeverlust je Rohr (Gl. 11.5):
$$T_{aus}=T_{B}+(T_{ein}-T_{B})\,e^{-U' L/(\dot m\, c_p)},\qquad T_{Knoten}=\frac{\sum \dot m_{ein}T_{aus}+\dot m_{Quelle}T_{Quelle}}{\sum \dot m_{ein}+\dot m_{Quelle}}$$
mit $U'=0{,}3$ W/(m K) je Strang (Framework-Wert), Bodentemperatur 10 °C (Stadtbach_topo.yaml). Es gibt nur die **Hauptleitungen** des Sektionsmodells, nicht die Verteilnetze; absolute Netzverluste sind daher unterschätzt. Das Modell ist **nicht dynamisch**: Speicherladezustand und Durchmischung im Behälter (Kap. 15.4) werden stundenweise quasistationär behandelt, Laufzeiten im Netz nicht.

In [ ]:
# ---------------------------------------------------------------- Quasistationäres Temperaturmodell (Aufwind-Mischung + Rohrwärmeverlust)
CP_J = 4190.0; U_LIN = 0.30; T_SOIL = 10.0             # W/(m K) Rohrpaar-Näherung je Strang (Framework-Wert), Bodentemperatur 10 °C (Stadtbach_topo.yaml)
E_VON = np.array([int(np.where(Ainc[:, j] == 1)[0][0]) for j in range(len(E_IDS))]); E_NACH = np.array([int(np.where(Ainc[:, j] == -1)[0][0]) for j in range(len(E_IDS))])

def transport_T(m_e, src, Tsrc, u_lin=U_LIN, t_soil=T_SOIL, iters=80):
    """Temperaturen an den Knoten (B,N) und Verlust je Kante [W]. m_e: (B,E) Massenstrom entlang von→nach; src: (B,N) Zufluss aus Quellen [kg/s] mit Temperatur Tsrc (B,N)."""
    B, N = src.shape; E = m_e.shape[1]; ar = np.arange(B)
    T = np.where(src > 0, Tsrc, np.nanmean(np.where(src > 0, Tsrc, np.nan), axis=1, keepdims=True))
    T = np.nan_to_num(T, nan=60.0)
    am = np.maximum(np.abs(m_e), 0.05); ULm = (u_lin*L_E)[None, :]/(CP_J*am)            # exp(-UL/(m cp))
    up = np.where(m_e >= 0, E_VON[None, :], E_NACH[None, :]); dn = np.where(m_e >= 0, E_NACH[None, :], E_VON[None, :])
    for _ in range(iters):
        num = src*Tsrc; den = src.copy(); Tout = np.empty((B, E))
        for e in range(E):
            Tu = T[ar, up[:, e]]; Tout[:, e] = t_soil + (Tu - t_soil)*np.exp(-ULm[:, e])
            np.add.at(num, (ar, dn[:, e]), np.abs(m_e[:, e])*Tout[:, e]); np.add.at(den, (ar, dn[:, e]), np.abs(m_e[:, e]))
        Tn = np.where(den > 1e-6, num/np.maximum(den, 1e-6), T)
        if np.max(np.abs(Tn - T)) < 1e-4: T = Tn; break
        T = Tn
    Tin = T[ar[:, None], up]; loss_W = (u_lin*L_E)[None, :]*(0.5*(Tin + Tout) - t_soil)*(np.abs(m_e) > 1e-3)
    return T, loss_W

def thermal(res, m_stor=None, T_h=None, Tsrc_v=None, Tsrc_r=None, u_lin=U_LIN):
    cs = res["cs"]; B = len(cs); inj_v = res["inj"]; ia = res["inj_all"]
    sv = np.zeros((B, len(NODES))); Tv0 = np.zeros((B, len(NODES)))
    for nd in ("HKW","AVA","GT","BM","HWS"): sv[:, nidx[nd]] = np.maximum(ia[:, nidx[nd]], 0.0); Tv0[:, nidx[nd]] = cs.T_vl.values if Tsrc_v is None else Tsrc_v[nd]
    ms = np.zeros(B) if m_stor is None else np.asarray(m_stor, float)
    sv[:, nidx[STOR_NODE]] += np.maximum(ms, 0.0); Tv0[:, nidx[STOR_NODE]] = np.where(ms > 0, (cs.T_vl.values if T_h is None else np.asarray(T_h, float)), Tv0[:, nidx[STOR_NODE]])
    Tv, lv = transport_T(res["mv"], sv, Tv0, u_lin)
    sr = np.zeros((B, len(NODES))); Tr0 = np.zeros((B, len(NODES)))
    for nd in NODES:
        if nd in ("HKW","AVA","GT","BM","HWS"): continue
        sr[:, nidx[nd]] = (-(inj_v[:, nidx[nd]] - ms) + np.maximum(-ms, 0.0)) if nd == STOR_NODE else np.maximum(-inj_v[:, nidx[nd]], 0.0)          # Rücklaufquelle = Kundenlast (+ Ladestrom am Speicherknoten)
        Tr0[:, nidx[nd]] = cs.T_rl.values if Tsrc_r is None else Tsrc_r
    Tr, lr = transport_T(res["mr"], sr, Tr0, u_lin)
    return dict(Tv=Tv, Tr=Tr, loss_MW=(lv.sum(1) + lr.sum(1))/1e6, loss_v_edge=lv, loss_r_edge=lr)

# ---------------------------------------------------------------- Plausibilitätsprüfung gegen gemessene Vorlauftemperaturen (Validierungszeitraum)
hv_ = pd.DatetimeIndex(H_VAL)[::max(1, len(H_VAL)//300)]; hv_ = hv_[BC.loc[hv_, "valid"].values]
cs_v = case_from_bc(hv_); res_v = solve_case(cs_v)
Ts = {"HKW": acron.loc[hv_, "HKW_Temp_VL"].values, "AVA": acron.loc[hv_, "AVA_Temp_VL"].fillna(acron.loc[hv_, "HKW_Temp_VL"]).values,
      "GT": acron.loc[hv_, "GT-Ost_Temp_VL"].fillna(acron.loc[hv_, "HKW_Temp_VL"]).values, "BM": acron.loc[hv_, "BMHKW_Temp_VL"].fillna(acron.loc[hv_, "HKW_Temp_VL"]).values,
      "HWS": acron.loc[hv_, "Heizwerk_Sued_Temp_VL"].fillna(acron.loc[hv_, "HKW_Temp_VL"]).values}
th_v = thermal(res_v, Tsrc_v=Ts)
rows = []
for j, r in MESS.iterrows():
    col = f"{r.station}_Temp_VL"
    if col not in acron.columns or r.station in ("BMHKW","Heizwerk_Sued"): continue
    meas = acron.loc[hv_, col].values; mod = th_v["Tv"][:, r.node]; m_ = np.isfinite(meas) & (meas > 40)
    if m_.sum() < 20: continue
    e = mod[m_] - meas[m_]; rows.append(dict(Station=r.station, Knoten=NODES[r.node], n=int(m_.sum()), Bias_K=e.mean(), RMSE_K=np.sqrt((e**2).mean()), r=np.corrcoef(mod[m_], meas[m_])[0, 1]))
tv_tab = pd.DataFrame(rows).round(2); display(tv_tab)
print(f"Vorlauftemperatur (Mischung + Rohrverlust, U' = {U_LIN} W/(m K)): mittlere Abweichung {tv_tab.Bias_K.mean():+.2f} K, RMSE (Mittel der Stationen) {tv_tab.RMSE_K.mean():.2f} K.")
print(f"Modellierte Netzwärmeverluste im Validierungszeitraum: {th_v['loss_MW'].mean():.1f} MW im Mittel (VL+RL, ohne Hausanschlüsse); Größenordnung typischer Fernwärmenetze 5–10 % der Einspeisung.")
tv_tab.to_csv(RESULTS/"thermik_validierung.csv", index=False)

### 15.4 Speicherparameter und hydraulische Machbarkeit der Anschlussvarianten
Zentrale Variante: **5 000 m³** (Projektwert der Topo-YAML), T_max 120 °C, **40 MW** Ent-/Ladeleistung, Anschluss **DN300** (= Kundenanschluss MAN_A), Anschlussstränge je 250 m (Annahme). Füllstandsgrenzen 5–95 %, nutzbarer Volumenanteil 85 % (Schichtung), Wandwärmeverlust über U = 0,15 W/(m² K). Die Speicherenergie folgt $E=\rho c_p V \Delta T \eta_{strat}$ (Test in Kap. 12.1: 5 000 m³ bei 10 K ≈ 57 MWh, nicht 84,5 MWh). Alle übrigen Größen (Leistung, Volumen, DN, Förderhöhe, Spreizung, Anfangsfüllstand) werden in Kap. 19.3 (S9) variiert.

In [ ]:
# ---------------------------------------------------------------- Speicherparameter (zentrale Variante; alle Werte Annahmen gemäß Katalog A-04/K-03/K-04)
SPS = dict(V_m3=5000.0,            # Projektwert aus Stadtbach_topo.yaml (Druckspeicher 5 000 m³, 10 bar)
           T_max_C=120.0,          # Auftraggeber: bis 120 °C im Winter
           h_tank_m=20.0,          # Behälterhöhe (Annahme; Zylinder H/D ≈ 1,7 → D ≈ 21 m)
           eta_strat=0.85,         # nutzbarer Anteil des Volumens (Temperaturschichtung/Mischzone), Sensitivität 0,70–0,95
           U_wall=0.15,            # W/(m² K) Wärmedämmung inkl. Dach/Boden (Annahme)
           Qd_max_MW=40.0, Qc_max_MW=40.0,          # Ent-/Ladeleistung (zentrale Variante, S9 variiert)
           soc_min=0.05, soc_max=0.95, soc0=0.90,   # Füllstandsgrenzen und Anfangsfüllstand
           dn_conn_mm=300.0, L_conn_m=250.0, zeta_conn=12.0,   # Anschluss je Strang (VL/RL): DN, Länge, Summe Einzelwiderstände (Regelventil, 2 Absperrarmaturen, Rückschlagklappe, Bögen)
           eta_pump=0.75, dp_valve_min_bar=0.5,      # Pumpenwirkungsgrad (Netz-Strom→Wasser); Mindest-Drosseldruck des Regelventils
           dp_kunde_min_bar=1.0, dT_boil=1.0)        # Kunden-Mindestdifferenzdruck (Annahme, WV640 Δp intern 0,8–1,3 bar), Siedeabstand p_abs − p_sat ≥ 1 bar
P_ATM = 1.013

def geom_tank(V, h):
    D = np.sqrt(4*V/(np.pi*h)); return D, np.pi*D*h + 2*np.pi*D**2/4       # Durchmesser, Oberfläche
D_t, A_t = geom_tank(SPS["V_m3"], SPS["h_tank_m"])
def storage_E_MWh(V, dT, T_mean=90.0, eta=None): return storage_capacity_MWh(V, dT, T_mean, SPS["eta_strat"] if eta is None else eta)
def storage_loss_MW(T_mean, T_amb=5.0, V=None):
    D, A = geom_tank(SPS["V_m3"] if V is None else V, SPS["h_tank_m"]); return SPS["U_wall"]*A*(T_mean - T_amb)/1e6

def conn_head_m(m, T, dn_mm=None, L=None, zeta=None):
    """Druckhöhenverlust [m] EINES Anschlussstrangs bei Massenstrom m [kg/s] (Reibung Darcy–Weisbach + Einzelverluste)."""
    dn = (SPS["dn_conn_mm"] if dn_mm is None else dn_mm)/1000.0; L = SPS["L_conn_m"] if L is None else L; zeta = SPS["zeta_conn"] if zeta is None else zeta
    m = np.abs(np.asarray(m, float)); rho = rho_w(T); A_ = np.pi*dn**2/4; v = m/(rho*A_)
    r = edge_resistance(np.maximum(m, 1e-3), L, dn, 1.0, T, 1e-4)
    return r*m*m + zeta*v**2/(2*G)
def conn_velocity(m, T, dn_mm=None): dn = (SPS["dn_conn_mm"] if dn_mm is None else dn_mm)/1000.0; return np.abs(m)/(rho_w(T)*np.pi*dn**2/4)

# Tests: Energie, Leistung–Massenstrom–Spreizung
E_ = storage_E_MWh(SPS["V_m3"], 62.0, 90.0); print(f"Nutzbare Speicherenergie {SPS['V_m3']:.0f} m³ (η_strat {SPS['eta_strat']}, ΔT 62 K): {E_:.0f} MWh = {E_/SPS['Qd_max_MW']:.1f} h bei {SPS['Qd_max_MW']:.0f} MW; Oberfläche {A_t:.0f} m², Wärmeverlust bei 100 °C mittlerer Temp. {storage_loss_MW(100.0)*1e3:.0f} kW ({100*storage_loss_MW(100.0)*24/E_:.2f} %/d).")
m_test = 40.0e3/(cp_w(90)*62.0); assert abs(m_test*cp_w(90)*62.0/1e3 - 40.0) < 1e-9
print(f"Anschluss DN{SPS['dn_conn_mm']:.0f}: 40 MW bei ΔT 62 K ⇒ {m_test:.0f} kg/s = {m_test/rho_w(90)*3600:.0f} m³/h, v = {conn_velocity(m_test, 90):.2f} m/s, Verlusthöhe je Strang {conn_head_m(m_test, 90):.1f} m ({conn_head_m(m_test, 90)*rho_w(90)*G/1e5:.2f} bar)")

# ---------------------------------------------------------------- Hydraulische Machbarkeit der Anschlussvarianten (gemessene Randbedingungen, S0-Stunden)
hrs_ = [pd.Timestamp("2025-02-14 09:00"), pd.Timestamp("2025-02-14 03:00")]; lab_ = ["Spitzenstunde 14.02.2025 09:00 (Q≈248 MW)", "Nachtstunde 14.02.2025 03:00 (Q≈184 MW)"]
m_grid = np.arange(-250.0, 251.0, 10.0)                      # >0 Entladen, <0 Laden
rows = []; DP_CURVE = {}
for h_, l_ in zip(hrs_, lab_):
    cs_ = pd.concat([case_from_bc([h_])]*len(m_grid), ignore_index=True); r_ = solve_case(cs_, m_stor=m_grid)
    iM = nidx["MAN"]; dHM = r_["Hv"][:, iM] - r_["Hr"][:, iM]; dpM = (r_["rho_v"]*G*(r_["Hv"][:, iM]-Z_NODE[iM]) - r_["rho_r"]*G*(r_["Hr"][:, iM]-Z_NODE[iM]))/1e5
    Tm = cs_.T_vl.values[0]; hl2 = 2*conn_head_m(m_grid, Tm)
    Hp_req = np.maximum(dHM + hl2, 0.0)                       # Entladen: Pumpe muss von RL-Druckhöhe auf VL-Druckhöhe heben + Verluste beider Stränge
    ch_ok = (dHM - hl2 - SPS["dp_valve_min_bar"]*1e5/(rho_w(Tm)*G)) >= 0                    # Laden ohne Pumpe: Netzdruck deckt Verluste + Regelventil
    DP_CURVE[l_] = pd.DataFrame(dict(m=m_grid, dpMAN=dpM, Hp=Hp_req, ok_charge=ch_ok, ok_ok=r_["ok"]))
    for m0 in (-150.0, -100.0, -50.0, 0.0, 50.0, 100.0, 150.0):
        i_ = int(np.argmin(np.abs(m_grid-m0))); Q_ = abs(m0)*cp_w(90)*62.0/1e3
        rows.append({"Fall": l_, "Massenstrom [kg/s]": m0, "≙ Leistung [MW]": round(Q_, 1), "Modus": "Entladen" if m0 > 0 else ("Laden" if m0 < 0 else "aus"),
                     "Δp am MAN-Knoten [bar]": round(dpM[i_], 2), "Verlust beider Stränge [bar]": round(hl2[i_]*rho_w(Tm)*G/1e5, 2),
                     "ohne Pumpe möglich": ("ja" if ch_ok[i_] else "nein") if m0 < 0 else ("–" if m0 == 0 else "nein (Pumpe nötig)"),
                     "erforderliche Pumpenförderhöhe [bar]": round(Hp_req[i_]*rho_w(Tm)*G/1e5, 2) if m0 > 0 else (0.0 if m0 <= 0 else np.nan)})
feas = pd.DataFrame(rows); display(feas)
mc_max = {l_: (d.m[d.ok_charge & (d.m < 0)].min() if (d.ok_charge & (d.m < 0)).any() else 0.0) for l_, d in DP_CURVE.items()}
for l_, v_ in mc_max.items(): print(f"Maximaler Ladestrom ohne Pumpe ({l_}): {abs(v_):.0f} kg/s ≈ {abs(v_)*cp_w(90)*62.0/1e3:.0f} MW")
feas.to_csv(RESULTS/"speicheranbindung_machbarkeit.csv", index=False)

## 16. Definition der Szenarien

### 16.1 Faire Vergleichsbedingungen und Basistag
Alle Szenarien werden **programmatisch aus einer zentralen Konfiguration** erzeugt (Tabelle in 16.3). Paarvergleiche (mit/ohne Speicher) verwenden **identische** Verbraucherlast, Außentemperatur, Heizkurve, Netzausbau, Erzeugerpark und Schaltzustand; dies wird durch den Test „S2 ≡ S1“ abgesichert. Abweichende Erzeuger-/Pumpenfahrweisen infolge der Optimierung werden in Kap. 18.2 explizit ausgewiesen.

* **Basistag** ist der 2025 beobachtete Tag mit der höchsten mittleren Wärmeleistung (14.02.2025, Ta ≈ −3,7 °C, Tagesmittel ≈ 215 MW, in der Größenordnung der Auslegungs-Tagesleistung der Lastextrapolation aus Kap. 8/10.7). Er liefert die stündlichen Lastprofile, Erzeuger GT-Ost/BMHKW/Heizwerk Süd, Westübergabe und MAN-Bezug.
* **Auslegungsbedingungen (S1 ff.):** Vorlauftemperatur nach der linearen Heizkurve bei −14 °C (= 120 °C); Massenströme aus $\dot Q=\dot m c_p\Delta T$ (höhere Spreizung ⇒ kleinerer Massenstrom); Erzeuger GT/BM/HWS behalten ihren Massenstromanteil, begrenzt auf das 2025 beobachtete Maximum (P99,9).
* **Ungünstige, aber beobachtete Erzeugerdruckhaltung:** Vorlaufdruck HKW/AVA −0,4 bar, Rücklaufdruck +0,25 bar gegenüber dem Basistag (≈ P5/P95 der Hochlaststunden 2025). Erzeugerdrücke werden als Randbedingung festgehalten (Erzeuger regeln Δp); Anhebung als Gegenmaßnahme in Kap. 19.2.
* **Grenzwerte (Annahmen, Sensitivität in Kap. 21):** Kunden-Δp ≥ 1,0 bar (2025: 1. Perzentil der Stationen ≈ 1,1–1,2 bar; WV640 Δp intern 0,8–1,3 bar); Siedeabstand p_abs − p_sat ≥ 1 bar; Lieferungstemperatur ≥ Heizkurve − 10 K; Erzeugerlimit = 2025 beobachteter Maximaldurchfluss (P99,9); Leitungsauslastung bezogen auf v = 2,5 m/s bzw. (bei Trennung) die dokumentierten Transportgrenzen 750/500/300 t/h für SL5/SL6/SL7.
* **Nicht versorgte Wärme (Näherung):** Kundenregelventile liefern bei Δp < Δp_min nur ≈ √(Δp/Δp_min) des Sollstroms. Rückwirkung auf das Netz wird vernachlässigt; der Wert dient dem Vergleich, nicht als Prognose einer Lieferunterbrechung.

### 16.2 Bias-Korrektur des Kunden-Differenzdrucks
Kap. 14 zeigt einen Differenzdruck-RMSE von ≈ 0,6 bar (Mitte/Süd). Zur belastbaren Bewertung absoluter Grenzwerte wird der Modell-Differenzdruck je Kundenknoten um den **Median der Abweichung Messung − Modell in den 10 % Hochlaststunden 2025** korrigiert (Knoten ohne Messstelle: keine Korrektur, Unsicherheit ± 0,6 bar). **Differenzen** zwischen Szenarien sind davon unabhängig.

In [ ]:
# ---------------------------------------------------------------- Bias-Korrektur der Kunden-Differenzdrücke (Modell − Messung bei hoher Last)
hq = BC.index[BC.valid & (BC.R >= BC.R[BC.valid].quantile(0.9))]                    # Hochlaststunden 2025 (oberes 10 %-Quantil der Restlast R)
def compute_dp_bias(th, verbose=True):
    sim_q = simulate(th, hq); pv_q, pr_q = station_pressures(sim_q); rows = []
    for j, r in MESS.iterrows():
        if r.vl is None or r.rl is None or r.station in EXCL_CAL or r.station in ("BMHKW",): continue
        meas = (acron.loc[hq, r.vl] - acron.loc[hq, r.rl]).values; mod = pv_q[:, j] - pr_q[:, j]; e = meas - mod; e = e[np.isfinite(e) & sim_q["ok"]]
        rows.append(dict(Station=r.station, Knoten=NODES[r.node], n=len(e), Bias_Messung_minus_Modell_bar=np.median(e), Streuung_P5_P95=f"{np.percentile(e,5):+.2f}…{np.percentile(e,95):+.2f}"))
    man_m = acron.loc[hq, "MAN_DiffDruck_Station"].values; man_mod = (sim_q["rho_v"]*G*(sim_q["Hv"][:, nidx["MAN"]]-Z_NODE[nidx["MAN"]]) - sim_q["rho_r"]*G*(sim_q["Hr"][:, nidx["MAN"]]-Z_NODE[nidx["MAN"]]))/1e5
    e = (man_m - man_mod)[np.isfinite(man_m)]; rows.append(dict(Station="MAN (Stationsmessung)", Knoten="MAN", n=len(e), Bias_Messung_minus_Modell_bar=np.median(e), Streuung_P5_P95=f"{np.percentile(e,5):+.2f}…{np.percentile(e,95):+.2f}"))
    hw_m = (acron.loc[hq, "HWW_Druck_VL_Primeinsp"] - acron.loc[hq, "HWW_Druck_RL_Primeinsp"]).values; iH_ = nidx["HWW"]
    hw_mod = (sim_q["rho_v"]*G*(sim_q["Hv"][:, iH_]-Z_NODE[iH_]) - sim_q["rho_r"]*G*(sim_q["Hr"][:, iH_]-Z_NODE[iH_]))/1e5
    e = (hw_m - hw_mod)[np.isfinite(hw_m)]; rows.append(dict(Station="HWW Primäreinspeisung", Knoten="HWW", n=len(e), Bias_Messung_minus_Modell_bar=np.median(e), Streuung_P5_P95=f"{np.percentile(e,5):+.2f}…{np.percentile(e,95):+.2f}"))
    bt = pd.DataFrame(rows); bt["Bias_Messung_minus_Modell_bar"] = bt.Bias_Messung_minus_Modell_bar.round(2); display(bt)
    DP_BIAS = np.zeros(len(KUNDEN)); DP_BIAS_N = {}
    for j, k_ in enumerate(KUNDEN):
        v = bt[bt.Knoten == k_].Bias_Messung_minus_Modell_bar.values
        if len(v): DP_BIAS[j] = float(np.median(v)); DP_BIAS_N[k_] = len(v)
    if verbose: print("Bias-Korrektur je Kundenknoten [bar] (Messung − Modell, Hochlast):", dict(zip(KUNDEN, DP_BIAS.round(2))), "| Knoten ohne Messstelle: Korrektur 0, Unsicherheit ±0,6 bar (mäßiger Ost-/Süd-Validierungsfehler, Kap. 14):", [k_ for k_ in KUNDEN if k_ not in DP_BIAS_N])
    if verbose: bt.to_csv(RESULTS/"dp_bias_kunden.csv", index=False)
    return DP_BIAS, bt
DP_BIAS, _bt = compute_dp_bias(theta_cal)

### 16.3 Hilfsfunktionen, Kennzahlen und Szenariokonfiguration
Schaltzustände S7 (aus den Armaturen der Pläne abgeleitet, Annahme): **S7-1** Ost getrennt bei Arm. 342.3 (MAN bleibt an der HKW-Seite), **S7-2** Ost getrennt bei 342.1/001.2 (MAN auf der Ost-Seite); jeweils zusätzlich Landungen SL6/SL7 in SL3 offen. Der Ostbereich wird dann nur von AVA, GT-Ost und BMHKW gespeist. Störfälle S8: GT-Ausfall, AVA-Ausfall, Koppelarmatur M_SL1_SL2 zu, Korridor SL1b/c auf 1/3 Kapazität, PSS ohne Förderhöhe, +8 K Rücklauftemperatur, leerer Speicher. Mehrfachausfälle werden nicht konstruiert.

In [ ]:
# ---------------------------------------------------------------- Zentrale Szenario-Hilfsfunktionen
T_CURVE = lambda Ta: np.clip(120.0 - (np.asarray(Ta, float) + 10.0), 90.0, 120.0)        # lineare Heizkurve: 120 °C @ −10 °C … 90 °C @ +20 °C
assert abs(float(T_CURVE(-10)) - 120) < 1e-9 and abs(float(T_CURVE(20)) - 90) < 1e-9 and abs(float(T_CURVE(-14)) - 120) < 1e-9
CAP = {"HKW": BC.loc[BC.valid, "m_hkw"].quantile(0.999), "AVA": BC.loc[BC.valid, "m_ava"].quantile(0.999),
       "GT": BC.loc[BC.valid, "m_gt"].quantile(0.999), "BM": BC.loc[BC.valid, "m_bm"].quantile(0.999), "HWS": BC.loc[BC.valid, "m_hws"].quantile(0.999)}   # in 2025 beobachtete Maxima (P99,9) als Erzeugerlimit [kg/s]
print("Erzeugerlimits (beobachtetes P99,9 2025, kg/s):", {k: round(v) for k, v in CAP.items()}, "≙ m³/h:", {k: round(v/0.95*3.6) for k, v in CAP.items()})
BASE_DAY = pd.Timestamp("2025-02-14")          # beobachteter Tag mit der höchsten Tagesmittelleistung (Q ≈ 215 MW, Ta ≈ −3,7 °C)
V_LIM = 2.5                                      # m/s: Bezugsgeschwindigkeit für die Leitungsauslastung (Annahme)
EDGE_CAP = lambda T: rho_w(T)*np.pi*D_E**2/4*V_LIM                   # kg/s je Kante
SEP_LIM = pd.Series(ka2.set_index("edge_id").max_t_h_separiert).reindex(E_IDS).values   # t/h, nur bei separiertem Betrieb (SL5/SL6/SL7)
OST_SEK = ["SL5M","SL5O","SL6S","SL7G"]

UNFAV = dict(vl=-0.4, rl=+0.25)      # ungünstige, aber beobachtete Erzeuger-Druckhaltung (≈ P5 des Vorlaufdrucks bzw. P95 des Rücklaufdrucks der Hochlaststunden 2025)
def base_day_case(day=BASE_DAY, heat_factor=1.0, design=True, Ta_des=-14.0, growth=0.0, growth_sec=None, dT_rl=0.0, dp_hkw=0.0, dp_ava=0.0, unfav=None, dp_rl=0.0):
    """24-h-Fall aus gemessenen Randbedingungen des Basistags, skaliert auf Auslegungsbedingungen.
    design=True: Vorlauftemperatur nach Heizkurve bei Auslegungs-Außentemperatur; Massenströme aus Wärmebedarf/ΔT; Erzeuger GT/BM/HWS mit demselben Massenstromfaktor, begrenzt auf das beobachtete Maximum.
    growth: Lastzuwachs (Bruchteil) auf Sektionen, MAN und West-Übergabe; growth_sec: {Sektion: Zuwachs} überschreibt die Verteilung je Sektion."""
    hrs = BC.index[BC.valid & (BC.index.date == day.date())]; assert len(hrs) == 24, f"{day.date()}: {len(hrs)} gültige Stunden"
    cs = case_from_bc(hrs); cs["hour"] = np.arange(24)
    Tv_obs, Tr_obs = cs.T_vl.values.copy(), cs.T_rl.values.copy()
    cs["T_rl"] = Tr_obs + dT_rl
    if design: cs["T_vl"] = T_CURVE(Ta_des); cs["Ta"] = Ta_des + (cs.Ta.values - cs.Ta.mean())
    f_m = heat_factor*(Tv_obs - Tr_obs)/(cs.T_vl.values - cs.T_rl.values)                     # Massenstromfaktor aus Wärmebilanz Q = ṁ·cp·ΔT
    cs["R"] = cs.R*f_m; cs["f_mass"] = f_m
    for c_, cap_ in (("m_gt","GT"),("m_bm","BM"),("m_hws","HWS")): cs[c_] = np.minimum(cs[c_]*f_m, CAP[cap_])
    g_all = growth
    cs["m_man"] = cs.m_man*f_m*(1+g_all); cs["m_wue"] = cs.m_wue*f_m*(1+g_all)
    for s_ in SEKN: cs[f"f_{s_}"] = 1.0 + (growth_sec.get(s_, 0.0) if growth_sec is not None else g_all)
    unfav = design if unfav is None else unfav; uv, ur = (UNFAV["vl"], UNFAV["rl"]) if unfav else (0.0, 0.0)
    cs["pVL_hkw"] = cs.pVL_hkw + dp_hkw + uv; cs["pVL_ava"] = cs.pVL_ava + dp_ava + uv; cs["pRL_hkw"] = cs.pRL_hkw + ur + dp_rl; cs["pRL_ava"] = cs.pRL_ava + ur + dp_rl
    return cs

def storage_sim(Qc, Qd, E0, Ecap, loss_MW=0.0, eta_c=0.98, eta_d=0.98):
    """Füllstand (nutzbare Energie, MWh) am Stundenende; Qc/Qd als Netto-Wärmeleistung am Netzanschluss [MW]."""
    E = np.empty(len(Qc)); e = E0
    for t in range(len(Qc)): e = storage_step(e, Qc[t], Qd[t], 1.0, eta_c, eta_d, loss_MW); E[t] = e
    return E

def periodic_E0(Qc, Qd, Ecap, loss_MW, eta_c=0.98, eta_d=0.98, soc_min=None):
    """Anfangsfüllstand (MWh) einer periodischen Tagesfahrweise: kleinster Füllstand = soc_min·Ecap."""
    soc_min = SPS["soc_min"] if soc_min is None else soc_min
    rel = storage_sim(Qc, Qd, 0.0, Ecap, loss_MW, eta_c, eta_d); return soc_min*Ecap - min(0.0, rel.min())

def dispatch_rule(R, Ecap, Qd_max, Qc_max, soc0=None, soc_min=None, soc_max=None, eta=0.98, loss_MW=0.0, periodic=True):
    """Regelbasierte Fahrweise: Entladen mit Qd_max in den k Stunden höchster Netzlast, Laden in den lastärmsten Stunden (periodischer Tageszyklus: Füllstand am Tagesende = Tagesanfang);
    k maximal, sodass die Füllstandsgrenzen eingehalten werden. periodic=False: Anfangsfüllstand soc0 fest, keine Wiederbefüllung im Betrachtungstag."""
    soc0 = SPS["soc0"] if soc0 is None else soc0; soc_min = SPS["soc_min"] if soc_min is None else soc_min; soc_max = SPS["soc_max"] if soc_max is None else soc_max
    R = np.asarray(R, float); n = len(R); order_hi = np.argsort(-R); order_lo = np.argsort(R); span = (soc_max - soc_min)*Ecap
    for k in range(n, -1, -1):
        Qd = np.zeros(n); Qc = np.zeros(n)
        for t in order_hi[:k]: Qd[t] = Qd_max
        if not periodic:
            avail = (soc0 - soc_min)*Ecap; used = 0.0; Qd[:] = 0.0
            for t in order_hi:
                q = min(Qd_max, (avail - used)*eta)
                if q <= 1e-9: break
                Qd[t] = q; used += q/eta
            return Qc, Qd
        need = (Qd.sum()/eta + loss_MW*n)/eta                                                   # erforderliche Ladeenergie [MWh] für geschlossenen Tageszyklus
        free = [t for t in order_lo if Qd[t] == 0]
        if len(free)*Qc_max < need - 1e-9: continue
        for t in free:
            q = min(Qc_max, need); Qc[t] = q; need -= q
            if need <= 1e-9: break
        E0 = periodic_E0(Qc, Qd, Ecap, loss_MW); E = storage_sim(Qc, Qd, E0, Ecap, loss_MW)
        if E.max() <= soc_max*Ecap + 1e-6 and E.min() >= soc_min*Ecap - 1e-6: return Qc, Qd
    return np.zeros(n), np.zeros(n)

def stor_flows(cs, Qc, Qd, T_h):
    """Massenströme am Speicher [kg/s] (>0 Entladen) aus Leistung und Spreizung: Entladen ṁ = Q/(cp (T_h − T_rl)), Laden ṁ = Q/(cp (T_vl − T_rl))."""
    Tv, Tr = cs.T_vl.values, cs.T_rl.values
    md = Qd*1e3/(cp_w(0.5*(T_h+Tr))*np.maximum(T_h - Tr, 5.0)); mc = Qc*1e3/(cp_w(0.5*(Tv+Tr))*np.maximum(Tv - Tr, 5.0))
    return md - mc

def pump_head_m(res, m_stor, T_ref=None):
    """Erforderliche Förderhöhe der Speicherpumpe [m]: Entladen: (H_VL−H_RL)_MAN + Verluste beider Stränge; Laden: nur bei Unterdeckung (Netz-Δp reicht nicht)."""
    iM = nidx[STOR_NODE]; dH = res["Hv"][:, iM] - res["Hr"][:, iM]; hl2 = 2*conn_head_m(m_stor, res["cs"].T_vl.values)
    Hd = np.maximum(dH + hl2, 0.0)
    Hc = np.maximum(hl2 + SPS["dp_valve_min_bar"]*1e5/(res["rho_v"]*G) - dH, 0.0)
    return np.where(m_stor > 0, Hd, np.where(m_stor < 0, Hc, 0.0))

def run_with_storage(cs, Qc, Qd, variant="B", cfg=None, T_h=None, dn_mm=None, dT_corr=None):
    """Speicherfall: variant 'A' = Regelventile ohne Pumpe (Entladen unmöglich, Laden begrenzt auf netzdruckgetriebenen Strom), 'B' = regelbare Pumpe."""
    T_h = SPS["T_max_C"] if T_h is None else T_h; T_h = min(T_h, cs.T_vl.max()) if T_h is not None else T_h
    Qc = np.array(Qc, float); Qd = np.array(Qd, float); cut = {"Entladung_unterdrückt_MWh": 0.0, "Ladung_begrenzt_MWh": 0.0}
    if variant == "A": cut["Entladung_unterdrückt_MWh"] = float(Qd.sum()); Qd[:] = 0.0
    m = stor_flows(cs, Qc, Qd, T_h); res = solve_case(cs, cfg=cfg, m_stor=m)
    if variant == "A":
        for _ in range(12):
            hl2 = 2*conn_head_m(m, cs.T_vl.values, dn_mm); iM = nidx[STOR_NODE]; dH = res["Hv"][:, iM] - res["Hr"][:, iM]
            bad = (m < 0) & ((dH - hl2 - SPS["dp_valve_min_bar"]*1e5/(res["rho_v"]*G)) < 0)
            if not bad.any(): break
            Qc_old = Qc.copy(); Qc[bad] *= 0.75; cut["Ladung_begrenzt_MWh"] += float((Qc_old - Qc).sum())
            m = stor_flows(cs, Qc, Qd, T_h); res = solve_case(cs, cfg=cfg, m_stor=m)
    return res, m, Qc, Qd, T_h

In [ ]:
# ---------------------------------------------------------------- Kennzahlen je Szenario (Kap. 19 des Arbeitsauftrags)
def evaluate(name, cs, cfg=None, storage=None, dp_min=None, Ecap=None, u_lin=U_LIN):
    """storage: None oder dict(Qc, Qd, variant, [dn_mm], [T_h]). Gibt dict mit Kennzahlen 'kpi' und Stundenreihen 'h' zurück."""
    cfg = dict(cfg or {}); dp_min = SPS["dp_kunde_min_bar"] if dp_min is None else dp_min
    if storage is None:
        m = np.zeros(len(cs)); res = solve_case(cs, cfg=cfg); Qc = Qd = np.zeros(len(cs)); T_h = None; variant = "-"
    else:
        variant = storage.get("variant", "B"); res, m, Qc, Qd, T_h = run_with_storage(cs, storage["Qc"], storage["Qd"], variant, cfg, storage.get("T_h"), storage.get("dn_mm"))
    th = thermal(res, m_stor=m, T_h=(None if T_h is None else np.full(len(cs), T_h)), u_lin=u_lin)
    pv, pr = node_pressures(res); B = len(cs); ok_z = ~np.isnan(Z_NODE)
    dpk_raw = (pv - pr)[:, KUNDEN_IDX]; dpk = dpk_raw + DP_BIAS[None, :]; dpmin_h = dpk.min(1); arg_dp = dpk.argmin(1)
    pvk = pv[:, ok_z]; prk = pr[:, ok_z]; nz = np.array(NODES)[ok_z]
    it_v = np.unravel_index(np.argmin(pvk), pvk.shape); it_r = np.unravel_index(np.argmin(prk), prk.shape)
    psat_n = psat_bar(np.clip(th["Tv"], 20, 150)); marg = (pv + P_ATM - psat_n)[:, ok_z]; it_m = np.unravel_index(np.argmin(marg), marg.shape)
    Tk = th["Tv"][:, KUNDEN_IDX]; Treq = T_CURVE(cs.Ta.values) - 10.0; Tlief_h = Tk.min(1)
    # nicht versorgte Wärme (erste Näherung): Kundenventile liefern bei Δp < Δp_min nur ≈ √(Δp/Δp_min) des Sollstroms
    dem = np.zeros((B, len(KUNDEN)))
    for j, k_ in enumerate(KUNDEN):
        if k_ in SEKN: dem[:, j] = (cs[f"w_{k_}"].values if f"w_{k_}" in cs else res["w"][SEKN.index(k_)])*cs[f"f_{k_}"].values*cs.R.values
        elif k_ == "MAN": dem[:, j] = cs.m_man.values
        elif k_ == "HWW": dem[:, j] = cs.m_wue.values
    load_MW = dem*cp_w(0.5*(cs.T_vl.values+cs.T_rl.values))[:, None]*(cs.T_vl.values - cs.T_rl.values)[:, None]/1e3
    unserv_MW = (load_MW*(1 - np.sqrt(np.clip(dpk/dp_min, 0, 1)))).sum(1)
    flow_hkw = res["inj_all"][:, nidx["HKW"]]; flow_ava = res["inj_all"][:, nidx["AVA"]]; dTn = (cs.T_vl.values - cs.T_rl.values); cpn = cp_w(0.5*(cs.T_vl.values+cs.T_rl.values))
    over_h = np.maximum(flow_hkw - CAP["HKW"], 0)*cpn*dTn/1e3; over_a = np.maximum(flow_ava - CAP["AVA"], 0)*cpn*dTn/1e3
    util = np.abs(res["mv"])/EDGE_CAP(cs.T_vl.values)[:, None] if False else np.abs(res["mv"])/(rho_w(cs.T_vl.values)[:, None]*np.pi*D_E[None, :]**2/4*V_LIM)
    util_sep = np.abs(res["mv"])*3.6/np.where(np.isfinite(SEP_LIM), SEP_LIM, np.inf)[None, :]
    sepmode = bool(cfg.get("separiert", False)); ut = util_sep if sepmode else util
    iu = np.unravel_index(np.argmax(np.where(np.isfinite(ut), ut, 0)), ut.shape)
    P_net = (np.abs(res["mv"][:, [eidx["SL4c"], eidx["W1"]]])*G*res["gain"][:, [eidx["SL4c"], eidx["W1"]]]/(SPS["eta_pump"]*1e6)).sum(1)
    Hp = pump_head_m(res, m) if storage is not None else np.zeros(B)
    P_sto = np.abs(m)*G*Hp/(SPS["eta_pump"]*1e6) if storage is not None else np.zeros(B)
    Tm = res["cs"].T_vl.values; dp_pump_bar = Hp*rho_w(Tm)*G/1e5
    Qd_h = Qd; Qc_h = Qc
    kpi = {"Szenario": name, "Variante": variant, "Löser ok": bool(res["ok"].all()),
           "p_VL,min [bar Ü]": pvk.min(), "Knoten/Stunde p_VL,min": f"{nz[it_v[1]]}/{it_v[0]:02d}h",
           "p_RL,min [bar Ü]": prk.min(), "Knoten/Stunde p_RL,min": f"{nz[it_r[1]]}/{it_r[0]:02d}h",
           "Δp_Kunde,min [bar]": dpmin_h.min(), "Δp_Kunde,min roh [bar]": dpk_raw.min(), "Knoten/Stunde Δp,min": f"{KUNDEN[arg_dp[dpmin_h.argmin()]]}/{int(dpmin_h.argmin()):02d}h",
           "p_max [bar Ü]": pv[:, ok_z].max(), "Siedeabstand min [bar]": marg.min(), "Knoten/Stunde Siede": f"{nz[it_m[1]]}/{it_m[0]:02d}h",
           "T_Lieferung,min [°C]": Tlief_h.min(), "T_RL,max HKW [°C]": th["Tr"][:, nidx["HKW"]].max(),
           "kritische Knoten [#]": int((dpk < dp_min - 1e-3).any(0).sum()), "kritische Knoten": ",".join(np.array(KUNDEN)[(dpk < dp_min - 1e-3).any(0)]),
           "Dauer Δp-Verletzung [h]": int((dpmin_h < dp_min - 1e-3).sum()), "Δp-Defizit [bar·h]": float(np.maximum(dp_min - dpk, 0).sum()), "Dauer T-Verletzung [h]": int((Tlief_h < Treq).sum()),
           "nicht versorgt,max [MW]": unserv_MW.max(), "nicht versorgt [MWh]": unserv_MW.sum(),
           "HKW-Durchfluss,max [m³/h]": flow_hkw.max()/rho_w(Tm[0])*3600, "AVA-Durchfluss,max [m³/h]": flow_ava.max()/rho_w(Tm[0])*3600,
           "Erzeugerlimit HKW überschritten [h]": int((over_h > 0.5).sum()), "Überschreitung HKW,max [MW]": over_h.max(),
           "Erzeugerlimit AVA überschritten [h]": int((over_a > 0.5).sum()), "Überschreitung AVA,max [MW]": over_a.max(),
           "Leitungsauslastung,max [-]": ut[iu], "Kante Auslastung": f"{E_IDS[iu[1]]}/{iu[0]:02d}h", "Kanten > 1,0 [#]": int((ut > 1.0).any(0).sum()),
           "Pumpstrom Netz [MWh]": P_net.sum(), "Pumpstrom Speicher [MWh]": P_sto.sum(), "Netzwärmeverlust [MWh]": th["loss_MW"].sum(),
           "Ladeenergie [MWh]": Qc_h.sum(), "Entladeenergie [MWh]": Qd_h.sum(), "Qd,max [MW]": Qd_h.max(), "Qc,max [MW]": Qc_h.max(),
           "Pumpenförderhöhe,max [bar]": dp_pump_bar.max(), "Massenstrom Speicher,max [kg/s]": np.abs(m).max()}
    hourly = pd.DataFrame(dict(h=np.arange(B), dp_min=dpmin_h, p_VL_min=pvk.min(1), p_RL_min=prk.min(1), T_lief=Tlief_h, flow_hkw_m3h=flow_hkw/rho_w(Tm)*3600, flow_ava_m3h=flow_ava/rho_w(Tm)*3600,
                               m_stor=m, Qc=Qc_h, Qd=Qd_h, unserv=unserv_MW, P_net=P_net, P_sto=P_sto, Hp_bar=dp_pump_bar, loss=th["loss_MW"], R=cs.R.values, Ta=cs.Ta.values))
    return dict(kpi=kpi, h=hourly, res=res, th=th, m=m, pv=pv, pr=pr, dpk=dpk, cs=cs, util=ut, name=name)

# ---------------------------------------------------------------- Test: Szenariogleichheit vor Aktivierung des Speichers (Arbeitsauftrag Kap. 22) und Erhaltungssätze
cs_t = base_day_case(design=True)
e1 = evaluate("S1_test", cs_t); e2 = evaluate("S2_test", cs_t, storage=dict(Qc=np.zeros(24), Qd=np.zeros(24), variant="B"))
assert np.allclose(e1["res"]["Hv"], e2["res"]["Hv"], atol=1e-9) and np.allclose(e1["res"]["mr"], e2["res"]["mr"], atol=1e-9)
assert np.abs(e1["res"]["inj_all"].sum(1)).max() < 1e-2
print("Tests bestanden: (i) S2 (angeschlossen, Nullstrom) ≡ S1 bis auf Rundung (max. Δ Druckhöhe 0), (ii) Massenbilanz je Stunde < 0,01 kg/s.")
# Test Speicherleistung ↔ Massenstrom ↔ Spreizung
Qd_t = np.zeros(24); Qd_t[8] = 40.0; mm = stor_flows(cs_t, np.zeros(24), Qd_t, 120.0)
Q_back = mm[8]*cp_w(0.5*(120+cs_t.T_rl[8]))*(120 - cs_t.T_rl[8])/1e3; assert abs(Q_back - 40.0) < 1e-6
print(f"Test bestanden: 40 MW Entladen ⇒ {mm[8]:.1f} kg/s bei T_h = 120 °C, T_RL = {cs_t.T_rl[8]:.1f} °C (Rückrechnung {Q_back:.2f} MW).")
print(pd.Series(e1["kpi"]).to_string())

In [ ]:
# ---------------------------------------------------------------- Zentrale Szenariokonfiguration (alle Szenarien werden programmatisch aus dieser Tabelle erzeugt)
GROW = {"a": 0.10, "b": 0.20, "c": 0.30, "d": 0.50}
OST_G30 = {s_: (0.30/sum(res_w for res_w in unpack(theta_cal)[1][[SEKN.index(x) for x in OST_SEK]]))*(1.0 if s_ in OST_SEK else 0.0) for s_ in SEKN}     # +30 % Gesamtlast, nur in den Ost-Sektionen
SEP1 = dict(closed=["SL5b","K_SL6_SL3","K_SL7_SL3"], separiert=True)       # Ost getrennt bei Arm. 342.3 (MAN bleibt auf HKW-Seite)
SEP2 = dict(closed=["SL5a","K_SL6_SL3","K_SL7_SL3"], separiert=True)       # Ost getrennt bei Arm. 342.1/001.2 (MAN auf Ost-Seite)
SCEN = [
 dict(id="S0",  name="Historische Baseline (14.02.2025, gemessene RB)",   base=dict(design=False), cfg={}, sto="none"),
 dict(id="S0E", name="Historische Baseline mit Speicher (Rückblick, opt., B)", base=dict(design=False), cfg={}, sto="opt_B"),
 dict(id="S1",  name="Kritischer Bestand ohne Speicher (Auslegungs-RB)",  base=dict(design=True), cfg={}, sto="none"),
 dict(id="S2",  name="Bestand, Speicher angeschlossen, inaktiv",           base=dict(design=True), cfg={}, sto="inactive"),
 dict(id="S3A", name="Bestand, Entladung, Variante A (Ventile, keine Pumpe)", base=dict(design=True), cfg={}, sto="rule_A"),
 dict(id="S3B", name="Bestand, Entladung, Variante B (Pumpe), regelbasiert", base=dict(design=True), cfg={}, sto="rule_B"),
 dict(id="S3O", name="Bestand, Variante B, optimierte Fahrweise",         base=dict(design=True), cfg={}, sto="opt_B"),
 dict(id="S4A", name="Speicherladung nachts, Variante A (Netz-Δp)",       base=dict(design=True), cfg={}, sto="charge_night_A"),
 dict(id="S4B", name="Speicherladung nachts, Variante B (Pumpe)",          base=dict(design=True), cfg={}, sto="charge_night_B"),
 dict(id="S4K", name="Kritische Ladung zur Lastspitze (07–11 Uhr), B",     base=dict(design=True), cfg={}, sto="charge_peak_B"),
] + [dict(id=f"S5{k}", name=f"Ausbau +{int(100*g)} % ohne Speicher", base=dict(design=True, growth=g), cfg={}, sto="none") for k, g in GROW.items()] + [
 dict(id="S5O", name="Ausbau +30 % (nur Ost-Sektionen) ohne Speicher", base=dict(design=True, growth=0.30, growth_sec=OST_G30), cfg={}, sto="none")
] + [dict(id=f"S6{k}", name=f"Ausbau +{int(100*g)} % mit optimiertem Speicher (B)", base=dict(design=True, growth=g), cfg={}, sto="opt_B") for k, g in GROW.items()] + [
 dict(id="S6O", name="Ausbau +30 % (nur Ost) mit optimiertem Speicher (B)", base=dict(design=True, growth=0.30, growth_sec=OST_G30), cfg={}, sto="opt_B"),
]
for tag, sepc, lab in (("1", SEP1, "Ost getrennt bei 342.3 (MAN an HKW-Seite)"), ("2", SEP2, "Ost getrennt bei 342.1 (MAN an Ost-Seite)")):
    for gid, g, gl in (("B", 0.0, "Bestand"), ("A", 0.30, "Ausbau +30 %")):
        SCEN += [dict(id=f"S7{tag}{gid}0", name=f"{lab}, {gl}, ohne Speicher", base=dict(design=True, growth=g), cfg=sepc, sto="none"),
                 dict(id=f"S7{tag}{gid}E", name=f"{lab}, {gl}, Entladung (opt., B)", base=dict(design=True, growth=g), cfg=sepc, sto="opt_B"),
                 dict(id=f"S7{tag}{gid}L", name=f"{lab}, {gl}, Nachtladung (B)", base=dict(design=True, growth=g), cfg=sepc, sto="charge_night_B")]
S8 = [("a", "GT-Ost fällt aus", dict(zero_gt=True), {}), ("b", "AVA fällt aus (HKW einziger Druckhalter)", {}, dict(ava_ref=False)),
      ("c", "Koppelarmatur M_SL1_SL2 geschlossen", {}, dict(closed=["M_SL1_SL2"])), ("d", "Korridor SL1b/SL1c auf 1/3 Kapazität (k×3)", {}, dict(mult={"SL1b": 3.0, "SL1c": 3.0})),
      ("e", "Pumpstation Süd (PSS) ohne Förderhöhe", {}, dict(pump_scale=0.0)), ("f", "Hohe Rücklauftemperatur (+8 K)", dict(dT_rl=8.0), {})]
for k_, lab, bk, cf in S8:
    SCEN += [dict(id=f"S8{k_}0", name=f"{lab}, ohne Speicher", base=dict(design=True, **bk), cfg=cf, sto="none"), dict(id=f"S8{k_}E", name=f"{lab}, Speicher (opt., B)", base=dict(design=True, **bk), cfg=cf, sto="opt_B")]
SCEN += [dict(id="S8g", name="Speicher leer zur Lastspitze (Füllstand 5 %)", base=dict(design=True), cfg={}, sto="opt_B_empty")]
cfg_tab = pd.DataFrame([dict(ID=s["id"], Szenario=s["name"], Basis="Bestand" if not s["base"].get("growth") else f"+{int(100*s['base']['growth'])} %" + (" (Ost)" if s["base"].get("growth_sec") else ""),
                             Schaltzustand="getrennt" if s["cfg"].get("separiert") else ("Sonderzustand" if s["cfg"] else "vermascht"), Speicher=s["sto"]) for s in SCEN])
print(f"{len(SCEN)} Szenarien aus zentraler Konfiguration erzeugt."); display(cfg_tab)
cfg_tab.to_csv(RESULTS/"szenarien_konfiguration.csv", index=False)

## 17. Optimierungsläufe

### 17.1 Formulierung (Fahrweise des Speichers je 24-h-Fall)
Das Framework-Optimierungsmodell (Pyomo-MILP) enthält weder Höhen noch Schleifengesetz, Kesselformel oder Speicherpumpe (Kap. 11.10). Die Fahrweise wird daher in diesem Notebook mit **Pyomo + HiGHS** über ein **sequentiell linearisiertes** Modell der *nichtlinearen* Netzhydraulik (Kap. 12) bestimmt; jede Lösung wird anschließend mit dem nichtlinearen Löser nachgerechnet (Ein Solverstatus ist kein physikalischer Nachweis).

**Variablen** (je Stunde t): $Q_{d,t}, Q_{c,t}\ge 0$ Entlade-/Ladeleistung [MW], $y_t\in\{0,1\}$ (Entladen/Laden exklusiv), Füllstand $E_t$ [MWh], Schlupf $s^{\Delta p}_{k,t}\ge0$ [bar] (Kunden-Δp-Unterschreitung je Knoten), $s^{cap}_t\ge0$ [kg/s] (HKW-Durchfluss über Erzeugerlimit = nicht gedeckte Last), $z$ (Spitzen-Durchfluss HKW), Hilfsvariablen $dch_t$, $dev_t$.
**Parameter:** Spreizungen $a_d,a_c$ (kg/s je MW), Netzsensitivitäten $\partial \Delta p_k/\partial \dot m_S$, $\partial \dot m_{HKW}/\partial \dot m_S$ aus Finite-Differenz-Probeläufen (±Probeschritt um den aktuellen Arbeitspunkt), Speicherkennwerte (Kap. 15.4), Grenzen $\Delta p_{min}$, Erzeugerlimit.
**Nebenbedingungen:** Füllstandsbilanz $E_{t+1}=E_t+\eta Q_{c,t}-Q_{d,t}/\eta-Q_{loss}$; periodischer Tageszyklus $E_{24}\ge E_0$ (Anfangsfüllstand frei; bei Sensitivität fest); $E\in[5\,\%,95\,\%]\cdot E_{cap}$; $Q\le Q_{max}$; Anschluss-DN: $v\le 3$ m/s; optional Pumpenförderhöhe $H_P\le H_{P,max}$ (linearisiert); Netz: $\Delta p_{k,t}^0+\sum S\,(\dot m_{S,t}-\dot m^{(i)}_{S,t}) + s^{\Delta p}_{k,t}\ge\Delta p_{min}$; Trust-Region $|\dot m_S-\dot m^{(i)}|\le 80$ kg/s.
**Zielfunktion (lexikographisch, 3 Stufen):** (1) $\min\sum s^{\Delta p}$ (Druckverletzungen); (2) bei festgehaltener Stufe 1: $\min\sum s^{cap}$ (unversorgte Last); (3) bei festgehaltenen Stufen 1–2: $\min\; w_{peak}z+w_{pump}P_{el}+w_{thr}\sum(Q_c+Q_d)+w_{chg}\sum dch$ (Spitze, Pumpstrom, Verluste über Durchsatz, Fahrweisenwechsel). Gewichte: 10 €/(kg/s), 120 €/MWh, 0,5 €/MWh, 0,2 €/MW. Die Stufen 1–2 bilden die in Kap. 14 des Auftrags geforderte Priorisierung (Druck vor unversorgter Last vor Wirtschaftlichkeit) **strikt**, nicht nur gewichtet. Thermische Unterversorgung und Siedesicherheit sind in der Fahrweisenoptimierung keine Stellgröße (bei T_h ≥ Heizkurve − 10 K nicht bindend) und werden in der Nachrechnung geprüft. Schlupfvariablen haben die Bedeutung: $s^{\Delta p}$ = fehlender Kunden-Differenzdruck, $s^{cap}$ = Durchfluss über dem 2025 beobachteten HKW-Maximum.
**Solver:** HiGHS (appsi_highs, Standardoptionen), MILP mit 24 Binärvariablen. **Konvergenz:** Abbruch, wenn der größte Schritt < 0,5 kg/s oder die nichtlinear bewertete Schlupfgröße über zwei Runden um < 0,5 % besser wird; beste Runde wird übernommen. **Initialisierung:** Nullfahrplan; Test mit regelbasierten und zufälligen Startfahrplänen (17.3).

In [ ]:
import pyomo.environ as pyo, time as _time
W_OPT = dict(w_peak=10.0, w_pump=120.0, w_thr=0.5, w_chg=0.2)    # Stufe 2: €/(kg/s) Spitzenstrom HKW, €/MWh Pumpstrom, €/MWh Durchsatz (Verluste), €/(MW) Fahrweisenwechsel
def probe(cs, cfg, m_vec):
    res = solve_case(cs, cfg=cfg, m_stor=m_vec); pv, pr = node_pressures(res); iM = nidx[STOR_NODE]
    dp = (pv - pr)[:, KUNDEN_IDX] + DP_BIAS[None, :]; dH = res["Hv"][:, iM] - res["Hr"][:, iM]
    return dp, res["inj_all"][:, nidx["HKW"]], res["inj_all"][:, nidx["AVA"]], dH, res["ok"]

def optimize_dispatch(cs, cfg=None, V_m3=None, Qd_max=None, Qc_max=None, soc0=None, dn_mm=None, rounds=14, trust=80.0, weights=None, T_h=None, m_start=None,
                      dp_min=None, cap_hkw=None, verbose=False, end_soc=True, periodic=True, solver="appsi_highs", Hp_max_bar=None, v_max=3.0):
    """Zweistufige (lexikographische) Optimierung der Speicherfahrweise für einen 24-h-Fall mit sequentieller Linearisierung der Netzhydraulik (Kap. 17).
    Stufe 1: Σ Δp-Unterschreitung bei Kunden (bar·h) minimal. Stufe 2: bei festgehaltener Stufe-1-Zielgröße: HKW-Durchfluss über Erzeugerlimit (nicht gedeckte Last, kg/s·h) minimal. Stufe 3: bei festgehaltenen Stufen 1–2: Spitzen-Durchfluss HKW, Pumpstrom, Durchsatz (Verluste), Fahrweisenwechsel."""
    t_start = _time.time(); w = dict(W_OPT, **(weights or {})); B = len(cs)
    V = SPS["V_m3"] if V_m3 is None else V_m3; Qd_max = SPS["Qd_max_MW"] if Qd_max is None else Qd_max; Qc_max = SPS["Qc_max_MW"] if Qc_max is None else Qc_max
    soc0 = SPS["soc0"] if soc0 is None else soc0; dp_min = SPS["dp_kunde_min_bar"] if dp_min is None else dp_min; cap_hkw = CAP["HKW"] if cap_hkw is None else cap_hkw
    T_h = min(SPS["T_max_C"], cs.T_vl.max()) if T_h is None else T_h
    Tv, Tr = cs.T_vl.values, cs.T_rl.values
    dn_c = SPS["dn_conn_mm"] if dn_mm is None else dn_mm; m_dn = rho_w(Tv)*np.pi*(dn_c/1000.0)**2/4*v_max               # Anschluss-DN begrenzt den Massenstrom (v ≤ v_max)
    ad = 1e3/(cp_w(0.5*(T_h+Tr))*np.maximum(T_h - Tr, 5.0)); ac = 1e3/(cp_w(0.5*(Tv+Tr))*np.maximum(Tv - Tr, 5.0))          # kg/s je MW
    Ecap = storage_capacity_MWh(V, T_h - Tr.mean(), 0.5*(T_h+Tr.mean()), SPS["eta_strat"]); loss = storage_loss_MW(0.5*(T_h+Tr.mean()), V=V)
    m_cur = np.zeros(B) if m_start is None else np.array(m_start, float); hist = []; cand = []; best_hist = []; conv = False
    K = len(KUNDEN)
    for rd in range(rounds):
        dpp, fhp, fap, dHp, _ = probe(cs, cfg, m_cur + trust*0.5); dpm, fhm, fam, dHm, _ = probe(cs, cfg, m_cur - trust*0.5); dp0, fh0, fa0, dH0, ok0 = probe(cs, cfg, m_cur)
        S_dp = (dpp - dpm)/trust; S_f = (fhp - fhm)/trust
        Hp_cur = np.maximum(dH0 + 2*conn_head_m(m_cur, Tv, dn_mm), 0.0)
        res = {}
        for stage in (1, 2, 3):
            M = pyo.ConcreteModel(); T_ = range(B); K_ = range(K)
            M.Qd = pyo.Var(T_, bounds=(0, Qd_max)); M.Qc = pyo.Var(T_, bounds=(0, Qc_max)); M.y = pyo.Var(T_, domain=pyo.Binary)
            M.E = pyo.Var(range(B+1), bounds=(SPS["soc_min"]*Ecap, SPS["soc_max"]*Ecap)); M.sd = pyo.Var(T_, K_, bounds=(0, None)); M.sc = pyo.Var(T_, bounds=(0, None))
            M.z = pyo.Var(bounds=(0, None)); M.dch = pyo.Var(T_, bounds=(0, None)); M.dev = pyo.Var(T_, bounds=(0, None))
            if not periodic: M.E0 = pyo.Constraint(expr=M.E[0] == soc0*Ecap)
            M.bal = pyo.Constraint(T_, rule=lambda m, t: m.E[t+1] == m.E[t] + 0.98*m.Qc[t] - m.Qd[t]/0.98 - loss)
            if end_soc: M.endc = pyo.Constraint(expr=M.E[B] >= (M.E[0] if periodic else soc0*Ecap) - 1e-3)
            M.ex1 = pyo.Constraint(T_, rule=lambda m, t: m.Qd[t] <= Qd_max*m.y[t]); M.ex2 = pyo.Constraint(T_, rule=lambda m, t: m.Qc[t] <= Qc_max*(1 - m.y[t]))
            mnet = lambda m, t: ad[t]*m.Qd[t] - ac[t]*m.Qc[t]
            M.dnl1 = pyo.Constraint(T_, rule=lambda m, t: ad[t]*m.Qd[t] <= m_dn[t]); M.dnl2 = pyo.Constraint(T_, rule=lambda m, t: ac[t]*m.Qc[t] <= m_dn[t])
            if Hp_max_bar is not None:
                Hmax_m = Hp_max_bar*1e5/(rho_w(Tv)*G); dHs = (dHp - dHm)/trust; hl0 = 2*conn_head_m(m_cur, Tv, dn_mm); hls = (2*conn_head_m(m_cur + 1.0, Tv, dn_mm) - hl0)
                M.hpc = pyo.Constraint(T_, rule=lambda m, t: dH0[t] + hl0[t] + (dHs[t] + hls[t])*(ad[t]*m.Qd[t] - ac[t]*m.Qc[t] - m_cur[t]) <= Hmax_m[t] + 1e3*(1 - m.y[t]))
            M.dv1 = pyo.Constraint(T_, rule=lambda m, t: m.dev[t] >= mnet(m, t) - m_cur[t]); M.dv2 = pyo.Constraint(T_, rule=lambda m, t: m.dev[t] >= m_cur[t] - mnet(m, t)); prox = sum(M.dev[t] for t in T_)      # Proximalterm: wählt unter gleichwertigen Optima das zur letzten Iteration nächste
            M.tr1 = pyo.Constraint(T_, rule=lambda m, t: mnet(m, t) - m_cur[t] <= trust); M.tr2 = pyo.Constraint(T_, rule=lambda m, t: mnet(m, t) - m_cur[t] >= -trust)
            M.dpc = pyo.Constraint(T_, K_, rule=lambda m, t, k: dp0[t, k] + S_dp[t, k]*(mnet(m, t) - m_cur[t]) + m.sd[t, k] >= dp_min)
            M.capc = pyo.Constraint(T_, rule=lambda m, t: fh0[t] + S_f[t]*(mnet(m, t) - m_cur[t]) - m.sc[t] <= cap_hkw)
            M.pk = pyo.Constraint(T_, rule=lambda m, t: m.z >= fh0[t] + S_f[t]*(mnet(m, t) - m_cur[t]))
            M.d1 = pyo.Constraint(range(1, B), rule=lambda m, t: m.dch[t] >= mnet(m, t) - mnet(m, t-1)); M.d2 = pyo.Constraint(range(1, B), rule=lambda m, t: m.dch[t] >= mnet(m, t-1) - mnet(m, t))
            viol_dp = sum(M.sd[t, k] for t in T_ for k in K_); viol_cap = sum(M.sc[t] for t in T_)
            pump = sum(G*Hp_cur[t]/(SPS["eta_pump"]*1e6)*ad[t]*M.Qd[t] for t in T_)                                   # MW (Stundenwert = MWh)
            if stage == 1: M.obj = pyo.Objective(expr=viol_dp + 1e-4*sum(M.Qc[t]+M.Qd[t] for t in T_) + 1e-4*prox)
            elif stage == 2:
                M.fix1 = pyo.Constraint(expr=viol_dp <= res["v1"] + 1e-6)
                M.obj = pyo.Objective(expr=viol_cap + 1e-4*sum(M.Qc[t]+M.Qd[t] for t in T_) + 1e-4*prox)
            else:
                M.fix1 = pyo.Constraint(expr=viol_dp <= res["v1"] + 1e-6); M.fix2 = pyo.Constraint(expr=viol_cap <= res["v2"] + 1e-6)
                M.obj = pyo.Objective(expr=w["w_peak"]*M.z + w["w_pump"]*pump + w["w_thr"]*sum(M.Qc[t] + M.Qd[t] for t in T_) + w["w_chg"]*sum(M.dch[t] for t in range(1, B)) + 1e-2*prox)
            sol = pyo.SolverFactory(solver); out = sol.solve(M, tee=False) if solver != "appsi_highs" else sol.solve(M)
            if stage == 1: res["v1"] = pyo.value(viol_dp)
            if stage == 2: res["v2"] = pyo.value(viol_cap)
        Qd = np.array([pyo.value(M.Qd[t]) for t in T_]); Qc = np.array([pyo.value(M.Qc[t]) for t in T_]); m_new = ad*Qd - ac*Qc
        dpn, fhn, _, _, _ = probe(cs, cfg, m_new); nl_v1 = float(np.maximum(dp_min - dpn, 0).sum()); nl_v2 = float(np.maximum(fhn - cap_hkw, 0).sum()); nl_pk = float(fhn.max())     # nichtlineare Bewertung des Kandidaten
        step = np.abs(m_new - m_cur).max(); hist.append(dict(runde=rd+1, stufe1_dp_schlupf_barh=res["v1"], stufe2_HKW_ueber_kgs_h=res["v2"], schlupf_nichtlinear_barh=nl_v1, max_schritt_kgs=step, peak_HKW_lin=pyo.value(M.z)))
        cand.append((round(nl_v1 + 1e-3*nl_v2, 3), nl_pk, Qd.copy(), Qc.copy(), m_new.copy()))
        m_cur = m_new
        cost = min(c[0] for c in cand); best_hist.append(cost)
        if step < 0.5 or (len(best_hist) >= 4 and best_hist[-3] - best_hist[-1] < 0.005*(1 + cost)): conv = True; break          # Konvergenz: Schritt < 0,5 kg/s oder bester nichtlinearer Schlupf über zwei Runden < 0,5 % verbessert
    best = min(cand, key=lambda c: (round(c[0], 3), c[1])); Qd, Qc, m_cur = best[2], best[3], best[4]
    hist_df = pd.DataFrame(hist)
    return dict(converged=conv, Qd=Qd, Qc=Qc, m=m_cur, Ecap=Ecap, hist=hist_df, T_h=T_h, t_s=_time.time()-t_start, status=str(out.solver.termination_condition) if hasattr(out, "solver") else "?", weights=w, ad=ad, ac=ac)

# ---------------------------------------------------------------- Test auf S1
t0 = _time.time(); csS1 = base_day_case(design=True); ev0 = evaluate("S1", csS1)
opt = optimize_dispatch(csS1); print("Optimierung S1:", opt["status"], f"{opt['t_s']:.1f} s, Runden: {len(opt['hist'])}"); display(opt["hist"].round(3))
ev1 = evaluate("S3-opt", csS1, storage=dict(Qc=opt["Qc"], Qd=opt["Qd"], variant="B"))
print(pd.DataFrame({"S1": ev0["kpi"], "S3-opt": ev1["kpi"]}).loc[["Δp_Kunde,min [bar]","Dauer Δp-Verletzung [h]","Δp-Defizit [bar·h]","nicht versorgt [MWh]","HKW-Durchfluss,max [m³/h]","Pumpstrom Speicher [MWh]","Entladeenergie [MWh]","Ladeenergie [MWh]","Pumpenförderhöhe,max [bar]"]])
print(np.round(opt["Qd"],1), np.round(opt["Qc"],1))

### 17.2 Alle Szenarien rechnen
Regelbasierte Fahrweisen (Entladen in den k lastreichsten Stunden, Laden in den lastärmsten, geschlossener Tageszyklus) und optimierte Fahrweisen werden mit demselben Code erzeugt; S3B (regelbasiert) dient als Vergleich zur Optimierung.

In [ ]:
import time as _time
ECAP_C = storage_capacity_MWh(SPS["V_m3"], SPS["T_max_C"] - BC.loc[BC.valid, "T_rl"].median(), 0.5*(SPS["T_max_C"] + BC.loc[BC.valid, "T_rl"].median()), SPS["eta_strat"])
def storage_plan(kind, cs, cfg, ev_noS=None, **over):
    V = over.get("V_m3", SPS["V_m3"]); Qd_max = over.get("Qd_max", SPS["Qd_max_MW"]); Qc_max = over.get("Qc_max", SPS["Qc_max_MW"]); soc0 = over.get("soc0", SPS["soc0"])
    T_h = over.get("T_h", min(SPS["T_max_C"], cs.T_vl.max())); Ecap = storage_capacity_MWh(V, T_h - cs.T_rl.mean(), 0.5*(T_h+cs.T_rl.mean()), SPS["eta_strat"]); loss = storage_loss_MW(0.5*(T_h+cs.T_rl.mean()), V=V)
    n = len(cs); z = np.zeros(n)
    if kind == "inactive": return dict(Qc=z, Qd=z, variant="B", T_h=T_h)
    if kind.startswith("rule"):
        Qc, Qd = dispatch_rule(cs.R.values, Ecap, Qd_max, Qc_max, soc0=soc0, loss_MW=loss); Qc = Qc if kind[-1] == "B" else Qc*0.0     # Variante A: Entladung ist ohne Pumpe nicht möglich; Wiederbefüllung erfolgt außerhalb des Betrachtungstags
        return dict(Qc=Qc, Qd=Qd, variant=kind[-1], T_h=T_h)
    if kind.startswith("charge_night"):
        Qc = z.copy(); idx = np.argsort(cs.R.values)[:6]; Qc[idx] = Qc_max; return dict(Qc=Qc, Qd=z, variant=kind[-1], T_h=T_h)
    if kind.startswith("charge_peak"):
        Qc = z.copy(); Qc[7:11] = Qc_max; return dict(Qc=Qc, Qd=z, variant=kind[-1], T_h=T_h)
    if kind.startswith("opt"):
        soc = 0.05 if kind.endswith("empty") else soc0
        o = optimize_dispatch(cs, cfg=cfg, V_m3=V, Qd_max=Qd_max, Qc_max=Qc_max, soc0=soc, T_h=T_h, dn_mm=over.get("dn_mm"), end_soc=not kind.endswith("empty"), periodic=not kind.endswith("empty"))
        return dict(Qc=o["Qc"], Qd=o["Qd"], variant="B", T_h=T_h, opt=o)
    raise ValueError(kind)

RUN = {}; t0 = _time.time()
for s in SCEN:
    bk = dict(s["base"]); zero_gt = bk.pop("zero_gt", False)
    cs = base_day_case(**bk)
    if zero_gt: cs["m_gt"] = 0.0
    plan = None if s["sto"] == "none" else storage_plan(s["sto"], cs, s["cfg"])
    RUN[s["id"]] = evaluate(f"{s['id']}", cs, cfg=s["cfg"], storage=plan); RUN[s["id"]]["plan"] = plan; RUN[s["id"]]["spec"] = s
print(f"{len(RUN)} Szenarien gerechnet in {_time.time()-t0:.0f} s; Löser konvergiert in allen Stunden: {all(v['kpi']['Löser ok'] for v in RUN.values())}")
KPI = pd.DataFrame([dict(ID=k, Szenario=v["spec"]["name"], **{c: x for c, x in v["kpi"].items() if c not in ("Szenario",)}) for k, v in RUN.items()])
KPI.to_csv(RESULTS/"szenarien_kennzahlen.csv", index=False)
cols = ["ID","Δp_Kunde,min [bar]","Knoten/Stunde Δp,min","Dauer Δp-Verletzung [h]","Δp-Defizit [bar·h]","nicht versorgt [MWh]","HKW-Durchfluss,max [m³/h]","Erzeugerlimit HKW überschritten [h]","Leitungsauslastung,max [-]","Entladeenergie [MWh]","Ladeenergie [MWh]","Pumpenförderhöhe,max [bar]"]
with pd.option_context("display.width", 250, "display.max_rows", 200): display(KPI[cols].round(2))

### 17.3 Optimierungsprotokoll, Initialisierung und Gewichtungstest

In [ ]:
# ---------------------------------------------------------------- Optimierungsläufe: Protokoll, Initialisierung und Linearisierungsfehler
rows = []
for sid, v in RUN.items():
    o = (v.get("plan") or {}).get("opt")
    if o is not None: rows.append({"Szenario": sid, "Status": o["status"], "Rechenzeit [s]": o["t_s"], "SLP-Runden": len(o["hist"]), "konvergiert": o["converged"], "letzter Schritt [kg/s]": o["hist"].max_schritt_kgs.iloc[-1], "Δp-Schlupf [bar·h]": o["hist"].stufe1_dp_schlupf_barh.iloc[-1],
                                  "HKW über Limit [kg/s·h]": o["hist"].stufe2_HKW_ueber_kgs_h.iloc[-1], "Spitze HKW linear [m³/h]": o["hist"].peak_HKW_lin.iloc[-1]*3600/rho_w(v["cs"].T_vl.iloc[0]), "Spitze HKW nichtlinear [m³/h]": v["kpi"]["HKW-Durchfluss,max [m³/h]"],
                                  "Δp-Defizit nichtlinear [bar·h]": v["kpi"]["Δp-Defizit [bar·h]"]})
OPTLOG = pd.DataFrame(rows); display(OPTLOG.round(2)); OPTLOG.to_csv(RESULTS/"optimierung_protokoll.csv", index=False)
print(f"Alle {len(OPTLOG)} Optimierungen: Status {sorted(set(OPTLOG.Status))}, Rechenzeit gesamt {OPTLOG['Rechenzeit [s]'].sum():.0f} s, Löser: HiGHS (MILP, appsi_highs), SLP-Abbruch bei Schritt < 0,5 kg/s oder stabiler Schlupfzielgröße (Änderung < 2 % über zwei Runden); Anzahl nicht konvergierter Läufe: {int((~OPTLOG['konvergiert']).sum())}.")
# Initialisierungs-Test: drei Startfahrweisen führen zu derselben Fahrweise (Konvexität der linearisierten Teilprobleme, Start nur für die Linearisierung)
cs_t = base_day_case(design=True, growth=0.20); res_st = {}
o_a = optimize_dispatch(cs_t); Qc_r, Qd_r = dispatch_rule(cs_t.R.values, o_a["Ecap"], SPS["Qd_max_MW"], SPS["Qc_max_MW"], loss_MW=0.02); o_b = optimize_dispatch(cs_t, m_start=stor_flows(cs_t, Qc_r, Qd_r, o_a["T_h"]))
o_c = optimize_dispatch(cs_t, m_start=np.random.default_rng(SEED).uniform(-60, 60, len(cs_t)))
ev_ = [evaluate("x", cs_t, storage=dict(Qc=o["Qc"], Qd=o["Qd"], variant="B"))["kpi"] for o in (o_a, o_b, o_c)]
st_tab = pd.DataFrame({"Start": ["Null", "regelbasiert", "zufällig (±60 kg/s)"], "Δp-Defizit [bar·h]": [k["Δp-Defizit [bar·h]"] for k in ev_], "HKW-Spitze [m³/h]": [k["HKW-Durchfluss,max [m³/h]"] for k in ev_], "Entladeenergie [MWh]": [k["Entladeenergie [MWh]"] for k in ev_],
                       "max. Abweichung Qd gegen Null-Start [MW]": [0.0, np.abs(o_b["Qd"]-o_a["Qd"]).max(), np.abs(o_c["Qd"]-o_a["Qd"]).max()]}).round(2); display(st_tab)
# Gewichtungs-Sensitivität der Stufe 3 (Stufen 1–2 bleiben unverändert) am kritischen Bestandstag ohne ungünstige Erzeugerdrücke (dort ist Stufe 1 nicht bindend)
rows = []
cs_w = base_day_case(design=True, unfav=False); rows = []
for lab, w_ in (("Standard", {}), ("Pumpstrom ×10", dict(w_pump=1200.0)), ("Spitze ×0,1", dict(w_peak=1.0)), ("Durchsatz ×10", dict(w_thr=5.0))):
    o = optimize_dispatch(cs_w, weights=w_); k = evaluate("x", cs_w, storage=dict(Qc=o["Qc"], Qd=o["Qd"], variant="B"))["kpi"]
    rows.append({"Gewichtung Stufe 3": lab, "Δp-Defizit [bar·h]": k["Δp-Defizit [bar·h]"], "HKW-Spitze [m³/h]": k["HKW-Durchfluss,max [m³/h]"], "Pumpstrom [MWh]": k["Pumpstrom Speicher [MWh]"], "Entladeenergie [MWh]": k["Entladeenergie [MWh]"]})
display(pd.DataFrame(rows).round(2))

## 18. Ergebnisvergleich ohne und mit Speicher

### 18.1 Paarvergleiche (identische Randbedingungen)

In [ ]:
# ---------------------------------------------------------------- Ergebnisvergleich mit und ohne Speicher (identische Randbedingungen je Paar)
PAIRS = [("S1","S2","Anschluss ohne Betrieb"), ("S1","S3A","Entladung, Variante A (ohne Pumpe)"), ("S1","S3B","Entladung regelbasiert, Variante B"), ("S1","S3O","Entladung optimiert, Variante B"), ("S1","S4A","Nachtladung A"), ("S1","S4B","Nachtladung B"), ("S1","S4K","kritische Ladung zur Spitze"),
         ("S5a","S6a","Ausbau +10 %"), ("S5b","S6b","Ausbau +20 %"), ("S5c","S6c","Ausbau +30 %"), ("S5d","S6d","Ausbau +50 %"), ("S5O","S6O","Ausbau +30 % (nur Ost)"),
         ("S71B0","S71BE","getrennt 342.3, Bestand, Entladung"), ("S71B0","S71BL","getrennt 342.3, Bestand, Ladung"), ("S72B0","S72BE","getrennt 342.1, Bestand, Entladung"), ("S71A0","S71AE","getrennt 342.3, +30 %, Entladung"), ("S72A0","S72AE","getrennt 342.1, +30 %, Entladung"),
         ("S8a0","S8aE","GT-Ausfall"), ("S8b0","S8bE","AVA-Ausfall"), ("S8c0","S8cE","Koppelarmatur zu"), ("S8d0","S8dE","Korridor eingeschränkt"), ("S8e0","S8eE","PSS ohne Förderhöhe"), ("S8f0","S8fE","hohe Rücklauftemperatur")]
KEYS = ["Δp_Kunde,min [bar]","Δp-Defizit [bar·h]","Dauer Δp-Verletzung [h]","nicht versorgt [MWh]","HKW-Durchfluss,max [m³/h]","Erzeugerlimit HKW überschritten [h]","Pumpstrom Speicher [MWh]","Netzwärmeverlust [MWh]","Pumpenförderhöhe,max [bar]"]
rows = []
for a, b, lab in PAIRS:
    ka, kb = RUN[a]["kpi"], RUN[b]["kpi"]; r = {"Paar": f"{a} → {b}", "Beschreibung": lab}
    for k in KEYS:
        r[k + " ohne"] = ka[k]; r[k + " mit"] = kb[k]
    rows.append(r)
CMP = pd.DataFrame(rows); CMP.to_csv(RESULTS/"vergleich_mit_ohne_speicher.csv", index=False)
show = pd.DataFrame({"Paar": CMP.Paar, "Beschreibung": CMP.Beschreibung, "Δp_min ohne": CMP["Δp_Kunde,min [bar] ohne"], "Δp_min mit": CMP["Δp_Kunde,min [bar] mit"], "Δ": CMP["Δp_Kunde,min [bar] mit"] - CMP["Δp_Kunde,min [bar] ohne"],
                     "Defizit ohne [bar·h]": CMP["Δp-Defizit [bar·h] ohne"], "Defizit mit": CMP["Δp-Defizit [bar·h] mit"], "unversorgt ohne [MWh]": CMP["nicht versorgt [MWh] ohne"], "unversorgt mit": CMP["nicht versorgt [MWh] mit"],
                     "HKW-Spitze ohne [m³/h]": CMP["HKW-Durchfluss,max [m³/h] ohne"], "HKW-Spitze mit": CMP["HKW-Durchfluss,max [m³/h] mit"], "Pumpenhöhe [bar]": CMP["Pumpenförderhöhe,max [bar] mit"]}).round(2)
with pd.option_context("display.width", 250, "display.max_rows", 100): display(show)

### 18.2 Zerlegung der Speicherwirkung
Thermische Wirkung, hydraulische Wirkung, aktive Pumpenwirkung, zeitliche Lastverschiebung und geänderter Erzeugereinsatz werden getrennt ausgewiesen. Die reine Anschlusswirkung ist S2 − S1 (muss 0 sein); die Pumpenwirkung ergibt sich aus S3A (Ventile) gegen S3B/S3O (Pumpe).

In [ ]:
# ---------------------------------------------------------------- Wirkungszerlegung (Arbeitsauftrag Kap. 16): thermisch / hydraulisch / aktive Pumpe / Lastverschiebung / Erzeugereinsatz
def decomp(base_id, sto_id, noA_id=None):
    a, b = RUN[base_id], RUN[sto_id]; ka, kb = a["kpi"], b["kpi"]; ha, hb = a["h"], b["h"]; cs = a["cs"]
    cpm = cp_w(0.5*(cs.T_vl.values + cs.T_rl.values)); dT = cs.T_vl.values - cs.T_rl.values
    Qhkw_a = ha.flow_hkw_m3h.values*rho_w(cs.T_vl.values)/3600*cpm*dT/1e3; Qhkw_b = hb.flow_hkw_m3h.values*rho_w(cs.T_vl.values)/3600*cpm*dT/1e3
    Qava_a = ha.flow_ava_m3h.values*rho_w(cs.T_vl.values)/3600*cpm*dT/1e3; Qava_b = hb.flow_ava_m3h.values*rho_w(cs.T_vl.values)/3600*cpm*dT/1e3
    disch = hb.Qd.values > 0.5; chg = hb.Qc.values > 0.5
    return {"Fall": f"{base_id} → {sto_id}",
            "Thermisch: lokal eingespeiste Speicherwärme [MWh]": float(hb.Qd.sum()), "Thermisch: Netz-Wärmeverlust Δ [MWh]": kb["Netzwärmeverlust [MWh]"] - ka["Netzwärmeverlust [MWh]"],
            "Hydraulisch: ΔHKW-Durchfluss in Entladestunden, Mittel [m³/h]": float((hb.flow_hkw_m3h - ha.flow_hkw_m3h)[disch].mean()) if disch.any() else 0.0,
            "Hydraulisch: Δp_min in Entladestunden, Mittel [bar]": float((hb.dp_min - ha.dp_min)[disch].mean()) if disch.any() else 0.0,
            "Aktive Pumpe: Förderhöhe max [bar]": kb["Pumpenförderhöhe,max [bar]"], "Aktive Pumpe: Pumpstrom Speicher [MWh]": kb["Pumpstrom Speicher [MWh]"],
            "Lastverschiebung: Δ HKW-Durchfluss in Ladestunden, Mittel [m³/h]": float((hb.flow_hkw_m3h - ha.flow_hkw_m3h)[chg].mean()) if chg.any() else 0.0,
            "Lastverschiebung: Δp_min in Ladestunden, Mittel [bar]": float((hb.dp_min - ha.dp_min)[chg].mean()) if chg.any() else 0.0,
            "Erzeugereinsatz: Δ HKW-Wärme [MWh]": float((Qhkw_b - Qhkw_a).sum()), "Erzeugereinsatz: Δ AVA-Wärme [MWh]": float((Qava_b - Qava_a).sum()), "Erzeugereinsatz: Δ HKW-Spitze [m³/h]": kb["HKW-Durchfluss,max [m³/h]"] - ka["HKW-Durchfluss,max [m³/h]"],
            "Gesamtwirkung Δp-Defizit [bar·h]": kb["Δp-Defizit [bar·h]"] - ka["Δp-Defizit [bar·h]"]}
DEC = pd.DataFrame([decomp("S1", "S3O"), decomp("S5a", "S6a"), decomp("S5b", "S6b"), decomp("S1", "S3B"), decomp("S1", "S4B")]).set_index("Fall").T.round(2)
display(DEC); DEC.to_csv(RESULTS/"wirkungszerlegung.csv")
# Isolierte Pumpenwirkung: S3A (Ventile, Entladung unmöglich) gegen S3B/S3O (Pumpe) – der gesamte Entlastungseffekt hängt an der Pumpe
print(f"Aktive Pumpenwirkung: ohne Pumpe (S3A) Δp-Defizit {RUN['S3A']['kpi']['Δp-Defizit [bar·h]']:.2f} bar·h (= S1 {RUN['S1']['kpi']['Δp-Defizit [bar·h]']:.2f}); mit Pumpe optimiert (S3O) {RUN['S3O']['kpi']['Δp-Defizit [bar·h]']:.2f} bar·h. Die Entlastung entsteht ausschließlich durch die Speicherpumpe ({RUN['S3O']['kpi']['Pumpenförderhöhe,max [bar]']:.1f} bar max.), nicht durch den Anschluss (S2 ≡ S1).")

### 18.3 Betriebs- und Auslegungsdruck, Siedesicherheit, PN-Stufen
Der Speicherdruck wird aus dem **maximal auftretenden Netzdruck** am MAN und der Pumpen-Förderhöhe abgeleitet, nicht aus 120 °C allein. Schnelle Druckstöße (Wasserhammer, Armaturenschaltung) bildet die quasistationäre Rechnung **nicht** ab (siehe Kap. 22/23).

In [ ]:
# ---------------------------------------------------------------- Betriebs- und Auslegungsdruck, Siedesicherheit, PN-Stufen am Speicher (direkte Netzeinbindung)
iM = nidx["MAN"]; z_s = Z_NODE[iM]; h_t = SPS["h_tank_m"]; T_h = SPS["T_max_C"]
pv_man = np.concatenate([v["pv"][:, iM] for v in RUN.values()]); pr_man = np.concatenate([v["pr"][:, iM] for v in RUN.values()])
pvmax_all = max(v["pv"][:, ~np.isnan(Z_NODE)].max() for v in RUN.values())
hp_max = max(v["kpi"]["Pumpenförderhöhe,max [bar]"] for v in RUN.values())
p_static_hkw = 8.4; rho90 = float(rho_w(90.0))
p_static_man = p_static_hkw - rho90*G*(z_s - 474.68)/1e5                                     # Ruhedruck WV640: 8,4 bar Ü am HKW (474,68 m NN) → am MAN-Höhenniveau
p_boil_top = float(psat_bar(T_h)) + SPS["dT_boil"] - P_ATM                                   # erforderlicher Überdruck am Speicherkopf bei 120 °C (p_sat + 1 bar Sicherheit)
p_top_min = pr_man.min() - rho90*G*h_t/1e5                                                   # Druck am Kopf, wenn der Speicherfuß auf dem kleinsten Rücklaufdruck liegt (Worst Case)
p_top_static = p_static_man - rho90*G*h_t/1e5
rows = [("Ruhedruck am MAN-Niveau (aus WV640, 8,4 bar Ü am HKW)", p_static_man, "bar Ü"), ("kleinster Rücklaufdruck am MAN-Knoten (alle Szenarien)", pr_man.min(), "bar Ü"), ("größter Vorlaufdruck am MAN-Knoten (alle Szenarien)", pv_man.max(), "bar Ü"),
        ("größter Vorlaufdruck im Netz (alle Szenarien, Knoten mit bekannter Höhe)", pvmax_all, "bar Ü"), ("größte erforderliche Speicherpumpen-Förderhöhe (alle Szenarien)", hp_max, "bar"),
        (f"Sättigungsdruck bei {T_h:.0f} °C (Überdruck, p_sat − 1,013)", float(psat_bar(T_h)) - P_ATM, "bar Ü"), (f"Mindest-Überdruck am Speicherkopf bei {T_h:.0f} °C (p_sat + 1 bar Abstand)", p_boil_top, "bar Ü"),
        (f"Druck am Speicherkopf bei statischem Ruhedruck ({h_t:.0f} m Behälterhöhe)", p_top_static, "bar Ü"), ("Druck am Speicherkopf im ungünstigsten Fall (Fuß auf kleinstem Rücklaufdruck)", p_top_min, "bar Ü"),
        ("Siedeabstand am Speicherkopf im ungünstigsten Fall (p_abs − p_sat)", p_top_min + P_ATM - float(psat_bar(T_h)), "bar")]
pd_tab = pd.DataFrame(rows, columns=["Größe", "Wert", "Einheit"]); pd_tab["Wert"] = pd_tab.Wert.round(2); display(pd_tab); pd_tab.to_csv(RESULTS/"druckauslegung_speicher.csv", index=False)
p_op_min = max(p_boil_top + rho90*G*h_t/1e5, 0.0)                                            # Betriebsdruck am Speicherfuß mindestens
p_des = 1.10*(pv_man.max() + 0.0) ; p_des_pump = 1.10*(pv_man.max() + hp_max)
print(f"Betriebsdruck am Speicherfuß: mindestens {p_op_min:.1f} bar Ü (Siedeabstand bei {T_h:.0f} °C), zulässig bis zum Netz-Vorlaufdruck am MAN (max. {pv_man.max():.1f} bar Ü). "
      f"Auslegungsdruck (Fuß, +10 % Reserve): {p_des:.1f} bar Ü ohne bzw. {p_des_pump:.1f} bar Ü mit Berücksichtigung der Pumpen-Förderhöhe (Absperrdruck der Pumpe ist separat nachzuweisen). "
      f"MAN liegt im PN25-Bereich (Kap. 9/10): {'PN25 wird nicht berührt' if p_des_pump < 25 else 'PN25 wird berührt'}; die PN16-Grenze (West, hydraulisch getrennt) wird {'nicht ' if p_des_pump < 16 else ''}überschritten.")
# Siedesicherheit im Netz (alle Szenarien): kleinster Abstand p_abs − p_sat an allen Knoten
marg = pd.DataFrame({k: [v["kpi"]["Siedeabstand min [bar]"], v["kpi"]["Knoten/Stunde Siede"]] for k, v in RUN.items()}, index=["Siedeabstand min [bar]", "Knoten/Stunde"]).T
print(f"Siedesicherheit im Netz: kleinster Abstand p_abs − p_sat über alle {len(RUN)} Szenarien: {marg['Siedeabstand min [bar]'].astype(float).min():.1f} bar (Anforderung ≥ {SPS['dT_boil']:.1f} bar) bei {marg.loc[marg['Siedeabstand min [bar]'].astype(float).idxmin(), 'Knoten/Stunde']} in {marg['Siedeabstand min [bar]'].astype(float).idxmin()}.")

### 18.4 Abbildungen

In [ ]:
# ---------------------------------------------------------------- Abbildungen: Netzkarten (schematisch, Koordinaten aus Kap. 10.6)
import matplotlib.cm as cm
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
POS = kn2.set_index("node_id")[["x_km", "y_km"]].reindex(NODES).values; TYP = kn2.set_index("node_id").typ.reindex(NODES).values
GCOL = {"SL1": "#1b7a6d", "SL5": "#c8115f", "OSTSTUB": "#e67e22", "SL6": "#2196f3", "SL7": "#7fb069", "SL2": "#8e44ad", "SL3": "#2c3fe0", "SL4": "#f08080", "M12": "#999", "M23": "#999", "M24": "#999", "M34": "#999", "LAND": "#555", "MANA": "#000", "W1": "#16a085"}
EDGE_G = [GN[grp_of_edge[j]] for j in range(len(E_IDS))]
def draw_edges(ax, widths=None, colors=None, cmap=None, norm=None, dashed=()):
    segs = [[POS[E_VON[j]], POS[E_NACH[j]]] for j in range(len(E_IDS))]
    lw = np.full(len(E_IDS), 2.0) if widths is None else widths
    col = [GCOL[g] for g in EDGE_G] if colors is None else cmap(norm(colors))
    lc = LineCollection(segs, linewidths=lw, colors=col, zorder=1); ax.add_collection(lc)
    for j in dashed: ax.plot(*zip(*segs[j]), color="k", lw=1.0, ls="--", zorder=2)
def label_nodes(ax, names, fs=6.5, dx=0.05, dy=0.05):
    for n in names: ax.annotate(n, POS[nidx[n]], xytext=(POS[nidx[n]][0]+dx, POS[nidx[n]][1]+dy), fontsize=fs, zorder=6)
def base_axes(ax, title):
    ax.set_title(title, fontsize=9); ax.set_xlim(-5.4, 5.6); ax.set_ylim(-5.2, 4.9); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s_ in ax.spines.values(): s_.set_visible(False)
def node_scatter(ax, vals, cmap, vmin, vmax, nodes=None, size=70, edge="k"):
    idx = np.arange(len(NODES)) if nodes is None else np.array([nidx[n] for n in nodes])
    return ax.scatter(POS[idx, 0], POS[idx, 1], c=vals, cmap=cmap, vmin=vmin, vmax=vmax, s=size, edgecolors=edge, linewidths=.6, zorder=5)

hp_ = 8   # Spitzenstunde im Basistag (08:00–09:00)
fig, axs = plt.subplots(2, 3, figsize=(17, 11.5))
# (a) Topologie, Infrastruktur, PN-Zonen, Messstellen, Separierungsstellen
ax = axs[0, 0]; base_axes(ax, "(a) Topologie: Stammleitungen, Erzeuger, Pumpstationen, MAN/UPM, Messstellen, Trennstellen"); draw_edges(ax)
sty = {"erzeuger": ("^", "#c0392b", 110), "pumpstation": ("D", "#16a085", 80), "verbraucher": ("o", "#2874a6", 60), "koppelstelle": ("o", "#ffffff", 40)}
for t_, (mk, c_, s_) in sty.items():
    ii = [i for i, tt in enumerate(TYP) if tt == t_]; ax.scatter(POS[ii, 0], POS[ii, 1], marker=mk, c=c_, s=s_, edgecolors="k", linewidths=.7, zorder=5)
ax.scatter(*POS[nidx["MAN"]], marker="*", c="gold", s=330, edgecolors="k", zorder=7); ax.annotate("MAN/UPM\nSpeicher (Annahme)", POS[nidx["MAN"]], xytext=(POS[nidx["MAN"]][0]-2.5, POS[nidx["MAN"]][1]+0.6), fontsize=8, arrowprops=dict(arrowstyle="-", lw=.6), zorder=8)
ms_nodes = sorted(set(MESS.node)); ax.scatter(POS[ms_nodes, 0] + 0.12, POS[ms_nodes, 1] - 0.12, marker="s", c="none", edgecolors="#34495e", s=22, zorder=6)
for gname, eid, txt in (("s1", "SL5b", "Trennung 342.3"), ("s2", "SL5a", "Trennung 342.1"), ("l1", "K_SL6_SL3", "Landung 338"), ("l2", "K_SL7_SL3", "Landung 818")):
    j = eidx[eid]; mid = 0.5*(POS[E_VON[j]] + POS[E_NACH[j]]); ax.scatter(*mid, marker="X", c="red", s=70, edgecolors="k", zorder=7); ax.annotate(txt, mid, xytext=(mid[0]+0.1, mid[1]-0.28), fontsize=6.5, color="red")
ax.add_patch(plt.Rectangle((POS[nidx["WEST"] if "WEST" in nidx else 0][0]-0.5, -0.0), 0, 0, fill=False)) if False else None
label_nodes(ax, [n for n in NODES if TYP[nidx[n]] in ("erzeuger", "pumpstation")], fs=7)
ax.text(-5.3, -4.9, "PN25: gesamtes Verbundnetz; PN16: Westnetz (hydraulisch über WÜ getrennt, nicht modelliert)", fontsize=7)
ax.legend(handles=[Line2D([], [], marker="^", c="#c0392b", ls="", mec="k", label="Erzeuger"), Line2D([], [], marker="D", c="#16a085", ls="", mec="k", label="Pumpstation"), Line2D([], [], marker="o", c="#2874a6", ls="", mec="k", label="Kundenknoten"),
                   Line2D([], [], marker="o", c="w", ls="", mec="k", label="Koppelstelle"), Line2D([], [], marker="s", c="none", ls="", mec="#34495e", label="Messstelle"), Line2D([], [], marker="X", c="red", ls="", mec="k", label="Trenn-/Schaltstelle (S7)")], loc="lower right", fontsize=7)
# (b) Massenstrom/Auslastung Vorlauf, S1 Spitzenstunde
ax = axs[0, 1]; base_axes(ax, "(b) Vorlauf-Massenstrom und Auslastung, S1 08 Uhr (Breite ∝ ṁ, Farbe = ṁ/ṁ(2,5 m/s))")
e1 = RUN["S1"]; mvh = np.abs(e1["res"]["mv"][hp_]); ut = e1["util"][hp_]; draw_edges(ax, widths=0.5 + 6*mvh/mvh.max(), colors=ut, cmap=cm.YlOrRd, norm=plt.Normalize(0, 1.6))
sm = cm.ScalarMappable(norm=plt.Normalize(0, 1.6), cmap=cm.YlOrRd); plt.colorbar(sm, ax=ax, fraction=.035, label="Auslastung [-]")
for j in np.argsort(-mvh)[:6]: mid = 0.5*(POS[E_VON[j]] + POS[E_NACH[j]]); ax.annotate(f"{E_IDS[j]} {mvh[j]:.0f} kg/s", mid, fontsize=6.5, color="#333")
ax.scatter(*POS[nidx["MAN"]], marker="*", c="gold", s=250, edgecolors="k", zorder=7)
# (c) Differenzdruck an Kundenknoten S1
def dp_panel(ax, sid, h, title):
    base_axes(ax, title); draw_edges(ax, colors=np.zeros(len(E_IDS)), cmap=lambda x: np.tile([0.8, 0.8, 0.8, 1.0], (len(E_IDS), 1)), norm=lambda x: x)
    v = RUN[sid]["dpk"][h]; sc = node_scatter(ax, v, cm.RdYlGn, 0.0, 2.0, KUNDEN, size=110)
    bad = [k for k, x in zip(KUNDEN, v) if x < SPS["dp_kunde_min_bar"]]
    for k, x in zip(KUNDEN, v): ax.annotate(f"{k}\n{x:.2f}", POS[nidx[k]], xytext=(POS[nidx[k]][0]+.08, POS[nidx[k]][1]+.05), fontsize=6.3, color=("red" if k in bad else "k"), zorder=8)
    return sc
sc = dp_panel(axs[0, 2], "S1", hp_, "(c) Kunden-Differenzdruck [bar], S1 08 Uhr (rot: < 1,0 bar)"); plt.colorbar(sc, ax=axs[0, 2], fraction=.035, label="Δp [bar]")
# (d) Differenz mit − ohne Speicher (optimierte Fahrweise) in der Stunde mit größter Entladung
h_d = int(np.argmax(RUN["S3O"]["h"].Qd.values)); ax = axs[1, 0]; base_axes(ax, f"(d) Δ(Δp) mit − ohne Speicher (S3O − S1), {h_d:02d} Uhr [bar]"); draw_edges(ax, colors=np.zeros(len(E_IDS)), cmap=lambda x: np.tile([0.8, 0.8, 0.8, 1.0], (len(E_IDS), 1)), norm=lambda x: x)
d = RUN["S3O"]["dpk"][h_d] - RUN["S1"]["dpk"][h_d]; sc = node_scatter(ax, d, cm.PuOr, -0.3, 0.3, KUNDEN, size=110); plt.colorbar(sc, ax=ax, fraction=.035, label="Δ(Δp) [bar]")
for k, x in zip(KUNDEN, d): ax.annotate(f"{k} {x:+.2f}", POS[nidx[k]], xytext=(POS[nidx[k]][0]+.08, POS[nidx[k]][1]+.05), fontsize=6.3, zorder=8)
ax.scatter(*POS[nidx["MAN"]], marker="*", c="gold", s=300, edgecolors="k", zorder=7, alpha=.5)
# (e) Vorlauftemperatur S3O
ax = axs[1, 1]; base_axes(ax, f"(e) Vorlauftemperatur [°C], S3O {h_d:02d} Uhr (Mischung + Rohrverlust)"); draw_edges(ax, colors=np.zeros(len(E_IDS)), cmap=lambda x: np.tile([0.8, 0.8, 0.8, 1.0], (len(E_IDS), 1)), norm=lambda x: x)
Tv = RUN["S3O"]["th"]["Tv"][h_d]; sc = node_scatter(ax, Tv, cm.coolwarm, 100, 122, None, size=90); plt.colorbar(sc, ax=ax, fraction=.035, label="T_VL [°C]")
for n in ("MAN", "SEC2", "SEC4", "SL5O", "SL7G", "HKW"): ax.annotate(f"{Tv[nidx[n]]:.1f}", POS[nidx[n]], xytext=(POS[nidx[n]][0]+.08, POS[nidx[n]][1]+.05), fontsize=6.5, zorder=8)
# (f) Ausbau +20 % ohne Speicher
sc = dp_panel(axs[1, 2], "S5b", hp_, "(f) Kunden-Differenzdruck [bar], Ausbau +20 % ohne Speicher, 08 Uhr"); plt.colorbar(sc, ax=axs[1, 2], fraction=.035, label="Δp [bar]")
fig.suptitle("Netzkarten (schematische Koordinaten, Stadtbach): Topologie, Fluss, Druck, Speicherwirkung", y=.995); fig.tight_layout(); fig.savefig(FIG_DIR/"18_netzkarten.png", dpi=120); plt.show()

In [ ]:
# ---------------------------------------------------------------- Zeitreihen, Dauerlinien, Rangliste
def soc_series(v):
    p = v["plan"]; Th = p["T_h"]; Ecap = storage_capacity_MWh(SPS["V_m3"], Th - v["cs"].T_rl.mean(), 0.5*(Th + v["cs"].T_rl.mean()), SPS["eta_strat"]); loss = storage_loss_MW(0.5*(Th + v["cs"].T_rl.mean()))
    E0 = periodic_E0(p["Qc"], p["Qd"], Ecap, loss); return storage_sim(p["Qc"], p["Qd"], E0, Ecap, loss)/Ecap, Ecap
fig, axs = plt.subplots(4, 2, figsize=(15, 13), sharex=True)
for c_, (a, b, lab) in enumerate((("S1", "S3O", "Bestand (Auslegungs-RB, ungünstige Erzeugerdrücke)"), ("S5a", "S6a", "Ausbau +10 %"))):
    ha, hb = RUN[a]["h"], RUN[b]["h"]; ax = axs[0, c_]
    ax.plot(ha.h, ha.dp_min, color=COL["grau"], label=f"{a} ohne Speicher"); ax.plot(hb.h, hb.dp_min, color=COL["ok"], label=f"{b} mit Speicher"); ax.axhline(SPS["dp_kunde_min_bar"], color="red", ls="--", lw=1, label="Δp_min Kunde")
    ax.fill_between(ha.h, ha.dp_min, SPS["dp_kunde_min_bar"], where=ha.dp_min < SPS["dp_kunde_min_bar"], color="red", alpha=.12); ax.set_ylabel("min. Kunden-Δp [bar]"); ax.set_title(lab); ax.legend(fontsize=7, ncol=3)
    ax = axs[1, c_]; ax.plot(ha.h, ha.flow_hkw_m3h, color=COL["grau"], label="HKW ohne"); ax.plot(hb.h, hb.flow_hkw_m3h, color=COL["vl"], label="HKW mit"); ax.plot(hb.h, hb.flow_ava_m3h, color=COL["rl"], ls=":", label="AVA mit")
    ax.axhline(CAP["HKW"]/rho_w(RUN[a]["cs"].T_vl.iloc[0])*3600, color="k", ls="--", lw=.8, label="HKW-Limit 2025 (P99,9)"); ax.set_ylabel("Durchfluss [m³/h]"); ax.legend(fontsize=7, ncol=2)
    ax = axs[2, c_]; ax.bar(hb.h, hb.Qd, color=COL["vl"], label="Entladen [MW]"); ax.bar(hb.h, -hb.Qc, color=COL["rl"], label="Laden [MW]"); ax2 = ax.twinx(); soc, Ec = soc_series(RUN[b]); ax2.plot(hb.h, 100*soc, color="k", label="Füllstand"); ax2.set_ylabel("Füllstand [%]"); ax2.grid(False); ax.set_ylabel("Leistung [MW]"); ax.legend(loc="upper left", fontsize=7)
    ax = axs[3, c_]; ax.plot(hb.h, hb.Hp_bar, color="#8e44ad", label="Pumpen-Förderhöhe [bar]"); ax.plot(hb.h, hb.T_lief/30, color="#d35400", label="T_Lieferung/30 [K]"); ax.set_ylabel("Förderhöhe [bar]"); ax.set_xlabel("Stunde des Tages"); ax.legend(fontsize=7)
fig.suptitle("Tagesgänge mit und ohne Speicher (optimierte Fahrweise, Variante B, 5 000 m³ / 40 MW)", y=.995); fig.tight_layout(); fig.savefig(FIG_DIR/"18_tagesgaenge.png"); plt.show()

# Dauerlinien (Druck, Differenzdruck, Temperatur): 24-h-Szenarien und gemessenes Jahr 2025
fig, axs = plt.subplots(1, 3, figsize=(16, 4.6))
for sid, c_, ls in (("S1", COL["grau"], "-"), ("S3O", COL["ok"], "-"), ("S5a", COL["warn"], "--"), ("S6a", COL["vl"], "--")):
    h = RUN[sid]["h"]; x = np.linspace(0, 100, len(h))
    axs[0].plot(x, np.sort(h.p_VL_min.values), ls, color=c_, label=sid); axs[1].plot(x, np.sort(h.dp_min.values), ls, color=c_, label=sid); axs[2].plot(x, np.sort(h.T_lief.values)[::-1], ls, color=c_, label=sid)
mid_st = ["Fuggerstr", "Schlettererstr", "Theodor-Heuss-Platz", "Lechhauser_Str"]; dpm = pd.concat([(acron[r.vl] - acron[r.rl]).rename(r.station) for _, r in MESS.iterrows() if r.station in mid_st], axis=1).min(axis=1).dropna()
axs[1].plot(np.linspace(0, 100, len(dpm)), np.sort(dpm.values), color="k", lw=1, label="Messung 2025, Jahr (min. Mitte)")
axs[1].axhline(SPS["dp_kunde_min_bar"], color="red", ls=":", lw=1)
for ax, t_, y_ in zip(axs, ("Kleinster Vorlaufdruck im Netz", "Kleinster Kunden-Differenzdruck", "Kleinste Lieferungstemperatur (Kundenknoten)"), ("p_VL,min [bar Ü]", "Δp [bar]", "T [°C]")): ax.set_title(t_); ax.set_ylabel(y_); ax.set_xlabel("Anteil der Stunden [%] (aufsteigend sortiert)"); ax.legend(fontsize=7)
axs[2].set_xlabel("Anteil der Stunden [%] (absteigend sortiert)"); fig.tight_layout(); fig.savefig(FIG_DIR/"18_dauerlinien.png"); plt.show()

# Kritische Knoten: Rangliste (kleinster Kunden-Δp je Knoten)
rk = pd.DataFrame({sid: RUN[sid]["dpk"].min(0) for sid in ("S1", "S3O", "S5a", "S6a", "S5b", "S6b")}, index=KUNDEN).sort_values("S1"); rk["Nutzen S1→S3O [bar]"] = rk.S3O - rk.S1; rk["Nutzen S5a→S6a [bar]"] = rk.S6a - rk.S5a
display(rk.round(2)); rk.to_csv(RESULTS/"kritische_knoten_rangliste.csv")
fig, ax = plt.subplots(figsize=(10, 4.5)); x = np.arange(len(rk)); w = .2
for k, (sid, c_) in enumerate((("S1", COL["grau"]), ("S3O", COL["ok"]), ("S5a", COL["warn"]), ("S6a", COL["vl"]))): ax.bar(x + (k-1.5)*w, rk[sid], w, color=c_, label=sid)
ax.axhline(SPS["dp_kunde_min_bar"], color="red", ls="--"); ax.set_xticks(x); ax.set_xticklabels(rk.index); ax.set_ylabel("min. Kunden-Δp über 24 h [bar]"); ax.set_title("Rangliste kritischer Knoten (aufsteigend nach S1) und Speichernutzen je Knoten"); ax.legend(ncol=4); fig.tight_layout(); fig.savefig(FIG_DIR/"18_rangliste_knoten.png"); plt.show()

In [ ]:
# Szenariodifferenzen (mit − ohne Speicher) und Heizkurve gegen Messdaten
fig, axs = plt.subplots(1, 3, figsize=(17, 5.2)); yy = np.arange(len(CMP))
d1 = CMP["Δp_Kunde,min [bar] mit"] - CMP["Δp_Kunde,min [bar] ohne"]; axs[0].barh(yy, d1, color=np.where(d1 >= 0, COL["ok"], COL["vl"])); axs[0].set_title("Δ kleinster Kunden-Δp [bar]")
d2 = 100*(CMP["Δp-Defizit [bar·h] mit"]/CMP["Δp-Defizit [bar·h] ohne"].replace(0, np.nan) - 1); axs[1].barh(yy, d2.fillna(0), color=np.where(d2.fillna(0) <= 0, COL["ok"], COL["vl"])); axs[1].set_title("Δ Δp-Defizit [%] (negativ = besser)")
d3 = CMP["HKW-Durchfluss,max [m³/h] mit"] - CMP["HKW-Durchfluss,max [m³/h] ohne"]; axs[2].barh(yy, d3, color=np.where(d3 <= 0, COL["ok"], COL["vl"])); axs[2].set_title("Δ HKW-Spitzendurchfluss [m³/h] (negativ = Entlastung)")
for ax in axs: ax.set_yticks(yy); ax.set_yticklabels(CMP.Paar + "  " + CMP.Beschreibung.str[:28], fontsize=6.5); ax.invert_yaxis(); ax.axvline(0, color="k", lw=.8)
fig.suptitle("Wirkung des Speichers: Szenariodifferenz (mit − ohne) unter identischen Randbedingungen", y=.995); fig.tight_layout(); fig.savefig(FIG_DIR/"18_szenariodifferenzen.png"); plt.show()
# Heizkurve gegen Messdaten und Auslegungspunkt
fig, ax = plt.subplots(figsize=(8, 4.4)); vv = BC[BC.valid]; ax.scatter(vv.Ta, vv.T_vl, s=3, alpha=.25, color=COL["grau"], label="Messung HKW-Vorlauf 2025 (stündlich)")
xs = np.linspace(-16, 25, 100); ax.plot(xs, T_CURVE(xs), color=COL["vl"], lw=2, label="lineare Heizkurve (120 °C @ −10 … 90 °C @ +20)"); ax.scatter([-14], [float(T_CURVE(-14))], marker="*", s=200, color="k", zorder=5, label="Auslegungspunkt S1 (−14 °C → 120 °C)")
ax.set_xlabel("Außentemperatur [°C]"); ax.set_ylabel("Vorlauftemperatur [°C]"); ax.legend(fontsize=7); ax.set_title("Heizkurve gegen Messdaten (Details Kap. 8)"); fig.tight_layout(); fig.savefig(FIG_DIR/"18_heizkurve_auslegung.png"); plt.show()

### 18.5 Historische Drucksituationen 2025 (S0)
Geprüft wird, ob die Messdaten 2025 anhaltende kritische Situationen enthalten und ob das Modell sie reproduziert.

In [ ]:
# ---------------------------------------------------------------- Historische kritische Drucksituationen 2025
mid_st = ["Fuggerstr","Schlettererstr","Theodor-Heuss-Platz","Lechhauser_Str","Hans-Boeckler-Str","Kurt-Schumacher-Str","Am_MIttleren_Moos"]
dpm = pd.concat([(acron[r.vl]-acron[r.rl]).rename(r.station) for _, r in MESS.iterrows() if r.station in mid_st], axis=1); mn = dpm.min(axis=1).dropna()
rows = []
for thr in (1.0, 0.8, 0.6, 0.3):
    low = mn[mn < thr]; runs = (low.index.to_series().diff() != pd.Timedelta("1h")).cumsum(); L = runs.value_counts().max() if len(low) else 0
    rows.append({"Schwelle [bar]": thr, "Stunden": len(low), "Tage": len(set(low.index.date)), "längste zusammenhängende Dauer [h]": int(L), "davon Nullwerte (0,0 bar)": int((dpm.loc[low.index].min(axis=1) == 0).sum())})
ev_tab = pd.DataFrame(rows); display(ev_tab)
low = mn[mn < 0.8]; hh = [t for t in low.index if t in BC.index and BC.valid[t]]
if hh:
    s = simulate(theta_cal, pd.DatetimeIndex(hh)); pv_, pr_ = station_pressures(s); out = []
    for j, r in MESS.iterrows():
        if r.station in mid_st:
            me = (acron.loc[hh, r.vl] - acron.loc[hh, r.rl]).values; mo = pv_[:, j] - pr_[:, j]; out.append(pd.DataFrame(dict(zeit=hh, station=r.station, gemessen=me, modell=mo)))
    evd = pd.concat(out); evd = evd[evd.gemessen < 0.8]
    print(f"Ereignisstunden mit gemessenem Kunden-Δp < 0,8 bar: {len(hh)}; an den betroffenen Stationen sagt das Modell im Mittel {evd.modell.mean():.2f} bar voraus (gemessen {evd.gemessen.mean():.2f} bar); Modell < 1,0 bar in {100*(evd.modell < 1.0).mean():.0f} % dieser Fälle.")
hq_ = BC.index[BC.valid & (BC.R >= BC.R[BC.valid].quantile(0.9))]; mq = mn.reindex(hq_).dropna()
print(f"Hochlaststunden (oberes 10 %-Quantil, n = {len(mq)}): gemessener kleinster Kunden-Δp Mitte/Ost P5 {mq.quantile(.05):.2f} bar, Minimum {mq.min():.2f} bar; Modell (S0, Basistag) kleinster Kunden-Δp {RUN['S0']['kpi']['Δp_Kunde,min [bar]']:.2f} bar (bias-korrigiert) in {RUN['S0']['kpi']['Dauer Δp-Verletzung [h]']} h unter 1,0 bar.")
EVENTS = ev_tab

## 19. Netzausbau und Sensitivitäten
### 19.1 Lastzuwachs 0…50 % (S5/S6) und Druckanhebung als Alternative
Konkrete Ausbauvorhaben liegen in den Unterlagen nicht vor (Daten­lücke); es werden daher transparente Zuwächse von 10/20/30/50 % (proportional auf alle Sektionen, MAN und West-Übergabe) sowie +30 % nur im Osten gerechnet. Erzeugerdrücke bleiben auf den Randwerten festgehalten; die dafür nötige **Druckanhebung** an HKW und AVA (Vorlauf) wird als Alternativmaßnahme ausgewiesen.

In [ ]:
# ---------------------------------------------------------------- Netzausbau: Lastzuwachs 0…50 % ohne/mit Speicher und Druckanhebung der Erzeuger als Alternative
GR = np.round(np.arange(0.0, 0.501, 0.05), 2); rows = []; t0 = _time.time()
for g in GR:
    cs = base_day_case(design=True, growth=g); e0 = evaluate("no", cs); o = optimize_dispatch(cs); e1 = evaluate("mit", cs, storage=dict(Qc=o["Qc"], Qd=o["Qd"], variant="B"))
    for lab, e in (("ohne Speicher", e0), ("mit Speicher (opt.)", e1)):
        k = e["kpi"]; rows.append({"Zuwachs": g, "Fall": lab, "Δp_Kunde,min [bar]": k["Δp_Kunde,min [bar]"], "Δp-Defizit [bar·h]": k["Δp-Defizit [bar·h]"], "Dauer Δp-Verletzung [h]": k["Dauer Δp-Verletzung [h]"], "nicht versorgt [MWh]": k["nicht versorgt [MWh]"],
                                   "HKW-Limit überschritten [h]": k["Erzeugerlimit HKW überschritten [h]"], "Überschreitung HKW,max [MW]": k["Überschreitung HKW,max [MW]"], "Überschreitung AVA,max [MW]": k["Überschreitung AVA,max [MW]"],
                                   "HKW-Durchfluss,max [m³/h]": k["HKW-Durchfluss,max [m³/h]"], "Leitungsauslastung,max [-]": k["Leitungsauslastung,max [-]"], "Entladeenergie [MWh]": k["Entladeenergie [MWh]"]})
GRW = pd.DataFrame(rows); GRW.to_csv(RESULTS/"ausbau_kennzahlen.csv", index=False)
print(f"Ausbau-Sensitivität in {_time.time()-t0:.0f} s"); display(GRW.pivot(index="Zuwachs", columns="Fall", values=["Δp-Defizit [bar·h]","nicht versorgt [MWh]","HKW-Limit überschritten [h]"]).round(1))

# Alternative Maßnahme: Anhebung des Vorlaufdrucks an HKW und AVA (Bisektion auf Δp-Defizit < 0,05 bar·h)
def needed_raise(g, storage=None, tol=0.05, hi=12.0):
    def f(dpv):
        cs = base_day_case(design=True, growth=g, dp_hkw=dpv, dp_ava=dpv); return evaluate("x", cs, storage=storage)["kpi"]
    k = f(0.0)
    if k["Δp-Defizit [bar·h]"] < tol: return 0.0, k
    if f(hi)["Δp-Defizit [bar·h]"] >= tol: return np.nan, k
    lo = 0.0
    for _ in range(14):
        mid = 0.5*(lo + hi)
        if f(mid)["Δp-Defizit [bar·h]"] < tol: hi = mid
        else: lo = mid
    return hi, k
rows = []
for g in (0.0, 0.10, 0.20, 0.30, 0.50):
    cs = base_day_case(design=True, growth=g); o = optimize_dispatch(cs); sto = dict(Qc=o["Qc"], Qd=o["Qd"], variant="B")
    r0, _ = needed_raise(g, None); r1, _ = needed_raise(g, sto)
    rows.append({"Zuwachs": g, "erforderliche Druckanhebung HKW+AVA ohne Speicher [bar]": r0, "… mit Speicher (opt.) [bar]": r1, "Einsparung [bar]": r0 - r1})
RAISE = pd.DataFrame(rows).round(2); display(RAISE); RAISE.to_csv(RESULTS/"druckanhebung_bedarf.csv", index=False)
print("Hinweis: Die Druckanhebung ist nur so weit zulässig, wie Erzeugerpumpen, Abschaltdrücke (WV640) und PN-Stufen es erlauben; diese Grenzen sind nicht vollständig bekannt (Datenlücke).")

### 19.2 Auslegungsvariation des Speichers (S9)
Variiert werden Volumen, Ent-/Ladeleistung, Anschlussdurchmesser, Pumpenförderhöhe (Begrenzung), nutzbare Vorlauftemperatur (Spreizung) und Anfangsfüllstand je Einzelparameter um die zentrale Variante (Bestand, +10 %, +20 %). Jede Variante wird neu optimiert und nichtlinear nachgerechnet. Varianten mit Lieferungstemperatur unter der Heizkurven-Grenze sind als thermisch unzulässig gekennzeichnet (Spalte T-Verletzung).

In [ ]:
import itertools, time as _time
def s9_case(cs, V=None, Qd=None, dn=None, Hp=None, Th=None, soc0=None, cfg=None, tag="S9"):
    """Optimierte Fahrweise und nichtlineare Bewertung für eine Speicherauslegung (Variante B)."""
    V = SPS["V_m3"] if V is None else V; Qd = SPS["Qd_max_MW"] if Qd is None else Qd; dn = SPS["dn_conn_mm"] if dn is None else dn
    Th = min(SPS["T_max_C"], cs.T_vl.max()) if Th is None else Th
    dn0 = SPS["dn_conn_mm"]; SPS["dn_conn_mm"] = dn
    try:
        o = optimize_dispatch(cs, cfg=cfg, V_m3=V, Qd_max=Qd, Qc_max=Qd, dn_mm=dn, Hp_max_bar=Hp, T_h=Th, soc0=soc0, periodic=soc0 is None, end_soc=soc0 is None)
        ev = evaluate(tag, cs, cfg=cfg, storage=dict(Qc=o["Qc"], Qd=o["Qd"], variant="B", dn_mm=dn, T_h=Th))
    finally: SPS["dn_conn_mm"] = dn0
    return ev, o
def reduction(k_no, k_with, key): return 1.0 - k_with[key]/k_no[key] if k_no[key] > 1e-9 else np.nan

S9_BASES = {"Bestand": dict(design=True), "Ausbau +10 %": dict(design=True, growth=0.10), "Ausbau +20 %": dict(design=True, growth=0.20)}
S9_ROWS = []; t0 = _time.time()
for bl, bk in S9_BASES.items():
    cs = base_day_case(**bk); ev0 = evaluate("ohne", cs); k0 = ev0["kpi"]
    grids = [("V_m3", [1000, 2500, 5000, 10000, 20000, 40000]), ("Qd", [10, 20, 40, 60, 80, 120]), ("dn", [150, 200, 250, 300, 400, 500]), ("Hp", [1.0, 2.0, 3.0, 4.0, 6.0]), ("Th", [90, 100, 110, 120]), ("soc0", [0.3, 0.5, 0.7, 0.9])]
    for par, vals in grids:
        for v in vals:
            ev, o = s9_case(cs, **{par if par != "V_m3" else "V": v}); k = ev["kpi"]
            S9_ROWS.append({"Basis": bl, "Parameter": par, "Wert": v, "Δp-Defizit [bar·h]": k["Δp-Defizit [bar·h]"], "Reduktion Defizit [%]": 100*reduction(k0, k, "Δp-Defizit [bar·h]"), "nicht versorgt [MWh]": k["nicht versorgt [MWh]"],
                            "Reduktion unversorgt [%]": 100*reduction(k0, k, "nicht versorgt [MWh]"), "Δp_Kunde,min [bar]": k["Δp_Kunde,min [bar]"], "HKW-Limit überschritten [h]": k["Erzeugerlimit HKW überschritten [h]"], "T-Verletzung [h]": k["Dauer T-Verletzung [h]"], "T_Lieferung,min [°C]": k["T_Lieferung,min [°C]"],
                            "HKW-Durchfluss,max [m³/h]": k["HKW-Durchfluss,max [m³/h]"], "Entladeenergie [MWh]": k["Entladeenergie [MWh]"], "Qd,max [MW]": k["Qd,max [MW]"], "Pumpenförderhöhe,max [bar]": k["Pumpenförderhöhe,max [bar]"],
                            "Pumpstrom Speicher [MWh]": k["Pumpstrom Speicher [MWh]"], "Massenstrom Speicher,max [kg/s]": k["Massenstrom Speicher,max [kg/s]"], "ohne Speicher: Defizit": k0["Δp-Defizit [bar·h]"], "ohne Speicher: unversorgt": k0["nicht versorgt [MWh]"], "ohne Speicher: HKW-Limit [h]": k0["Erzeugerlimit HKW überschritten [h]"]})
S9 = pd.DataFrame(S9_ROWS); S9.to_csv(RESULTS/"s9_auslegungsvariation.csv", index=False)
print(f"S9: {len(S9)} Auslegungsvarianten in {_time.time()-t0:.0f} s gerechnet")
with pd.option_context("display.width", 250, "display.max_rows", 200): display(S9[S9.Basis == "Bestand"][["Parameter","Wert","Reduktion Defizit [%]","Reduktion unversorgt [%]","Δp_Kunde,min [bar]","HKW-Limit überschritten [h]","Entladeenergie [MWh]","Qd,max [MW]","Pumpenförderhöhe,max [bar]","T-Verletzung [h]"]].round(1))

In [ ]:
# ---------------------------------------------------------------- Auswertung S9: ab welcher Auslegung werden die Probleme ausreichend reduziert?
GOAL = 0.8                                        # Annahme: „ausreichend“ = Δp-Defizit um ≥ 80 % reduziert bei thermisch zulässiger Fahrweise (keine T-Verletzung)
ok = S9[(S9["T-Verletzung [h]"] == 0)]
best = ok.loc[ok.groupby("Basis")["Reduktion Defizit [%]"].idxmax()][["Basis", "Parameter", "Wert", "Reduktion Defizit [%]", "Reduktion unversorgt [%]", "Qd,max [MW]", "Pumpenförderhöhe,max [bar]"]].round(1)
display(best)
for bl in S9_BASES:
    sub = ok[(ok.Basis == bl)]; hit = sub[sub["Reduktion Defizit [%]"] >= 100*GOAL]
    base_def = sub["ohne Speicher: Defizit"].iloc[0]
    print(f"{bl}: Defizit ohne Speicher {base_def:.1f} bar·h; größte Reduktion bei einer Einzelvariation {sub['Reduktion Defizit [%]'].max():.0f} % ({sub.loc[sub['Reduktion Defizit [%]'].idxmax(), 'Parameter']} = {sub.loc[sub['Reduktion Defizit [%]'].idxmax(), 'Wert']}); "
          + (f"Ziel ≥ {100*GOAL:.0f} % erreicht bei: " + ", ".join(f"{r.Parameter}={r.Wert}" for r in hit.itertuples()) if len(hit) else f"Ziel ≥ {100*GOAL:.0f} % von keiner Einzelvariation erreicht."))
fig, axs = plt.subplots(len(S9_BASES), 6, figsize=(21, 3.6*len(S9_BASES)), sharey="row")
for r_, bl in enumerate(S9_BASES):
    for c_, (par, lab) in enumerate((("V_m3", "Volumen [m³]"), ("Qd", "Ent-/Ladeleistung [MW]"), ("dn", "Anschluss-DN [mm]"), ("Hp", "max. Förderhöhe [bar]"), ("Th", "Speichertemperatur [°C]"), ("soc0", "Anfangsfüllstand [-]"))):
        d = S9[(S9.Basis == bl) & (S9.Parameter == par)]; ax = axs[r_, c_]
        ax.plot(d.Wert, d["Reduktion Defizit [%]"], "o-", color=COL["ok"], label="Δp-Defizit"); ax.plot(d.Wert, d["Reduktion unversorgt [%]"], "s--", color=COL["rl"], label="unversorgt")
        bad = d[d["T-Verletzung [h]"] > 0]; ax.scatter(bad.Wert, bad["Reduktion Defizit [%]"], marker="x", color="red", s=70, zorder=5, label="T-Grenze verletzt")
        ax.axhline(100*GOAL, color="k", ls=":", lw=.8); ax.set_xlabel(lab); ax.set_title(bl if c_ == 0 else "", loc="left", fontsize=9)
        if par in ("V_m3", "Qd", "dn"): ax.set_xscale("log")
    axs[r_, 0].set_ylabel("Reduktion [%]")
axs[0, 0].legend(fontsize=7); fig.suptitle("S9: Wirkung der Speicherauslegung (je Einzelvariation, optimierte Fahrweise)", y=1.0); fig.tight_layout(); fig.savefig(FIG_DIR/"19_s9_auslegung.png"); plt.show()
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for lab, c_ in (("ohne Speicher", COL["grau"]), ("mit Speicher (opt.)", COL["ok"])):
    d = GRW[GRW.Fall == lab]; axs[0].plot(100*d.Zuwachs, d["Δp-Defizit [bar·h]"], "o-", color=c_, label=lab); axs[1].plot(100*d.Zuwachs, d["HKW-Limit überschritten [h]"], "o-", color=c_, label=lab)
axs[0].set_xlabel("Lastzuwachs [%]"); axs[0].set_ylabel("Δp-Defizit [bar·h]"); axs[1].set_xlabel("Lastzuwachs [%]"); axs[1].set_ylabel("Stunden über HKW-Limit"); axs[0].legend(); axs[0].set_title("Netzausbau: Druckdefizit"); axs[1].set_title("Netzausbau: Erzeugerlimit HKW")
fig.tight_layout(); fig.savefig(FIG_DIR/"19_ausbau.png"); plt.show()

### 19.4 Standortvariation des Speichers
Bisher war MAN/UPM gesetzt. Hier wird derselbe Speicher (5 000 m³, 40 MW, Pumpe, optimierte Fahrweise) an anderen Knoten des Sektionsmodells eingebunden (Einspeisung im Vorlauf-, Entnahme im Rücklaufknoten). Zusätzlich wird der **Netzhebel** ausgewiesen: Δp-Gewinn aller Kundenknoten je 100 kg/s lokaler Einspeisung. Die Sektionsknoten stehen für ein Gebiet; reale Flächen, Genehmigung und Anschlusspunkte sind unbekannt.

In [ ]:
# ---------------------------------------------------------------- 19.4 Standortvariation des Speichers (5 000 m³, 40 MW, Variante B, optimierte Fahrweise)
CAND = ["MAN", "SL1G", "SEC2", "SEC3", "SEC4", "SUED", "HWS", "SL5M", "SL5O", "SL6S", "SL7G", "HWW"]
NOTE = {"HWS": "Erzeugerstandort (Heizwerk Süd)", "HWW": "Westübergabe", "SEC2": "Innenstadt DOM+CITY", "SEC4": "SL4/Ulrich", "SEC3": "STAWA", "SUED": "Süd", "SL1G": "SL1-Gebiet", "SL5M": "SL5 Mitte", "SL5O": "SL5 Ost", "SL6S": "SL6 Ost+Süd", "SL7G": "SL7-Gebiet", "MAN": "MAN/UPM (Auftrag)"}
SB = {"Bestand": dict(design=True), "Ausbau +10 %": dict(design=True, growth=0.10), "Ausbau +20 %": dict(design=True, growth=0.20)}
rows = []; t0 = _time.time()
for bl, bk in SB.items():
    cs_ = base_day_case(**bk); k0 = evaluate("ohne", cs_)["kpi"]
    for nd in CAND:
        with with_stor_node(nd):
            o = optimize_dispatch(cs_); k = evaluate("mit", cs_, storage=dict(Qc=o["Qc"], Qd=o["Qd"], variant="B"))["kpi"]
        rows.append({"Basis": bl, "Standort": nd, "Gebiet": NOTE[nd], "Defizit ohne [bar·h]": k0["Δp-Defizit [bar·h]"], "Defizit mit [bar·h]": k["Δp-Defizit [bar·h]"], "Reduktion Defizit [%]": 100*(1 - k["Δp-Defizit [bar·h]"]/k0["Δp-Defizit [bar·h]"]) if k0["Δp-Defizit [bar·h]"] > 0 else np.nan,
                     "Reduktion unversorgt [%]": 100*(1 - k["nicht versorgt [MWh]"]/k0["nicht versorgt [MWh]"]) if k0["nicht versorgt [MWh]"] > 0 else np.nan, "Δp_min mit [bar]": k["Δp_Kunde,min [bar]"], "Pumpenförderhöhe [bar]": k["Pumpenförderhöhe,max [bar]"],
                     "HKW-Spitze mit [m³/h]": k["HKW-Durchfluss,max [m³/h]"], "T_Lieferung,min [°C]": k["T_Lieferung,min [°C]"], "Entladeenergie [MWh]": k["Entladeenergie [MWh]"]})
LOC = pd.DataFrame(rows); LOC.to_csv(RESULTS/"standortvariation.csv", index=False)
print(f"Standortvariation: {len(LOC)} Optimierungen in {_time.time()-t0:.0f} s")
piv = LOC.pivot(index="Standort", columns="Basis", values="Reduktion Defizit [%]")[list(SB)].sort_values("Bestand", ascending=False); display(piv.round(1))
# Hebel: Δp-Gewinn aller Kundenknoten je 100 kg/s lokaler Einspeisung in der Spitzenstunde (Netzsensitivität)
cs_s = base_day_case(design=True); lev = {}
for nd in CAND:
    with with_stor_node(nd):
        r_ = solve_case(cs_s.iloc[[8, 8]].reset_index(drop=True), m_stor=np.array([0.0, 100.0]))
    pv_, pr_ = node_pressures(r_); dp_ = (pv_ - pr_)[:, KUNDEN_IDX] + DP_BIAS; lev[nd] = (dp_[1] - dp_[0])
LEV = pd.DataFrame(lev, index=KUNDEN).T; LEV["Σ Gewinn kritischer Knoten [bar]"] = LEV[["SL1G", "SEC2", "SEC3", "SEC4", "HWW"]].sum(axis=1)
display(LEV.round(2).sort_values("Σ Gewinn kritischer Knoten [bar]", ascending=False))
fig, axs = plt.subplots(1, 2, figsize=(15, 4.8)); x = np.arange(len(piv)); w = .27
for k, (bl, c_) in enumerate(zip(SB, (COL["ok"], COL["warn"], COL["vl"]))): axs[0].bar(x + (k-1)*w, piv[bl], w, color=c_, label=bl)
axs[0].set_xticks(x); axs[0].set_xticklabels(piv.index); axs[0].set_ylabel("Reduktion Δp-Defizit [%]"); axs[0].legend(frameon=False, fontsize=8); axs[0].axvline(list(piv.index).index("MAN") if "MAN" in piv.index else 0, color="k", lw=.5, alpha=0)
axs[0].set_title("Wirkung des 5 000 m³/40 MW-Speichers je Standort (optimierte Fahrweise)", loc="left", fontsize=9)
ls = LEV["Σ Gewinn kritischer Knoten [bar]"].sort_values(); axs[1].barh(ls.index, ls.values, color=[COL["ok"] if i != "MAN" else "gold" for i in ls.index]); axs[1].set_xlabel("Σ Δp-Gewinn der kritischen Knoten je 100 kg/s Einspeisung [bar]"); axs[1].set_title("Netzhebel je Einspeiseknoten (Spitzenstunde, S1)", loc="left", fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR/"19_standortvariation.png"); plt.show()

### 19.5 Druckanhebung im Detail
Erforderliche Änderung des Erzeugerdrucks (Vorlauf anheben bzw. Rücklauf absenken – im Modell wirkt nur die Differenz) für Kunden-Δp ≥ 1,0 bar, je Lastzuwachs, ohne Speicher und mit Speicher (MAN bzw. SEC2), sowie das daraus folgende Druckniveau im Netz.

In [ ]:
# ---------------------------------------------------------------- 19.5 Druckanhebung im Detail: Hebel, Bedarf je Ausbaustufe, Druckniveau im Netz
def deficit(g, dpv=0.0, dpr=0.0, storage=None, node=None):
    cs_ = base_day_case(design=True, growth=g, dp_hkw=dpv, dp_ava=dpv, dp_rl=dpr)
    if node:
        with with_stor_node(node): return evaluate("x", cs_, storage=storage)["kpi"]
    return evaluate("x", cs_, storage=storage)["kpi"]
def need(g, lever, storage=None, node=None, tol=0.05, hi=8.0):
    f_ = (lambda x: deficit(g, dpv=x, storage=storage, node=node)) if lever == "VL" else ((lambda x: deficit(g, dpr=-x, storage=storage, node=node)) if lever == "RL" else (lambda x: deficit(g, dpv=x/2, dpr=-x/2, storage=storage, node=node)))
    k0 = f_(0.0)
    if k0["Δp-Defizit [bar·h]"] < tol: return 0.0, k0
    if f_(hi)["Δp-Defizit [bar·h]"] >= tol: return np.nan, k0
    lo_, hi_ = 0.0, hi
    for _ in range(13):
        mid = 0.5*(lo_ + hi_)
        if f_(mid)["Δp-Defizit [bar·h]"] < tol: hi_ = mid
        else: lo_ = mid
    return hi_, f_(hi_)
GR2 = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]; rows = []
for g in GR2:
    cs_ = base_day_case(design=True, growth=g)
    with with_stor_node("SEC2"): o2 = optimize_dispatch(cs_)
    o1 = optimize_dispatch(cs_); st1 = dict(Qc=o1["Qc"], Qd=o1["Qd"], variant="B"); st2 = dict(Qc=o2["Qc"], Qd=o2["Qd"], variant="B")
    rv, kv = need(g, "VL"); rr, _ = need(g, "RL"); rm, _ = need(g, "VL", storage=st1); rs, _ = need(g, "VL", storage=st2, node="SEC2")
    rows.append({"Zuwachs": g, "VL-Anhebung ohne Speicher [bar]": rv, "RL-Absenkung ohne Speicher [bar]": rr, "VL-Anhebung mit Speicher MAN [bar]": rm, "VL-Anhebung mit Speicher SEC2 [bar]": rs, "p_VL,max im Netz nach Anhebung [bar Ü]": kv["p_max [bar Ü]"]})
DR = pd.DataFrame(rows); DR.to_csv(RESULTS/"druckanhebung_fein.csv", index=False); display(DR.round(2))
fig, axs = plt.subplots(1, 2, figsize=(14, 4.8))
for col, c_, ls_, lab in (("VL-Anhebung ohne Speicher [bar]", COL["grau"], "-", "nur Vorlaufdruck anheben"), ("RL-Absenkung ohne Speicher [bar]", COL["rl"], "-", "nur Rücklaufdruck absenken"), ("VL-Anhebung mit Speicher MAN [bar]", COL["warn"], "--", "Vorlauf + Speicher MAN"), ("VL-Anhebung mit Speicher SEC2 [bar]", COL["ok"], "--", "Vorlauf + Speicher SEC2")):
    axs[0].plot(100*DR.Zuwachs, DR[col], "o"+ls_, color=c_, ms=4, label=lab)
axs[0].set_xlabel("Lastzuwachs [%]"); axs[0].set_ylabel("erforderliche Druckänderung [bar]"); axs[0].legend(frameon=False, fontsize=8); axs[0].set_title("Bedarf an Erzeugerdruck-Maßnahme für Δp ≥ 1,0 bar bei allen Kunden", loc="left", fontsize=9)
axs[1].plot(100*DR.Zuwachs, DR["p_VL,max im Netz nach Anhebung [bar Ü]"], "o-", color=COL["vl"], ms=4, label="größter Vorlaufdruck im Netz nach Anhebung"); axs[1].axhline(16, color=COL["grau"], ls=":", label="PN16"); axs[1].axhline(25, color=COL["grau"], ls="--", label="PN25")
axs[1].set_xlabel("Lastzuwachs [%]"); axs[1].set_ylabel("p_VL,max [bar Ü]"); axs[1].set_ylim(10, 27); axs[1].legend(frameon=False, fontsize=8); axs[1].set_title("Druckniveau im Netz bleibt weit unter PN25", loc="left", fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR/"19_druckanhebung_fein.png"); plt.show()

### 19.6 Räumliche Verteilung des Ausbaus
Da konkrete Ausbauvorhaben fehlen, wird derselbe Zuwachs von +10 % der Gesamtlast jeweils vollständig in eine einzelne Sektion gelegt. Das zeigt, welche Gebiete beim Ausbau am empfindlichsten sind.

In [ ]:
# ---------------------------------------------------------------- 19.6 Räumliche Verteilung des Ausbaus: +10 % Gesamtlast, konzentriert auf je eine Sektion
w_cal_ = unpack(theta_cal)[1]; rows = []
for s_, w_s in zip(SEKN, w_cal_):
    gs = {x: (0.10/w_s if x == s_ else 0.0) for x in SEKN}; cs_ = base_day_case(design=True, growth=0.0, growth_sec=gs); k0 = evaluate("o", cs_)["kpi"]
    o = optimize_dispatch(cs_); k1 = evaluate("m", cs_, storage=dict(Qc=o["Qc"], Qd=o["Qd"], variant="B"))["kpi"]
    with with_stor_node("SEC2"): o2 = optimize_dispatch(cs_); k2 = evaluate("m2", cs_, storage=dict(Qc=o2["Qc"], Qd=o2["Qd"], variant="B"))["kpi"]
    rows.append({"Sektion mit Zuwachs": s_, "Lastanteil heute [%]": 100*w_s, "Zuwachs in der Sektion [%]": 100*0.10/w_s, "Defizit ohne [bar·h]": k0["Δp-Defizit [bar·h]"], "Δp_min ohne [bar]": k0["Δp_Kunde,min [bar]"], "kritischer Knoten": k0["Knoten/Stunde Δp,min"], "Defizit mit Speicher MAN": k1["Δp-Defizit [bar·h]"], "Defizit mit Speicher SEC2": k2["Δp-Defizit [bar·h]"], "HKW-Limit überschritten [h]": k0["Erzeugerlimit HKW überschritten [h]"]})
DIST = pd.DataFrame(rows).sort_values("Defizit ohne [bar·h]", ascending=False); DIST.to_csv(RESULTS/"ausbauverteilung.csv", index=False); display(DIST.round(2))
fig, ax = plt.subplots(figsize=(10, 4.4)); x = np.arange(len(DIST)); w = .27
ax.bar(x - w, DIST["Defizit ohne [bar·h]"], w, color=COL["grau"], label="ohne Speicher"); ax.bar(x, DIST["Defizit mit Speicher MAN"], w, color=COL["warn"], label="Speicher MAN"); ax.bar(x + w, DIST["Defizit mit Speicher SEC2"], w, color=COL["ok"], label="Speicher SEC2")
ax.axhline(RUN["S1"]["kpi"]["Δp-Defizit [bar·h]"], color="k", ls=":", lw=.8); ax.text(len(DIST)-.5, RUN["S1"]["kpi"]["Δp-Defizit [bar·h]"] + 1, "heute (S1)", ha="right", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(DIST["Sektion mit Zuwachs"]); ax.set_ylabel("Δp-Defizit [bar·h]"); ax.legend(frameon=False, fontsize=8); ax.set_title("+10 % Gesamtlast, konzentriert auf eine Sektion: wo der Ausbau stattfindet, entscheidet über die Engpässe", loc="left", fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR/"19_ausbauverteilung.png"); plt.show()

### 19.7 Modellstruktur: Kalibrierung mit Gewicht auf Hochlast-Differenzdruck
Test (nicht Hauptkalibrierung): zusätzliche Residuen für den Kunden-Differenzdruck in 220 Hochlaststunden Jan–Sep (R ≥ P85); Validierung an den Hochlaststunden Okt–Dez. Die Bias-Korrektur (Kap. 16.2) wird je Kalibrierung neu bestimmt. Ziel ist eine Aussage, wie stark Defizit und Speicherwirkung von der Kalibrierung abhängen.

In [ ]:
# ---------------------------------------------------------------- 19.7 Modellstruktur: Alternativkalibrierung mit Gewicht auf Hochlast-Differenzdruck (Test, ersetzt die Hauptkalibrierung nicht)
import time as _time
rng_ = np.random.default_rng(SEED)
cand_ = BC.index[BC.valid & (BC.index <= "2025-09-30 23:00") & (BC.R >= BC.R[BC.valid].quantile(0.85))]; H_HI = pd.DatetimeIndex(sorted(rng_.choice(cand_, size=min(220, len(cand_)), replace=False)))
cand_v = BC.index[BC.valid & (BC.index >= "2025-10-01") & (BC.R >= BC.R[BC.valid].quantile(0.85))]; H_HV = pd.DatetimeIndex(cand_v)
ST_DP = [(j, r) for j, r in MESS.iterrows() if r.vl is not None and r.rl is not None and r.station not in EXCL_CAL and r.station != "BMHKW"]
def dp_resid(theta, hours, sig=0.25):
    sim = simulate(theta, hours); pv, pr = station_pressures(sim); out = []
    for j, r in ST_DP:
        me = (acron.loc[hours, r.vl] - acron.loc[hours, r.rl]).values; e = (pv[:, j] - pr[:, j]) - me; out.append(np.where(np.isfinite(e) & sim["ok"], e/sig, np.nan))
    return np.concatenate(out)
def objective_hi(theta):
    base = objective(theta); extra = np.nan_to_num(dp_resid(theta, H_HI), nan=0.0); return np.r_[base, extra]
cache2 = RESULTS/"kalibrierung_hochlast.json"; theta_hi = None
if cache2.exists():
    cj = json.loads(cache2.read_text())
    if cj.get("signatur") == sig_in + "_hi": theta_hi = np.array(cj["theta"]); print("Hochlast-Kalibrierung aus Zwischenspeicher geladen")
if theta_hi is None:
    t0 = _time.time(); sol2 = least_squares(objective_hi, theta_cal, bounds=(lo, hi), method="trf", diff_step=1e-3, max_nfev=25, verbose=0); theta_hi = sol2.x
    print(f"Hochlast-Kalibrierung: Status {sol2.status}, {sol2.nfev} Auswertungen, {_time.time()-t0:.0f} s, Kosten {sol2.cost:.0f}"); cache2.write_text(json.dumps(dict(signatur=sig_in + "_hi", theta=list(map(float, theta_hi)))))
def dp_stats(theta, hours):
    sim = simulate(theta, hours); pv, pr = station_pressures(sim); rows = []
    for j, r in ST_DP:
        me = (acron.loc[hours, r.vl] - acron.loc[hours, r.rl]).values; e = (pv[:, j] - pr[:, j]) - me; e = e[np.isfinite(e) & sim["ok"]]
        rows.append({"Station": r.station, "Bias [bar]": e.mean(), "RMSE [bar]": np.sqrt((e**2).mean())})
    return pd.DataFrame(rows).set_index("Station")
dpA, dpB = dp_stats(theta_cal, H_HV), dp_stats(theta_hi, H_HV); cmpv = pd.concat([dpA.add_suffix(" (Haupt)"), dpB.add_suffix(" (Hochlast)")], axis=1).round(2); display(cmpv)
print(f"Validierung Hochlast Okt–Dez ({len(H_HV)} Stunden): Differenzdruck-RMSE im Mittel der Stationen {dpA['RMSE [bar]'].mean():.2f} → {dpB['RMSE [bar]'].mean():.2f} bar, mittlerer Betrag des Bias {dpA['Bias [bar]'].abs().mean():.2f} → {dpB['Bias [bar]'].abs().mean():.2f} bar.")
# Gesamtvalidierung (alle Größen) – Verschlechterung an anderer Stelle?
rc_a, _ = residuals(theta_cal, H_VAL, detail=True); rc_b, _ = residuals(theta_hi, H_VAL, detail=True)
def rm(rc): return rc.assign(f=rc.modell - rc.messung).groupby("art").f.apply(lambda e: np.sqrt(np.nanmean(e**2)))
tab = pd.DataFrame({"Haupt": rm(rc_a), "Hochlast": rm(rc_b)}).round(2); display(tab)
# Auswirkung auf die Speicheraussage (S1, Fahrplan der Hauptkalibrierung unverändert)
cs_h = base_day_case(design=True); o_h = optimize_dispatch(cs_h); sto_h = dict(Qc=o_h["Qc"], Qd=o_h["Qd"], variant="B"); rows = []
for lab, th in (("Hauptkalibrierung", theta_cal), ("Hochlast-Kalibrierung", theta_hi)):
    with with_theta(th):
        _keep = DP_BIAS.copy(); DP_BIAS[:] = compute_dp_bias(th, False)[0]          # Bias-Korrektur zur jeweiligen Kalibrierung neu bestimmen
        try: k0 = evaluate("a", cs_h)["kpi"]; k1 = evaluate("b", cs_h, storage=sto_h)["kpi"]
        finally: DP_BIAS[:] = _keep
    rows.append({"Kalibrierung": lab, "Δp_min ohne [bar]": k0["Δp_Kunde,min [bar]"], "Defizit ohne [bar·h]": k0["Δp-Defizit [bar·h]"], "Defizit mit Speicher MAN [bar·h]": k1["Δp-Defizit [bar·h]"], "Reduktion [%]": 100*(1 - k1["Δp-Defizit [bar·h]"]/k0["Δp-Defizit [bar·h]"]), "Speicherwirkung Δp_min [bar]": k1["Δp_Kunde,min [bar]"] - k0["Δp_Kunde,min [bar]"], "kritische Knoten ohne": k0["kritische Knoten"]})
HL = pd.DataFrame(rows).round(2); display(HL); HL.to_csv(RESULTS/"hochlast_kalibrierung_vergleich.csv", index=False)

### 19.8 Maßnahmenvergleich (hydraulisch, ohne Kosten)
Neun Maßnahmen(-Kombinationen) werden unter denselben Randbedingungen für heute, +10 %, +20 % und +30 % verglichen: Erzeugerdruck +1 bar, Verstärkung der Ost-Trassen bzw. des Innenstadtverbunds (Annahme: halber Reibungswiderstand, z. B. Parallelleitung), Speicher am MAN bzw. in der Innenstadt (SEC2) in zwei Größen, sowie Kombinationen. Zusätzlich wird die Speichergröße am besten Standort variiert. Kosten sind nicht Teil dieser Rechnung.

In [ ]:
# ---------------------------------------------------------------- 19.8 Maßnahmenvergleich ohne Kosten: Druckanhebung ↔ Leitungsverstärkung ↔ Speicher (MAN/Innenstadt) ↔ Kombination
INK = "#2b2b2b"
VERST_OST = {"SL5": 0.5, "SL6": 0.5, "SL7": 0.5, "OSTSTUB": 0.5}                 # Parallelleitung/Verstärkung: halber Reibungswiderstand der Ost-Trassen (Annahme)
VERST_IN = {"SL2": 0.5, "M12": 0.5, "M23": 0.5, "M24": 0.5, "M34": 0.5}            # Verstärkung Innenstadt-Verbund und SL2
MASS = [("M0", "keine Maßnahme", dict(), None, None), ("M1", "Erzeugerdruck +1,0 bar", dict(dpv=1.0), None, None), ("M2", "Verstärkung Ost-Trassen (½ Widerstand)", dict(cfg=dict(k_scale=VERST_OST)), None, None),
        ("M3", "Verstärkung Innenstadt-Verbund/SL2 (½ Widerstand)", dict(cfg=dict(k_scale=VERST_IN)), None, None), ("M4", "Speicher MAN 5 000 m³/40 MW", dict(), "MAN", (5000, 40)),
        ("M5", "Speicher Innenstadt (SEC2) 5 000 m³/40 MW", dict(), "SEC2", (5000, 40)), ("M6", "Speicher Innenstadt (SEC2) 10 000 m³/80 MW", dict(), "SEC2", (10000, 80)),
        ("M7", "Druck +1,0 bar + Speicher SEC2 5 000 m³/40 MW", dict(dpv=1.0), "SEC2", (5000, 40)), ("M8", "Verstärkung Ost + Innenstadt + Druck +1,0 bar", dict(dpv=1.0, cfg=dict(k_scale={**VERST_OST, **VERST_IN})), None, None)]
GRM = [0.0, 0.10, 0.20, 0.30]; rows = []; t0 = _time.time()
for g in GRM:
    for mid, lab, kw, node, (sz) in [(m[0], m[1], m[2], m[3], m[4]) for m in MASS]:
        cs_ = base_day_case(design=True, growth=g, dp_hkw=kw.get("dpv", 0.0), dp_ava=kw.get("dpv", 0.0)); cfg_ = kw.get("cfg", {}); sto = None
        if node:
            with with_stor_node(node):
                o = optimize_dispatch(cs_, cfg=cfg_, V_m3=sz[0], Qd_max=sz[1], Qc_max=sz[1]); k = evaluate("x", cs_, cfg=cfg_, storage=dict(Qc=o["Qc"], Qd=o["Qd"], variant="B"))["kpi"]
        else: k = evaluate("x", cs_, cfg=cfg_)["kpi"]
        rows.append({"Zuwachs": g, "ID": mid, "Maßnahme": lab, "Δp-Defizit [bar·h]": k["Δp-Defizit [bar·h]"], "nicht versorgt [MWh]": k["nicht versorgt [MWh]"], "Δp_min [bar]": k["Δp_Kunde,min [bar]"], "HKW-Limit überschritten [h]": k["Erzeugerlimit HKW überschritten [h]"],
                     "Leitungsauslastung,max [-]": k["Leitungsauslastung,max [-]"], "Pumpenförderhöhe [bar]": k["Pumpenförderhöhe,max [bar]"]})
MS = pd.DataFrame(rows); MS.to_csv(RESULTS/"massnahmenvergleich.csv", index=False); print(f"Maßnahmenvergleich: {len(MS)} Fälle in {_time.time()-t0:.0f} s")
piv = MS.pivot(index="Maßnahme", columns="Zuwachs", values="Δp-Defizit [bar·h]").reindex([m[1] for m in MASS]); display(piv.round(1))
fig, axs = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw=dict(width_ratios=[1.25, 1])); im = axs[0].imshow(np.log10(piv.values + 1), cmap="YlOrRd", aspect="auto")
axs[0].set_xticks(range(len(GRM))); axs[0].set_xticklabels([f"+{int(100*g)} %" if g else "heute" for g in GRM]); axs[0].set_yticks(range(len(piv))); axs[0].set_yticklabels(piv.index, fontsize=8); axs[0].grid(False)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]): axs[0].text(j, i, f"{piv.values[i, j]:.0f}", ha="center", va="center", fontsize=8, color=("white" if np.log10(piv.values[i, j] + 1) > 1.7 else INK))
axs[0].set_title("Verbleibendes Δp-Defizit [bar·h] je Maßnahme und Lastzuwachs (Farbe logarithmisch)", loc="left", fontsize=9)
for mid, c_, mk in (("M1", COL["grau"], "o"), ("M2", COL["rl"], "s"), ("M3", COL["mess"], "s"), ("M4", COL["warn"], "^"), ("M5", COL["ok"], "^"), ("M6", "#1b5e20", "^"), ("M8", COL["vl"], "D")):
    d = MS[MS.ID == mid]; axs[1].plot(100*d.Zuwachs, d["Δp-Defizit [bar·h]"], mk + "-", color=c_, ms=5, label=mid + " " + d.Maßnahme.iloc[0][:34])
d0 = MS[MS.ID == "M0"]; axs[1].plot(100*d0.Zuwachs, d0["Δp-Defizit [bar·h]"], "k--", label="M0 keine Maßnahme"); axs[1].set_xlabel("Lastzuwachs [%]"); axs[1].set_ylabel("Δp-Defizit [bar·h]"); axs[1].set_yscale("symlog", linthresh=1); axs[1].legend(fontsize=6.5, frameon=False)
fig.tight_layout(); fig.savefig(FIG_DIR/"19_massnahmenvergleich.png"); plt.show()
# Speichergröße/-leistung am besten Standort (SEC2), +10 %
rows = []; cs_ = base_day_case(design=True, growth=0.10); k0 = evaluate("o", cs_)["kpi"]
for V in (2500, 5000, 10000, 20000):
    for Q in (20, 40, 80):
        with with_stor_node("SEC2"):
            o = optimize_dispatch(cs_, V_m3=V, Qd_max=Q, Qc_max=Q); k = evaluate("x", cs_, storage=dict(Qc=o["Qc"], Qd=o["Qd"], variant="B"))["kpi"]
        rows.append({"V [m³]": V, "Q [MW]": Q, "Reduktion Defizit [%]": 100*(1 - k["Δp-Defizit [bar·h]"]/k0["Δp-Defizit [bar·h]"]), "Pumpenförderhöhe [bar]": k["Pumpenförderhöhe,max [bar]"]})
SZ = pd.DataFrame(rows); pz = SZ.pivot(index="V [m³]", columns="Q [MW]", values="Reduktion Defizit [%]"); display(pz.round(1)); SZ.to_csv(RESULTS/"standort_sec2_groesse.csv", index=False)

## 20. pandapipes-Kontrollrechnung
### 20.1 Altbestand im Repository

In [ ]:
# ---------------------------------------------------------------- Befunde der alten pandapipes-Studie (Memmingen-Crosscheck im Repository)
import json as _json, re as _re
_p = ROOT/"configs"/"pressure"/"Memmingen_pandapipes_crosscheck.ipynb"
_nb = _json.loads(_p.read_text(encoding="utf-8")); _code = "\n".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
def _has(pat): return bool(_re.search(pat, _code))
old = pd.DataFrame([
 ("pandapipes-Version", "zur Laufzeit ausgegeben (print(pp.__version__)), nicht festgeschrieben", "nicht reproduzierbar dokumentiert"),
 ("Netzaufbau", "15 Knoten, ein Erzeuger (ext_grid, Sollwert 4,8 bar), Rohre mit k = 0,1 mm, nur Hydraulik (mode='hydraulics')", "Hydraulik ohne Wärmetransport"),
 ("Höhen", "kein height_m an Junctions" if not _has(r"height_m") else "height_m verwendet", "Geodäsie fehlt: für Stadtbach nicht übertragbar"),
 ("Druckeinheit", "p_bar direkt aus Sollwert (pandapipes rechnet absolut); kein Überdruck/Absolut-Abgleich" if not _has(r"1\.013|101325") else "Abgleich vorhanden", "für Stadtbach: explizit +1,013 bar"),
 ("Pumpen", "keine pandapipes-Pumpe; Pumpenleistung nur aus dem MILP" if not _has(r"create_pump") else "Pumpen vorhanden", "für Stadtbach: Pumpen mit konstanter Kennlinie (Kap. 20.2)"),
 ("Verbraucher/Erzeuger", "sink = Bedarf, source = lokale Erzeugung (Workaround nach falscher Druckabsenkung an j_12)", "Quellen/Senken aus dem Optimierungsergebnis"),
 ("Reibungsmodell", "MILP: konstantes λ = 0,02; pandapipes: Colebrook–White", "hier: Swamee–Jain ↔ Colebrook/Nikuradse (Kap. 20.2)"),
 ("Bekannte Workarounds", "Regularisierung des Drucks am gemischten Erzeuger j_12; Druck-Ceiling-Artefakt", "für Stadtbach nicht übernommen"),
 ("Ergebnisse", "RMSE 0,0013 bar (9/15 Knoten), Rest nach Fix ≈ 0", "wurden nicht übernommen, nur Vorgehen"),
], columns=["Aspekt", "Befund im Altbestand", "Konsequenz für diese Studie"])
with pd.option_context("display.max_colwidth", 110, "display.width", 250): display(old)
old.to_csv(RESULTS/"pandapipes_altstudie_befunde.csv", index=False)

### 20.2 Neuer Crosscheck aus demselben kanonischen Datensatz
Das pandapipes-Netz (0.14) wird aus denselben Objekten wie das Eigenmodell aufgebaut (`NODES`, `E_IDS`, Längen, Durchmesser, Knotenhöhen `Z_NODE`, Multiplikatoren, Randbedingungen, Einspeisungen und Speicherstrom je Stunde). Vorlauf und Rücklauf sind getrennte Teilnetze mit Druckrändern an HKW und AVA (absolute Drücke = Überdruck + 1,013 bar). Kalibriermultiplikatoren werden als **wirksame Länge** abgebildet (Reibungshöhe ∝ L), Pumpen (SL4c, W1) als pandapipes-Pumpen mit konstanter Kennlinie, der Speicher als konsistente Ein-/Ausspeisung am Knoten MAN (Quelle/Senke). Verglichen werden Knotendruck (Vorlauf/Rücklauf/Differenz), Leitungsmassenstrom und Flussrichtung. **Temperatur/Wärmeverlust:** pandapipes konvergierte für das vermaschte Netz mit Fremdeinspeisern (Quellen ohne Temperaturangabe) im Wärmetransport nicht; daher wird der Wärmeverlust-Ansatz am Einzelstrang gegen pandapipes und analytisch geprüft, und der Netz-Wärmetransport des Eigenmodells über eine Energiebilanz kontrolliert.

In [ ]:
import pandapipes as pp, logging
logging.getLogger("pandapipes").setLevel(logging.ERROR)
def build_pp(res, i, side="VL", thermal_mode=False, u_lin=U_LIN):
    """pandapipes-Netz (Vorlauf 'VL' bzw. Rücklauf 'RL') für Stunde i aus demselben kanonischen Datensatz (NODES, E_IDS, L_E, D_E, Z_NODE, Multiplikatoren) und denselben Randbedingungen wie das Eigenmodell."""
    cs = res["cs"]; T = float(cs.T_vl.iloc[i] if side == "VL" else cs.T_rl.iloc[i]); Hh = res["Hv" if side == "VL" else "Hr"][i]; rho = float(res["rho_v" if side == "VL" else "rho_r"][i])
    m_e = (res["mv"] if side == "VL" else res["mr"])[i]; inj = res["inj"][i]*(1 if side == "VL" else -1)
    net = pp.create_empty_network(fluid="water"); J = {}
    for n_, nm in enumerate(NODES):
        p_abs = rho*G*(Hh[n_] - Z_NODE[n_])/1e5 + P_ATM
        J[nm] = pp.create_junction(net, pn_bar=p_abs, tfluid_k=T + 273.15, height_m=float(Z_NODE[n_]), name=nm)
    closed = set(res["cfg"].get("closed", []))
    for j, e in enumerate(E_IDS):
        if e in closed: continue
        a, b = NODES[E_VON[j]], NODES[E_NACH[j]]
        gain_bar = float(res["gain"][i, j]*rho*G/1e5) if side == "VL" else 0.0
        kw = dict(length_km=float(res["mult"][j]*L_E[j]/1000.0), diameter_m=float(D_E[j]), k_mm=0.1, name=e, sections=1)      # Kalibriermultiplikator k als wirksame Länge (Reibungshöhe ∝ L)
        if gain_bar > 1e-6:
            jm = pp.create_junction(net, pn_bar=1.0, tfluid_k=T + 273.15, height_m=float(Z_NODE[E_VON[j]]), name=f"{e}_pumpe")
            pp.create_pump_from_parameters(net, J[a], jm, f"konst_{e}", pressure_list=[gain_bar, gain_bar], flowrate_list=[1.0, 20000.0], reg_polynomial_degree=1, name=f"Pumpe_{e}")
            a_j = jm
        else: a_j = J[a]
        pp.create_pipe_from_parameters(net, a_j, J[b], **kw)
    for n_, nm in enumerate(NODES):
        if nm in res["ref_names"]: continue
        q = inj[n_]
        if q > 1e-9: pp.create_source(net, J[nm], mdot_kg_per_s=float(q), name=f"Quelle_{nm}")
        elif q < -1e-9: pp.create_sink(net, J[nm], mdot_kg_per_s=float(-q), name=f"Senke_{nm}")
    for k_, nm in enumerate(res["ref_names"]):
        p_abs = rho*G*(Hh[nidx[nm]] - Z_NODE[nidx[nm]])/1e5 + P_ATM
        pp.create_ext_grid(net, J[nm], p_bar=float(p_abs), t_k=T + 273.15, type=("pt" if k_ == 0 else "p"), name=f"Rand_{nm}")
    return net, J

def run_pp(res, i, side="VL", friction="colebrook", thermal=False):
    res["ref_names"] = [NODES[r_] for r_ in (REFS if res["cfg"].get("ava_ref", True) else [nidx["HKW"]])]
    net, J = build_pp(res, i, side, thermal_mode=thermal)
    for kw in (dict(max_iter_hyd=200, tol_p=1e-7, tol_m=1e-7), dict(max_iter_hyd=500, tol_p=1e-6, tol_m=1e-6), dict(max_iter_hyd=800, tol_p=1e-5, tol_m=1e-5, alpha=0.5), dict(max_iter_hyd=800, tol_p=1e-4, tol_m=1e-4, alpha=0.3)):
        try: pp.pipeflow(net, mode="hydraulics", friction_model=friction, **kw); net.tol_used = kw["tol_p"]; return net, J
        except pp.PipeflowNotConverged: continue
    raise RuntimeError(f"pandapipes konvergiert nicht ({side}, Stunde {i})")

In [ ]:
# ---------------------------------------------------------------- Kontrollrechnung: 8 Betriebspunkte (Eigenmodell ↔ pandapipes), Vorlauf und Rücklauf
PP_POINTS = [("P1", "Historischer Normalbetrieb (S0, 03 Uhr)", "S0", 3), ("P2", "Historische Höchstlast 14.02.2025 09 Uhr (S0)", "S0", 9),
             ("P3", "Winterspitze Bestand ohne Speicher (S1, 08 Uhr)", "S1", 8), ("P4", "Dieselbe Stunde mit Speicherentladung (S3O)", "S3O", 8),
             ("P5", "Kritische Speicherladung zur Lastspitze (S4K)", "S4K", 8), ("P6", "Ausbau +20 % ohne Speicher (S5b)", "S5b", 8),
             ("P7", "Ausbau +20 % mit Speicher (S6b)", "S6b", 8), ("P8", "Separierter Zustand Ost bei 342.3, Bestand (S71B0)", "S71B0", 8)]
def cmp_point(pid, lab, sid, i, friction="colebrook"):
    res = RUN[sid]["res"]; out = {}
    for side in ("VL", "RL"):
        net, J = run_pp(res, i, side, friction); Hh = res["Hv" if side == "VL" else "Hr"][i]; rho = res["rho_v" if side == "VL" else "rho_r"][i]
        p_my = rho*G*(Hh - Z_NODE)/1e5; p_pp = np.array([net.res_junction.p_bar[J[n]] - P_ATM for n in NODES])
        names = list(net.pipe.name); m_pp = np.array([net.res_pipe.mdot_from_kg_per_s[net.pipe.index[names.index(e)]] if e in names else 0.0 for e in E_IDS])
        m_my = (res["mv"] if side == "VL" else res["mr"])[i]
        out[side] = dict(p_my=p_my, p_pp=p_pp, m_my=m_my, m_pp=m_pp)
    return out
PPR = {}; rows = []
for pid, lab, sid, i in PP_POINTS:
    o = cmp_point(pid, lab, sid, i); PPR[pid] = o
    dpv = o["VL"]["p_pp"] - o["VL"]["p_my"]; dpr = o["RL"]["p_pp"] - o["RL"]["p_my"]; dpd = (o["VL"]["p_pp"]-o["RL"]["p_pp"]) - (o["VL"]["p_my"]-o["RL"]["p_my"])
    mm = np.r_[o["VL"]["m_my"], o["RL"]["m_my"]]; mp = np.r_[o["VL"]["m_pp"], o["RL"]["m_pp"]]; sel = np.abs(mm) > 5.0
    dirx = int(((np.sign(mm) != np.sign(mp)) & (np.abs(mm) > 1.0) & (np.abs(mp) > 1.0)).sum())
    rows.append({"Punkt": pid, "Betriebspunkt": lab, "max|Δp_VL| [bar]": np.abs(dpv).max(), "max|Δp_RL| [bar]": np.abs(dpr).max(), "max|Δ(Δp)| [bar]": np.abs(dpd).max(), "mittl. Δp_VL [bar]": dpv.mean(),
                 "max|Δṁ| [kg/s]": np.abs(mm-mp).max(), "max rel. Δṁ (|ṁ|>5) [%]": 100*np.max(np.abs(mm[sel]-mp[sel])/np.abs(mm[sel])), "Flussrichtung abweichend [#]": dirx})
pp_tab = pd.DataFrame(rows).round(3); display(pp_tab); pp_tab.to_csv(RESULTS/"pandapipes_vergleich.csv", index=False)
# Ursachenanalyse: Reibungsmodell (Colebrook ↔ Nikuradse) und Dichte
o2 = cmp_point("P3", "", "S1", 8, friction="nikuradse"); o1 = PPR["P3"]
print("P3 Reibungsmodell Colebrook: max|Δp_VL| {:.3f} bar, max|Δṁ| {:.2f} kg/s | Nikuradse: max|Δp_VL| {:.3f} bar, max|Δṁ| {:.2f} kg/s".format(np.abs(o1["VL"]["p_pp"]-o1["VL"]["p_my"]).max(), np.abs(o1["VL"]["m_pp"]-o1["VL"]["m_my"]).max(), np.abs(o2["VL"]["p_pp"]-o2["VL"]["p_my"]).max(), np.abs(o2["VL"]["m_pp"]-o2["VL"]["m_my"]).max()))
worst = pp_tab.loc[pp_tab["max|Δp_VL| [bar]"].idxmax()]; print(f"Größte Abweichung im Vorlaufdruck: {worst['Punkt']} mit {worst['max|Δp_VL| [bar]']:.3f} bar; in allen Punkten Flussrichtungs-Abweichungen bei |ṁ|>1 kg/s: {int(pp_tab['Flussrichtung abweichend [#]'].sum())}.")
# ---------------------------------------------------------------- Temperatur und Wärmeverlust: Einzelstrang gegen pandapipes (Netz-Wärmetransport in pandapipes konvergiert für das vermaschte Netz mit Fremdeinspeisern nicht)
def pp_chain(mdot, T0=120.0, L=3000.0, dn=0.3, ulin=U_LIN, n=3):
    net = pp.create_empty_network(fluid="water"); js = [pp.create_junction(net, pn_bar=10.0, tfluid_k=T0+273.15) for _ in range(n+1)]
    pp.create_ext_grid(net, js[0], p_bar=10.0, t_k=T0+273.15, type="pt")
    for k in range(n): pp.create_pipe_from_parameters(net, js[k], js[k+1], length_km=L/n/1000, diameter_m=dn, k_mm=0.1, u_w_per_m2k=ulin/(np.pi*dn), text_k=T_SOIL+273.15, sections=1)
    pp.create_sink(net, js[-1], mdot_kg_per_s=mdot); pp.pipeflow(net, mode="sequential", friction_model="colebrook", iter=100)
    return net.res_junction.t_k.values[-1] - 273.15
rows = []
for md in (5.0, 10.0, 30.0, 100.0):
    Tp = pp_chain(md); Te = T_SOIL + (120 - T_SOIL)*np.exp(-U_LIN*3000/(md*CP_J))
    rows.append({"ṁ [kg/s]": md, "T_aus pandapipes [°C]": Tp, "T_aus analytisch [°C]": Te, "Δ pandapipes–analytisch [K]": Tp - Te})
display(pd.DataFrame(rows).round(3))

In [ ]:
# Energiebilanz des Netz-Temperaturmodells (Eigenmodell): Σ Quellenenthalpie − Σ Senkenenthalpie − Verluste = 0
def vl_energy_check(ev, h):
    res, th = ev["res"], ev["th"]; ia = res["inj_all"][h]; src = np.zeros(len(NODES)); Tsr = np.zeros(len(NODES)); cs = ev["cs"]
    for nd in ("HKW", "AVA", "GT", "BM", "HWS"): src[nidx[nd]] = max(ia[nidx[nd]], 0.0); Tsr[nidx[nd]] = cs.T_vl.iloc[h]
    src[nidx["MAN"]] = max(ev["m"][h], 0.0); Tsr[nidx["MAN"]] = (ev["plan"]["T_h"] if ev.get("plan") else cs.T_vl.iloc[h])
    sink = src - ia; Q_in = (src*Tsr).sum()*CP_J; Q_out = (sink*th["Tv"][h]).sum()*CP_J; loss = th["loss_v_edge"][h].sum()
    return Q_in, Q_out, loss
rows = []
for sid, h in (("S1", 8), ("S3O", 8), ("S5b", 8), ("S71B0", 8)):
    qi, qo, ql = vl_energy_check(RUN[sid], h); rows.append({"Szenario/Stunde": f"{sid}/{h:02d}", "Σ Quelle [MW]": qi/1e6, "Σ Senke [MW]": qo/1e6, "Verlust Vorlauf [MW]": ql/1e6, "Bilanzfehler [MW]": (qi - qo - ql)/1e6})
eb = pd.DataFrame(rows).round(4); display(eb); assert eb["Bilanzfehler [MW]"].abs().max() < 0.05
print(f"Energiebilanz des Vorlauf-Temperaturmodells erfüllt (max. Fehler {eb['Bilanzfehler [MW]'].abs().max():.4f} MW, bezogen auf Enthalpie ab 0 °C).")

### 20.3 Abbildung und Ursachen

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.6)); x = np.arange(len(NODES))
for pid, *_ in PP_POINTS:
    o = PPR[pid]; axs[0].plot(x, o["VL"]["p_pp"] - o["VL"]["p_my"], "o-", ms=3, lw=.8, label=pid)
axs[0].set_xticks(x); axs[0].set_xticklabels(NODES, rotation=90, fontsize=7); axs[0].set_ylabel("p_pandapipes − p_Eigenmodell [bar]"); axs[0].set_title("Räumliche Verteilung der Druckabweichung (Vorlauf)"); axs[0].legend(ncol=4, fontsize=6)
mm = np.concatenate([np.r_[PPR[p]["VL"]["m_my"], PPR[p]["RL"]["m_my"]] for p, *_ in PP_POINTS]); mp = np.concatenate([np.r_[PPR[p]["VL"]["m_pp"], PPR[p]["RL"]["m_pp"]] for p, *_ in PP_POINTS])
axs[1].scatter(mm, mp, s=8, alpha=.5); lim = [min(mm.min(), mp.min()), max(mm.max(), mp.max())]; axs[1].plot(lim, lim, "k-", lw=.6); axs[1].set_xlabel("Massenstrom Eigenmodell [kg/s]"); axs[1].set_ylabel("Massenstrom pandapipes [kg/s]"); axs[1].set_title("Leitungsmassenströme (alle Punkte, VL+RL)")
d = np.concatenate([np.abs(np.r_[PPR[p]["VL"]["p_pp"] - PPR[p]["VL"]["p_my"], PPR[p]["RL"]["p_pp"] - PPR[p]["RL"]["p_my"]]) for p, *_ in PP_POINTS]); axs[2].hist(d, bins=30, color=COL["rl"]); axs[2].set_xlabel("|Δp| Knoten [bar]"); axs[2].set_title("Verteilung der Knotendruckabweichungen")
fig.tight_layout(); fig.savefig(FIG_DIR/"20_pandapipes_vergleich.png"); plt.show()
print(f"Knotendruck: Median |Δp| {np.median(d):.4f} bar, P95 {np.percentile(d,95):.4f} bar, Maximum {d.max():.4f} bar; Massenstrom: max. |Δṁ| {np.abs(mm-mp).max():.2f} kg/s.")
print("Ursachen der Restabweichung: (i) Reibungsmodell (Swamee–Jain mit k = 0,1 mm im Eigenmodell gegen Colebrook in pandapipes; mit Nikuradse 3× größer, Kap. 20.2), (ii) Dichte: Eigenmodell konstant je Teilnetz, pandapipes je Knoten, (iii) Löser-Toleranzen (Eigenmodell 1e-5 kg/s, pandapipes 1e-7…1e-4 bar), (iv) relative Fehler konzentrieren sich auf Leitungen mit kleinem Massenstrom (Nenner).")

## 21. Unsicherheitsbewertung
Unterschieden werden **Datenunsicherheit** (Lastniveau, Rücklauftemperatur, Erzeugerdrücke), **Parameterunsicherheit** (Rauheit, Kalibriermultiplikatoren, Lastanteile, Wärmeverlust, Δp_min, Anschlusslänge), **Modellstrukturunsicherheit** (Ost-Kanten an den Kalibriergrenzen) und **Szenariounsicherheit** (Ausbaulast). Teil 1: Einzelsensitivitäten (Tornado) für Bestand und +20 %; Teil 2: gemeinsame Monte-Carlo-Streuung (60 Läufe je Fall, Seed fest) mit den Fahrplänen der Nominal-Optimierung (Betreiber kennt die Parameterfehler nicht).

In [ ]:
import contextlib, time as _time
@contextlib.contextmanager
def with_theta(th):
    global theta_cal
    old = theta_cal; theta_cal = th
    try: yield
    finally: theta_cal = old
@contextlib.contextmanager
def with_sps(**kw):
    old = {k: SPS[k] for k in kw}; SPS.update(kw)
    try: yield
    finally: SPS.update(old)

BASES_U = {"S1 Bestand": dict(design=True), "S5 +20 %": dict(design=True, growth=0.20)}
NOM = {}
for bl, bk in BASES_U.items():
    cs0 = base_day_case(**bk); o0 = optimize_dispatch(cs0); NOM[bl] = dict(bk=bk, Qc=o0["Qc"], Qd=o0["Qd"])
def eval_pair(bl, theta=None, rough=1e-4, dT_rl=0.0, dpv=0.0, hf=1.0, dpmin=None, u=U_LIN, k_scale=None, growth=None, extra_cfg=None, sps=None, hours_pump=None):
    bk = dict(NOM[bl]["bk"]); bk.update(heat_factor=hf, dT_rl=dT_rl, dp_hkw=dpv, dp_ava=dpv)
    if growth is not None: bk["growth"] = growth
    cs = base_day_case(**bk); cfg = dict(extra_cfg or {}, rough=rough); 
    if k_scale: cfg["k_scale"] = k_scale
    sto = dict(Qc=NOM[bl]["Qc"], Qd=NOM[bl]["Qd"], variant="B")
    with with_theta(theta_cal if theta is None else theta), with_sps(**(sps or {})):
        e0 = evaluate("no", cs, cfg=cfg, dp_min=dpmin, u_lin=u); e1 = evaluate("mit", cs, cfg=cfg, storage=sto, dp_min=dpmin, u_lin=u)
    k0, k1 = e0["kpi"], e1["kpi"]
    return {"Δp_min ohne [bar]": k0["Δp_Kunde,min [bar]"], "Δp_min mit [bar]": k1["Δp_Kunde,min [bar]"], "Speicherwirkung Δp_min [bar]": k1["Δp_Kunde,min [bar]"] - k0["Δp_Kunde,min [bar]"],
            "Defizit ohne [bar·h]": k0["Δp-Defizit [bar·h]"], "Defizit mit [bar·h]": k1["Δp-Defizit [bar·h]"], "HKW-Spitze ohne [m³/h]": k0["HKW-Durchfluss,max [m³/h]"], "HKW-Spitze mit [m³/h]": k1["HKW-Durchfluss,max [m³/h]"],
            "Spitzenentlastung HKW [m³/h]": k0["HKW-Durchfluss,max [m³/h]"] - k1["HKW-Durchfluss,max [m³/h]"], "Pumpenförderhöhe [bar]": k1["Pumpenförderhöhe,max [bar]"], "T_Lieferung,min mit [°C]": k1["T_Lieferung,min [°C]"]}
# ---------------------------------------------------------------- (1) Einzelsensitivitäten (Tornado)
def prior_theta():
    th = theta_cal.copy(); 
    for g_ in ("SL6", "SL7", "SL3"): th[GN_FREE.index(g_)] = 0.0          # Kanten an den Parametergrenzen auf Plan-/Sollwert k = 1 zurücksetzen (Modellstruktur)
    return th
def ost_share(f):
    th = theta_cal.copy()
    for s_ in OST_SEK: th[NGF + SEKN.index(s_)] += np.log(f)
    return th
FACT = [("Datenunsicherheit", "Lastniveau ±10 %", dict(hf=0.9), dict(hf=1.1)), ("Datenunsicherheit", "Rücklauftemperatur ±3 K", dict(dT_rl=-3.0), dict(dT_rl=3.0)),
        ("Datenunsicherheit", "Erzeuger-Vorlaufdruck ±0,3 bar", dict(dpv=-0.3), dict(dpv=0.3)),
        ("Parameterunsicherheit", "Rohrrauheit 0,05…0,5 mm", dict(rough=5e-5), dict(rough=5e-4)), ("Parameterunsicherheit", "Kalibriermultiplikatoren aller Gruppen ×0,8/×1,2", dict(k_scale={g_: 0.8 for g_ in GN_FREE}), dict(k_scale={g_: 1.2 for g_ in GN_FREE})),
        ("Parameterunsicherheit", "Lastanteil Ost-Sektionen ×0,8/×1,2", dict(theta="ost0.8"), dict(theta="ost1.2")), ("Parameterunsicherheit", "Wärmeverlust U' 0,2…0,4 W/(m K)", dict(u=0.2), dict(u=0.4)),
        ("Parameterunsicherheit", "Mindest-Kundendifferenzdruck 0,8…1,2 bar", dict(dpmin=0.8), dict(dpmin=1.2)),
        ("Parameterunsicherheit", "Speicheranschluss: Länge 100…500 m", dict(sps=dict(L_conn_m=100.0)), dict(sps=dict(L_conn_m=500.0))),
        ("Modellstruktur", "Ost-Kanten (SL3/SL6/SL7) auf k = 1 statt Grenzwert", dict(theta="prior"), dict(theta="prior")),
        ("Szenariounsicherheit", "Ausbaulast ±5 Prozentpunkte", dict(growth="-"), dict(growth="+"))]
rows = []
for bl in BASES_U:
    base = eval_pair(bl)
    for kat, lab, lo_s, hi_s in FACT:
        out = []
        for spec in (lo_s, hi_s):
            sp = dict(spec)
            if sp.get("theta") == "ost0.8": sp["theta"] = ost_share(0.8)
            elif sp.get("theta") == "ost1.2": sp["theta"] = ost_share(1.2)
            elif sp.get("theta") == "prior": sp["theta"] = prior_theta()
            if sp.get("growth") in ("-", "+"): sp["growth"] = NOM[bl]["bk"].get("growth", 0.0) + (0.05 if sp["growth"] == "+" else -0.05)
            try: out.append(eval_pair(bl, **sp))
            except Exception as ex: out.append({k: np.nan for k in base})
        rows.append({"Basis": bl, "Kategorie": kat, "Faktor": lab, "Δp_min ohne (niedrig/hoch)": f"{out[0]['Δp_min ohne [bar]']:.2f} / {out[1]['Δp_min ohne [bar]']:.2f}", "Δp_min ohne, Spanne [bar]": abs(out[1]["Δp_min ohne [bar]"] - out[0]["Δp_min ohne [bar]"]),
                     "Speicherwirkung Δp_min (niedrig/hoch) [bar]": f"{out[0]['Speicherwirkung Δp_min [bar]']:.2f} / {out[1]['Speicherwirkung Δp_min [bar]']:.2f}", "Speicherwirkung, Spanne [bar]": abs(out[1]["Speicherwirkung Δp_min [bar]"] - out[0]["Speicherwirkung Δp_min [bar]"]),
                     "HKW-Spitzenentlastung (niedrig/hoch) [m³/h]": f"{out[0]['Spitzenentlastung HKW [m³/h]']:.0f} / {out[1]['Spitzenentlastung HKW [m³/h]']:.0f}"})
TOR = pd.DataFrame(rows); TOR.to_csv(RESULTS/"sensitivitaet_einzeln.csv", index=False)
BASE_VAL = {bl: eval_pair(bl) for bl in BASES_U}
print("Referenzwerte:", {bl: {k: round(v, 2) for k, v in d.items()} for bl, d in BASE_VAL.items()})
with pd.option_context("display.width", 250, "display.max_colwidth", 60): display(TOR.sort_values(["Basis","Speicherwirkung, Spanne [bar]"], ascending=[True, False]).round(2))

In [ ]:
# ---------------------------------------------------------------- (2) Gemeinsame Monte-Carlo-Streuung (Parameter-, Daten- und Strukturunsicherheit; feste Fahrpläne aus der Nominal-Optimierung)
rng = np.random.default_rng(SEED); N_MC = 60; sig = np.clip(se, 0.05, 0.4)
MC = []; t0 = _time.time()
for bl in BASES_U:
    for n in range(N_MC):
        th = np.clip(theta_cal + sig*rng.standard_normal(len(theta_cal)), lo, hi)
        if rng.random() < 0.25: th[[GN_FREE.index(g_) for g_ in ("SL6", "SL7", "SL3")]] = 0.0          # 25 %: Ost-Kanten strukturell auf Sollwert
        pr = dict(theta=th, rough=float(np.exp(rng.uniform(np.log(5e-5), np.log(5e-4)))), dT_rl=float(rng.uniform(-3, 3)), dpv=float(rng.normal(0, 0.2)), hf=float(rng.uniform(0.9, 1.1)), dpmin=float(rng.uniform(0.8, 1.2)), u=float(rng.uniform(0.2, 0.4)),
                  sps=dict(L_conn_m=float(rng.uniform(100, 500)), zeta_conn=float(rng.uniform(6, 20))))
        try: MC.append(dict(Basis=bl, **eval_pair(bl, **pr)))
        except Exception as ex: pass
MCD = pd.DataFrame(MC); MCD.to_csv(RESULTS/"monte_carlo.csv", index=False)
print(f"Monte-Carlo: {len(MCD)} gültige Läufe in {_time.time()-t0:.0f} s")
q = MCD.groupby("Basis")[["Δp_min ohne [bar]","Δp_min mit [bar]","Speicherwirkung Δp_min [bar]","Defizit ohne [bar·h]","Defizit mit [bar·h]","Spitzenentlastung HKW [m³/h]","Pumpenförderhöhe [bar]"]].quantile([0.05, 0.5, 0.95]).round(2)
display(q)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax, bl in zip(axs[:2], BASES_U):
    t = TOR[TOR.Basis == bl].sort_values("Speicherwirkung, Spanne [bar]"); colr = {"Datenunsicherheit": "#d68910", "Parameterunsicherheit": "#2874a6", "Modellstruktur": "#c0392b", "Szenariounsicherheit": "#7f8c8d"}
    ax.barh(t.Faktor.str[:48], t["Speicherwirkung, Spanne [bar]"], color=[colr[k] for k in t.Kategorie]); ax.set_title(f"{bl}: Spanne der Speicherwirkung (Δp_min mit − ohne)"); ax.set_xlabel("Spanne [bar]"); ax.tick_params(axis="y", labelsize=7)
axs[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colr.values()], labels=list(colr), fontsize=7, loc="lower right")
for bl, c_ in zip(BASES_U, (COL["ok"], COL["vl"])): axs[2].hist(MCD[MCD.Basis == bl]["Speicherwirkung Δp_min [bar]"], bins=15, alpha=.6, color=c_, label=bl)
axs[2].set_xlabel("Speicherwirkung Δp_min [bar]"); axs[2].set_title("Monte-Carlo (60 Läufe je Fall)"); axs[2].legend(); fig.tight_layout(); fig.savefig(FIG_DIR/"21_unsicherheit.png"); plt.show()

## 22. Technische Schlussfolgerungen
Die Antworten folgen den 20 Fragen des Arbeitsauftrags (Kap. 24 des Auftrags). Alle Zahlen werden aus den Ergebnissen dieses Notebooks berechnet.

In [ ]:
K = lambda sid, key: RUN[sid]["kpi"][key]
ANS = {}
def A(n, title, text): ANS[n] = (title, text); print(f"**{n}. {title}**\n{text}\n")
from IPython.display import Markdown, display
_out = []
def A(n, title, text): ANS[n] = (title, text); _out.append(f"**{n}. {title}** {text}")
# 1
A(1, "Niedrigste Drücke heute", f"Im kritischen Bestandsfall S1 (Auslegung −14 °C, 120 °C, ungünstige Erzeugerdrücke) liegen der kleinste Vorlaufdruck bei {K('S1','p_VL,min [bar Ü]'):.1f} bar Ü ({K('S1','Knoten/Stunde p_VL,min')}), der kleinste Rücklaufdruck bei {K('S1','p_RL,min [bar Ü]'):.1f} bar Ü ({K('S1','Knoten/Stunde p_RL,min')}) und der kleinste Kunden-Differenzdruck bei {K('S1','Δp_Kunde,min [bar]'):.2f} bar ({K('S1','Knoten/Stunde Δp,min')}; kritische Knoten: {K('S1','kritische Knoten')}). Absolute Drücke sind überall weit oberhalb der Siedegrenze; das Problem ist der **Differenzdruck**, nicht der Absolutdruck. Bias-korrigierte Modellunsicherheit ≈ ±0,3–0,6 bar (Ost/Süd größer).")
# 2
ev1 = EVENTS.set_index("Schwelle [bar]")
A(2, "Historische kritische Situationen", f"Die Messdaten 2025 enthalten **keine anhaltende Druckkrise**: Kunden-Δp < 0,8 bar trat in {int(ev1.loc[0.8,'Stunden'])} Einzelstunden an {int(ev1.loc[0.8,'Tage'])} verschiedenen Tagen auf (längste Dauer {int(ev1.loc[0.8,'längste zusammenhängende Dauer [h]'])} h, davon Nullwerte: {int(ev1.loc[0.8,'davon Nullwerte (0,0 bar)'])}); das sind überwiegend Einzelausreißer bzw. lokale Ereignisse, die ein Sektionsmodell nicht reproduziert (und die vermutlich teilweise Messfehler sind). Reproduziert wird die **Hochlast-Belastung**: S0 (Basistag) zeigt {K('S0','Dauer Δp-Verletzung [h]')} Stunden unter 1,0 bar (kleinster Wert {K('S0','Δp_Kunde,min [bar]'):.2f} bar, SEC4/SL1G), gemessen blieben die Stationen ≥ ca. 1,1 bar. Das Modell ist dort etwas pessimistisch.")
# 3
hv = RAISE.set_index("Zuwachs")
A(3, "Wahrscheinlichste Ursachen", f"(i) Hohe Winterlast im Innenstadt-/Süd-Bereich (kritische Knoten siehe 1) bei knapper Erzeugerdruckdifferenz; im Modell sind vor allem die Ost-Trassen stark ausgelastet (größte Auslastung {K('S1','Leitungsauslastung,max [-]'):.2f} auf {K('S1','Kante Auslastung')}); (ii) **knappe Erzeugerdruckhaltung**: bereits eine Anhebung von Vorlauf-Druck HKW/AVA um {hv.loc[0.0].iloc[0]:.1f} bar würde in S1 alle Δp-Verletzungen beseitigen (Daten: Erzeugerdrücke variieren 2025 nur um ≈ ±0,5 bar); (iii) der Ost-Bereich (SL5–SL7) ist im Modell überlastet und strukturell unsicher (Kap. 13/21).")
# 4
g30 = GRW[(GRW.Zuwachs == 0.30) & (GRW.Fall == 'ohne Speicher')].iloc[0]; g10 = GRW[(GRW.Zuwachs == 0.10) & (GRW.Fall == 'ohne Speicher')].iloc[0]
A(4, "Wirkung des Netzausbaus", f"Die Probleme wachsen überproportional: Δp-Defizit {K('S1','Δp-Defizit [bar·h]'):.0f} bar·h (heute) → {g10['Δp-Defizit [bar·h]']:.0f} (+10 %) → {GRW[(GRW.Zuwachs==0.2)&(GRW.Fall=='ohne Speicher')].iloc[0]['Δp-Defizit [bar·h]']:.0f} (+20 %) → {g30['Δp-Defizit [bar·h]']:.0f} bar·h (+30 %); ab ≈ +15 % wird das HKW-Erzeugerlimit (2025-Maximum) in einzelnen Stunden überschritten, ab +30 % ganztägig; bei +30 % wird der Kunden-Differenzdruck rechnerisch {K('S5c','Δp_Kunde,min [bar]'):.2f} bar (negativ = Versorgung mit den Erzeugerdrücken von 2025 nicht mehr möglich). Ohne Gegenmaßnahmen an Erzeugern/Leitungen ist ein Ausbau über ca. +5…10 % nicht ohne Druckverletzungen möglich (Annahme: Erzeugerdrücke wie 2025).")
# 5
d = DEC["S1 → S3O"]; sens = {}
_cs = base_day_case(design=True); _r = solve_case(_cs.iloc[[8]*2].reset_index(drop=True), m_stor=np.array([0.0, 100.0])); _pv, _pr = node_pressures(_r); _dp = (_pv - _pr)[:, KUNDEN_IDX] + DP_BIAS
A(5, "Entlastung vorgelagerter Leitungen", f"Je 100 kg/s (≙ {100*cp_w(90)*62/1e3:.0f} MW bei ΔT 62 K) lokal eingespeister Entladung sinkt der HKW-Durchfluss um {(_r['inj_all'][0, nidx['HKW']] - _r['inj_all'][1, nidx['HKW']]):.0f} kg/s; der Kunden-Differenzdruck steigt an den kritischen Knoten um nur ≈ {np.median(_dp[1] - _dp[0]):.2f} bar (MAN selbst {(_dp[1]-_dp[0])[KUNDEN.index('MAN')]:.2f} bar). Mit der optimierten Fahrweise (5 000 m³, 40 MW) sinkt das Δp-Defizit in S1 von {K('S1','Δp-Defizit [bar·h]'):.1f} auf {K('S3O','Δp-Defizit [bar·h]'):.1f} bar·h ({100*(1-K('S3O','Δp-Defizit [bar·h]')/K('S1','Δp-Defizit [bar·h]')):.0f} %), die nicht versorgte Wärme (Näherung) von {K('S1','nicht versorgt [MWh]'):.0f} auf {K('S3O','nicht versorgt [MWh]'):.0f} MWh. Die Wirkung ist eine Mengen-/Lokaleinspeisung, **keine** Druckhaltung.")
# 6, 7
gain = rk["Nutzen S1→S3O [bar]"].sort_values(ascending=False)
A(6, "Profitierende Netzbereiche", f"Größter Nutzen (Δ kleinster Kunden-Δp, S1→S3O): {', '.join(f'{k} {v:+.2f} bar' for k, v in gain.head(4).items())}. Profitiert der gesamte HKW-seitig vermaschte Bereich (Innenstadt, Süd, SL5), überwiegend mit 0,1–0,2 bar.")
neg = rk[rk["Nutzen S1→S3O [bar]"] < -0.005]
A(7, "Nicht/negativ beeinflusste Bereiche", (f"Negativ beeinflusst in der Nominalfahrweise: {', '.join(neg.index)}. " if len(neg) else "In der optimierten Fahrweise wird kein Kundenknoten im Minimum verschlechtert. ") + f"Ohne Wirkung ist der Speicher bei **Trennung an Arm. 342.1** (MAN auf der Ost-Seite: Δ = {RUN['S72BE']['kpi']['Δp_Kunde,min [bar]'] - RUN['S72B0']['kpi']['Δp_Kunde,min [bar]']:+.2f} bar) für die HKW-Seite, und die Ost-Sektionen (SL6/SL7) sind nur schwach angekoppelt. Die Wirkung ist im Ostbereich wegen der strukturellen Modellunsicherheit nicht belastbar.")
# 8
A(8, "Neue Probleme durch Laden", f"Laden in Schwachlast (S4A/B) erzeugt keine neuen Differenzdruckprobleme, erhöht aber den HKW-Durchfluss auf {K('S4B','HKW-Durchfluss,max [m³/h]'):.0f} m³/h (Lastverschiebung, Erzeugerlimit {CAP['HKW']/0.95*3.6:.0f} m³/h) und den Lokal-Δp-Bedarf am MAN (kleinster Wert {K('S4B','Δp_Kunde,min [bar]'):.2f} bar). **Laden zur Lastspitze (S4K) ist kritisch**: Δp_min {K('S4K','Δp_Kunde,min [bar]'):.2f} bar ({K('S4K','Knoten/Stunde Δp,min')}), {K('S4K','Erzeugerlimit HKW überschritten [h]')} Stunden über dem HKW-Limit. Temperatur: Rücklauf am HKW max. {K('S4B','T_RL,max HKW [°C]'):.1f} °C (S1: {K('S1','T_RL,max HKW [°C]'):.1f} °C) – kein thermisches Problem.")
# 9-11
hpmax = max(v['kpi']['Pumpenförderhöhe,max [bar]'] for k, v in RUN.items() if k.startswith(('S3', 'S6', 'S7', 'S8')))
A(9, "Speicherpumpe erforderlich?", f"**Ja.** Entladen ist bei direkter Einbindung ohne Pumpe hydraulisch unmöglich (p_VL > p_RL am MAN; S3A ≡ S1: Δp-Defizit {K('S3A','Δp-Defizit [bar·h]'):.1f} bar·h). Variante A (Ventile) eignet sich nur zum Laden mit Netz-Δp (max. ≈ {abs(list(mc_max.values())[0])*cp_w(90)*62/1e3:.0f}–{abs(list(mc_max.values())[1])*cp_w(90)*62/1e3:.0f} MW je nach Lastzustand).")
A(10, "Förderhöhe und Massenstrom", f"Für 40 MW Entladung (≈ {stor_flows(cs_t, np.zeros(24), np.r_[np.zeros(8), 40.0, np.zeros(15)], 120.0)[8]:.0f} kg/s) sind über alle Entladeszenarien bis zu **{hpmax:.1f} bar** Förderhöhe nötig (Lastspitze ≈ 3–4,5 bar; Gl. 15.1), Pumpstrom ≈ {K('S3O','Pumpstrom Speicher [MWh]'):.2f} MWh/Tag (η = 0,75).")
d_ = S9[(S9.Basis == 'Bestand') & (S9.Parameter == 'dn')]; dn_ok = d_[d_["Reduktion Defizit [%]"] >= 0.95*d_["Reduktion Defizit [%]"].max()].Wert.min()
A(11, "Anschlussdurchmesser", f"Hydraulisch sinnvoll ist **DN{dn_ok:.0f}–DN300** (v ≤ 3 m/s bei 40 MW); kleinere Anschlüsse (DN150/200) begrenzen die nutzbare Leistung auf {d_[d_.Wert==150]['Qd,max [MW]'].iloc[0]:.0f}/{d_[d_.Wert==200]['Qd,max [MW]'].iloc[0]:.0f} MW, größere bringen keinen Nutzen, senken nur die Pumpenförderhöhe ({d_[d_.Wert==300]['Pumpenförderhöhe,max [bar]'].iloc[0]:.1f} → {d_[d_.Wert==500]['Pumpenförderhöhe,max [bar]'].iloc[0]:.1f} bar).")
v_ = S9[(S9.Basis == 'Bestand') & (S9.Parameter == 'V_m3')]; v_ok = v_[v_["Reduktion Defizit [%]"] >= 0.90*v_["Reduktion Defizit [%]"].max()].Wert.min(); v_sat = v_[v_["Reduktion Defizit [%]"] >= 0.99*v_["Reduktion Defizit [%]"].max()].Wert.min()
q_ = S9[(S9.Basis == 'Bestand') & (S9.Parameter == 'Qd')]; q_ok = q_[q_["Reduktion Defizit [%]"] >= 0.90*q_["Reduktion Defizit [%]"].max()].Wert.min(); q_sat = q_[q_["Reduktion Defizit [%]"] >= 0.99*q_["Reduktion Defizit [%]"].max()].Wert.min()
A(12, "Speichergröße und Entladeleistung", f"Im Bestand erreicht die Wirkung ≥ 90 % ihres Maximums ab ≈ **{v_ok:.0f} m³** bzw. ≈ **{q_ok:.0f} MW** (Sättigung bei {v_sat:.0f} m³ bzw. {q_sat:.0f} MW) (bis {v_['Reduktion Defizit [%]'].max():.0f} % Reduktion des Δp-Defizits). **Die untersuchten Engpässe werden damit nicht vollständig beseitigt**: das Ziel ≥ 80 % Reduktion wird von keiner Einzelvariation erreicht; die Reserve ist durch die lokale Wirkung (0,1–0,2 bar je 100 kg/s) begrenzt, nicht durch das Volumen.")
# 13-15
A(13, "Betriebs-/Auslegungsdruck", f"Betriebsdruck am Speicherfuß ≥ {p_op_min:.1f} bar Ü (Siedeabstand bei 120 °C auf 20 m Behälterhöhe), zulässig bis zum Netz-Vorlaufdruck am MAN (max. {pv_man.max():.1f} bar Ü). **Auslegungsdruck ≈ {p_des_pump:.0f} bar Ü** (größter Netzdruck am MAN + Pumpenförderhöhe, +10 %); er ist aus dem maximalen Netzdruck abgeleitet, nicht aus p_sat(120 °C) = {psat_bar(120.0)-P_ATM:.1f} bar Ü. Die Projektvorgabe der Topo-YAML (10 bar) ist **zu niedrig** gegenüber dem Netzdruck.")
A(14, "Siedesicherheit bei 120 °C", f"Ja: kleinster Siedeabstand im Netz {marg['Siedeabstand min [bar]'].astype(float).min():.1f} bar (Anforderung ≥ 1 bar); am Speicherkopf im ungünstigsten Fall {pd_tab.iloc[-1]['Wert']:.1f} bar. Kritisch wäre nur ein Druckeinbruch unter ≈ {p_boil_top:.1f} bar Ü am Kopf (Pumpenausfall, Druckhaltungsverlust).")
A(15, "PN16/PN25", f"MAN liegt im PN25-Bereich; {'PN25' if p_des_pump >= 25 else 'die PN25-Grenze'} wird bei {p_des_pump:.0f} bar Ü Auslegungsdruck {'berührt' if p_des_pump >= 25 else 'nicht berührt'}. PN16 gilt für das hydraulisch getrennte Westnetz (nicht betroffen; größter Netzdruck im Verbund {pvmax_all:.1f} bar Ü liegt bereits unter PN16, die Speicher-Auslegung von ≈ {p_des_pump:.0f} bar Ü auch).")
# 16, 17, 18
sepA = [(k, RUN[k]['kpi']) for k in ('S71B0', 'S71BE', 'S72B0', 'S72BE')]
A(16, "Separierter Betrieb", f"Bei Trennung der Ost-Trassen (S7) steigen Defizit und HKW-Durchfluss stark (Bestand: Δp_min {K('S71B0','Δp_Kunde,min [bar]'):.2f} bar, HKW {K('S71B0','HKW-Durchfluss,max [m³/h]'):.0f} m³/h; Transportgrenzen SL5/6/7 überschritten: Auslastung {K('S71B0','Leitungsauslastung,max [-]'):.1f}). Der Speicher hilft nur, wenn MAN auf der HKW-Seite liegt (342.3: Δp-Defizit {K('S71B0','Δp-Defizit [bar·h]'):.0f} → {K('S71BE','Δp-Defizit [bar·h]'):.0f} bar·h); bei 342.1 keine Wirkung. **Der Speicher reicht bei separiertem Betrieb nicht.** Laden im getrennten Zustand verschlechtert die Lage (S71BL: Δp_min {K('S71BL','Δp_Kunde,min [bar]'):.2f} bar).")
rem = K('S3O', 'Δp-Defizit [bar·h]')
A(17, "Verbleibende Probleme trotz Speicher", f"Im Bestand bleibt ein Δp-Defizit von {rem:.1f} bar·h ({K('S3O','Dauer Δp-Verletzung [h]')} h unter 1,0 bar); bei +10 % {GRW[(GRW.Zuwachs==0.1)&(GRW.Fall=='mit Speicher (opt.)')].iloc[0]['Δp-Defizit [bar·h]']:.0f} bar·h, bei +30 % {GRW[(GRW.Zuwachs==0.3)&(GRW.Fall=='mit Speicher (opt.)')].iloc[0]['Δp-Defizit [bar·h]']:.0f} bar·h; bei GT-Ausfall (S8a) {K('S8aE','Δp-Defizit [bar·h]'):.0f} bar·h; Erzeugerlimit HKW bleibt ab +30 % ganztägig überschritten.")
A(18, "Zusätzliche Maßnahmen", f"**Ja.** Die wirksamste und einfachste Maßnahme ist die Anhebung der Erzeuger-Vorlaufdrücke: {hv.loc[0.0].iloc[0]:.1f} bar (heute), {hv.loc[0.1].iloc[0]:.1f} / {hv.loc[0.2].iloc[0]:.1f} / {hv.loc[0.3].iloc[0]:.1f} bar für +10/20/30 % – der Speicher spart dabei nur {hv.loc[0.2].iloc[2]:.2f} bar (+20 %). Weitere Hebel: Ertüchtigung der Ost-Trassen (SL6/SL7), Erzeugerkapazität ab +15 %, Pumpenregelung (PSS/PSW). (Grenzen der Anhebung: Erzeugerpumpen/Abschaltdrücke WV640 – Datenlücke.)")
# 19
A(19, "Übereinstimmung Modell – Messung – pandapipes", f"Eigenmodell ↔ pandapipes: max. Knotendruckabweichung {pp_tab[['max|Δp_VL| [bar]','max|Δp_RL| [bar]']].max().max():.3f} bar, Massenstrom ≤ {pp_tab['max|Δṁ| [kg/s]'].max():.1f} kg/s, keine Flussrichtungsabweichung (8 Betriebspunkte, VL+RL). Modell ↔ Messung (Validierung Okt–Dez 2025): Vorlaufdruck MAN RMSE {val.loc['p_VL'].loc['MAN'].RMSE:.2f} bar, Mitte/Süd ≈ 0,25 bar, Ost ≈ 0,8 bar, Kunden-Δp ≈ 0,6 bar (Mitte/Süd). Die Temperaturkontrolle ist nur am Einzelstrang mit pandapipes möglich; im Netz gilt RMSE {tv_tab.RMSE_K.mean():.1f} K gegen Messung (Hauptstrang ±1–4 K, Stichleitungen bis 14 K).")
# 20
sp = TOR[TOR.Basis == 'S1 Bestand']["Speicherwirkung, Spanne [bar]"].max(); ab = TOR[TOR.Basis == 'S1 Bestand']["Δp_min ohne, Spanne [bar]"].max()
q1 = MCD[MCD.Basis == 'S1 Bestand']["Speicherwirkung Δp_min [bar]"].quantile([.05, .5, .95]).values
A(20, "Belastbar vs. annahmeabhängig", f"**Belastbar:** (a) Der Speicher wirkt **nur mit Pumpe** (Physik, Gl. 15.1); (b) seine Wirkung ist **klein bis mäßig und positiv**: Δ kleinster Kunden-Δp {q1[1]:+.2f} bar (P5–P95 der Monte-Carlo-Streuung: {q1[0]:+.2f}…{q1[2]:+.2f} bar); (c) Auslegungsdruck/Siedesicherheit; (d) Laden zur Lastspitze ist schädlich. **Annahmeabhängig:** absolute Druckwerte und Verletzungsdauern (einzelne Faktoren verschieben Δp_min um bis zu {ab:.1f} bar), Ost-Bereich (strukturell ±1 bar), Lastextrapolation, Δp_min = 1,0 bar, Ausbaulast, Erzeugerdruckhaltung.")
# 21–22 (Vertiefungen Kap. 19.4–19.7)
pv_loc = LOC.pivot(index="Standort", columns="Basis", values="Reduktion Defizit [%]"); best_loc = pv_loc["Bestand"].idxmax(); lq = LOC[(LOC.Basis == "Bestand")].set_index("Standort")
A(21, "Standort des Speichers", f"Der Standort ist **der wichtigste Hebel des Speichers**: bei gleicher Größe (5 000 m³, 40 MW) reduziert er das Δp-Defizit im Bestand am MAN um {pv_loc.loc['MAN','Bestand']:.0f} %, im Innenstadt-/West-Bereich dagegen um {pv_loc.loc['SEC2','Bestand']:.0f} % (SEC2) bzw. {pv_loc.loc['SL1G','Bestand']:.0f} % (SL1G), bei nur {lq.loc['SEC2','Pumpenförderhöhe [bar]']:.1f}–{lq.loc['SL1G','Pumpenförderhöhe [bar]']:.1f} bar Förderhöhe (MAN {lq.loc['MAN','Pumpenförderhöhe [bar]']:.1f} bar) und {lq.loc['SEC2','Entladeenergie [MWh]']:.0f}–{lq.loc['SL1G','Entladeenergie [MWh]']:.0f} MWh statt {lq.loc['MAN','Entladeenergie [MWh]']:.0f} MWh Entladeenergie. Grund: die kritischen Knoten (SEC2, SEC4, SL1G, HWW) liegen im Innenstadt-/Südbereich, und der Netzhebel je 100 kg/s ist dort ca. 3–6× größer als am MAN. Bei +20 % Ausbau bleibt jeder Standort bei ≤ {pv_loc['Ausbau +20 %'].max():.0f} % Reduktion. Einschränkung: Grundstück, Genehmigung und Anschluss eines 5 000 m³-Druckspeichers in der Innenstadt sind unbekannt; die Sektionsknoten repräsentieren ein Gebiet, keinen Punkt.")
A(22, "Ausbauverteilung und Kalibrierung", f"(a) Wo der Ausbau stattfindet, entscheidet: +10 % Gesamtlast, konzentriert in Süd, erzeugt {DIST.set_index('Sektion mit Zuwachs').loc['SUED','Defizit ohne [bar·h]']:.0f} bar·h Defizit, im Ost-Bereich (SL5O) nur {DIST.set_index('Sektion mit Zuwachs').loc['SL5O','Defizit ohne [bar·h]']:.0f} bar·h (heute {K('S1','Δp-Defizit [bar·h]'):.0f}). (b) Eine zusätzliche Kalibrierung mit Gewicht auf Hochlast-Differenzdruck verbessert die Hochlast-Validierung (Δp-RMSE {dpA['RMSE [bar]'].mean():.2f} → {dpB['RMSE [bar]'].mean():.2f} bar), senkt aber das berechnete Bestandsdefizit deutlich ({HL.iloc[0]['Defizit ohne [bar·h]']:.0f} → {HL.iloc[1]['Defizit ohne [bar·h]']:.0f} bar·h); die **relative** Speicherwirkung bleibt bei {HL.iloc[1]['Reduktion [%]']:.0f}–{HL.iloc[0]['Reduktion [%]']:.0f} %. Absolute Defizite sind kalibrierungsabhängig, relative Aussagen robuster. (c) Druckanhebung: Bedarf {DR.set_index('Zuwachs').loc[0.0].iloc[0]:.1f} bar heute, {DR.set_index('Zuwachs').loc[0.3].iloc[0]:.1f} bar bei +30 %; der größte Netzdruck bleibt dabei unter {DR.iloc[:,-1].max():.0f} bar Ü (PN25 nicht berührt); Absenken des Rücklaufdrucks wirkt gleich (nur Δp zählt).")
mp = MS.pivot(index="ID", columns="Zuwachs", values="Δp-Defizit [bar·h]")
A(23, "Maßnahmenvergleich (ohne Kosten)", f"Reihenfolge der Wirksamkeit (verbleibendes Δp-Defizit, heute / +10 % / +20 % / +30 %): Erzeugerdruck +1,0 bar {mp.loc['M1'].iloc[0]:.0f} / {mp.loc['M1'].iloc[1]:.0f} / {mp.loc['M1'].iloc[2]:.0f} / {mp.loc['M1'].iloc[3]:.0f} bar·h; Verstärkung Ost {mp.loc['M2'].iloc[0]:.0f} / {mp.loc['M2'].iloc[1]:.0f} / {mp.loc['M2'].iloc[2]:.0f} / {mp.loc['M2'].iloc[3]:.0f}; Innenstadt-Verbund {mp.loc['M3'].iloc[0]:.0f} / {mp.loc['M3'].iloc[1]:.0f} / {mp.loc['M3'].iloc[2]:.0f} / {mp.loc['M3'].iloc[3]:.0f}; Speicher MAN {mp.loc['M4'].iloc[0]:.0f} / {mp.loc['M4'].iloc[1]:.0f} / {mp.loc['M4'].iloc[2]:.0f} / {mp.loc['M4'].iloc[3]:.0f}; Speicher SEC2 {mp.loc['M5'].iloc[0]:.0f} / {mp.loc['M5'].iloc[1]:.0f} / {mp.loc['M5'].iloc[2]:.0f} / {mp.loc['M5'].iloc[3]:.0f} (ohne Maßnahme {mp.loc['M0'].iloc[0]:.0f} / {mp.loc['M0'].iloc[1]:.0f} / {mp.loc['M0'].iloc[2]:.0f} / {mp.loc['M0'].iloc[3]:.0f}). **Die Erzeugerdruckanhebung schlägt jede einzelne bauliche Maßnahme**; der Speicher wirkt nur im heutigen Bestand nennenswert, bei Ausbau kaum. Ein größerer oder stärkerer Speicher am besten Standort ändert wenig (+10 %: {pz.min().min():.0f}–{pz.max().max():.0f} % Reduktion über alle Größen 2 500–20 000 m³ / 20–80 MW). Bauliche Verstärkung der Ost-Trassen oder des Innenstadtverbunds reduziert um ≈ 30–40 %, löst aber allein nichts; die Kombination Druck +1 bar mit Verstärkung beseitigt das Defizit bis +30 %. Kosten fehlen (Datenlücke): die Rangfolge ist rein hydraulisch.")
Markdown("\n\n".join(_out))

### 22.1 Grafische Zusammenfassung der Kernbefunde
Jede Abbildung beantwortet eine Frage: **G1** warum eine Pumpe nötig ist; **G2** wie klein der Speichernutzen gegenüber der Druckanhebung an den Erzeugern ist; **G3** in welchen Fällen der Speicher hilft oder schadet; **G4** wie robust die Speicherwirkung gegen Unsicherheiten ist; **G5** welches Druckfenster der Speicher braucht; **G6** welche Netzbereiche profitieren. Die Abbildungen werden nach `results/studie_MAN_UPM/abbildungen/` geschrieben.

In [ ]:
# ---------------------------------------------------------------- Grafische Zusammenfassung der Kernbefunde (je Abbildung eine Aussage)
INK = "#2b2b2b"
# G1 – Warum eine Pumpe? Netz-Δp am MAN gegen Speicherstrom, erforderliche Förderhöhe, Laden ohne Pumpe
lab_pk = [k for k in DP_CURVE if k.startswith("Spitzen")][0]; dc = DP_CURVE[lab_pk]; Tm_ = float(cs_t.T_vl.iloc[8]); bar_per_m = float(rho_w(Tm_)*G/1e5)
P_MW = -dc.m.values*cp_w(90)*62.0/1e3                                   # Leistung: >0 Entladen, <0 Laden (m_grid: >0 Entladen → hier Vorzeichen für die Achse: Entladen rechts)
P_MW = dc.m.values*cp_w(90)*62.0/1e3
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.axvspan(0, P_MW.max(), color=COL["vl"], alpha=.07); ax.axvspan(P_MW.min(), 0, color=COL["rl"], alpha=.07)
ax.plot(P_MW, dc.dpMAN, color=INK, lw=2, label="Netz-Δp am MAN-Knoten (Spitzenstunde)")
ch = dc[dc.m < 0]; ax.plot(P_MW[dc.m < 0], (ch.Hp*0 + 0.5).values, color="none")
ax.plot(P_MW[dc.m > 0], dc.Hp[dc.m > 0]*bar_per_m, color=COL["vl"], lw=2.2, label="erforderliche Pumpen-Förderhöhe beim Entladen")
ok_c = dc[(dc.m < 0) & dc.ok_charge]; lim = P_MW[(dc.m < 0) & dc.ok_charge].min() if len(ok_c) else 0
ax.axvline(lim, color=COL["rl"], ls="--", lw=1.2); ax.text(lim - 0.8, 4.6, f"Laden ohne Pumpe\nmöglich bis ≈ {abs(lim):.0f} MW", color=INK, fontsize=8, ha="right")
ax.text(P_MW.max()*0.55, 0.25, "Entladen: ohne Pumpe hydraulisch\nunmöglich (p_VL > p_RL)", fontsize=9, color=INK, ha="center")
ax.text(P_MW.min()*0.6, 0.25, "Laden", fontsize=9, color=INK, ha="center")
ax.set_xlabel("Speicherleistung am Netzanschluss [MW]   (← Laden | Entladen →)"); ax.set_ylabel("Druckdifferenz / Förderhöhe [bar]"); ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.set_title("G1 | Entladen braucht eine Pumpe: Förderhöhe ≈ Netz-Δp + Anschlussverluste", loc="left", fontsize=10); fig.tight_layout(); fig.savefig(FIG_DIR/"22_G1_pumpe.png"); plt.show()

# G2 – Speicher vs. einfache Gegenmaßnahme: erforderliche Anhebung des Erzeuger-Vorlaufdrucks
R2 = RAISE.set_index("Zuwachs"); x = np.arange(len(R2)); w = .36
fig, ax = plt.subplots(figsize=(8.6, 4.4))
b1 = ax.bar(x - w/2, R2.iloc[:, 0], w, color=COL["grau"], label="ohne Speicher"); b2 = ax.bar(x + w/2, R2.iloc[:, 1], w, color=COL["ok"], label="mit optimiertem Speicher (5 000 m³, 40 MW)")
for i, (a_, b_) in enumerate(zip(R2.iloc[:, 0], R2.iloc[:, 1])): ax.text(i - w/2, a_ + .03, f"{a_:.2f}", ha="center", fontsize=8, color=INK); ax.text(i + w/2, b_ + .03, f"{b_:.2f}", ha="center", fontsize=8, color=INK)
ax.set_xticks(x); ax.set_xticklabels([f"+{int(100*g)} %" if g else "heute" for g in R2.index]); ax.set_xlabel("Lastzuwachs"); ax.set_ylabel("erforderliche Anhebung Vorlaufdruck HKW+AVA [bar]"); ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("G2 | Der Speicher spart nur 0,1–0,25 bar – die Druckanhebung an den Erzeugern bleibt der Haupthebel", loc="left", fontsize=10); fig.tight_layout(); fig.savefig(FIG_DIR/"22_G2_druckanhebung.png"); plt.show()

# G3 – Ampelübersicht: Wirkung mit − ohne Speicher je Paar und Kennzahl
def rel(a, b): return np.where(a > 1e-9, 100*(b/np.where(a > 1e-9, a, 1) - 1), 0.0)
M = pd.DataFrame({"Δ kleinster Kunden-Δp [bar]": CMP["Δp_Kunde,min [bar] mit"] - CMP["Δp_Kunde,min [bar] ohne"], "Δ Δp-Defizit [%]": rel(CMP["Δp-Defizit [bar·h] ohne"].values, CMP["Δp-Defizit [bar·h] mit"].values),
                  "Δ nicht versorgt [%]": rel(CMP["nicht versorgt [MWh] ohne"].values, CMP["nicht versorgt [MWh] mit"].values), "Δ HKW-Spitze [%]": rel(CMP["HKW-Durchfluss,max [m³/h] ohne"].values, CMP["HKW-Durchfluss,max [m³/h] mit"].values)}); M.index = CMP.Paar + "  " + CMP.Beschreibung.str[:30]
fig, axs = plt.subplots(1, 4, figsize=(15, 8), sharey=True, gridspec_kw=dict(wspace=.08))
for ax, (col, sign, vm) in zip(axs, (("Δ kleinster Kunden-Δp [bar]", 1, 0.7), ("Δ Δp-Defizit [%]", -1, 60), ("Δ nicht versorgt [%]", -1, 60), ("Δ HKW-Spitze [%]", -1, 40))):
    v = M[[col]].values*sign; im = ax.imshow(np.clip(v, -vm, vm), cmap="RdYlGn", vmin=-vm, vmax=vm, aspect="auto"); ax.set_xticks([0]); ax.set_xticklabels([col], fontsize=8); ax.grid(False)
    for i, val in enumerate(M[col].values): ax.text(0, i, f"{val:+.2f}" if "bar" in col else f"{val:+.0f}", ha="center", va="center", fontsize=7.5, color=INK)
axs[0].set_yticks(range(len(M))); axs[0].set_yticklabels(M.index, fontsize=7.5)
fig.suptitle("G3 | Wirkung des Speichers je Szenariopaar (grün = besser mit Speicher, rot = schlechter): nützlich bei Entladung, schädlich bei Laden zur Spitze und bei Trennung", x=.02, ha="left", fontsize=10); fig.subplots_adjust(left=.27, right=.98, top=.93, bottom=.06); fig.savefig(FIG_DIR/"22_G3_ampel.png"); plt.show()

# G4 – Robustheit: Speicherwirkung (Δp_min mit − ohne) unter Einzelvariationen und Monte-Carlo-Band
def two(s): a, b = s.split("/"); return float(a), float(b)
fig, axs = plt.subplots(1, 2, figsize=(14, 5.2), sharex=True)
for ax, bl in zip(axs, BASES_U):
    t = TOR[TOR.Basis == bl].copy(); lo_, hi_ = zip(*[two(s) for s in t["Speicherwirkung Δp_min (niedrig/hoch) [bar]"]]); yy = np.arange(len(t)); nomv = BASE_VAL[bl]["Speicherwirkung Δp_min [bar]"]
    qq = MCD[MCD.Basis == bl]["Speicherwirkung Δp_min [bar]"].quantile([.05, .95]).values; ax.axvspan(qq[0], qq[1], color=COL["ok"], alpha=.12, label="Monte-Carlo P5–P95 (alle Faktoren gemeinsam)")
    for y_, a_, b_ in zip(yy, lo_, hi_): ax.plot([a_, b_], [y_, y_], color=COL["mess"], lw=1.6); ax.scatter([a_, b_], [y_, y_], s=22, color=COL["mess"], zorder=3)
    ax.axvline(nomv, color=COL["vl"], lw=1.2, label=f"Nominal {nomv:+.2f} bar"); ax.axvline(0, color="k", lw=.6)
    ax.set_yticks(yy); ax.set_yticklabels(t.Faktor.str[:46], fontsize=7.5); ax.set_xlabel("Speicherwirkung: Δ kleinster Kunden-Δp [bar]"); ax.set_title(bl, loc="left", fontsize=9); ax.legend(fontsize=7, frameon=False, loc="upper right")
fig.suptitle("G4 | Die Speicherwirkung bleibt unter allen Unsicherheiten positiv und klein (≈ 0,0–0,3 bar) – Absolutwerte streuen deutlich mehr", x=.02, ha="left", fontsize=10); fig.tight_layout(rect=(0, 0, 1, .95)); fig.savefig(FIG_DIR/"22_G4_robustheit.png"); plt.show()

# G5 – Druckfenster des Speichers (Überdruck am Speicherfuß bzw. -kopf)
marks = [(float(psat_bar(120.0)) - P_ATM + SPS["dT_boil"], "Siedegrenze Kopf bei 120 °C (p_sat + 1 bar)", COL["vl"]), (p_op_min, "Mindest-Betriebsdruck Fuß (Kopf 20 m höher)", COL["warn"]), (pr_man.min(), "kleinster Rücklaufdruck MAN (alle Szenarien)", COL["rl"]),
         (p_static_man, "Ruhedruck am MAN-Niveau", COL["rl"]), (pv_man.max(), "größter Vorlaufdruck MAN", COL["vl"]), (10.0, "Projektvorgabe Topo-YAML", COL["grau"]), (16.0, "PN16 (Westnetz, getrennt)", COL["grau"]), (p_des_pump, "Auslegungsdruck (+Pumpe, +10 %)", COL["ok"]), (25.0, "PN25", COL["grau"])]
fig, ax = plt.subplots(figsize=(12, 4.6)); ax.axhspan(-.45, .45, color="#eeeeee"); ax.axvspan(p_op_min, pv_man.max(), ymin=.2, ymax=.55, color=COL["ok"], alpha=.3, label="zulässiges Betriebsfenster am Speicherfuß")
short = {0: "Siedegrenze Kopf 120 °C"}
for i, (v, t_, c_) in enumerate(sorted(marks)):
    lv = i % 5; ax.plot([v, v], [-.3, .3 + .28*lv], color=c_, lw=1.6); ax.text(v + .12, .3 + .28*lv, f"{t_} – {v:.1f} bar Ü", fontsize=7, ha="left", va="bottom", color=INK)
ax.set_xlim(0, 30); ax.set_ylim(-.6, 2.2); ax.set_yticks([]); ax.set_xlabel("Überdruck [bar]"); ax.legend(loc="lower right", fontsize=8, frameon=False)
ax.set_title("G5 | Druckfenster: Auslegung folgt dem Netzdruck (≈ 17 bar Ü), nicht der Siedegrenze; Projektvorgabe 10 bar ist zu niedrig", loc="left", fontsize=10); fig.tight_layout(); fig.savefig(FIG_DIR/"22_G5_druckfenster.png"); plt.show()

# G6 – Wo hilft der Speicher? Nutzen je Kundenknoten (S1→S3O) mit Ausgangslage
rk2 = rk.sort_values("Nutzen S1→S3O [bar]"); fig, ax = plt.subplots(figsize=(9, 4.4)); yy = np.arange(len(rk2))
ax.barh(yy, rk2["Nutzen S1→S3O [bar]"], color=np.where(rk2["Nutzen S1→S3O [bar]"] >= 0, COL["ok"], COL["vl"]), height=.55)
for y_, (k_, r_) in zip(yy, rk2.iterrows()): ax.text(max(r_["Nutzen S1→S3O [bar]"], 0) + .004, y_, f"{r_['Nutzen S1→S3O [bar]']:+.2f} bar (Ausgang S1: {r_['S1']:.2f} bar)", va="center", fontsize=7.5, color=INK)
ax.set_yticks(yy); ax.set_yticklabels(rk2.index); ax.set_xlabel("Δ kleinster Kunden-Δp über 24 h, S3O − S1 [bar]"); ax.set_xlim(min(rk2["Nutzen S1→S3O [bar]"].min() - .02, -.02), rk2["Nutzen S1→S3O [bar]"].max() + .3)
ax.set_title("G6 | Nutzen je Netzbereich: gleichmäßig klein; Ostknoten (SL5–SL7) am höchsten, aber dort am unsichersten", loc="left", fontsize=10); fig.tight_layout(); fig.savefig(FIG_DIR/"22_G6_nutzen_knoten.png"); plt.show()

## 23. Empfehlungen und offene Datenlücken

In [ ]:
# ---------------------------------------------------------------- Empfehlung (Kategorien des Arbeitsauftrags) – regelbasiert aus den Ergebnissen
red = 100*(1 - K('S3O', 'Δp-Defizit [bar·h]')/K('S1', 'Δp-Defizit [bar·h]')); s3a_same = abs(K('S3A', 'Δp-Defizit [bar·h]') - K('S1', 'Δp-Defizit [bar·h]')) < 1e-6
hyd = "ja" if red >= 80 else ("teilweise" if red >= 15 else "nein")
Qd_loc = K('S3O', 'Entladeenergie [MWh]'); ther = "teilweise" if K('S3O', 'HKW-Durchfluss,max [m³/h]') >= K('S1', 'HKW-Durchfluss,max [m³/h]') else "ja"
pump = "ja" if (s3a_same and red > 0) else ("wahrscheinlich" if red > 0 else "nein")
g10_with = GRW[(GRW.Zuwachs == 0.1) & (GRW.Fall == 'mit Speicher (opt.)')].iloc[0]; netz = "ja" if (rem > 1.0 or g10_with['Δp-Defizit [bar·h]'] > 1.0) else "wahrscheinlich"
reco = pd.DataFrame([
 ("Speicher hydraulisch wirksam", hyd, f"Δp-Defizit −{red:.0f} % im kritischen Bestand (nur mit Pumpe); Wirkung 0,1–0,2 bar je 100 kg/s"),
 ("Speicher thermisch wirksam", ther, f"{Qd_loc:.0f} MWh/Tag lokal eingespeist; HKW-Spitzendurchfluss {K('S1','HKW-Durchfluss,max [m³/h]'):.0f} → {K('S3O','HKW-Durchfluss,max [m³/h]'):.0f} m³/h (Lastverschiebung auf Ladestunden)"),
 ("Speicherpumpe erforderlich", pump, "Entladen ohne Pumpe hydraulisch unmöglich"),
 ("Netzverstärkung zusätzlich erforderlich", netz, "verbleibendes Defizit auch mit Speicher; Ausbau ab ≈ +10 % nicht ohne Erzeugerdruckanhebung/Ostausbau"),
 ("empfohlene Speicherleistung", f"ca. {q_ok:.0f} MW Entladung (DN300 begrenzt auf ≈ 52 MW)", f"≥ 90 % der maximalen Wirkung ab {q_ok:.0f} MW, Sättigung ab {q_sat:.0f} MW (S9)"),
 ("empfohlener Speichervolumenbereich", f"{v_[v_.Wert < v_ok].Wert.max() if (v_.Wert < v_ok).any() else v_ok:.0f}–{v_ok:.0f} m³", "≥ 90 % der Wirkung ab unterer Grenze, Sättigung ab oberer; Energie ≈ 5–8 h bei 40 MW"),
 ("empfohlener Anschlussdurchmesser", f"DN{dn_ok:.0f} bis DN300 (Anschlussstränge ≤ 250–300 m); DN > 300 nur zur Pumpenentlastung", "v ≤ 3 m/s bei 40 MW"),
 ("erforderliche Pumpenförderhöhe", f"{hpmax:.1f} bar bei 40 MW (Lastspitze), ≈ 3 bar für 35 MW", "H_P = Δp_MAN + Verluste (Gl. 15.1)"),
 ("erforderlicher Betriebs-/Auslegungsdruck", f"Betrieb ≥ {p_op_min:.1f} bar Ü (Fuß), Auslegung ≈ {p_des_pump:.0f} bar Ü", "aus maximalem Netzdruck am MAN + Pumpe, nicht aus 120 °C"),
 ("zulässiges Betriebsfenster Entladen", "lastreichste Stunden (Kunden-Δp < ≈ 1,2 bar), Füllstand 5–95 %, T ≥ 110 °C", "T < Heizkurve − 10 K verletzt die Lieferungstemperatur (S9: 12–13 h)"),
 ("zulässiges Betriebsfenster Laden", "Schwachlast (Netz-Δp am MAN ≥ ≈ 1,5 bar), nicht zur Lastspitze, nicht im getrennten Zustand", "S4K/S71BL: neue Verletzungen"),
 ("empfohlener Standort (zu prüfen)", f"Innenstadt/West (SEC2/SL1G) statt MAN: −{pv_loc.loc['SEC2','Bestand']:.0f}/−{pv_loc.loc['SL1G','Bestand']:.0f} % Defizit gegenüber −{pv_loc.loc['MAN','Bestand']:.0f} % am MAN", f"Förderhöhe {lq.loc['SEC2','Pumpenförderhöhe [bar]']:.1f}–{lq.loc['SL1G','Pumpenförderhöhe [bar]']:.1f} bar; Flächen-/Genehmigungslage unbekannt"),
 ("Rangfolge der Maßnahmen (hydraulisch)", "1. Erzeugerdruck anheben (Grenze klären), 2. Ostbereich/Innenstadtverbund verstärken, 3. Speicher (nur bei Pumpe, bevorzugt Innenstadt/West)", "Defizit-Reduktion je Maßnahme in Kap. 19.8; Kosten ausstehend"),
 ("verbleibende Risiken", "Ost-Modellunsicherheit, Schaltzustände 2025 unbekannt, Höhen ±3–5 m, Erzeugerpumpen-Grenzen unbekannt, Druckstöße nicht untersucht", "siehe Datenlücken"),
], columns=["Kategorie", "Empfehlung", "Begründung"])
with pd.option_context("display.max_colwidth", 120, "display.width", 250): display(reco)
reco.to_csv(RESULTS/"empfehlung.csv", index=False)
gaps = pd.DataFrame([
 ("Einbindungsplanung Speicher (Anschlusspunkt, Höhe, Leitungslängen, Pumpe)", "Plan/YAML: keine; gesucht in WV640/650/660, Topo-YAML, Ordner Stadtbach", "Anschlussverluste, Pumpenhöhe, PN-Zone", "Variantenrechnung A/B, Länge ±, Kap. 21"),
 ("Druckhaltung/Abschaltdrücke/Pumpenkennlinien der Erzeuger", "WV640 nur Ruhedruck und Δp intern", "Grenze der Druckanhebung (Kap. 19.1)", "Anhebung als Bedarf ausgewiesen, nicht als Limit"),
 ("Schaltzustände 2025 und Betriebsanweisung Trennung", "nicht in Daten", "S7-Zuordnung (Arm. 342.1/342.3) ist Annahme", "beide Varianten gerechnet"),
 ("Geländehöhen der Stationen", "nur Differenzen aus Niedriglast (±3–5 m)", "absolute Drücke ±0,3–0,5 bar", "Differenzen statt Absolutwerte"),
 ("Konkrete Ausbauvorhaben (Orte, Leistungen)", "keine räumlichen Daten", "Verteilung des Zuwachses", "+10…50 % und +30 % nur Ost"),
 ("Kundenlasten je Sektion", "nicht identifizierbar (Kap. 13)", "Lastverteilung Ost/Innenstadt", "Sensitivität Ost-Anteil ±20 %"),
 ("Ost-Kanten (SL3/SL6/SL7) und Stichleitungen (Beethovenpark)", "Kalibrierung an Grenzen", "Ost-Aussagen nur ±1 bar", "strukturelle Variante k = 1 (Kap. 21)"),
 ("Transiente Drücke (Wasserhammer, Armaturenschaltung, Pumpenausfall am Speicher)", "quasistationäres Modell", "Auslegungsdruck, Sicherheitsventile", "**separate Druckstoßstudie empfohlen**"),
 ("Wärmeverluste der Verteilnetze, Speicherdämmung, Durchmischung", "nicht in Daten", "Temperaturaussagen (Stichleitungen ±14 K)", "U' 0,2–0,4; Schichtungsfaktor 0,7–0,95 (Kap. 19/21)"),
 ("Zusätzlich benötigte Messungen", "–", "Mess- und Kalibrierqualität", "Δp-Messung an SEC2/SEC4/HWW mit Stundenmaximum, Erzeuger-Pumpendrehzahlen, Schaltzustände, Druck an Armaturen 342.x"),
], columns=["Fehlende Information", "Gesucht in / Stand", "Einfluss", "Behandlung"])
with pd.option_context("display.max_colwidth", 90, "display.width", 250): display(gaps)
gaps.to_csv(RESULTS/"datenluecken.csv", index=False)

### 23.1 Plan für das weitere Vorgehen
**Leitgedanke:** Die Rechnungen zeigen, dass der Speicher am MAN nur eine kleine, pumpenabhängige Entlastung (≈ 0,1–0,2 bar) bringt, während die Anhebung der Erzeugerdrücke und die Ertüchtigung des Ostbereichs die Hauptthemen sind. Deshalb zuerst **Daten klären und den Ostbereich schärfen**, dann **Maßnahmen vergleichen**, und erst wenn der Speicher in dieser Rangfolge bestehen bleibt, die **Detailauslegung** (Pumpe, Druckstoß, Schichtung) angehen.

| Phase | Ziel | Kernfrage | Entscheidung am Ende |
|---|---|---|---|
| **A Daten/Annahmen (Wo 0–8)** | Betreiberdaten einholen: Einbindungskonzept, Erzeugerpumpen/Abschaltdrücke, Schaltzustände, Höhen, Ausbauvorhaben, Hochlast-Δp-Messungen | Wie weit darf der Erzeugerdruck real angehoben werden? Wo liegt MAN real? | **G1 (Wo 3):** reichen die Daten, um den Ostbereich zu verfeinern? |
| **B Modell schärfen (Wo 3–9)** | Ostbereich (Erzeuger-Δp-Regelung, Stichleitungen, zeitvariable Lastanteile), Hochlast-Kalibrierung, Unsicherheitsbänder, Temperaturvalidierung | Ost ±1 bar → ±0,5 bar? Bias < 0,3 bar? | **G2 (Wo 8):** Modellgüte ausreichend für Maßnahmenvergleich? |
| **C Maßnahmenvergleich (Wo 8–13)** | Druckanhebung ↔ Leitungsverstärkung Ost ↔ Speicher (MAN und alternative Standorte) ↔ Kombination; Kosten-Nutzen | Welche Maßnahme löst welchen Engpass je Ausbaustufe am günstigsten? | **G3 (Wo 13):** Speicher weiterverfolgen? |
| **D Speicher-Detailauslegung (Wo 12–18)** | Pumpenauslegung (NPSH, Regelung), Druckhaltung, Druckstoßstudie, Schichtungsmodell, Regelstrategie | Welcher Auslegungsdruck/welche Pumpe ist sicher? | **G4 (Wo 20):** Auslegungsfreigabe |
| **E Framework/Bericht (Wo 14–21)** | Höhen, Schleifengesetz, Speicherpumpe in Calion (oder SLP-Kopplung), Regressionstests, Abschlussbericht | Läuft die Hydraulik im Framework reproduzierbar? | Abnahme |

**Sofort (ohne Betreiberdaten) möglich:** B2/B3 (Kalibrierung, Unsicherheitsbänder), Standortvariation des Speichers im Modell, Druckanhebungs-Szenarien in feineren Stufen, Sensitivität der Ausbauverteilung.
**Nicht ohne Betreiberdaten möglich:** Grenze der Druckanhebung (A2), reale Trennzustände (A3), absolute Drücke (A4).

In [ ]:
# ---------------------------------------------------------------- Plan für das weitere Vorgehen (Zeitachse in Wochen = grobe Schätzung, abhängig von Datenlieferung)
PLAN = pd.DataFrame([
 ("A", "A1 Einbindungskonzept Speicher (Anschlusspunkt, Höhe, Leitungslängen, Pumpenart)", 0, 3, "Betreiber/Planer", "Anschlussverluste, Pumpenhöhe, PN-Zone belastbar"),
 ("A", "A2 Erzeuger: Pumpenkennlinien, Abschaltdrücke, Δp-Regelung (HKW, AVA, GT, BM)", 0, 3, "Betreiber", "Grenze der Druckanhebung (Haupthebel, Kap. 19.1)"),
 ("A", "A3 Schaltzustände 2025 und Betriebsanweisung für Trennung (342.1/342.3)", 0, 3, "Betreiber", "S7 auf reale Zustände statt Annahme"),
 ("A", "A4 Geländehöhen der Stationen (DEM/Vermessung) und Ausbauvorhaben mit Ort/Leistung", 0, 4, "Betreiber/GIS", "absolute Drücke, räumliche Lastverteilung"),
 ("A", "A5 Zusatzmessung: Kunden-Δp (Stundenmaxima/-minima) SEC2/SEC4/HWW, Druck an Arm. 342.x", 1, 8, "Betreiber", "Validierung der Hochlast-Differenzdrücke"),
 ("B", "B1 Ostbereich: Erzeuger-Δp-Regelung, Stichleitungen, zeitvariable Lastanteile", 3, 6, "wir", "Ost-Unsicherheit ±1 bar → Ziel ±0,5 bar"),
 ("B", "B2 Kalibrierung auf Hochlast-Differenzdruck (Bias < 0,3 bar), Unsicherheitsbänder aus Jacobian", 4, 4, "wir", "weniger Bias-Korrektur, Bandbreiten je Knoten"),
 ("B", "B3 Temperaturmodell validieren (Verteilnetz, Stichleitungen, Erzeugertemperaturen)", 5, 4, "wir", "Temperaturaussagen ±2 K statt bis 14 K"),
 ("C", "C1 Maßnahmenvergleich: Druckanhebung ↔ Leitungsverstärkung Ost ↔ Speicher (MAN und alternative Standorte)", 8, 5, "wir", "Rangfolge der Maßnahmen je Ausbaustufe"),
 ("C", "C2 Kosten-Nutzen (Investition, Pumpstrom, vermiedene Erzeugerkosten) – Kostendaten einholen", 9, 4, "gemeinsam", "Wirtschaftlichkeit Speicher vs. Alternativen"),
 ("D", "D1 Pumpenauslegung (Kennlinie, NPSH, Regelung), Druckhaltung und Sicherheitskonzept am Speicher", 12, 5, "wir + Planer", "Auslegungsdruck, Sicherheitsventile"),
 ("D", "D2 Transiente Druckstoßstudie (Wasserhammer, Pumpenausfall, Armaturenschaltung)", 12, 6, "wir/extern", "Freigabe Auslegungsdruck"),
 ("D", "D3 Schichtungs-/Durchmischungsmodell (1-D) und Regelstrategie (Prognose, Füllstandsführung)", 13, 5, "wir", "nutzbare Kapazität, Fahrplan-Regeln"),
 ("E", "E1 Höhen, Schleifengesetz und Speicherpumpe in das Calion-Framework übernehmen (oder SLP-Kopplung)", 14, 6, "wir", "Framework-Optimierung mit Hydraulik"),
 ("E", "E2 Regressionstests, Reproduzierbarkeit, Abschlussbericht", 18, 3, "wir", "abnahmefähige Studie"),
], columns=["Phase", "Arbeitspaket", "Start [Wo]", "Dauer [Wo]", "Verantwortung", "Ergebnis / Nutzen"])
STATUS = {"B2": "Teilschritt erledigt (Kap. 19.7: Hochlast-Test)", "C1": "hydraulisch erledigt (Kap. 19.4–19.8); Kosten ausstehend (C2)"}
PLAN["Status"] = [STATUS.get(a.split(" ")[0], "offen") for a in PLAN.Arbeitspaket]
GATES = [(3, "G1 Datenlage: reichen Erzeuger-/Schaltdaten? → Ost-Verfeinerung ja/nein"), (8, "G2 Modellgüte: Ost ≤ ±0,5 bar, Δp-Bias < 0,3 bar?"), (13, "G3 Maßnahmenrangfolge: Speicher weiterverfolgen?"), (20, "G4 Auslegungsfreigabe (Druckstoß, Pumpe, Kosten)")]
PCOL = {"A": "#d68910", "B": "#2874a6", "C": "#1e8449", "D": "#8e44ad", "E": "#7f8c8d"}
fig, ax = plt.subplots(figsize=(13, 6.6)); yy = np.arange(len(PLAN))[::-1]
for y_, r in zip(yy, PLAN.itertuples()):
    ax.barh(y_, r[4], left=r[3], color=PCOL[r.Phase], height=.55); ax.text(r[3] + r[4] + .2, y_, r.Arbeitspaket.split(" ", 1)[1][:100], va="center", fontsize=7, color=INK)
for g_, t_ in GATES: ax.axvline(g_, color="k", ls="--", lw=.8); ax.text(g_ + .1, len(PLAN) - .2, t_.split(" ")[0], fontsize=8, va="bottom", fontweight="bold")
ax.set_yticks(yy); ax.set_yticklabels(PLAN.Arbeitspaket.str.split(" ").str[0], fontsize=8); ax.set_xlim(0, 36); ax.set_xlabel("Woche (grobe Schätzung; A-Pakete hängen von der Datenlieferung des Betreibers ab)")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in PCOL.values()], labels=["A Daten/Annahmen", "B Modell schärfen", "C Maßnahmenvergleich", "D Speicher-Detailauslegung", "E Framework/Bericht"], loc="lower right", fontsize=8, frameon=False)
ax.set_title("Fahrplan: erst Daten und Ostmodell, dann Maßnahmenvergleich, erst danach Detailauslegung des Speichers (Gates G1–G4)", loc="left", fontsize=10); fig.tight_layout(); fig.savefig(FIG_DIR/"23_fahrplan.png"); plt.show()
display(PLAN); PLAN.to_csv(RESULTS/"plan_arbeitspakete.csv", index=False)
(RESULTS/"plan_weiteres_vorgehen.md").write_text("# Plan für das weitere Vorgehen\n\n" + "\n".join(["| " + " | ".join(PLAN.columns) + " |", "|" + "---|"*len(PLAN.columns)] + ["| " + " | ".join(str(x) for x in r) + " |" for r in PLAN.itertuples(index=False)]) + "\n\n## Entscheidungspunkte\n" + "\n".join(f"* Woche {w}: {t}" for w, t in GATES) + "\n", encoding="utf-8")

## 24. Reproduzierbarer Ergebnisexport

In [ ]:
import hashlib, datetime as _dt
# ---------------------------------------------------------------- Management Summary (aus den berechneten Ergebnissen erzeugt; wird in Kap. 1 angezeigt)
ms = f"""**Management Summary (automatisch aus Kap. 15–23 erzeugt, Stand {_dt.date.today().isoformat()}, Seed {SEED})**

* **Speicherwirkung:** hydraulisch **{reco.set_index('Kategorie').loc['Speicher hydraulisch wirksam','Empfehlung']}**, thermisch **{reco.set_index('Kategorie').loc['Speicher thermisch wirksam','Empfehlung']}**; Speicherpumpe **{reco.set_index('Kategorie').loc['Speicherpumpe erforderlich','Empfehlung']}** (Entladen ohne Pumpe ist bei direkter Einbindung hydraulisch unmöglich).
* **Bestand (kritischer Auslegungstag):** kleinster Kunden-Δp {K('S1','Δp_Kunde,min [bar]'):.2f} bar ({K('S1','Knoten/Stunde Δp,min')}), {K('S1','Dauer Δp-Verletzung [h]')} h unter 1,0 bar; mit optimiertem Speicher (5 000 m³, 40 MW, Pumpe {K('S3O','Pumpenförderhöhe,max [bar]'):.1f} bar) Δp-Defizit −{red:.0f} %. Die Entlastung beträgt nur ≈ 0,1–0,2 bar an den kritischen Knoten.
* **Standort:** Der Standort ist der größte Hebel: Innenstadt/West (SEC2/SL1G) reduziert das Defizit um {pv_loc.loc['SEC2','Bestand']:.0f}/{pv_loc.loc['SL1G','Bestand']:.0f} % gegenüber {pv_loc.loc['MAN','Bestand']:.0f} % am MAN (bei niedrigerer Förderhöhe); Genehmigungs-/Flächenlage unbekannt.
* **Maßnahmenrangfolge (hydraulisch):** Erzeugerdruck +1 bar beseitigt das Defizit bis ≈ +10 % Ausbau (Rest bei +30 %: {mp.loc['M1'].iloc[3]:.0f} bar·h); bauliche Verstärkungen ≈ −30–40 %; der Speicher wirkt bei Ausbau kaum. Kosten fehlen.
* **Ausbau:** Probleme wachsen überproportional (+10 %: {g10['Δp-Defizit [bar·h]']:.0f}, +30 %: {g30['Δp-Defizit [bar·h]']:.0f} bar·h); der Speicher reduziert sie nur um wenige Prozent. **Netzverstärkung/Erzeugerdruckanhebung ist zusätzlich erforderlich** (Bedarf {hv.loc[0.2].iloc[0]:.1f} bar bei +20 %, Speicher spart {hv.loc[0.2].iloc[2]:.2f} bar).
* **Druck/Siedesicherheit:** Auslegungsdruck ≈ {p_des_pump:.0f} bar Ü (aus Netzdruck + Pumpe), Siedesicherheit bei 120 °C überall gegeben; PN25 nicht berührt. Transiente Druckstöße: separate Studie empfohlen.
* **Belastbarkeit:** Modell ↔ pandapipes ≤ {pp_tab[['max|Δp_VL| [bar]','max|Δp_RL| [bar]']].max().max():.3f} bar; Modell ↔ Messung: MAN ±0,1 bar, Mitte/Süd ±0,3 bar, Ost ±1 bar. Differenzen (mit − ohne) sind belastbarer als Absolutwerte.
* **Empfehlung:** {reco.set_index('Kategorie').loc['empfohlene Speicherleistung','Empfehlung']}, {reco.set_index('Kategorie').loc['empfohlener Speichervolumenbereich','Empfehlung']}, {reco.set_index('Kategorie').loc['empfohlener Anschlussdurchmesser','Empfehlung']}; Pumpe {reco.set_index('Kategorie').loc['erforderliche Pumpenförderhöhe','Empfehlung']}.
"""
(RESULTS/"management_summary.md").write_text(ms, encoding="utf-8"); display(Markdown(ms))
(RESULTS/"antworten_20_fragen.md").write_text("\n\n".join(_out), encoding="utf-8")
# ---------------------------------------------------------------- Abnahmekriterien (Kap. 25 des Auftrags) – maschinell prüfbare Teile
chk = {
 "S2 ≡ S1 (Speicher angeschlossen, inaktiv)": bool(np.allclose(RUN["S1"]["res"]["Hv"], RUN["S2"]["res"]["Hv"])),
 "Löser in allen Szenarien/Stunden konvergiert": all(v["kpi"]["Löser ok"] for v in RUN.values()),
 "Massenbilanz je Szenario < 0,01 kg/s": all(np.abs(v["res"]["inj_all"].sum(1)).max() < 1e-2 for v in RUN.values()),
 "Optimierungen alle optimal und konvergiert": bool((OPTLOG.Status == "optimal").all() and OPTLOG["konvergiert"].all()),
 "pandapipes: max. Knotendruckabweichung < 0,05 bar": bool(pp_tab[["max|Δp_VL| [bar]", "max|Δp_RL| [bar]"]].max().max() < 0.05),
 "Energiebilanz Temperaturmodell": bool(eb["Bilanzfehler [MW]"].abs().max() < 0.05),
 "Heizkurven-Endpunkte (120 °C @ −10, 90 °C @ +20)": bool(abs(float(T_CURVE(-10)) - 120) < 1e-9 and abs(float(T_CURVE(20)) - 90) < 1e-9),
 "Speicherenergie 5 000 m³ @ 10 K ≈ 56–58 MWh": bool(55 < storage_capacity_MWh(5000, 10) < 59),
 "Vorlauf/Rücklauf getrennt modelliert (Kap. 10–12)": True, "Kalibrierung/Validierung zeitlich getrennt (Kap. 13/14)": True,
 "mit/ohne Speicher unter gleichen Randbedingungen (Paare)": bool(len(PAIRS) >= 20), "Bestand und Ausbau, vermascht und separiert gerechnet": bool(any(k.startswith("S5") for k in RUN) and any(k.startswith("S7") for k in RUN)),
 "Laden und Entladen getrennt bewertet": bool({"S3O", "S4B", "S4K"} <= set(RUN)), "Speicher- und Pumpenwirkung getrennt (S3A/S3B/S3O)": bool({"S3A", "S3B", "S3O"} <= set(RUN)),
 "Unsicherheiten dargestellt (Tornado, Monte-Carlo)": bool(len(TOR) > 0 and len(MCD) > 0)}
chk_tab = pd.DataFrame({"Kriterium": list(chk), "erfüllt": list(chk.values())}); display(chk_tab); assert all(chk.values()), "Abnahmekriterium nicht erfüllt"
# ---------------------------------------------------------------- Ergebnisexport mit Prüfsummen
files = sorted(p for p in RESULTS.glob("*") if p.is_file() and p.suffix in (".csv", ".md", ".json"))
man = pd.DataFrame([{"Datei": p.name, "Größe [kB]": round(p.stat().st_size/1024, 1), "SHA-256 (8)": hashlib.sha256(p.read_bytes()).hexdigest()[:8]} for p in files]); man.to_csv(RESULTS/"manifest_ergebnisse.csv", index=False)
figs = sorted(FIG_DIR.glob("*.png")); print(f"{len(files)} Ergebnisdateien und {len(figs)} Abbildungen in {RESULTS.relative_to(ROOT).as_posix()} (nicht versioniert: NDA).")
display(man.tail(12))

---
**Ende des Notebooks.** Alle Ergebnisdateien liegen unter `results/studie_MAN_UPM/` (nicht versioniert, NDA-Daten).